# TBP — Primera prueba con series reales de Argentina

Este notebook calcula el TBP por cohorte (1950–2035) con las series de `datos/procesados/` (ver `datos/README.md`). Es una **prueba exploratoria**: sirve para ver si los datos se comportan, no para concluir.

**Qué es dato y qué es supuesto** (todo queda a la vista):

| Variable | Valor | Origen | Tipo |
|---|---|---|---|
| $N_t$ | nacimientos | DEIS (≤2024); ONU proyección (2025+) | observado / proyectado |
| $1-\mu$ | supervivencia de la **cohorte** hasta 65 | ONU WPP 2024, tabla de mortalidad (producto de $p_x$ por edad y año calendario) | estimado |
| $J_{t+65}$ | población 65+ en $t+65$ | ONU WPP 2024 | proyectado |
| $E_r$ | esperanza de vida a los 65 en $t+65$ | ONU WPP 2024 (de período) | proyectado |
| $\tau$ | 11 % personal + 10,77 % patronal = **21,77 %** | Anuario ANSES 2008–2023, p. 39 | normativa (sin aportes a PAMI/obra social) |
| $\rho$ | **40,3 %** (haber medio SIPA / salario imponible medio, dic-2023) | Anuario ANSES, p. 91 | observado, **constante** |
| $\bar A$ | **14,2** (obs.) o **30** (legal) años | ANSES p. 46 (varones 64 años: 15,8; mujeres 59 años: 12,6; promedio simple) / requisito legal | **supuesto — escenario** |

> ⚠️ $\bar A$ observado está **sesgado a la baja**: ANSES solo cuenta aportes desde jul-1994 y a personas con ≥12 meses de aporte. El 30 es el requisito legal (techo). La verdad está entre ambos. **Por eso el nivel del TBP no es interpretable todavía; la forma de la curva y su tendencia sí.**

In [ ]:
import os, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
try: display
except NameError: display = print

# --- Carga de datos -------------------------------------------------------
# Orden de busqueda: 1) carpeta del repo (../datos/procesados), 2) Drive (opcional), 3) descarga desde GitHub.
# En Colab con Drive: descomentar las dos lineas siguientes para montarlo.
# from google.colab import drive; drive.mount('/content/drive')
ARCHIVOS = ["series_tbp_extraidas.csv", "un_wpp2024_argentina_tabla_mortalidad_abreviada.csv"]
# Repositorio del proyecto
RAW_BASE = "https://raw.githubusercontent.com/carolinamsfelipe/Tp-Indicadores/main/datos/procesados"
CANDIDATOS = ["../datos/procesados", "datos/procesados",
              "/content/drive/MyDrive/JUANDE/CURSAR/06. Taller de indicadores/datos_tbp",
              "/content/drive/MyDrive/datos_tbp", "/content/datos_tbp", "."]
DATA = next((p for p in CANDIDATOS if os.path.exists(os.path.join(p, "series_tbp_extraidas.csv"))), None)
if DATA is None:   # no hay copia local: descargar desde GitHub (raw)
    DATA = "datos_tbp_descargados"; os.makedirs(DATA, exist_ok=True)
    for f in ARCHIVOS:
        urllib.request.urlretrieve(f"{RAW_BASE}/{f}", os.path.join(DATA, f))
print("Datos en:", os.path.abspath(DATA))

# Las salidas (CSV y PNG) se escriben en ./salidas_tbp para no pisar los datos versionados.
OUT = "salidas_tbp"; os.makedirs(OUT, exist_ok=True)

s = pd.read_csv(f"{DATA}/series_tbp_extraidas.csv").set_index("year")
mort = pd.read_csv(f"{DATA}/un_wpp2024_argentina_tabla_mortalidad_abreviada.csv", low_memory=False)
mort = mort[mort.Sex == "Total"]
display(s.loc[[1950, 1990, 2010, 2024, 2040, 2091], ["deis_nacimientos", "un_births_miles", "un_e65", "un_pob_65mas_miles"]])

## 1. Supervivencia de la cohorte (1−μ)

Para la cohorte nacida en $t$ se multiplica la probabilidad de sobrevivir cada grupo de edad ($0, 1\text{–}4, 5\text{–}9,\dots,60\text{–}64$) usando la tabla de mortalidad del **año calendario en que ese grupo vive**. Es más correcto que usar la tabla del año de nacimiento ("de período"), que ignora que la mortalidad va bajando.

In [ ]:
px = mort.pivot_table(index="Time", columns="AgeGrpStart", values="px")
grp = mort[mort.Time == 2000].sort_values("AgeGrpStart")[["AgeGrpStart", "AgeGrpSpan"]]
grp = grp[grp.AgeGrpStart < 65]

def surv65_cohorte(t):
    S = 1.0
    for a, sp in grp.itertuples(index=False):
        y = int(np.floor(t + a + sp/2))
        if y > px.index.max() or y < px.index.min():
            return np.nan
        S *= px.loc[y, a]
    return S

print("Ejemplo cohorte 2000: 1-mu =", round(surv65_cohorte(2000), 4), "| tabla de período del año 2000 (ignora mejoras futuras):", round(s.un_S65_periodo[2000], 4))

## 2. Cálculo por cohorte

$\tau$ y $\rho$ son constantes (los datos de ANSES no dan serie anual utilizable, ver `anses_valores_extraidos.csv`). **Todo el movimiento de la curva viene de la demografía** ($N_t$, $1-\mu$, $J$, $E_r$).

In [ ]:
TAU = 0.11 + 0.1077     # Anuario ANSES p.39 (aporte personal 11% + contribución patronal SIPA 10,77% actividades inc. a)
RHO = 0.403             # Anuario ANSES p.91: haber medio SIPA / salario imponible medio, dic-2023
A_OBS, A_LEGAL = 14.2, 30.0

rows = []
for t in range(1950, 2036):
    y = t + 65
    N_deis = s.deis_nacimientos.get(t, np.nan)
    N_un = s.un_births_miles[t] * 1000
    rows.append(dict(cohorte=t, anio_65=y,
        N_t=N_deis if not np.isnan(N_deis) else N_un,
        fuente_N="DEIS" if not np.isnan(N_deis) else "ONU proyectado",
        N_un=N_un, S65=surv65_cohorte(t),
        J=s.un_pob_65mas_miles[y]*1000, E65=s.un_e65[y], pob_edad65=s.un_pob_edad65_miles[y]*1000))
d = pd.DataFrame(rows)

for nom, A in [("obs", A_OBS), ("legal", A_LEGAL)]:
    d[f"TBP_A_{nom}"] = d.N_t*d.S65*A*TAU/(d.J*d.E65*RHO)     # fórmula original del grupo
    d[f"TBP_C_{nom}"] = d.N_t*d.S65*A*TAU/(d.J*RHO)           # en años (sin E_r)
    d[f"TBP_B_{nom}"] = A*TAU/(d.E65*RHO)                     # cohorte pura, sin N
d["D"] = d.N_t*d.S65/d.J                                       # factor demográfico

d.to_csv(f"{OUT}/tbp_argentina_cohortes_1950_2035.csv", index=False)
display(d[d.cohorte.isin([1950,1975,1990,2000,2010,2015,2020,2024,2030,2035])][
    ["cohorte","N_t","fuente_N","S65","J","E65","D","TBP_A_obs","TBP_A_legal","TBP_C_legal","TBP_B_legal"]])

## 3. Grupo de control con datos reales

Tres chequeos que **no dependen** de los supuestos de $\bar A$, $\tau$ ni $\rho$:

1. **Identidades algebraicas:** $TBP_A = D\cdot B$ y $TBP_A = TBP_C/E_r$.
2. **Consistencia demográfica:** la población de exactamente 65 años en $t+65$ (ONU) debería ser ≈ $N_t \cdot (1-\mu)$ (nacidos que sobreviven; la diferencia es migración y error de las fuentes).
3. **Consistencia entre fuentes:** nacimientos DEIS vs ONU.

In [ ]:
# 1) identidades
B = d.TBP_B_legal
assert np.allclose(d.TBP_A_legal, d.D * B)
assert np.allclose(d.TBP_A_legal, d.TBP_C_legal / d.E65)
print("✅ Identidades TBP_A = D×B y TBP_A = TBP_C/E_r: OK en las 86 cohortes")

# 2) cohorte reconstruida vs población observada/proyectada a los 65
d["ratio_pob65_vs_NS"] = d.pob_edad65 / (d.N_t * d.S65)
r = d.ratio_pob65_vs_NS
print(f"\n2) población(65 años en t+65) / (N_t × S65): media {r.mean():.3f} | min {r.min():.3f} | max {r.max():.3f}")
fuera = d[(r < 0.85) | (r > 1.20)][["cohorte", "fuente_N", "ratio_pob65_vs_NS"]]
print("   Cohortes fuera de [0.85, 1.20]:", "ninguna" if fuera.empty else "")
if not fuera.empty: display(fuera)

# 3) DEIS vs ONU
c = d[d.fuente_N == "DEIS"].copy()
c["ONU/DEIS"] = c.N_un / c.N_t
print(f"\n3) Nacimientos ONU/DEIS ({c.cohorte.min()}–{c.cohorte.max()}): media {c['ONU/DEIS'].mean():.3f} | min {c['ONU/DEIS'].min():.3f} | max {c['ONU/DEIS'].max():.3f}")
display(c.loc[c.cohorte.isin([1950,1980,2000,2015,2020,2022,2023,2024]), ["cohorte", "N_t", "N_un", "ONU/DEIS"]])

## 4. Gráficos

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
ax[0].plot(d.cohorte, d.N_t/1e3, label="Usado (DEIS ≤2024, ONU después)")
ax[0].plot(d.cohorte, d.N_un/1e3, "--", label="ONU WPP 2024")
ax[0].axvline(2024.5, color="gray", ls=":"); ax[0].set_title("Nacimientos (miles)"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

ax[1].plot(d.cohorte, d.D, color="tab:green")
ax[1].axvline(2024.5, color="gray", ls=":"); ax[1].set_title("D = N·(1−μ) / J₆₅₊ (factor demográfico)"); ax[1].grid(alpha=.3)

ax[2].plot(d.cohorte, d.TBP_A_legal, label="TBP_A  (Ā = 30, techo)")
ax[2].plot(d.cohorte, d.TBP_A_obs, label="TBP_A  (Ā = 14,2, piso sesgado)")
ax[2].plot(d.cohorte, d.TBP_B_legal, "--", label="TBP_B (cohorte pura, Ā = 30)")
ax[2].axhline(1, color="gray", ls=":"); ax[2].axvline(2024.5, color="gray", ls=":")
ax[2].set_title("TBP por cohorte"); ax[2].legend(fontsize=8); ax[2].grid(alpha=.3)
for a in ax: a.set_xlabel("año de nacimiento de la cohorte")
plt.tight_layout(); plt.savefig(f"{OUT}/tbp_argentina_graficos.png", dpi=130); plt.show()

## 5. Qué muestran los datos (lectura preliminar)

* **Caída de la natalidad real:** de ~756 mil (2010) a 413 mil (2024). La cohorte 2024 tiene el factor demográfico $D$ más bajo de toda la serie.
* **Rebote artificial después de 2024:** la ONU proyecta ~507 mil nacimientos en 2024 y ~520 mil hacia 2030. Eso **no refleja** la caída registrada por DEIS. El repunte de las cohortes 2025–2035 es un efecto del supuesto ONU, **no un dato**: hay que reemplazarlo con un escenario propio de natalidad.
* **Longevidad:** $E_{65}$ pasa de 17,9 (cohorte 1950) a ~24,6 (cohorte 2035). Eso empuja el TBP hacia abajo por dos vías (en $J$ y en $E_r$), el problema de doble conteo detectado en la auditoría.
* **La curva de $TBP_B$** (sin natalidad) baja suavemente por longevidad; la de $TBP_A$ baja mucho más y tiene la caída de natalidad encima. La diferencia entre ambas es el aporte de la natalidad.
* **Todavía no se puede afirmar:** niveles absolutos ($\bar A$ incierto), umbral "TBP=1", ni proyecciones más allá de 2024 basadas en natalidad ONU.

## 6. Fe de erratas (02-10-2026)

Las secciones 1 a 5 de arriba se dejan **tal como se presentaron originalmente**. Una revisión posterior detectó los puntos siguientes; se corrigen acá, al final, para que quede registro de la evolución del trabajo.

| # | Dónde | Decía | Corrección | Por qué |
|---|---|---|---|---|
| 1 | Sección 5, *Longevidad* | $E_{65}$ pasa de 17,9 a ~24,6 | $E_{65}$ va de **16,1** (cohorte 1956) a 24,6 (cohorte 2035); 17,9 corresponde a la cohorte 1950 | La cohorte 1956 cumple 65 en 2021 y $E_r$ sale de la tabla de *período* de ese año (COVID): pico atípico, no un error del cálculo |
| 2 | Sección 5, *curva de $TBP_B$* | "baja suavemente"; (en el resumen se citaron rangos 0,91→0,66 y 0,02–0,06) | $TBP_B$ va de **0,66 a 1,01**; $TBP_A$ de **0,018 a 0,066**; $TBP_C$ de 0,43 a 1,13 años. El máximo de los tres está en la cohorte 1956 | El resumen miró solo los extremos y omitió ese pico |
| 3 | Introducción y gráfico (*"Ā = 30, techo"*) | 30 años = techo legal | 30 años es el **requisito mínimo** para jubilarse, **no un máximo**: un promedio con ceros puede quedar muy por debajo | Error de concepto |
| 4 | Sección 5, natalidad | (no se mencionaba) | $N_t$ salta de 413.135 (DEIS 2024) a 508.067 (ONU 2025) | Es el cambio de fuente, no un hecho demográfico: el rebote posterior es un artefacto del supuesto ONU |

La celda siguiente **recalcula** esas cifras desde los datos, para que cualquiera pueda verificarlas.

In [ ]:
c = d.set_index("cohorte")
print(f"E_65: mín = {d.E65.min():.2f} (cohorte {int(d.loc[d.E65.idxmin(), 'cohorte'])}) | máx = {d.E65.max():.2f} (cohorte {int(d.loc[d.E65.idxmax(), 'cohorte'])}) | cohorte 1950 = {c.E65[1950]:.2f}")
print(f"TBP_B (Ā=30): {d.TBP_B_legal.min():.3f} a {d.TBP_B_legal.max():.3f} (máx. en la cohorte {int(d.loc[d.TBP_B_legal.idxmax(), 'cohorte'])})")
print(f"TBP_A (Ā=30): {d.TBP_A_legal.min():.4f} a {d.TBP_A_legal.max():.4f}")
print(f"TBP_C (Ā=30, años): {d.TBP_C_legal.min():.3f} a {d.TBP_C_legal.max():.3f}")
print(f"N_t usado: cohorte 2024 = {c.N_t[2024]:,.0f} ({c.fuente_N[2024]}) -> cohorte 2025 = {c.N_t[2025]:,.0f} ({c.fuente_N[2025]})")

## 7. Solución 1 — Sensibilidad por bandas del parámetro $\bar A$

**Opción 1 de la propuesta del grupo.** En vez de fijar $\bar A$ en un único valor, corremos el modelo para $\bar A \in \{15, 20, 25, 30\}$ años y graficamos una **banda de incertidumbre**.

Contexto: hoy $\bar A$ tiene dos escenarios: 14,2 (observado ANSES, **sesgado a la baja**: ANSES cuenta aportes solo desde jul-1994 y a personas con ≥12 meses de aporte) y 30 (requisito legal de referencia: es el mínimo para jubilarse, no un máximo). La verdad está entre ambos, pero no sabemos dónde.

$$TBP_A = \frac{N_t(1-\mu)\,\bar A\,\tau}{J_{t+65}\,E_r\,\rho},\qquad TBP_C = \frac{N_t(1-\mu)\,\bar A\,\tau}{J_{t+65}\,\rho}\ \text{(años)},\qquad TBP_B=\frac{\bar A\,\tau}{E_r\,\rho}\ \text{(cohorte pura)}$$

Constantes: $\tau = 0{,}2177$ (11 % personal + 10,77 % patronal; Anuario ANSES p. 39) y $\rho = 0{,}403$ (tasa de sustitución ANSES, dic-2023, p. 91). Todo lo demás sale del CSV `tbp_argentina_cohortes_1950_2035.csv`; no se inventa ningún dato.

> **Qué NO hace este notebook:** no corrige el sesgo de $\bar A$ ni lo estima. Es una forma honesta de *mostrar* la incertidumbre de nivel. La estimación va por la Opción 2 (calibración con EPH) y la Opción 3 ($\bar A_t$ dinámico).

### 7.2 Parámetros y cálculo para cada escenario de $\bar A$

In [ ]:
OUT = globals().get("OUT", DATA)   # carpeta de salidas del notebook
TAU = 0.11 + 0.1077     # Anuario ANSES p.39
RHO = 0.403             # Anuario ANSES p.91 (dic-2023), constante
A_ESCENARIOS = [15, 20, 25, 30]     # parametrizable
CORTE = 2024.5          # DEIS observado (<=2024) vs ONU proyectado (>=2025)

# N_t * S65 / J = D (factor demografico, ya en el CSV); lo reconstruimos desde las columnas base
d["D_calc"] = d.N_t * d.S65 / d.J
assert np.allclose(d.D_calc, d.D)

filas = []
for A in A_ESCENARIOS:
    t = pd.DataFrame({"cohorte": d.cohorte, "anio_65": d.anio_65, "fuente_N": d.fuente_N, "A": A})
    t["TBP_A"] = d.N_t * d.S65 * A * TAU / (d.J * d.E65 * RHO)
    t["TBP_C"] = d.N_t * d.S65 * A * TAU / (d.J * RHO)
    t["TBP_B"] = A * TAU / (d.E65 * RHO)
    for v in ["TBP_A", "TBP_B", "TBP_C"]:
        base = t.loc[t.cohorte == 1950, v].iloc[0]
        t[v + "_idx"] = 100 * t[v] / base          # indice base cohorte 1950 = 100
    filas.append(t)
sens = pd.concat(filas, ignore_index=True)
sens = sens.rename(columns={"A": "A_barra"})
sens.head()

### 7.3 Test de control (asserts)

1. **Consistencia con el CSV existente:** nuestro cálculo debe reproducir `TBP_*_legal` (Ā = 30) y `TBP_*_obs` (Ā = 14,2) del CSV.
2. **Linealidad en $\bar A$:** $TBP(2\bar A) = 2\cdot TBP(\bar A)$.
3. **Invarianza del índice normalizado:** el índice base 1950 = 100 es idéntico en todos los escenarios.

In [ ]:
def tbp(A, tipo):
    if tipo == "A": return d.N_t * d.S65 * A * TAU / (d.J * d.E65 * RHO)
    if tipo == "C": return d.N_t * d.S65 * A * TAU / (d.J * RHO)
    if tipo == "B": return A * TAU / (d.E65 * RHO)

# 1) reproduce el CSV existente
for tipo in "ABC":
    assert np.allclose(tbp(30.0, tipo), d[f"TBP_{tipo}_legal"])
    assert np.allclose(tbp(14.2, tipo), d[f"TBP_{tipo}_obs"])
print("OK 1) reproduce TBP_*_legal (A=30) y TBP_*_obs (A=14,2) del CSV")

# 2) linealidad
for tipo in "ABC":
    for A in [7.5, 10, 12.5, 15]:
        assert np.allclose(tbp(2 * A, tipo), 2 * tbp(A, tipo))
print("OK 2) linealidad: TBP(2A) = 2*TBP(A) para A, B y C")

# 3) indice normalizado identico entre escenarios
for v in ["TBP_A_idx", "TBP_B_idx", "TBP_C_idx"]:
    piv = sens.pivot(index="cohorte", columns="A_barra", values=v)
    ref = piv.iloc[:, 0]
    for c in piv.columns:
        assert np.allclose(piv[c], ref)
print("OK 3) el indice normalizado (base 1950=100) es identico entre escenarios")

### 7.4 Hallazgo clave: $\bar A$ cambia el **nivel**, no la **historia**

Como $\bar A$ entra en el numerador como factor multiplicativo y no depende de la cohorte $t$ (en este modelo es una constante), el TBP es **lineal homogéneo** en $\bar A$:

$$TBP_C(t;\bar A) = \bar A\cdot\underbrace{\frac{N_t(1-\mu_t)\,\tau}{J_{t+65}\,\rho}}_{k_C(t)\ \text{(no depende de }\bar A)}$$

Entonces, para dos escenarios cualesquiera $\bar A_1,\bar A_2$ y dos cohortes $t,s$:

$$\frac{TBP_C(t;\bar A_1)}{TBP_C(s;\bar A_1)}=\frac{\bar A_1\,k_C(t)}{\bar A_1\,k_C(s)}=\frac{k_C(t)}{k_C(s)}=\frac{\bar A_2\,k_C(t)}{\bar A_2\,k_C(s)}=\frac{TBP_C(t;\bar A_2)}{TBP_C(s;\bar A_2)}$$

$\bar A$ se cancela. Lo mismo vale para $TBP_A$ ($k_A=k_C/E_r$) y $TBP_B$. Por lo tanto:

* El **índice normalizado** (cohorte 1950 = 100) es **idéntico** para cualquier $\bar A$: $\text{Idx}(t)=100\,k(t)/k(1950)$.
* Las **variaciones porcentuales**, los **cocientes entre cohortes**, los **puntos de inflexión** y las **tendencias relativas** no dependen de $\bar A$.
* Lo que sí cambia con $\bar A$ es el **nivel**: la curva se estira verticalmente en un factor $\bar A_2/\bar A_1$ (entre $\bar A=15$ y $\bar A=30$, un factor 2).

**Consecuencia para la presentación:** la incertidumbre sobre $\bar A$ es incertidumbre de *nivel*, no de *comportamiento*. La historia (caída por natalidad y longevidad, ver abajo) es robusta; el valor absoluto, no.

> **Advertencia sobre la parte proyectada:** a partir de la cohorte 2025 los nacimientos $N_t$ son proyección ONU, no dato DEIS. La ONU proyecta ~507 mil nacimientos en 2024 vs ~413 mil observados por DEIS; el **rebote posterior a 2024 es un artefacto del supuesto ONU**, no un dato. No debe leerse como recuperación.
>
> **Sobre el nivel:** no interpretamos "TBP = 1" como umbral fiscal; el nivel depende de $\bar A$, de $\tau$ y de $\rho$ constantes, y no de un análisis de sostenibilidad.

### 7.5 Gráficos

In [ ]:
COL = plt.cm.viridis(np.linspace(0.1, 0.85, len(A_ESCENARIOS)))

def banda(ax, var, titulo, ylabel):
    piv = sens.pivot(index="cohorte", columns="A_barra", values=var)
    ax.fill_between(piv.index, piv.min(axis=1), piv.max(axis=1), color="tab:blue", alpha=0.15,
                    label=f"banda min-max (A = {min(A_ESCENARIOS)} a {max(A_ESCENARIOS)})")
    for c, A in zip(COL, A_ESCENARIOS):
        ax.plot(piv.index, piv[A], color=c, lw=1.6, label=f"A = {A}")
    ax.axvline(CORTE, color="gray", ls="--", lw=1)
    ax.text(CORTE + 0.5, ax.get_ylim()[1] * 0.97, "2024/2025: DEIS | ONU proyectado",
            fontsize=7.5, color="gray", va="top")
    ax.set_title(titulo, fontsize=10); ax.set_ylabel(ylabel); ax.set_xlabel("año de nacimiento de la cohorte")
    ax.grid(alpha=.3); ax.legend(fontsize=7.5, loc="upper right", bbox_to_anchor=(1, 0.88))

fig, ax = plt.subplots(1, 3, figsize=(19, 5.2))
banda(ax[0], "TBP_C", "TBP_C (versión recomendada, en años)\nbanda de incertidumbre de NIVEL", "TBP_C (años)")
banda(ax[1], "TBP_A", "TBP_A (fórmula original, con $E_r$)\nbanda de incertidumbre de NIVEL", "TBP_A")

# Panel (b): indice normalizado
for c, A in zip(COL, A_ESCENARIOS):
    ss = sens[sens.A_barra == A]
    ax[2].plot(ss.cohorte, ss.TBP_C_idx, color=c, lw=3.5 - 0.7 * A_ESCENARIOS.index(A), alpha=.8, label=f"A = {A}")
ax[2].axvline(CORTE, color="gray", ls="--", lw=1)
ax[2].axhline(100, color="gray", ls=":", lw=.8)
ax[2].set_title("Índice normalizado de TBP_C (cohorte 1950 = 100)\nlas 4 curvas se superponen: Ā no cambia la forma", fontsize=10)
ax[2].set_xlabel("año de nacimiento de la cohorte"); ax[2].set_ylabel("índice (1950 = 100)")
ax[2].grid(alpha=.3); ax[2].legend(fontsize=7.5)
ax[2].text(CORTE + 0.5, ax[2].get_ylim()[1] * 0.97, "post-2024: supuesto ONU\n(rebote = artefacto)", fontsize=7.5, color="gray", va="top")

fig.suptitle("Sensibilidad al parámetro Ā (años de aporte promedio): cambia el nivel, no la historia", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(f"{OUT}/tbp_sensibilidad_A.png", dpi=130)
plt.show()

### 7.6 Resultados: rango de $TBP_C$ por escenario en cohortes seleccionadas

In [ ]:
sel = [1950, 1975, 1990, 2000, 2010, 2024]
tabla = sens[sens.cohorte.isin(sel)].pivot(index="cohorte", columns="A_barra", values="TBP_C").round(3)
tabla["max/min"] = (tabla[max(A_ESCENARIOS)] / tabla[min(A_ESCENARIOS)]).round(2)
display(tabla)

idx = sens[(sens.A_barra == A_ESCENARIOS[0]) & sens.cohorte.isin(sel)].set_index("cohorte")["TBP_C_idx"].round(1)
print("Índice TBP_C (1950=100), igual en todos los escenarios:")
print(idx.to_string())

### 7.7 Exportar

`tbp_sensibilidad_A.csv`: una fila por cohorte × escenario, con TBP_A, TBP_B, TBP_C en nivel y como índice base 1950 = 100.

In [ ]:
sens.to_csv(f"{OUT}/tbp_sensibilidad_A.csv", index=False)
print("Escrito:", f"{OUT}/tbp_sensibilidad_A.csv", sens.shape)
print("Escrito:", f"{OUT}/tbp_sensibilidad_A.png")

### 7.8 Lectura y límites

* **Robusto (no depende de Ā):** forma de la curva, caída relativa entre cohortes, tendencias. Se comunica con el índice normalizado.
* **Incierto (sí depende de Ā):** el nivel. Se comunica con la banda 15–30 (factor 2 entre extremos).
* **No resuelto:** el sesgo de $\bar A$. Las cuatro cifras 15/20/25/30 son una grilla ilustrativa, no una distribución de probabilidad: la banda no es un intervalo de confianza.
* **Post-2024:** el rebote es un artefacto del supuesto ONU de nacimientos.
* **Próximos pasos:** Opción 2 (calibrar $\bar A$ con EPH) para acotar el nivel; Opción 3 ($\bar A_t$ dinámico por cohorte) que, a diferencia de esta, **sí** puede cambiar la forma de la curva.

## 8. Solución 2 — Calibración de Ā con la EPH (método; sin datos reales todavía)

<h1 style="color:#b00020;border:4px solid #b00020;padding:12px;text-align:center">
DATOS SINTÉTICOS — no son Argentina<br>(las celdas de prueba usan un dataset inventado; sirven solo para verificar el código)
</h1>

**Objetivo.** Estimar Ā (años de aporte promedio por persona de la cohorte, con ceros incluidos) como

Ā_c = Σ_{a=18}^{64} S_c(a) · d_c(a)

donde S_c(a) = P(sobrevivir hasta la edad a) y d_c(a) = P(aportar a la edad a | vivo), aproximada con la **tasa de empleo registrado** de la EPH tratada como pseudo-panel por año de nacimiento (cohorte = ANO4 − CH06) y edad.

**Estado:** este notebook NO produce ningún valor de Ā para Argentina hasta que se coloquen los microdatos reales de la EPH en `datos/eph/` (ver `datos/eph/README.md` y `docs/calibracion_A_EPH.md`). Sin esos archivos termina limpio con el mensaje "faltan datos".

Estructura: (1) configuración y funciones, (2) prueba con datos sintéticos + asserts, (3) prueba del lector de archivos, (4) corrida real (solo si hay datos).

In [ ]:
# ============ 1. CONFIGURACIÓN Y FUNCIONES ============
import os, re, io, glob, zipfile, tempfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EDAD_MIN, EDAD_MAX = 18, 64
EDADES = np.arange(EDAD_MIN, EDAD_MAX + 1)          # 47 edades simples
RUTAS_CANDIDATAS = ["datos/eph", "../datos/eph", "../../datos/eph"]
RUTAS_S65 = ["datos/procesados/tbp_argentina_cohortes_1950_2035.csv",
             "../datos/procesados/tbp_argentina_cohortes_1950_2035.csv"]
MIN_N_CELDA = 20        # mínimo de observaciones (sin ponderar) por celda cohorte x edad
MIN_EDADES_COHORTE = 3  # mínimo de edades observadas para estimar el efecto de una cohorte
GOMPERTZ_B = 0.085      # pendiente de la mortalidad adulta (supuesto; ver docs)
# Regla para cuentapropistas/patrones (la base individual NO trae una variable inequívoca de
# aporte de independientes: a verificar contra el diseño de registro). Opciones:
#   "solo_asalariados": independientes cuentan como NO aportantes (cota inferior)
#   "pp07i":            independientes aportan si PP07I == 1 (uso a verificar; ver docs)
#   "escenario":        independientes aportan con probabilidad IND_SHARE (supuesto explícito)
IND_RULE_DEFAULT = "solo_asalariados"
IND_SHARE_DEFAULT = None
NEEDED = {"ANO4", "TRIMESTRE", "CH04", "CH06", "ESTADO", "CAT_OCUP", "PP07H", "PP07I", "PONDERA"}

def banner(txt="DATOS SINTÉTICOS — no son Argentina"):
    line = "#" * 78
    print("\n".join([line, "##" + txt.center(74) + "##", line]))

# ---------- (a) lectura de archivos EPH ----------
def find_eph_files(rutas=RUTAS_CANDIDATAS):
    """Busca bases individuales (usu_individual_T*.txt/.csv, o EPH_usu_*.zip) en las rutas candidatas."""
    out = []
    for r in rutas:
        if not os.path.isdir(r):
            continue
        for p in sorted(glob.glob(os.path.join(r, "**", "*"), recursive=True)):
            b = os.path.basename(p).lower()
            if os.path.isfile(p) and re.search(r"usu_individual.*\.(txt|csv)$", b):
                out.append({"path": p, "member": None})
            elif os.path.isfile(p) and b.endswith(".zip") and "eph_usu" in b:
                with zipfile.ZipFile(p) as z:
                    for m in z.namelist():
                        if re.search(r"usu_individual.*\.(txt|csv)$", m.lower()):
                            out.append({"path": p, "member": m})
    return out

def _read_csv_flex(handle_or_path):
    df = pd.read_csv(handle_or_path, sep=";", usecols=lambda c: c.strip().upper() in NEEDED,
                     low_memory=False, encoding="latin-1")
    return df

def read_individual(src):
    """Lee UNA base individual trimestral (solo columnas necesarias). Devuelve DataFrame con columnas en MAYÚSCULAS."""
    if src["member"] is None:
        df = _read_csv_flex(src["path"])
    else:
        with zipfile.ZipFile(src["path"]) as z, z.open(src["member"]) as f:
            df = _read_csv_flex(io.TextIOWrapper(f, encoding="latin-1"))
    df.columns = [c.strip().upper() for c in df.columns]
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    faltan = {"ANO4", "CH06", "ESTADO", "PONDERA"} - set(df.columns)
    if faltan:
        raise ValueError(f"Faltan columnas obligatorias {sorted(faltan)} en {src}. Revisar el diseño de registro.")
    for opt in ("CAT_OCUP", "PP07H", "PP07I", "CH04", "TRIMESTRE"):
        if opt not in df.columns:
            df[opt] = np.nan
    return df

def read_all(srcs):
    return pd.concat([read_individual(s) for s in srcs], ignore_index=True)

# ---------- (b) pseudo-panel: densidad por edad x cohorte ----------
def add_flags(df, ind_rule=IND_RULE_DEFAULT, ind_share=IND_SHARE_DEFAULT, sexo=None):
    """Filtra 18-64 con entrevista individual realizada y construye 'reg' (prob. de estar aportando).
    Asalariado (CAT_OCUP==3): reg = (PP07H==1). Independiente (1,2): según ind_rule. Resto: 0."""
    d = df[(df["CH06"] >= EDAD_MIN) & (df["CH06"] <= EDAD_MAX) & (df["ESTADO"].isin([1, 2, 3]))].copy()
    if sexo is not None:
        d = d[d["CH04"] == sexo]
    ocupado = d["ESTADO"] == 1
    asal = ocupado & (d["CAT_OCUP"] == 3)
    indep = ocupado & d["CAT_OCUP"].isin([1, 2])
    reg = np.zeros(len(d))
    reg[asal.values] = (d.loc[asal, "PP07H"] == 1).astype(float).values   # Ns/Nr (9) o 2 => no registrado
    if ind_rule == "pp07i":
        reg[indep.values] = (d.loc[indep, "PP07I"] == 1).astype(float).values
    elif ind_rule == "escenario":
        if ind_share is None or not (0 <= ind_share <= 1):
            raise ValueError("ind_rule='escenario' requiere ind_share en [0,1] (supuesto explícito).")
        reg[indep.values] = float(ind_share)
    elif ind_rule != "solo_asalariados":
        raise ValueError(f"ind_rule desconocida: {ind_rule}")
    d["reg"] = reg
    d["edad"] = d["CH06"].astype(int)
    d["cohorte"] = (d["ANO4"] - d["CH06"]).astype(int)     # error de +-1 año (no se usa la fecha exacta)
    d["w"] = d["PONDERA"].astype(float)
    return d[["cohorte", "edad", "w", "reg"]]

def build_cells(d, min_n=MIN_N_CELDA):
    """Densidad ponderada por (cohorte, edad), agregando todos los trimestres/años en que esa cohorte tuvo esa edad."""
    d = d.assign(wreg=d["w"] * d["reg"])
    g = d.groupby(["cohorte", "edad"]).agg(n=("w", "size"), w=("w", "sum"), wreg=("wreg", "sum")).reset_index()
    g["dens"] = g["wreg"] / g["w"]
    g = g[g["n"] >= min_n].reset_index(drop=True)
    return g

# ---------- modelo logit aditivo cohorte + edad (supuesto: perfil por edad estable entre cohortes) ----------
def fit_age_cohort_logit(cells, iters=100, ridge=1e-3):
    coh = np.sort(cells["cohorte"].unique()); eds = np.sort(cells["edad"].unique())
    ci = {c: i for i, c in enumerate(coh)}; ei = {a: i for i, a in enumerate(eds)}
    X = np.zeros((len(cells), len(coh) + len(eds) - 1))
    for r, (c, a) in enumerate(zip(cells["cohorte"], cells["edad"])):
        X[r, ci[c]] = 1.0
        if ei[a] > 0:
            X[r, len(coh) + ei[a] - 1] = 1.0
    y = cells["dens"].values; m = cells["n"].values.astype(float)
    yy = np.clip(y, 0.01, 0.99)
    beta = np.linalg.solve(X.T @ (m[:, None] * X) + ridge * np.eye(X.shape[1]), X.T @ (m * np.log(yy / (1 - yy))))
    for _ in range(iters):
        eta = X @ beta
        mu = np.clip(1 / (1 + np.exp(-eta)), 1e-6, 1 - 1e-6)
        W = m * mu * (1 - mu)
        z = eta + (y - mu) / (mu * (1 - mu))
        new = np.linalg.solve(X.T @ (W[:, None] * X) + ridge * np.eye(X.shape[1]), X.T @ (W * z))
        if np.max(np.abs(new - beta)) < 1e-9:
            beta = new; break
        beta = new
    alpha = pd.Series(beta[:len(coh)], index=coh)
    f = pd.Series(np.r_[0.0, beta[len(coh):]], index=eds)
    n_ed = cells.groupby("cohorte")["edad"].nunique().reindex(coh)
    return {"alpha": alpha, "f": f, "n_edades": n_ed, "cells": cells}

def _sig(x): return 1 / (1 + np.exp(-x))

def extrapolar_alpha(fit, cohortes, modo="ultimas_k", k=5, ventana_tendencia=15):
    """Efecto de cohorte para cohortes sin datos suficientes (jóvenes o futuras). Supuesto explícito."""
    ok = fit["alpha"][fit["n_edades"] >= MIN_EDADES_COHORTE]
    out = {}
    for c in cohortes:
        if c in ok.index:
            out[c] = ok[c]
        elif c > ok.index.max():
            if modo == "ultimas_k":
                out[c] = ok.iloc[-k:].mean()
            elif modo == "tendencia":
                w = ok.iloc[-ventana_tendencia:]; b = np.polyfit(w.index.values, w.values, 1)
                out[c] = np.polyval(b, c)
            else:
                raise ValueError(modo)
        else:
            out[c] = ok.iloc[:k].mean()
    return pd.Series(out)

def densidad_modelada(fit, alpha_c):
    """Matriz cohorte x edad (18..64) de d_c(a). Edades sin datos usan el perfil f(a) del modelo."""
    f = fit["f"].reindex(EDADES)
    f = f.interpolate(limit_direction="both")           # edades sin ninguna observación: interpola/extiende el perfil
    return pd.DataFrame(_sig(alpha_c.values[:, None] + f.values[None, :]), index=alpha_c.index, columns=EDADES)

# ---------- (c) supervivencia y Ā ----------
def supervivencia_gompertz(s65, b=GOMPERTZ_B):
    """S(a) desde el nacimiento para a=18..64, Gompertz con pendiente b calibrado para que S(65)=s65 (por cohorte)."""
    s65 = np.asarray(s65, float)
    a = -np.log(s65) * b / (np.exp(65 * b) - 1)
    return np.exp(-(a[:, None] / b) * (np.exp(b * EDADES[None, :]) - 1))

def abar_por_cohorte(dens, s65):
    S = supervivencia_gompertz(s65.reindex(dens.index).values)
    return pd.Series((S * dens.values).sum(axis=1), index=dens.index, name="A_bar"), S

def pipeline(df_ind, s65, ind_rule=IND_RULE_DEFAULT, ind_share=IND_SHARE_DEFAULT, sexo=None,
             modo_extrap="ultimas_k", cohortes=None):
    """df individual EPH -> tabla por cohorte. Idéntico para datos sintéticos y reales."""
    d = add_flags(df_ind, ind_rule, ind_share, sexo)
    cells = build_cells(d)
    fit = fit_age_cohort_logit(cells)
    cohortes = list(cohortes) if cohortes is not None else list(s65.index)
    alpha_c = extrapolar_alpha(fit, cohortes, modo=modo_extrap)
    dens = densidad_modelada(fit, alpha_c)
    abar, S = abar_por_cohorte(dens, s65)
    tabla = pd.DataFrame({"A_bar": abar,
                          "n_edades_obs": fit["n_edades"].reindex(abar.index).fillna(0).astype(int),
                          "S65": s65.reindex(abar.index),
                          "A_max_sin_densidad": S.sum(axis=1)})
    return {"tabla": tabla, "dens": dens, "fit": fit, "cells": cells}

def chequeos(res):
    t, dens, cells = res["tabla"], res["dens"], res["cells"]
    assert np.all((cells["dens"] >= 0) & (cells["dens"] <= 1)), "densidad observada fuera de [0,1]"
    assert np.all((dens.values >= 0) & (dens.values <= 1)), "densidad modelada fuera de [0,1]"
    assert np.all((t["A_bar"] >= 0) & (t["A_bar"] <= 47)), "A_bar fuera de [0,47]"
    assert np.all(t["A_bar"] <= t["A_max_sin_densidad"] + 1e-9), "A_bar supera sum S(a) (densidad=1)"
    # monotonía: con la misma densidad, mayor S65 => mayor A_bar
    s_alta = (t["S65"] + 0.02).clip(upper=0.999)
    a2, _ = abar_por_cohorte(dens, s_alta)
    assert np.all(a2.values > t["A_bar"].values - 1e-12), "monotonía en S65 violada"
    return True
print("Funciones cargadas.")

### 8.2 Prueba con DATASET SINTÉTICO

<h1 style="color:#b00020;border:4px solid #b00020;padding:12px;text-align:center">DATOS SINTÉTICOS — no son Argentina</h1>

Se simula una EPH "de juguete" (columnas con los mismos nombres que la base individual de INDEC) con una verdad conocida (efecto de cohorte + perfil por edad + una pequeña oscilación de período), para comprobar que el pipeline recupera Ā. **Los números de esta sección no dicen nada sobre Argentina.**

In [ ]:
# ============ 2. DATASET SINTÉTICO + ASSERTS ============
banner("DATOS SINTÉTICOS — no son Argentina")

def f_edad_sintetica(a):  # perfil por edad inventado (logit)
    return 1.2 * np.exp(-((a - 40) / 18.0) ** 2) - 1.0

def alpha_sintetico(c):   # efecto de cohorte inventado (logit)
    return -0.2 + 0.008 * (np.asarray(c) - 1970)

def periodo_sintetico(y):
    return 0.10 * np.sin((np.asarray(y) - 2003) / 3.0)

def s65_sintetico(c):
    return np.clip(0.70 + 0.0035 * (np.asarray(c) - 1950), 0.5, 0.97)

def generar_eph_sintetica(anios=range(2003, 2025), n_por_edad=25, seed=123):
    rng = np.random.default_rng(seed)
    filas = []
    for y in anios:
        for q in (1, 2, 3, 4):
            for a in range(14, 71):
                p = _sig(alpha_sintetico(y - a) + f_edad_sintetica(a) + periodo_sintetico(y))
                u = rng.random(n_por_edad)
                reg = u < p
                otro = ~reg
                v = rng.random(n_por_edad)
                estado = np.where(reg, 1, np.where(v < 0.45, 1, 3))   # no registrados: 45% ocupados, resto inactivos
                cat = np.where(estado == 1, np.where(reg | (v < 0.30), 3, 2), np.nan)  # parte son cuentapropistas
                pp07h = np.where(cat == 3, np.where(reg, 1, 2), np.nan)
                pp07i = np.where(cat == 2, 2, np.nan)
                filas.append(pd.DataFrame({"ANO4": y, "TRIMESTRE": q, "CH04": rng.integers(1, 3, n_por_edad),
                                           "CH06": a, "ESTADO": estado, "CAT_OCUP": cat, "PP07H": pp07h,
                                           "PP07I": pp07i, "PONDERA": rng.integers(300, 900, n_por_edad)}))
    return pd.concat(filas, ignore_index=True)

df_sint = generar_eph_sintetica()
print("Filas sintéticas:", len(df_sint))
cohortes_sint = np.arange(1940, 2036)
s65_sint = pd.Series(s65_sintetico(cohortes_sint), index=cohortes_sint)
res_s = pipeline(df_sint, s65_sint, cohortes=cohortes_sint)
t = res_s["tabla"]

# verdad conocida
dens_true = pd.DataFrame(_sig(alpha_sintetico(cohortes_sint)[:, None] + f_edad_sintetica(EDADES)[None, :]),
                         index=cohortes_sint, columns=EDADES)
a_true, _ = abar_por_cohorte(dens_true, s65_sint)
t["A_bar_verdad_sintetica"] = a_true
t["error"] = t["A_bar"] - t["A_bar_verdad_sintetica"]

# --- asserts de control ---
chequeos(res_s)
obs = t[t["n_edades_obs"] >= MIN_EDADES_COHORTE]
mae = obs["error"].abs().mean()
print(f"[sintético] error absoluto medio de Ā en cohortes observadas: {mae:.3f} años")
assert mae < 1.5, "el pipeline no recupera la verdad sintética (tolerancia 1,5 años; el efecto de período sintético se confunde parcialmente con cohorte/edad: problema APC)"
fut = t[t["n_edades_obs"] < MIN_EDADES_COHORTE]
print(f"[sintético] error medio en cohortes SIN datos (extrapoladas con 'ultimas_k'): {fut['error'].mean():+.3f} años "
      "(sesgo esperado si hay tendencia entre cohortes: es el costo del supuesto de extrapolación)")
# monotonía en S65 a igual densidad (test directo)
a_lo, _ = abar_por_cohorte(res_s["dens"], s65_sint - 0.05)
a_hi, _ = abar_por_cohorte(res_s["dens"], s65_sint + 0.02)
assert np.all(a_hi.values > a_lo.values), "Ā debería crecer con S65"
# densidad cero => Ā cero ; densidad 1 => Ā = sum S(a) <= 47
z, _ = abar_por_cohorte(res_s["dens"] * 0.0, s65_sint); assert np.allclose(z, 0)
u, S1 = abar_por_cohorte(res_s["dens"] * 0.0 + 1.0, s65_sint); assert np.all(u <= 47) and np.all(u > 0)
print("Todos los asserts sintéticos OK.")

# gráficos
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for c, col in [(1950, "C0"), (1970, "C1"), (1990, "C2"), (2000, "C3")]:
    cc = res_s["cells"][res_s["cells"]["cohorte"] == c]
    ax[0].scatter(cc["edad"], cc["dens"], s=10, color=col, alpha=.6)
    ax[0].plot(EDADES, res_s["dens"].loc[c].values, color=col, label=f"cohorte {c}")
ax[0].set_xlabel("edad"); ax[0].set_ylabel("densidad d(a)"); ax[0].legend(); ax[0].set_title("SINTÉTICO: puntos observados vs modelo")
ax[1].plot(t.index, t["A_bar"], label="estimado"); ax[1].plot(t.index, t["A_bar_verdad_sintetica"], "--", label="verdad sintética")
ax[1].set_xlabel("cohorte"); ax[1].set_ylabel("Ā (años)"); ax[1].legend()
ax[1].set_title("SINTÉTICO: Ā por cohorte")
fig.suptitle("DATOS SINTÉTICOS — no son Argentina", color="red", fontsize=16, fontweight="bold")
plt.tight_layout(); plt.show()
banner("FIN PRUEBA SINTÉTICA — estos números NO son Argentina")

### 8.3 Prueba del lector de archivos (con un archivo de juguete en formato INDEC)

Escribe un subconjunto del dataset **sintético** en una carpeta temporal con el nombre `usu_individual_T100.txt` (separador `;`), y verifica que `find_eph_files` + `read_all` + `pipeline` dan lo mismo que con el DataFrame en memoria. No toca `datos/eph/`.

In [ ]:
banner("DATOS SINTÉTICOS — prueba del lector, no son Argentina")
with tempfile.TemporaryDirectory() as tmp:
    sub = df_sint[df_sint["ANO4"].isin([2010, 2011, 2012, 2013])].copy()
    sub.columns = [c.lower() if c in ("CH04", "CH06", "ESTADO") else c for c in sub.columns]  # mezcla de mayúsculas/minúsculas
    sub.to_csv(os.path.join(tmp, "usu_individual_T100.txt"), sep=";", index=False)
    srcs = find_eph_files([tmp]); assert len(srcs) == 1
    leido = read_all(srcs)
    assert set(NEEDED) <= set(leido.columns)
    assert len(leido) == len(sub)
    # zip con el nombre típico de INDEC
    zp = os.path.join(tmp, "EPH_usu_1_Trim_2012_txt.zip")
    with zipfile.ZipFile(zp, "w") as z:
        z.write(os.path.join(tmp, "usu_individual_T100.txt"), "usu_individual_T112.txt")
    os.remove(os.path.join(tmp, "usu_individual_T100.txt"))
    srcs2 = find_eph_files([tmp]); assert len(srcs2) == 1 and srcs2[0]["member"] is not None
    assert len(read_all(srcs2)) == len(sub)
print("Lector OK (txt y zip) con datos sintéticos.")
# find_eph_files sobre una carpeta inexistente => lista vacía, sin error
assert find_eph_files(["/ruta/que/no/existe"]) == []

### 8.4 Corrida con datos REALES (solo si están en `datos/eph/`)

Si no hay archivos, el notebook termina limpio con "faltan datos". Si los hay, aplica el mismo `pipeline` y guarda `salidas_tbp/abar_eph_por_cohorte.csv` (carpeta ignorada por git). Las cifras de referencia 14,2 (ANSES, observado, sesgado a la baja) y 30 (requisito legal de referencia, no un máximo) se muestran solo como contexto; **no son objetivo de calibración**.

In [ ]:
# ============ 4. CORRIDA REAL ============
srcs_reales = find_eph_files(RUTAS_CANDIDATAS)
HAY_DATOS = len(srcs_reales) > 0

if not HAY_DATOS:
    print("=" * 78)
    print("FALTAN DATOS: no se encontraron bases individuales de la EPH en", RUTAS_CANDIDATAS)
    print("Qué hacer: bajar las bases individuales (usu_individual_T*.txt o EPH_usu_*_txt.zip) desde INDEC")
    print("y dejarlas en datos/eph/ (ver datos/eph/README.md). No se calculó ningún Ā para Argentina.")
    print("=" * 78)
else:
    s65_path = next((p for p in RUTAS_S65 if os.path.exists(p)), None)
    if s65_path is None:
        print("FALTAN DATOS: no se encontró tbp_argentina_cohortes_1950_2035.csv (S65 por cohorte).")
        HAY_DATOS = False
if HAY_DATOS:
    print(f"Archivos EPH encontrados: {len(srcs_reales)}")
    df_real = read_all(srcs_reales)
    print("Años cubiertos:", int(df_real["ANO4"].min()), "-", int(df_real["ANO4"].max()), "| filas:", len(df_real))
    s65_real = pd.read_csv(s65_path).set_index("cohorte")["S65"]
    coh_real = s65_real.index
    escenarios = {"solo_asalariados": dict(ind_rule="solo_asalariados"),
                  "pp07i": dict(ind_rule="pp07i"),
                  "indep_50pct (supuesto)": dict(ind_rule="escenario", ind_share=0.5)}
    salida = {}
    for nombre, kw in escenarios.items():
        r = pipeline(df_real, s65_real, cohortes=coh_real, **kw)
        chequeos(r)
        salida[nombre] = r["tabla"]["A_bar"]
        salida["n_edades_obs"] = r["tabla"]["n_edades_obs"]
    out = pd.DataFrame(salida)
    os.makedirs("salidas_tbp", exist_ok=True)
    out.to_csv("salidas_tbp/abar_eph_por_cohorte.csv")
    print(out.loc[[c for c in (1960, 1970, 1980, 1990, 2000) if c in out.index]].round(2))
    print("Referencias de contexto (NO objetivo): ANSES observado 14,2 (sesgado a la baja); requisito legal 30 (techo).")

## 9. 🆕 Escenario propio de natalidad (RENAPER + DEIS + INDEC)

**Problema.** Para las cohortes 2025 en adelante, la serie principal usa los nacimientos proyectados por la ONU (WPP julio 2024): 508 mil en 2025, un **rebote artificial**. La ONU proyectó 2023 y 2024 *antes* de tener el dato: estimó 504 mil y 507 mil, y los nacimientos reales (DEIS) fueron **460.902** y **413.135**. Los datos recientes del RENAPER (2025 provisorio) y de la TGF indican que la caída continúa.

**Qué hace esta sección** (fragmento autónomo; depende solo de `d`, `s`, `TAU`, `RHO`, `DATA`, `OUT` en memoria):

1. arma un dato base 2025 **estimado** (RENAPER ajustado por la relación RENAPER/DEIS);
2. proyecta nacimientos 2025–2035 con tres escenarios de TGF **propuestos para que el grupo los valide**;
3. exporta `natalidad_escenarios_2025_2035.csv`;
4. recalcula $TBP_C$ y $TBP_A$ para las cohortes 2025–2035 y los compara con la serie actual (N de la ONU).

> **Aviso 1.** Los valores del RENAPER son los de la **descarga CSV del tablero** (nacimientos identificados y TGF, total país). El de **2025 es provisorio** y no equivale a los nacimientos *ocurridos* del DEIS.
> **Aviso 2.** $J$ (población 65+) **sigue siendo el de la ONU y NO está re-ponderado todavía**: eso se hace en la sección 10. Mientras tanto, el nivel de $TBP$ de las cohortes ≥2025 es parcial.

### 9.1 Carga de datos

Se cargan dos archivos de `datos/procesados/`: los datos del RENAPER y la población por edad simple de la ONU (mujeres, variante media). Rutas candidatas: `../datos/procesados`, `datos/procesados`, `DATA` y `datos_tbp`; si no están, se descargan del repositorio (URL *raw*) a la carpeta de salidas.

In [ ]:
import os, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt
try: display
except NameError: display = print

RAW_BASE_9 = "https://raw.githubusercontent.com/carolinamsfelipe/Tp-Indicadores/main/datos/procesados"
CANDS_9 = ["../datos/procesados", "datos/procesados", DATA, "datos_tbp", "../datos_tbp"]

def buscar_archivo_9(nombre):
    for c in CANDS_9:
        p = os.path.join(c, nombre)
        if os.path.exists(p):
            return p
    os.makedirs(OUT, exist_ok=True)          # fallback: descarga desde GitHub (raw)
    p = os.path.join(OUT, nombre)
    urllib.request.urlretrieve(f"{RAW_BASE_9}/{nombre}", p)
    return p

f_ren = buscar_archivo_9("renaper_natalidad_2012_2025.csv")
f_pob = buscar_archivo_9("un_wpp2024_argentina_poblacion_edad_simple.csv")
print("RENAPER :", os.path.abspath(f_ren)); print("Poblacion ONU:", os.path.abspath(f_pob))

ren = pd.read_csv(f_ren).set_index("anio")
ren["nac"] = ren["nacimientos_identificados_miles"] * 1000      # personas
pob = pd.read_csv(f_pob, low_memory=False)
if "Variant" in pob.columns:
    pob = pob[pob["Variant"] == "Medium"]
pob["edad"] = pd.to_numeric(pob["AgeGrpStart"], errors="coerce")
# M_t: mujeres de 15 a 49 anios en t (ONU medio), en miles
M = pob[(pob.edad >= 15) & (pob.edad <= 49)].groupby("Time")["PopFemale"].sum()

deis = s["deis_nacimientos"]; onu = s["un_births_miles"] * 1000
display(pd.DataFrame({"DEIS": deis.loc[2019:2024], "RENAPER": ren.nac.loc[2019:2024],
                      "ONU": onu.loc[2019:2024]}).round(0))
print("TGF RENAPER:", ren.tgf_hijos_por_mujer.loc[2022:2025].to_dict())
print("Mujeres 15-49 (miles, ONU medio):", M.loc[[2025, 2030, 2035]].round(0).to_dict())

### 9.2 Dato base 2025: ESTIMACIÓN a partir de un dato provisorio

Los nacimientos del RENAPER son *personas identificadas* y no coinciden exactamente con los *nacimientos ocurridos* (DEIS). Para llevar el dato RENAPER a la escala DEIS (la que usa la serie hasta 2024) se lo divide por la **relación RENAPER/DEIS promedio 2022–2024**:

$$N_{2025}^{est} = \frac{N_{2025}^{RENAPER}}{\overline{\left(N^{RENAPER}/N^{DEIS}\right)}_{2022\text{–}2024}}$$

**Por qué esa ventana:** la relación es estable (0,98–1,05 en 2012–2024) y los años recientes son los más comparables con 2025 (misma cobertura de DNI digital). Es una **ESTIMACIÓN**, **no un dato oficial**: combina un valor provisorio con una lectura a ojo de un gráfico.

In [ ]:
# ---- Parametros del dato base (modificables) ----
VENTANA_RATIO = (2022, 2024)        # anios usados para la relacion RENAPER/DEIS
ANIO_BASE = 2025

ratio = (ren.nac / deis).loc[2012:2024]
r_med = ratio.loc[VENTANA_RATIO[0]:VENTANA_RATIO[1]].mean()
N_REN_2025 = ren.nac[ANIO_BASE]
N_2025_EST = N_REN_2025 / r_med
TGF_2025_OBS = ren.tgf_hijos_por_mujer[ANIO_BASE]

print("Relacion RENAPER/DEIS 2012-2024:"); print(ratio.round(3).to_string())
print(f"\nMedia {VENTANA_RATIO[0]}-{VENTANA_RATIO[1]}: {r_med:.3f}")
print(f"RENAPER 2025 (provisorio): {N_REN_2025:,.0f}")
print(f"N_2025 ESTIMADO = {N_2025_EST:,.0f}  <- ESTIMACION a partir de un dato provisorio, NO dato oficial")
print(f"(ONU proyecta {onu[2025]:,.0f} para 2025: {onu[2025]/N_2025_EST - 1:+.0%} respecto de la estimacion)")

### 9.3 Escenarios 2025–2035 (propuesta a validar por el grupo)

**Método.** Los nacimientos de un año son (TGF × mujeres en edad fértil) por una constante que convierte "TGF" a "nacimientos":

$$N_t = TGF_t \times K \times \frac{M_t}{M_{2025}},\qquad K = \frac{N_{2025}^{est}}{TGF_{2025}}$$

* $M_t$ = mujeres de 15–49 años en $t$ (ONU medio, `PopFemale`). Capta el *efecto de composición* (cuántas mujeres hay en edad de tener hijos): crece ~3% entre 2025 y 2035.
* $K$ calibra el nivel en 2025, de modo que los tres escenarios **coinciden en 2025** y difieren solo por la trayectoria de la TGF. Es una aproximación: supone que la estructura de edades de la fecundidad no cambia (la TGF real usa tasas por edad).

**Los tres escenarios** (parámetros editables en la celda siguiente):

* **(a) Persistencia:** la TGF se queda en 1,04 (nivel observado 2025). Referencia "nada cambia".
* **(b) Recuperación lenta:** sube linealmente de 1,04 (2025) a 1,32 (2035), que es el valor de la **variante media de INDEC para 2035** (INDEC, proyecciones 2022–2040, pp. 26–30). Es el escenario "optimista".
* **(c) Caída adicional:** baja a 0,95 en 2028 y se estabiliza. Es una cota inferior ilustrativa, no una predicción.

**Contexto INDEC.** Las proyecciones de INDEC usaron nacimientos DEIS solo hasta 2023 y suponen una TGF media de 1,27 (2025), 1,24 (2030) y 1,32 (2035). La TGF observada 2025 del RENAPER (~1,04) queda **por debajo incluso del mínimo de la variante baja de INDEC (1,17)**. Por eso INDEC aparece solo como TGF de referencia (no se calculan nacimientos con la TGF INDEC: su denominador de mujeres y su calibración son otros).

In [ ]:
# ============ PARAMETROS DE LOS ESCENARIOS (modificables) ============
TGF_2025 = TGF_2025_OBS         # 1,04: TGF RENAPER 2025 (provisorio)
TGF_REC_2035 = 1.32             # (b) valor INDEC variante media 2035
TGF_CAIDA_MIN = 0.95            # (c) piso de la caida adicional
ANIO_CAIDA = 2028               # (c) anio en que se alcanza el piso
ANIOS = list(range(2025, 2036))
A_LEGAL_9, A_OBS_9 = 30.0, 14.2  # 30 = requisito legal de referencia (NO es un maximo); 14,2 = observado ANSES
# Referencia INDEC (proyecciones 2022-2040, pp. 26-30, ya verificado)
INDEC_MEDIA = {2025: 1.27, 2030: 1.24, 2035: 1.32, 2040: 1.40}
INDEC_MIN_BAJA = {2025: 1.28, 2030: 1.22, 2035: 1.17}     # minimos citados de la variante baja
# =====================================================================

K = N_2025_EST / TGF_2025

def tgf_escenario(nombre, t):
    if nombre == "persistencia":
        return TGF_2025
    if nombre == "recuperacion_lenta":
        return TGF_2025 + (TGF_REC_2035 - TGF_2025) * (t - ANIOS[0]) / (ANIOS[-1] - ANIOS[0])
    if nombre == "caida_adicional":
        return TGF_2025 + (TGF_CAIDA_MIN - TGF_2025) * min(t - ANIOS[0], ANIO_CAIDA - ANIOS[0]) / (ANIO_CAIDA - ANIOS[0])
    raise ValueError(nombre)

ESCENARIOS = ["persistencia", "recuperacion_lenta", "caida_adicional"]
filas = []
for e in ESCENARIOS:
    for t in ANIOS:
        tg = tgf_escenario(e, t)
        filas.append(dict(anio=t, escenario=e, tgf=tg, nacimientos=tg * K * M[t] / M[ANIO_BASE]))
# referencias: ONU (nacimientos) e INDEC media (solo TGF, interpolada linealmente entre anios publicados)
for t in ANIOS:
    filas.append(dict(anio=t, escenario="onu_medio", tgf=np.nan, nacimientos=onu[t]))
ks = sorted(INDEC_MEDIA)
for t in ANIOS:
    filas.append(dict(anio=t, escenario="indec_media_tgf", tgf=np.interp(t, ks, [INDEC_MEDIA[k] for k in ks]), nacimientos=np.nan))
esc = pd.DataFrame(filas)

piv = esc.pivot(index="anio", columns="escenario", values="nacimientos")[ESCENARIOS + ["onu_medio"]]
display((piv / 1000).round(1).loc[[2025, 2026, 2028, 2030, 2035]].rename_axis("miles de nacimientos"))
print(f"TGF observada 2025 = {TGF_2025:.2f} < minimo variante baja INDEC 2035 = {INDEC_MIN_BAJA[2035]:.2f} y 2025 = {INDEC_MIN_BAJA[2025]:.2f}")

### 9.4 Salida: `natalidad_escenarios_2025_2035.csv`

Formato largo: `anio, escenario, tgf, nacimientos`. Incluye los tres escenarios y dos referencias: `onu_medio` (solo nacimientos; la TGF de la ONU no se cargó acá) e `indec_media_tgf` (solo TGF, interpolada linealmente entre 2025, 2030 y 2035; **nacimientos vacíos a propósito**: no se tiene la serie de nacimientos de INDEC y calcularla con otra metodología mezclaría supuestos).

In [ ]:
DEST = next((p for p in ["../datos/procesados", "datos/procesados"] if os.path.isdir(p)), OUT)
os.makedirs(DEST, exist_ok=True)
f_out = os.path.join(DEST, "natalidad_escenarios_2025_2035.csv")
esc.round({"tgf": 4, "nacimientos": 0}).to_csv(f_out, index=False)
print("Escrito:", os.path.abspath(f_out), esc.shape)

### 9.5 Recalcular el TBP con cada escenario

Se reemplaza $N_t$ por el del escenario **solo para las cohortes ≥2025** (cohortes ≤2024 quedan idénticas: DEIS). El resto de los insumos es el de `d`: $S_{65}$, $E_{65}$ y $J$.

$$TBP_C=\frac{N(1-\mu)\bar A\tau}{J\rho},\qquad TBP_A=\frac{TBP_C}{E_r}$$

> **J NO está re-ponderado todavía** (sigue siendo la población 65+ de la ONU, que descansa en los nacimientos ONU anteriores). Con menos nacimientos, la población 65+ de 2090–2100 que proyecta la ONU también sería menor, así que el denominador está sobreestimado y el efecto aquí es una **cota parcial**. La corrección va en la sección 10.

In [ ]:
base9 = d.set_index("cohorte")[["N_t", "S65", "J", "E65", "TBP_C_legal", "TBP_A_legal", "TBP_C_obs", "TBP_A_obs"]].copy()
N_snapshot = base9["N_t"].copy()                      # para el control "no cambia <=2024"

def serie_N(nombre):
    # N por cohorte: DEIS hasta 2024 (igual que d); desde 2025 el escenario (o la ONU, serie actual)
    N = base9["N_t"].copy()
    if nombre != "actual_onu":
        N.loc[ANIOS] = esc[esc.escenario == nombre].set_index("anio")["nacimientos"].loc[ANIOS].values
    return N

def tbp_9(N, A, tipo):
    c = N * base9.S65 * A * TAU / (base9.J * RHO)
    return c if tipo == "C" else c / base9.E65

NOMBRES = ["actual_onu"] + ESCENARIOS
N9 = {n: serie_N(n) for n in NOMBRES}
T9 = {(n, A, tipo): tbp_9(N9[n], A, tipo) for n in NOMBRES for A in (A_LEGAL_9, A_OBS_9) for tipo in "CA"}

# la serie "actual" reproduce la de d (control de consistencia)
assert np.allclose(T9[("actual_onu", A_LEGAL_9, "C")], base9.TBP_C_legal)
assert np.allclose(T9[("actual_onu", A_OBS_9, "A")], base9.TBP_A_obs)
print("OK: la serie 'actual_onu' recalculada reproduce TBP_C/TBP_A de d")

### 9.6 Gráficos

(i) Nacimientos 2012–2035: DEIS, RENAPER (sin ajustar), ONU y los tres escenarios (el punto 2025 común es la **estimación**). (ii) $TBP_C$ por cohorte 2000–2035 para $\bar A=30$ y $\bar A=14{,}2$, con los tres escenarios y la serie actual; la línea vertical marca el corte 2024/2025. **$\bar A=30$ es el requisito legal de referencia, no un máximo.** En ambos paneles de $TBP_C$, $J$ es el de la ONU (no re-ponderado).

In [ ]:
COL9 = {"persistencia": "#d55e00", "recuperacion_lenta": "#009e73", "caida_adicional": "#cc79a7", "actual_onu": "#444444"}
ETQ9 = {"persistencia": "(a) Persistencia (TGF 1,04)", "recuperacion_lenta": "(b) Recuperación lenta (a 1,32)",
        "caida_adicional": "(c) Caída adicional (a 0,95)", "actual_onu": "Serie actual (N de la ONU)"}

fig, ax = plt.subplots(1, 3, figsize=(20, 5.4))
a = ax[0]
a.plot(deis.loc[2012:2024].index, deis.loc[2012:2024] / 1e3, color="#0072b2", lw=2, label="DEIS (observado)")
a.plot(ren.index, ren.nac / 1e3, "s--", color="#e69f00", ms=4, lw=1.2, label="RENAPER (2025 prov.)")
a.plot(onu.loc[2012:2035].index, onu.loc[2012:2035] / 1e3, color="#444444", ls=":", lw=2, label="ONU WPP 2024 (medio)")
for e in ESCENARIOS:
    p = piv[e]; a.plot(p.index, p / 1e3, color=COL9[e], lw=2, label=ETQ9[e])
a.plot([2025], [N_2025_EST / 1e3], "o", color="k", ms=6, label=f"N 2025 estimado ({N_2025_EST/1e3:.0f} mil)")
a.axvline(2024.5, color="gray", ls="--", lw=1); a.set_title("(i) Nacimientos 2012–2035 (miles)")
a.set_xlabel("año"); a.grid(alpha=.3); a.legend(fontsize=7.5)

for a, AA, tit in [(ax[1], A_LEGAL_9, "Ā = 30 (requisito legal de referencia, no un máximo)"),
                   (ax[2], A_OBS_9, "Ā = 14,2 (observado ANSES, sesgado a la baja)")]:
    for n in NOMBRES:
        x = T9[(n, AA, "C")].loc[2000:2035]
        a.plot(x.index, x, color=COL9[n], lw=2.4 if n == "actual_onu" else 1.8, ls=":" if n == "actual_onu" else "-", label=ETQ9[n])
    a.axvline(2024.5, color="gray", ls="--", lw=1)
    a.text(2024.8, a.get_ylim()[1] * 0.98, "2024/2025", fontsize=8, color="gray", va="top")
    a.set_title(f"(ii) TBP_C por cohorte, {tit}", fontsize=9.5); a.set_ylabel("TBP_C (años)")
    a.set_xlabel("año de nacimiento de la cohorte"); a.grid(alpha=.3); a.legend(fontsize=7.5, loc="upper right")
fig.suptitle("Escenarios propios de natalidad 2025–2035 — J todavía NO re-ponderado (sección 10)", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.95])
os.makedirs(OUT, exist_ok=True)
plt.savefig(f"{OUT}/seccion_09_natalidad.png", dpi=130); plt.show()

### 9.7 Tabla resumen: cohortes 2025, 2030 y 2035

Nacimientos y $TBP$ por escenario frente a la serie actual (N de la ONU). Columna `Δ vs actual` = variación porcentual del $TBP_C$ respecto de la serie actual. **$J$ de la ONU, no re-ponderado.**

In [ ]:
COHS = [2025, 2030, 2035]
res = []
for n in NOMBRES:
    for c in COHS:
        r = dict(cohorte=c, escenario=n, N=N9[n][c])
        for A, lab in [(A_LEGAL_9, "A30"), (A_OBS_9, "A14,2")]:
            r[f"TBP_C ({lab})"] = T9[(n, A, "C")][c]
            r[f"TBP_A ({lab})"] = T9[(n, A, "A")][c]
        r["Δ vs actual (%)"] = 100 * (T9[(n, A_LEGAL_9, "C")][c] / T9[("actual_onu", A_LEGAL_9, "C")][c] - 1)
        res.append(r)
resumen = pd.DataFrame(res).sort_values(["cohorte", "escenario"]).set_index(["cohorte", "escenario"])
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
display(resumen.round({"N": 0, "Δ vs actual (%)": 1}))
print("Nota: J = poblacion 65+ de la ONU (NO re-ponderado; se hace en la seccion 10). Resultados parciales.")

### 9.8 Controles (asserts)

1. Relación RENAPER/DEIS en [0,95; 1,06] para 2012–2024 (se excluye 2020, año de pandemia).
2. Nacimientos de todos los escenarios positivos.
3. Los tres escenarios coinciden en 2025 (y valen $N_{2025}^{est}$).
4. El ajuste no cambia las cohortes ≤2024 (ni `N_t` ni el TBP).

In [ ]:
r_chk = ratio.drop(2020)
assert r_chk.between(0.95, 1.06).all(), r_chk[~r_chk.between(0.95, 1.06)]
print(f"OK 1) RENAPER/DEIS en [0,95; 1,06]: min {r_chk.min():.3f} | max {r_chk.max():.3f} (2012-2024 sin 2020; 2020 = {ratio[2020]:.3f})")

for e in ESCENARIOS:
    assert (esc[esc.escenario == e].nacimientos > 0).all()
print("OK 2) nacimientos > 0 en los 3 escenarios, 2025-2035")

n25 = esc[(esc.anio == 2025) & esc.escenario.isin(ESCENARIOS)].nacimientos
assert np.allclose(n25, N_2025_EST) and np.ptp(n25) < 1e-6
print(f"OK 3) los 3 escenarios coinciden en 2025: {n25.iloc[0]:,.0f}")

ant = base9.index <= 2024
for n in ESCENARIOS:
    assert np.array_equal(N9[n][ant].values, N_snapshot[ant].values)
    assert np.allclose(T9[(n, A_LEGAL_9, "C")][ant], base9.TBP_C_legal[ant])
    assert np.allclose(T9[(n, A_OBS_9, "A")][ant], base9.TBP_A_obs[ant])
assert np.array_equal(d.set_index("cohorte")["N_t"].values, N_snapshot.values)   # d no fue modificado
assert esc.anio.min() == 2025
print("OK 4) cohortes <=2024 sin cambios (N_t y TBP) y `d` intacto")

### 9.9 Lectura y límites

* **Qué se puede decir:** la serie actual (N de la ONU) **sobrestima** los nacimientos de las cohortes 2025+ respecto de lo que muestran RENAPER y la TGF observada; incluso con una recuperación lenta hasta el valor INDEC 2035, las cohortes 2025–2030 quedan por debajo de lo que supone la ONU. Con persistencia o caída, $TBP_C$ de las cohortes recientes cae en forma sostenida frente a la serie actual.
* **Qué NO se puede decir:** que estos escenarios sean pronósticos; que 390 mil sea el dato oficial 2025 (es una estimación sobre un dato provisorio); que el nivel absoluto del $TBP$ de las cohortes ≥2025 sea definitivo (**$J$ no está re-ponderado**: sección 10).
* **Supuestos a validar por el grupo:** (i) estructura de edades de la fecundidad constante (la proporcionalidad TGF × mujeres 15–49); (ii) ventana 2022–2024 para la relación RENAPER/DEIS; (iii) los tres valores de TGF (1,04 / 1,32 / 0,95 en 2028).
* **Pendiente:** reemplazar el 2025 provisorio por el definitivo cuando el RENAPER lo publique.

## 10. 🆕 Consistencia de J: reponderación con nacimientos propios

**El problema (circularidad).** $J_{t+65}$ es la población de 65+ años que la ONU proyecta para el año $t+65$. Esa población está construida con los **nacimientos de la ONU** de las cohortes que la componen (nacidas entre $t-35$ y $t$). Pero en $N_t$ ya no usamos los nacimientos de la ONU: usamos DEIS (observado, hasta 2024) y, desde 2025, los escenarios propios de la sección 9. Es decir, el numerador y el denominador del $TBP$ cuentan **historias de natalidad distintas**: $N$ dice "nacieron pocos", $J$ supone "nacieron los que dijo la ONU". Si la natalidad cae, la población 65+ del futuro también tiene que ser menor, y el denominador está sobreestimado.

**Qué hace esta sección** (fragmento autónomo; depende solo de `d`, `s`, `TAU`, `RHO`, `A_OBS`, `A_LEGAL`, `DATA`, `OUT` en memoria y de los archivos que carga él mismo):

1. reconstruye $J$ como suma por edad de la población ONU y la **repondera por cohorte** con $w(c)=N_{propio}(c)/N_{ONU}(c)$;
2. compara $J$ original vs reponderado, para cada escenario;
3. recalcula $TBP_C$ y $TBP_A$ en tres versiones: (i) serie actual, (ii) $N$ propio con $J$ ONU (inconsistente), (iii) $N$ propio con $J$ reponderado (consistente);
4. exporta `tbp_cohortes_J_reponderado.csv` y verifica con asserts.

### 10.1 Carga de datos

Se cargan dos archivos de `datos/procesados/`: la población ONU por edad simple (`PopTotal`, variante media, miles) y los escenarios de natalidad de la sección 9. Rutas candidatas: `../datos/procesados`, `datos/procesados` y `DATA`; si no están, se descargan del repositorio (URL *raw*) a la carpeta de salidas. Los nacimientos ONU y DEIS salen de `s`.

In [ ]:
import os, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt
try: display
except NameError: display = print

RAW_BASE_10 = "https://raw.githubusercontent.com/carolinamsfelipe/Tp-Indicadores/main/datos/procesados"
CANDS_10 = ["../datos/procesados", "datos/procesados", DATA]

def buscar_archivo_10(nombre):
    for c in CANDS_10:
        p = os.path.join(c, nombre)
        if os.path.exists(p):
            return p
    os.makedirs(OUT, exist_ok=True)          # fallback: descarga desde GitHub (raw) a la carpeta de salidas
    p = os.path.join(OUT, nombre)
    urllib.request.urlretrieve(f"{RAW_BASE_10}/{nombre}", p)
    return p

f_pob10 = buscar_archivo_10("un_wpp2024_argentina_poblacion_edad_simple.csv")
f_esc10 = buscar_archivo_10("natalidad_escenarios_2025_2035.csv")
print("Poblacion ONU :", os.path.abspath(f_pob10)); print("Escenarios    :", os.path.abspath(f_esc10))

pop10 = pd.read_csv(f_pob10, low_memory=False)
if "Variant" in pop10.columns:
    pop10 = pop10[pop10["Variant"] == "Medium"]
# PopTotal por edad simple (columnas 0..100; 100 = grupo abierto 100+) y anio, en PERSONAS
P = pop10.pivot_table(index="Time", columns="AgeGrpStart", values="PopTotal") * 1000
P.columns = P.columns.astype(int)

esc10 = pd.read_csv(f_esc10)
ESC10 = ["persistencia", "recuperacion_lenta", "caida_adicional"]      # escenarios propios (seccion 9)
assert set(ESC10 + ["onu_medio"]) <= set(esc10.escenario.unique()), esc10.escenario.unique()
NAC10 = esc10.pivot(index="anio", columns="escenario", values="nacimientos")[ESC10 + ["onu_medio"]]

N_onu10 = s["un_births_miles"] * 1000          # nacimientos ONU (personas), 1950-2100
N_deis10 = s["deis_nacimientos"]               # nacimientos DEIS observados, 1914-2024
print("Edades en la tabla ONU:", P.columns.min(), "a", P.columns.max(), "| anios:", P.index.min(), "a", P.index.max())
print("Escenarios cargados:", ESC10, "+ onu_medio | anios", NAC10.index.min(), "a", NAC10.index.max())
display((NAC10 / 1000).round(1).loc[[2025, 2030, 2035]].rename_axis("miles de nacimientos"))

### 10.2 Método: reponderación por cohorte

La población de $a$ años en el año calendario $y$ pertenece a la cohorte $c=y-a$. Si esa cohorte tuvo, según nuestros datos, una fracción $w(c)$ de los nacimientos que supuso la ONU, asumimos que **le quedan $w(c)$ veces los sobrevivientes** que proyecta la ONU. Entonces:

$$J^{rw}(y)=\sum_{a=65}^{100+} \text{PopTotal}_{ONU}(a,y)\cdot w(y-a),\qquad w(c)=\frac{N_{propio}(c)}{N_{ONU}(c)}$$

con $N_{propio}$ = DEIS para $c\le 2024$ y el escenario para $c\ge 2025$. Se aplica a $y=t+65$ para $t=1950,\dots,2035$.

**Por qué esta forma (didáctica).** Es una regla de tres por cohorte: si una generación fue 20% más chica de lo que suponía la ONU, la población de esa generación a los 70 años también lo es. Se corrige solo lo que la circularidad contamina (los nacimientos) y se deja intacto el resto de la proyección ONU (mortalidad, migración).

**Convenciones (documentadas):**

* $w=1$ para cohortes $c<1950$: la serie `un_births_miles` empieza en 1950, no hay nacimientos ONU contra los cuales comparar.
* $w=1$ para 1971–1974: la serie DEIS tiene un hueco y el cuaderno principal completa $N_t$ con la ONU (`fuente_N`); se usa la misma regla.
* El grupo abierto **100+** es una sola fila de la tabla ONU; se le asigna $w(y-100)$ (cohorte más joven del grupo, la más numerosa). Pesa muy poco en $J$.
* La población ONU es de **mitad de año**, así que "edad $a$ en $y$" mezcla las cohortes $y-a$ e $y-a-1$. Se usa $c=y-a$ (la misma convención del cuaderno principal, donde la cohorte $t$ cumple 65 en $t+65$) y se chequea la robustez promediando ambas.

**Supuestos y límites.** (a) Mortalidad y migración de cada cohorte proporcionales a los de la proyección ONU (se reescala el tamaño, no el comportamiento). (b) Para $c\le 2024$, $w=$ DEIS/ONU mezcla la corrección por la circularidad con la **diferencia de nivel entre registro (DEIS) y estimación ONU**, que existe también en cohortes viejas (media DEIS/ONU 1950–2024: 0,97). Por eso se muestra también una variante que ajusta solo 2023–2024.

In [ ]:
AGES10 = np.arange(65, 101)            # 65..99 edades simples + 100 (= grupo abierto 100+)

def pesos_cohorte(escenario=None, solo_2023_24=False):
    """w(c) = N_propio(c) / N_ONU(c).  w=1 para c<1950 (no hay nacimientos ONU en la serie).
    N_propio = DEIS para 1950<=c<=2024 y el escenario para 2025<=c<=2035.
    Cohortes 1971-1974: la serie DEIS tiene un hueco; el cuaderno principal las completa con la ONU (fuente_N),
    asi que alli w = 1 (misma regla que N_t de d).
    escenario=None -> w = 1 en TODAS las cohortes (control: debe reproducir J de la ONU).
    solo_2023_24=True -> variante de referencia: w=1 salvo 2023 y 2024 (DEIS) y los escenarios (>=2025)."""
    w = pd.Series(1.0, index=np.arange(1900, 2101))
    if escenario is None:
        return w
    for c in range(1950, 2025):
        if ((not solo_2023_24) or c >= 2023) and not np.isnan(N_deis10.get(c, np.nan)):
            w[c] = N_deis10[c] / N_onu10[c]
    for c in range(2025, 2036):
        w[c] = NAC10.loc[c, escenario] / N_onu10[c]
    assert w.notna().all()
    return w

def J_reponderado(w, anios, mezcla=0.0):
    """J_rw(y) = sum_{a=65}^{100} PopTotal_ONU(a, y) * w(y - a).
    Convencion para el grupo abierto 100+: se le asigna w(y - 100) (la cohorte mas joven del grupo; es
    la mas numerosa y, por tamanio del grupo, el error es de segundo orden).
    mezcla=0 -> cohorte = y - a (convencion del cuaderno principal: cohorte t llega a 65 en t+65).
    mezcla=0.5 -> promedia las cohortes y-a e y-a-1 (control de robustez: la poblacion ONU es de mitad de anio)."""
    out = {}
    for y in anios:
        c = y - AGES10
        wy = (1 - mezcla) * w.reindex(c).values + mezcla * w.reindex(c - 1).values
        out[y] = float((P.loc[y, AGES10].values * wy).sum())
    return pd.Series(out)

dd = d.set_index("cohorte")
ANIOS65 = [int(c) + 65 for c in dd.index]                  # 2015..2100
J_onu10 = pd.Series(dd.J.values, index=dd.index)         # J de la ONU, indexado por cohorte t (anio t+65)

def J_rw_coh(escenario, **kw):
    r = J_reponderado(pesos_cohorte(escenario, kw.pop("solo_2023_24", False)), ANIOS65, **kw)
    r.index = dd.index
    return r

JRW = {e: J_rw_coh(e) for e in ESC10 + ["onu_medio"]}      # onu_medio = DEIS<=2024 y ONU despues
print("Pesos w(c) = N_propio/N_ONU en cohortes recientes:")
display(pd.DataFrame({"w DEIS/ONU": pesos_cohorte("persistencia").loc[[2000, 2010, 2020, 2022, 2023, 2024]],
                      }).T.round(3))
display(pd.DataFrame({e: pesos_cohorte(e).loc[2025:2035:5] for e in ESC10}).T.round(3).rename_axis("w(c), c = 2025/2030/2035"))

### 10.3 J reponderado vs J ONU

Diferencia porcentual $J^{rw}/J^{ONU}-1$. Para $t\le 2024$ todos los escenarios dan lo mismo (el stock 65+ de $t+65$ solo incluye cohortes $\le t$). Los escenarios solo afectan a las cohortes $t\ge 2025$.

In [ ]:
# Diferencia porcentual J_rw vs J_ONU, cohortes t <= 2024 (stock 2015-2089)
def difpct(Jrw, rango):
    x = 100 * (Jrw / J_onu10 - 1).loc[rango[0]:rango[1]]
    return x

rows = []
for nombre, kw in [("Método principal (DEIS 1950-2024)", {}),
                   ("Variante: solo 2023-2024 con DEIS", {"solo_2023_24": True}),
                   ("Robustez: cohorte = promedio de y-a e y-a-1", {"mezcla": 0.5})]:
    x = difpct(J_rw_coh("persistencia", **kw), (1950, 2024))
    rows.append({"variante": nombre, "media (%)": x.mean(), "media |Δ| (%)": x.abs().mean(),
                 "máx |Δ| (%)": x.abs().max(), "cohorte del máx": int(x.abs().idxmax())})
tab_dif = pd.DataFrame(rows).set_index("variante")
print(tab_dif.round(2).to_string())
x_main = difpct(JRW["persistencia"], (1950, 2024))
print(f"t <= 2024, método principal: ΔJ media = {x_main.mean():+.2f}% | media del valor absoluto = {x_main.abs().mean():.2f}% | "
      f"máx |Δ| = {x_main.abs().max():.2f}% (cohorte {int(x_main.abs().idxmax())}, J ONU {J_onu10[int(x_main.abs().idxmax())]/1e6:.2f} M -> "
      f"{JRW['persistencia'][int(x_main.abs().idxmax())]/1e6:.2f} M)")
print("Cohortes 2020-2024 (ΔJ %):", x_main.loc[2020:2024].round(2).to_dict())
post = pd.DataFrame({e: 100 * (JRW[e] / J_onu10 - 1) for e in ESC10 + ["onu_medio"]}).loc[[2025, 2030, 2035]]
print("\nΔJ % para cohortes >= 2025 (J de la cohorte t = poblacion 65+ en t+65):")
display(post.round(2))

### 10.4 Recalcular el TBP en tres versiones

$$TBP_C=\frac{N(1-\mu)\bar A\tau}{J\rho},\qquad TBP_A=\frac{TBP_C}{E_r}$$

* **(i) serie actual:** $N$ = DEIS hasta 2024 y ONU después; $J$ ONU.
* **(ii) inconsistente:** $N$ de la sección 9; $J$ ONU sin reponderar.
* **(iii) consistente:** $N$ de la sección 9; $J$ reponderado con el mismo escenario.

Se usan $\bar A=30$ (requisito legal de referencia, **no un máximo**) y $\bar A=14{,}2$ (observado ANSES). $S_{65}$, $E_{65}$, $\tau$ y $\rho$ son los de `d`.

In [ ]:
A_LIST = [("A30", A_LEGAL), ("A14_2", A_OBS)]

def N_escenario(e):
    N = dd.N_t.copy()                                     # DEIS <= 2024 ; ONU >= 2025 (serie actual)
    N.loc[2025:2035] = NAC10.loc[2025:2035, e].values
    return N

def tbp(N, J, A, tipo):
    c = N * dd.S65 * A * TAU / (J * RHO)                 # TBP_C (años)
    return c if tipo == "C" else c / dd.E65             # TBP_A = TBP_C / E_r

# (i) serie actual: N ONU post-2024, J ONU.  (ii) N seccion 9, J ONU sin reponderar (inconsistente).  (iii) N seccion 9, J reponderado.
V = {}
for A_lab, A in A_LIST:
    for tp in "CA":
        V[("i", None, A_lab, tp)] = tbp(dd.N_t, dd.J, A, tp)
        for e in ESC10:
            V[("ii", e, A_lab, tp)]  = tbp(N_escenario(e), dd.J, A, tp)
            V[("iii", e, A_lab, tp)] = tbp(N_escenario(e), JRW[e], A, tp)

# la serie (i) reproduce d
assert np.allclose(V[("i", None, "A30", "C")], dd.TBP_C_legal) and np.allclose(V[("i", None, "A14_2", "A")], dd.TBP_A_obs)
print("OK: (i) reproduce TBP_C / TBP_A de d (Ā = 30 y 14,2)")

### 10.5 Gráficos

(a) $J$ original vs reponderado en año calendario ($t+65$, 2015–2100) y su diferencia porcentual. (b) $TBP_C$ por cohorte 2000–2035 con (i), (ii) y (iii), para cada escenario y para $\bar A=30$ y $14{,}2$. La línea vertical marca el corte 2024/2025.

In [ ]:
COL10 = {"i": "#444444", "ii": "#d55e00", "iii": "#0072b2"}
ETQ_E = {"persistencia": "Persistencia (TGF 1,04)", "recuperacion_lenta": "Recuperación lenta (a 1,32)",
         "caida_adicional": "Caída adicional (a 0,95)"}
COLE = {"persistencia": "#d55e00", "recuperacion_lenta": "#009e73", "caida_adicional": "#cc79a7"}

# ---- Figura 1: J original vs reponderado (anio calendario 2015-2100) ----
anio = np.array(ANIOS65)
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
ax[0].plot(anio, J_onu10.values / 1e6, color="#444444", lw=2.4, ls=":", label="J ONU (original)")
for e in ESC10:
    ax[0].plot(anio, JRW[e].values / 1e6, color=COLE[e], lw=1.8, label=f"J reponderado: {ETQ_E[e]}")
ax[0].set_title("Población 65+ (millones), año calendario t+65"); ax[0].set_xlabel("año calendario (t+65)")
ax[0].set_ylabel("millones"); ax[0].grid(alpha=.3); ax[0].legend(fontsize=8)
ax[0].axvline(2024 + 65 + .5, color="gray", ls="--", lw=1)
ax[0].text(2090.1, ax[0].get_ylim()[0] + 0.5, "cohortes\n≥2025", fontsize=7.5, color="gray")
for e in ESC10 + ["onu_medio"]:
    ax[1].plot(anio, 100 * (JRW[e] / J_onu10 - 1).values, color=COLE.get(e, "#444444"), lw=1.8,
               ls=":" if e == "onu_medio" else "-", label="Solo DEIS ≤2024 (ONU después)" if e == "onu_medio" else ETQ_E[e])
ax[1].axhline(0, color="gray", lw=.8); ax[1].axvline(2024 + 65 + .5, color="gray", ls="--", lw=1)
ax[1].set_title("J reponderado vs J ONU (diferencia %)"); ax[1].set_xlabel("año calendario (t+65)")
ax[1].set_ylabel("%"); ax[1].grid(alpha=.3); ax[1].legend(fontsize=8)
fig.suptitle("Sección 10: población 65+ original (ONU) vs reponderada con nacimientos propios", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.savefig(f"{OUT}/seccion_10_J_reponderado.png", dpi=130); plt.show()

# ---- Figura 2: TBP_C por cohorte 2000-2035, (i), (ii), (iii) ----
fig, ax = plt.subplots(2, 3, figsize=(19, 9), sharex=True)
for i, (A_lab, A, tit) in enumerate([("A30", A_LEGAL, "Ā = 30 (requisito legal de referencia, no un máximo)"),
                                     ("A14_2", A_OBS, "Ā = 14,2 (observado ANSES)")]):
    for j, e in enumerate(ESC10):
        a = ax[i, j]
        a.plot(V[("i", None, A_lab, "C")].loc[2000:2035], color=COL10["i"], lw=2.2, ls=":", label="(i) serie actual (N ONU, J ONU)")
        a.plot(V[("ii", e, A_lab, "C")].loc[2000:2035], color=COL10["ii"], lw=1.8, label="(ii) N propio, J ONU (inconsistente)")
        a.plot(V[("iii", e, A_lab, "C")].loc[2000:2035], color=COL10["iii"], lw=2.2, label="(iii) N propio, J reponderado (consistente)")
        a.axvline(2024.5, color="gray", ls="--", lw=1)
        a.set_title(f"{ETQ_E[e]}\nTBP_C, {tit}", fontsize=9); a.grid(alpha=.3)
        if j == 0: a.set_ylabel("TBP_C (años)")
        if i == 1: a.set_xlabel("año de nacimiento de la cohorte")
        a.legend(fontsize=7.5, loc="lower left")
fig.suptitle("TBP_C por cohorte 2000–2035: serie actual vs N propio con J sin reponderar vs J reponderado", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.savefig(f"{OUT}/seccion_10_TBP_C_reponderado.png", dpi=130); plt.show()

### 10.6 Tabla: cohortes 2024, 2030 y 2035

La columna `TBP_C(A30) vs (i) %` compara con la serie actual. Debajo, el efecto aislado de reponderar $J$ ((iii) vs (ii), mismo $N$) y la sensibilidad a la convención de pesos.

In [ ]:
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
filas = []
for c in [2024, 2030, 2035]:
    filas.append(dict(cohorte=c, escenario="(i) serie actual", N=dd.N_t[c], J_M=dd.J[c] / 1e6,
                      **{f"TBP_C ({l})": V[("i", None, l, "C")][c] for l, _ in A_LIST},
                      **{f"TBP_A ({l})": V[("i", None, l, "A")][c] for l, _ in A_LIST}))
    for e in ESC10:
        for ver in ["ii", "iii"]:
            Jv = dd.J[c] if ver == "ii" else JRW[e][c]
            filas.append(dict(cohorte=c, escenario=f"({ver}) {e}", N=N_escenario(e)[c], J_M=Jv / 1e6,
                              **{f"TBP_C ({l})": V[(ver, e, l, "C")][c] for l, _ in A_LIST},
                              **{f"TBP_A ({l})": V[(ver, e, l, "A")][c] for l, _ in A_LIST}))
tabla10 = pd.DataFrame(filas)
ref = {c: V[("i", None, "A30", "C")][c] for c in [2024, 2030, 2035]}
tabla10["TBP_C(A30) vs (i) %"] = [100 * (r["TBP_C (A30)"] / ref[r["cohorte"]] - 1) for _, r in tabla10.iterrows()]
display(tabla10.set_index(["cohorte", "escenario"]).round({"N": 0, "J_M": 3}).round(3))

# efecto aislado de reponderar J: (iii) vs (ii), mismo N
ef = pd.DataFrame({e: {c: 100 * (V[("iii", e, "A30", "C")][c] / V[("ii", e, "A30", "C")][c] - 1) for c in [2024, 2030, 2035]} for e in ESC10})
print("Efecto de reponderar J (iii vs ii, mismo N), TBP_C en %:"); display(ef.round(2).rename_axis("cohorte"))

# robustez del resultado (iii) a la convención de pesos, escenario persistencia, TBP_C (Ā = 30)
rob = {}
for nombre, kw in [("principal: DEIS 1950-2024", {}), ("solo 2023-2024 con DEIS", {"solo_2023_24": True}),
                   ("cohorte = prom. de y-a e y-a-1", {"mezcla": 0.5})]:
    Jv = J_rw_coh("persistencia", **kw)
    rob[nombre] = {c: tbp(N_escenario("persistencia"), Jv, A_LEGAL, "C")[c] for c in [2024, 2030, 2035]}
rob["(ii) J ONU"] = {c: V[("ii", "persistencia", "A30", "C")][c] for c in [2024, 2030, 2035]}
rob["(i) serie actual"] = {c: V[("i", None, "A30", "C")][c] for c in [2024, 2030, 2035]}
print("TBP_C (Ā=30), persistencia: sensibilidad de (iii) a la convención de pesos"); display(pd.DataFrame(rob).T.round(3).rename_axis("versión"))

### 10.7 Controles (asserts)

1. Con $w\equiv 1$ el método reproduce exactamente $J$ ONU de `d` en las 86 cohortes.
2. $J^{rw}>0$ y finito en todos los escenarios.
3. Para $t\le 2024$ todos los escenarios dan el mismo $J^{rw}$; para $t\ge 2025$ difieren.
4. Con nacimientos propios menores a los de la ONU, $J^{rw}\le J^{ONU\text{ después de }2024}$ para $t\ge 2025$.
5. (i)=(ii) hasta 2024; (iii)=(ii)·$J^{ONU}/J^{rw}$; linealidad en $\bar A$.
6. $w=1$ para $c<1950$ y 1971–74; $w\cdot N_{ONU}=N_t$ de `d` hasta 2024; un caso puntual coincide con la suma explícita.

In [ ]:
# 1) con w = 1 el método reproduce EXACTAMENTE J de la ONU en d, para todas las cohortes
J_w1 = J_reponderado(pesos_cohorte(None), ANIOS65); J_w1.index = dd.index
assert np.allclose(J_w1.values, dd.J.values, rtol=1e-12, atol=1e-6)
assert np.allclose(J_w1.values, (pop10.assign(e=pd.to_numeric(pop10.AgeGrpStart)).query("e >= 65")
                                 .groupby("Time").PopTotal.sum().loc[ANIOS65].values) * 1000)
print("OK 1) con w ≡ 1: J_rw = J de la ONU (d.J) en las 86 cohortes (y = suma PopTotal 65+ de la tabla de edad simple)")

# 2) J_rw > 0 en todos los escenarios y cohortes
for e, Jr in JRW.items():
    assert (Jr > 0).all() and np.isfinite(Jr.values).all(), e
print("OK 2) J_rw > 0 y finito en todos los escenarios y cohortes")

# 3) para t <= 2024 los escenarios no cambian J_rw (el stock 65+ de y=t+65 solo incluye cohortes c <= t <= 2024)
for e in ESC10[1:] + ["onu_medio"]:
    assert np.allclose(JRW[e].loc[:2024].values, JRW["persistencia"].loc[:2024].values, rtol=1e-12)
assert not np.allclose(JRW["recuperacion_lenta"].loc[2025:].values, JRW["persistencia"].loc[2025:].values)
print("OK 3) t <= 2024: J_rw idéntico en todos los escenarios; t >= 2025: los escenarios sí difieren")

# 4) 'onu_medio' (w = 1 desde 2025) y 'solo DEIS<=2024' coinciden con el método principal hasta t=2024 y, para t>=2025, J_rw(onu_medio) > J_rw(persistencia)
assert (JRW["onu_medio"].loc[2025:] >= JRW["persistencia"].loc[2025:] - 1e-6).all()
print("OK 4) con nacimientos propios menores a la ONU, J_rw(persistencia) <= J_rw(ONU después de 2024) para t >= 2025")

# 5) las versiones coinciden donde corresponde: (i)=(ii) para t<=2024 ; (ii)=(iii) NO coinciden (J cambia) ; (iii) usa el mismo N que (ii)
for e in ESC10:
    assert np.allclose(V[("i", None, "A30", "C")].loc[:2024], V[("ii", e, "A30", "C")].loc[:2024])
    assert np.allclose(V[("ii", e, "A30", "C")].loc[2025:] * dd.J.loc[2025:] / JRW[e].loc[2025:], V[("iii", e, "A30", "C")].loc[2025:])
assert np.allclose(V[("iii", "persistencia", "A14_2", "C")] * 30 / 14.2, V[("iii", "persistencia", "A30", "C")])   # linealidad en Ā
print("OK 5) (i)=(ii) hasta 2024; (iii)=(ii)·J_ONU/J_rw; TBP lineal en Ā")

# 6) peso w(c) = 1 para c < 1950, y la identidad sum(PopTotal·w) con w=DEIS/ONU en un caso puntual
w_p = pesos_cohorte("persistencia")
assert (w_p.loc[:1949] == 1).all() and np.isclose(w_p[2024], N_deis10[2024] / N_onu10[2024])
y0 = 2040
manual = sum(P.loc[y0, a] * w_p[y0 - a] for a in range(65, 101))
assert np.isclose(manual, J_reponderado(w_p, [y0])[y0])
assert (w_p.loc[1971:1974] == 1).all()                                    # hueco DEIS 1971-74: N_t de d usa ONU
assert np.allclose((w_p * N_onu10.reindex(w_p.index)).loc[1950:2024].values, dd.N_t.loc[1950:2024].values)   # w·N_ONU = N_t de d
print("OK 6) w(c)=1 para c<1950 y 1971-74 (hueco DEIS); w·N_ONU = N_t de d hasta 2024; cálculo puntual de J_rw(2040) coincide con la suma explícita")

### 10.8 Salida: `tbp_cohortes_J_reponderado.csv`

Formato largo, una fila por cohorte y escenario (`persistencia`, `recuperacion_lenta`, `caida_adicional` y `onu_medio` = DEIS hasta 2024 y ONU después, la referencia). Columnas: `J_onu`, `J_reponderado`, `N_propio`, y $TBP_C$ / $TBP_A$ para $\bar A=30$ y $14{,}2$ con $J$ reponderado (versión iii) y, con sufijo `_sinrepond`, con $J$ ONU (versión ii).

In [ ]:
partes = []
for e in ESC10 + ["onu_medio"]:
    t = pd.DataFrame({"cohorte": dd.index, "escenario": e, "anio_65": dd.anio_65.values,
                      "N_propio": (N_escenario(e) if e != "onu_medio" else dd.N_t).values,
                      "J_onu": dd.J.values, "J_reponderado": JRW[e].values})
    for ver, Jv, suf in [("iii", t.J_reponderado.values, ""), ("ii", t.J_onu.values, "_sinrepond")]:
        for A_lab, A in A_LIST:
            N = t.N_propio.values
            c = N * dd.S65.values * A * TAU / (Jv * RHO)
            t[f"TBP_C_{A_lab}{suf}"] = c
            t[f"TBP_A_{A_lab}{suf}"] = c / dd.E65.values
    partes.append(t)
out10 = pd.concat(partes, ignore_index=True)
assert out10.escenario.nunique() == 4 and len(out10) == 4 * len(dd)
DEST10 = next((p for p in ["../datos/procesados", "datos/procesados"] if os.path.isdir(p)), OUT)
os.makedirs(DEST10, exist_ok=True)
f10 = os.path.join(DEST10, "tbp_cohortes_J_reponderado.csv")
out10.round({"N_propio": 0, "J_onu": 0, "J_reponderado": 0}).round(6).to_csv(f10, index=False)
print("Escrito:", os.path.abspath(f10), out10.shape); print("Escrito:", f"{OUT}/seccion_10_J_reponderado.png y seccion_10_TBP_C_reponderado.png")
out10.head(3)

### 10.9 Lectura y límites

* **Qué cambia en la conclusión.** La curva post-2024 **sigue siendo más baja que la serie actual** con persistencia y con caída adicional, pero **menos** de lo que mostraba la versión inconsistente: reponderar $J$ sube $TBP_C$ de las cohortes recientes (cohorte 2035, $\bar A=30$, persistencia: 0,463 -> 0,513; serie actual 0,603). Con recuperación lenta la cohorte 2035 (0,623) queda ligeramente **por encima** de la serie actual (0,603): la ONU y ese escenario terminan en nacimientos parecidos y el $J$ menor empuja hacia arriba.
* **Tamaño del efecto.** Para $t\ge 2025$ $J$ cae hasta -9,9% (persistencia), -5,8% (recuperación) y -12,0% (caída) en la cohorte 2035. El rebote artificial de 2025 desaparece en los dos casos de natalidad baja.
* **Para $t\le 2024$** $J$ baja 2,4% en promedio (máx 4,8%, cohorte 1999), sobre todo por la diferencia de nivel DEIS-ONU en cohortes viejas, no por 2023-24; con la variante que ajusta solo 2023-24 la diferencia es 0,0% (máx 0,9%). El nivel de $TBP$ de las cohortes ≤2024 sube ~5% con el método principal y la forma de la curva no cambia.
* **Qué NO se puede decir:** que los escenarios sean pronósticos; que la reponderación sea exacta (supone mortalidad y migración proporcionales a la ONU, trata 100+ como una cohorte y usa $c=y-a$ pese a que la población es de mitad de año); que la corrección de nivel para cohortes viejas (DEIS vs ONU) sea inequívoca.
* **Pendiente:** validar los escenarios con el grupo y reemplazar el dato provisorio de 2025 del RENAPER por el definitivo.

## 11. 🆕 Sensibilidad (ρ, τ, N), hipótesis testeables y control

Esta sección **depende solo de variables ya en memoria** del notebook principal: `d` (cohortes 1950–2035), `s` (series por año), `sens` (sensibilidad de $\bar A$, sección 7), `TAU`, `RHO`, `A_OBS`, `A_LEGAL`, `DATA` y `OUT`. Escribe tablas y gráficos en `OUT/salidas/`.

**Qué es y qué no es.** El TBP es una **identidad determinística**: no hay variable resultado ni error aleatorio. Por eso **no corresponde un test de significancia**. La validación se hace por (i) **estática comparativa** (mover un parámetro y mirar cuánto cambia el índice), (ii) **escenarios** y (iii) **comparación con un indicador convencional** (razón de dependencia). No se afirma novedad.

Recordatorio algebraico: $TBP_C = \dfrac{N(1-\mu)\,\bar A\,\tau}{J\,\rho}$ (años), $TBP_A = TBP_C / E_r$, $TBP_A = D\times B$ con $D = N(1-\mu)/J$ y $B = \bar A\tau/(E_r\rho)$.

In [ ]:
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt

SAL = os.path.join(OUT, "salidas")          # CSV y PNG de esta seccion (NO datos/procesados)
os.makedirs(SAL, exist_ok=True)

# ---- Parametros con fuente: Anuario Estadistico ANSES 2008-2023 (datos_tbp/Anuario Estadístico_final.pdf) ----
#  p.91: tasa de sustitucion = haber medio SIPA (dic) / salario imponible medio (nov)
#  p.39: aporte personal 11%; contribucion patronal SIPA 10,77% (inc. a), 12,35% (inc. b), 16% (admin. publica)
ANSES = dict(rho_dic2009=0.371, rho_dic2023=0.403,
             rho_sin_moratoria_2009=0.425, rho_sin_moratoria_2023=0.563,
             aporte_personal=0.11, contrib_inc_a=0.1077, contrib_inc_b=0.1235, contrib_admin_publica=0.16)
TAU_A  = ANSES["aporte_personal"] + ANSES["contrib_inc_a"]          # 21,77 %
TAU_B  = ANSES["aporte_personal"] + ANSES["contrib_inc_b"]          # 23,35 %
TAU_AP = ANSES["aporte_personal"] + ANSES["contrib_admin_publica"]  # 27 %
RHO_09, RHO_23, RHO_SM = ANSES["rho_dic2009"], ANSES["rho_dic2023"], ANSES["rho_sin_moratoria_2023"]

# ---- Control 4c: los valores citados coinciden con los del notebook y con lo documentado ----
assert np.isclose(TAU, 0.2177) and np.isclose(TAU, TAU_A)
assert np.isclose(TAU_B, 0.2335) and np.isclose(TAU_AP, 0.27)
assert np.isclose(RHO, 0.403) and np.isclose(RHO, RHO_23)
assert np.isclose(RHO_09, 0.371) and np.isclose(RHO_SM, 0.563)
assert A_OBS == 14.2 and A_LEGAL == 30
print("OK 4c) tau=21,77% (23,35% / 27% alternativos), rho=40,3% (37,1% / 56,3% alternativos), A=14,2 y 30: coinciden con lo citado")

c = d.set_index("cohorte")
def tbp_C(N, S, J, A, tau, rho):            return N * S * A * tau / (J * rho)
def tbp_A(N, S, J, E, A, tau, rho):         return N * S * A * tau / (J * E * rho)

# consistencia con el notebook principal (CSV y seccion 7)
assert np.allclose(tbp_C(c.N_t, c.S65, c.J, A_LEGAL, TAU, RHO), c.TBP_C_legal)
assert np.allclose(tbp_A(c.N_t, c.S65, c.J, c.E65, A_OBS, TAU, RHO), c.TBP_A_obs)
_chk = sens[(sens.A_barra == 30) & (sens.cohorte == 2024)].TBP_C.iloc[0]
assert np.isclose(_chk, c.TBP_C_legal[2024])
print(f"OK) funciones reproducen TBP_C/TBP_A del notebook principal; TBP_C(2024, A=30) = {c.TBP_C_legal[2024]:.3f} años")

### 11.1 Sensibilidad a $\rho$ y $\tau$ (y $\bar A$, $N$)

**Rangos justificados** (todos con fuente, salvo lo rotulado como ilustración):

* $\rho$: 0,371 (dic-2009), 0,403 (dic-2023, **base**) y 0,563 (haber contributivo **sin moratoria**, 2023) — Anuario ANSES p. 91.
* $\tau$: 0,2177 (11% + 10,77%, **base**), 0,2335 (11% + 12,35%, inciso b) y 0,27 (11% + 16%, administración pública) — Anuario ANSES p. 39.
* $\bar A$: 15 y 30 (30 es un requisito legal de referencia, **no un máximo**; 14,2 es un valor observado sesgado).

**Qué esperar:** $TBP_C$ es **lineal** en $\bar A$, $\tau$ y $1/\rho$ ⇒ estos parámetros mueven el **nivel** de toda la curva por un factor constante, no su forma. Lo que *sí* cambiaría la forma es que un parámetro **dependa de la cohorte** (ejemplo ilustrativo más abajo, **no es un dato**).

In [ ]:
RHOS = [RHO_09, RHO, RHO_SM]; TAUS = [TAU, TAU_B, TAU_AP]; AS = [15.0, 30.0]; COH = [2000, 2010, 2024]
filas = []
for t in COH:
    for A in AS:
        for rho in RHOS:
            for tau in TAUS:
                filas.append(dict(cohorte=t, A_barra=A, rho=rho, tau=tau,
                                  TBP_C=tbp_C(c.N_t[t], c.S65[t], c.J[t], A, tau, rho)))
grid = pd.DataFrame(filas)

# ---- Control 4a: linealidad en rho (TBP*rho constante) y en tau (TBP/tau constante) y en A ----
for (t, A), g in grid.groupby(["cohorte", "A_barra"]):
    assert np.allclose(g.TBP_C * g.rho / g.tau, g.TBP_C.iloc[0] * g.rho.iloc[0] / g.tau.iloc[0])   # TBP*rho/tau = const
for rho in RHOS:
    for tau in TAUS:
        g = grid[(grid.rho == rho) & (grid.tau == tau)].pivot(index="cohorte", columns="A_barra", values="TBP_C")
        assert np.allclose(g[30.0], 2 * g[15.0])                       # lineal en A
base_ = lambda t: tbp_C(c.N_t[t], c.S65[t], c.J[t], A_LEGAL, TAU, RHO)
for t in COH:
    for k in (0.5, 2.0):                                                # escalar tau o 1/rho por k escala TBP por k
        assert np.isclose(tbp_C(c.N_t[t], c.S65[t], c.J[t], A_LEGAL, k * TAU, RHO), k * base_(t))
        assert np.isclose(tbp_C(c.N_t[t], c.S65[t], c.J[t], A_LEGAL, TAU, RHO / k), k * base_(t))
print("OK 4a) linealidad: TBP_C*rho/tau constante, TBP_C(k*tau)=k*TBP_C, TBP_C(rho/k)=k*TBP_C, TBP_C(2A)=2*TBP_C")

tabla = grid.pivot_table(index=["A_barra", "cohorte", "rho"], columns="tau", values="TBP_C")
tabla.columns = [f"tau={x:.4f}" for x in tabla.columns]
print("TBP_C (años) por cohorte x Ā x ρ x τ  (tau=0,2177 base; 0,2335 inc. b; 0,27 admin. pública):")
display(tabla.round(3))

rng = grid.groupby("cohorte").TBP_C.agg(["min", "max"]); rng["max/min"] = rng["max"] / rng["min"]
print("\nRango de TBP_C en toda la grilla (3 ρ × 3 τ × 2 Ā):"); display(rng.round(3))
print("Solo ρ (τ y Ā base, Ā=30): TBP_C(ρ=0,563)/TBP_C(ρ=0,371) =", round(RHO_09 / RHO_SM, 3), "-> factor constante en todas las cohortes")
tabla.reset_index().to_csv(os.path.join(SAL, "s11_grilla_rho_tau_A.csv"), index=False)

In [ ]:
# ---- Curvas por cohorte para 3 valores de rho (Ā=30, tau base) + ejemplo ILUSTRATIVO no lineal ----
coh = c.index.values
fig, ax = plt.subplots(1, 3, figsize=(19, 5.2))
for rho, col, lab in zip(RHOS, ["tab:green", "tab:blue", "tab:red"],
                         ["ρ=0,371 (dic-2009)", "ρ=0,403 (dic-2023, base)", "ρ=0,563 (sin moratoria 2023)"]):
    ax[0].plot(coh, tbp_C(c.N_t, c.S65, c.J, A_LEGAL, TAU, rho), color=col, label=lab)
ax[0].set_title("TBP_C por cohorte según ρ (Ā=30, τ=0,2177)\nρ constante ⇒ cambia el NIVEL, no la forma", fontsize=10)

# ILUSTRATIVO: rho cae para cohortes jovenes por menor cobertura. NO ES UN DATO.
def rho_ilus(t):
    return RHO * (1 - 0.40 * np.clip((np.asarray(t, dtype=float) - 1975) / (2000 - 1975), 0, 1))
tb_const = tbp_C(c.N_t, c.S65, c.J, A_LEGAL, TAU, RHO)
tb_ilus  = tbp_C(c.N_t, c.S65, c.J, A_LEGAL, TAU, rho_ilus(coh))
ax[1].plot(coh, tb_const, color="tab:blue", label="ρ constante = 0,403")
ax[1].plot(coh, tb_ilus, color="tab:orange", ls="--", label="ρ(cohorte) ILUSTRATIVO")
ax[1].set_title("EJEMPLO ILUSTRATIVO — NO ES DATO: ρ cae 40% entre\ncohortes 1975 y 2000 (supuesto inventado, menor cobertura)", fontsize=10, color="firebrick")
idx_c = 100 * tb_const / tb_const[1950]; idx_i = 100 * tb_ilus / tb_ilus[1950]
ax[2].plot(coh, idx_c, color="tab:blue", label="ρ constante (índice 1950=100)")
ax[2].plot(coh, idx_i, color="tab:orange", ls="--", label="ρ(cohorte) ILUSTRATIVO")
ax[2].set_title("Índice 1950=100: con ρ(cohorte) la FORMA sí cambia\n(ilustrativo, no es dato)", fontsize=10, color="firebrick")
for a in ax:
    a.axvline(2024.5, color="gray", ls=":"); a.grid(alpha=.3); a.legend(fontsize=8); a.set_xlabel("año de nacimiento de la cohorte")
ax[0].set_ylabel("TBP_C (años)")
plt.tight_layout(); plt.savefig(os.path.join(SAL, "s11_curvas_rho.png"), dpi=130); plt.show()

# Control: con rho constante el indice normalizado es identico entre los 3 rho; con rho(cohorte) NO lo es
idx_rho = [100 * tbp_C(c.N_t, c.S65, c.J, A_LEGAL, TAU, r) / tbp_C(c.N_t, c.S65, c.J, A_LEGAL, TAU, r)[1950] for r in RHOS]
assert all(np.allclose(idx_rho[0], x) for x in idx_rho[1:])
assert np.ptp(idx_i - idx_c) > 5          # la forma cambia de manera apreciable
print(f"OK) índice 2024 (1950=100): rho constante = {idx_c[2024]:.1f} (igual para cualquier rho) | rho(cohorte) ilustrativo = {idx_i[2024]:.1f}")
print("Lectura: TBP menor ρ ⇒ MAYOR TBP (la prestación es más barata respecto del aporte); TBP alto no significa 'mejor' para el jubilado.")

In [ ]:
# ---- Tornado para la cohorte 2024: variacion % de TBP_C al mover cada parametro dentro de su rango ----
t0 = 2024
N0, S0, J0 = c.N_t[t0], c.S65[t0], c.J[t0]
base = tbp_C(N0, S0, J0, A_LEGAL, TAU, RHO)          # base: A=30 (referencia legal), rho=0,403, tau=0,2177
rangos = [   # (nombre, valor bajo, valor alto, funcion TBP(valor))
    ("Ā  [15 – 30]",                         15.0, 30.0,       lambda v: tbp_C(N0, S0, J0, v, TAU, RHO)),
    ("ρ  [0,371 – 0,563]",                   RHO_SM, RHO_09,   lambda v: tbp_C(N0, S0, J0, A_LEGAL, TAU, v)),   # rho alto => TBP bajo
    ("τ  [0,2177 – 0,27]",                   TAU, TAU_AP,      lambda v: tbp_C(N0, S0, J0, A_LEGAL, v, RHO)),
    ("J  ±10% (ILUSTRATIVO, proyección ONU)", 1.10 * J0, 0.90 * J0, lambda v: tbp_C(N0, S0, v, A_LEGAL, TAU, RHO)),
    ("N  ±10% (ILUSTRATIVO)",                0.90 * N0, 1.10 * N0, lambda v: tbp_C(v, S0, J0, A_LEGAL, TAU, RHO)),
]
torn = pd.DataFrame([dict(parametro=n, valor_en_extremo_bajo_TBP=lo, valor_en_extremo_alto_TBP=hi,
                          var_pct_bajo=100 * (f(lo) / base - 1), var_pct_alto=100 * (f(hi) / base - 1))
                     for n, lo, hi, f in rangos])
torn["amplitud_pp"] = torn.var_pct_alto - torn.var_pct_bajo
torn = torn.sort_values("amplitud_pp")
display(torn.round(2))

fig, ax = plt.subplots(figsize=(9, 4.2))
y = np.arange(len(torn))
ax.barh(y, torn.var_pct_bajo, color="tab:red", alpha=.8, label="extremo que baja TBP")
ax.barh(y, torn.var_pct_alto, color="tab:blue", alpha=.8, label="extremo que sube TBP")
ax.set_yticks(y); ax.set_yticklabels(torn.parametro, fontsize=8.5); ax.axvline(0, color="k", lw=.8)
ax.set_xlabel(f"variación % de TBP_C (cohorte 2024)\nbase = {base:.2f} años (Ā=30, ρ=0,403, τ=0,2177)", fontsize=9)
ax.set_title("Tornado: cohorte 2024 (N de DEIS; J y S65 proyección ONU)", fontsize=10); ax.grid(alpha=.3, axis="x"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(SAL, "s11_tornado_cohorte2024.png"), dpi=130); plt.show()
# controles aritmeticos del tornado (lineales / hiperbolica)
assert np.isclose(torn.set_index("parametro").loc["Ā  [15 – 30]", "var_pct_bajo"], -50.0)
assert np.isclose(torn.set_index("parametro").loc["N  ±10% (ILUSTRATIVO)", "var_pct_bajo"], -10.0)
assert np.isclose(torn.set_index("parametro").loc["ρ  [0,371 – 0,563]", "var_pct_bajo"], 100 * (RHO / RHO_SM - 1))
torn.to_csv(os.path.join(SAL, "s11_tornado_cohorte2024.csv"), index=False)
print("Nota: la base usa Ā=30 (referencia legal, no un máximo); con otra base de Ā los porcentajes de Ā cambian, los de ρ, τ y N no.")

### 11.2 Hipótesis testeables y estática comparativa

Cada hipótesis se formula con **signo esperado**, **criterio de rechazo** y **la parte del modelo** que la pone a prueba. **Honestidad metodológica:** como el TBP es una identidad, las "pruebas" que siguen son **comparaciones de estática comparativa** (¿el signo y la magnitud calculados con las cohortes reales coinciden con lo esperado?), **no pruebas estadísticas**: no hay p-valores ni intervalos de confianza, y las cohortes sucesivas no son observaciones independientes.

| | Hipótesis | Signo esperado | Parte del modelo | Criterio de rechazo |
|---|---|---|---|---|
| H1 | Una caída sostenida de $N$ respecto de las cohortes que integran $J$ reduce el TBP | − (vía $D$) | numerador $N$ y denominador $J$ de $D$ | La contribución de $\Delta\ln N$ a $\Delta\ln TBP$ tiene signo contrario, o $N$ cae y TBP no cae con el resto fijo |
| H2 | Mayor longevidad ($E_r$, $J$) reduce el TBP | − (doble canal en $TBP_A$; un canal en $TBP_C$) | $E_r$ y $J$ | La elasticidad de $TBP_A$ a $E_r$ no es −1, o la contribución de $-\Delta\ln J$ no es negativa cuando $J$ crece |
| H3 | Mayor densidad de aportes $\bar A$ aumenta el TBP (**nivel, no forma**) | + | $\bar A$ | Elasticidad ≠ +1, o el índice normalizado cambia con $\bar A$ |
| H4 | Mayor formalidad (más $\bar A$ y/o $\tau$ efectiva) aumenta el TBP | + | $\bar A$, $\tau$ | Igual que H3/H5; **la formalidad por cohorte requiere EPH (sección 8, sin datos reales todavía)** |
| H5 | Mayor tasa de aporte $\tau$ aumenta el TBP | + | $\tau$ | Elasticidad ≠ +1 |
| H6 | Mayor tasa de reemplazo $\rho$ reduce el TBP | − | $\rho$ | Elasticidad ≠ −1 |

Nota: $S_{65}=1-\mu$ (supervivencia hasta los 65) **aumenta** el TBP: más longevidad *antes* de 65 y más longevidad *después* de 65 empujan en sentidos opuestos. Por eso H2 se refiere a la longevidad **posterior** a los 65 ($E_r$) y al stock de mayores $J$.

In [ ]:
# ---- Elasticidades discretas (arco, +10%) en la cohorte 2024: eps = ln(TBP1/TBP0) / ln(x1/x0) ----
t0 = 2024
P0 = dict(N=c.N_t[t0], S=c.S65[t0], J=c.J[t0], E=c.E65[t0], A=A_LEGAL, tau=TAU, rho=RHO)
def f_C(p): return tbp_C(p["N"], p["S"], p["J"], p["A"], p["tau"], p["rho"])
def f_A(p): return tbp_A(p["N"], p["S"], p["J"], p["E"], p["A"], p["tau"], p["rho"])
esp = {"N": (1, 1), "S": (1, 1), "J": (-1, -1), "E": (0, -1), "A": (1, 1), "tau": (1, 1), "rho": (-1, -1)}
et = []
for k in P0:
    p1 = dict(P0); p1[k] = 1.10 * P0[k]
    eC = np.log(f_C(p1) / f_C(P0)) / np.log(1.10); eA = np.log(f_A(p1) / f_A(P0)) / np.log(1.10)
    assert np.isclose(eC, esp[k][0]) and np.isclose(eA, esp[k][1])
    et.append(dict(parametro=k, elasticidad_TBP_C=eC, elasticidad_TBP_A=eA, hipotesis={"N":"H1","S":"-","J":"H1/H2","E":"H2","A":"H3/H4","tau":"H4/H5","rho":"H6"}[k]))
elas = pd.DataFrame(et)
display(elas.round(3))
print("OK) signos y magnitudes de H1–H6 coinciden con lo esperado (elasticidades ±1; E_r: 0 en TBP_C, −1 en TBP_A).")
print("Ojo: estas elasticidades son propiedades del diseño del índice (cada factor entra con exponente ±1), no un resultado empírico.")

In [ ]:
# ---- Descomposición logarítmica EXACTA de la caída del TBP entre cohortes ----
#  ln TBP_C = ln N + ln S65 + ln Ā + ln τ − ln J − ln ρ  =>  con Ā, τ, ρ constantes:
#  Δln TBP_C = Δln N + Δln S65 − Δln J ;   Δln TBP_A = Δln TBP_C − Δln E_r
def descomponer(t0, t1, tipo="C"):
    comp = {"N (natalidad)": np.log(c.N_t[t1] / c.N_t[t0]),
            "S65 (supervivencia a 65)": np.log(c.S65[t1] / c.S65[t0]),
            "J (stock 65+; envejecimiento)": -np.log(c.J[t1] / c.J[t0])}
    if tipo == "A": comp["E_r (longevidad a los 65)"] = -np.log(c.E65[t1] / c.E65[t0])
    col = "TBP_C_legal" if tipo == "C" else "TBP_A_legal"
    total = np.log(c[col][t1] / c[col][t0])           # calculado desde el indice, independiente de los componentes
    assert np.isclose(sum(comp.values()), total, atol=1e-12)                      # control 4b
    sh = {k: 100 * v / total for k, v in comp.items()}
    assert np.isclose(sum(sh.values()), 100.0)
    return comp, total, sh

res = []
for (t0_, t1_) in [(1990, 2024), (2000, 2024), (2010, 2024)]:
    for tipo in "CA":
        comp, total, sh = descomponer(t0_, t1_, tipo)
        row = dict(desde=t0_, hasta=t1_, indice="TBP_" + tipo, dlnTBP=total, variacion_pct=100 * (np.exp(total) - 1))
        for k in comp: row["dln " + k] = comp[k]; row["% " + k] = sh[k]
        row["suma_%"] = sum(sh.values()); res.append(row)
desc = pd.DataFrame(res)
print("OK 4b) la suma de las contribuciones = ΔlnTBP (100%) en todas las comparaciones")
display(desc[["desde", "hasta", "indice", "variacion_pct", "dlnTBP"] + [x for x in desc.columns if x.startswith("% ")] + ["suma_%"]].round(2))

comp, total, sh = descomponer(2000, 2024, "C")
print(f"\nTBP_C cohorte 2000 -> 2024: {c.TBP_C_legal[2000]:.3f} -> {c.TBP_C_legal[2024]:.3f} años ({100*(np.exp(total)-1):.1f}%); Δln = {total:.4f}")
for k in comp: print(f"   {k:34s} Δln = {comp[k]:+.4f}  -> {sh[k]:6.1f}% de la caída")

# Detalle de H1: N de la cohorte vs. promedio de N de las cohortes que integran J (nacidas t-35..t, 65 a 100 años en t+65)
Nref = pd.Series({t: c.N_t.loc[t - 35:t].mean() for t in range(1985, 2036)})
for (t0_, t1_) in [(2000, 2024)]:
    rel = np.log((c.N_t[t1_] / Nref[t1_]) / (c.N_t[t0_] / Nref[t0_])); ref = np.log(Nref[t1_] / Nref[t0_])
    assert np.isclose(rel + ref, np.log(c.N_t[t1_] / c.N_t[t0_]))
    print(f"\nDetalle H1 ({t0_}->{t1_}): Δln N = {comp['N (natalidad)']:+.4f} = {rel:+.4f} (N relativo al promedio de las 36 cohortes que integran J) + {ref:+.4f} (nivel de ese promedio)")
print("Lectura: J agrega natalidad y longevidad de cohortes anteriores; por eso la contribución de J no es 'longevidad pura'.")
desc.to_csv(os.path.join(SAL, "s11_descomposicion_log.csv"), index=False)
elas.to_csv(os.path.join(SAL, "s11_elasticidades_cohorte2024.csv"), index=False)

fig, ax = plt.subplots(figsize=(8, 4))
comps = {"C": descomponer(2000, 2024, "C")[0], "A": descomponer(2000, 2024, "A")[0]}
labels = list(comps["A"].keys()); xx = np.arange(len(labels)); w = .38
ax.bar(xx - w/2, [comps["C"].get(k, 0) for k in labels], w, label="TBP_C (recomendada)", color="tab:blue")
ax.bar(xx + w/2, [comps["A"][k] for k in labels], w, label="TBP_A (con E_r)", color="tab:orange")
ax.axhline(0, color="k", lw=.8); ax.set_xticks(xx); ax.set_xticklabels(labels, fontsize=8, rotation=10)
ax.set_ylabel("contribución a Δln TBP (2000 → 2024)"); ax.set_title("Descomposición logarítmica exacta de la caída 2000→2024", fontsize=10)
ax.grid(alpha=.3, axis="y"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(SAL, "s11_descomposicion_2000_2024.png"), dpi=130); plt.show()

In [ ]:
# ---- Tabla de hipotesis con la evidencia numerica de estatica comparativa ----
cC, tC, shC = descomponer(2000, 2024, "C"); cA, tA, shA = descomponer(2000, 2024, "A")
e = elas.set_index("parametro")
cf_N = tbp_C(c.N_t[2000], c.S65[2024], c.J[2024], A_LEGAL, TAU, RHO) / c.TBP_C_legal[2024] - 1
hip = pd.DataFrame([
 dict(id="H1", hipotesis="Caída sostenida de N respecto de las cohortes que integran J reduce TBP (vía D)", signo="−",
      parte_del_modelo="N y J en D", criterio_rechazo="Contribución de Δln N con signo opuesto, o N cae y TBP_C no cae (resto fijo)",
      evidencia=f"2000→2024: N {c.N_t[2000]:,.0f}→{c.N_t[2024]:,.0f}; contribución de N = {shC['N (natalidad)']:.1f}% de la caída de TBP_C; con N de 2000 y el resto de 2024, TBP_C(2024) sería {100*cf_N:+.0f}% mayor",
      estado="no rechazada (comparación de estática comparativa)"),
 dict(id="H2", hipotesis="Mayor longevidad (E_r y J) reduce TBP (doble canal en TBP_A, uno en TBP_C)", signo="−",
      parte_del_modelo="E_r y J", criterio_rechazo="Elasticidad de TBP_A a E_r ≠ −1 o contribución de J no negativa cuando J crece",
      evidencia=f"elasticidad TBP_A a E_r = {e.loc['E','elasticidad_TBP_A']:.0f}, a J = {e.loc['J','elasticidad_TBP_A']:.0f}; 2000→2024: contrib. de J = {shC['J (stock 65+; envejecimiento)']:.1f}% (TBP_C), de E_r = {shA['E_r (longevidad a los 65)']:.1f}% (TBP_A); S65 empuja en sentido contrario ({shC['S65 (supervivencia a 65)']:.1f}%)",
      estado="no rechazada (E_r casi no cambia; el canal J es el relevante)"),
 dict(id="H3", hipotesis="Mayor densidad de aportes Ā aumenta TBP (nivel, no forma)", signo="+",
      parte_del_modelo="Ā", criterio_rechazo="Elasticidad ≠ +1 o el índice normalizado depende de Ā",
      evidencia=f"elasticidad = {e.loc['A','elasticidad_TBP_C']:.0f}; Ā 15→30 duplica TBP_C; índice 1950=100 idéntico (sección 7)", estado="no rechazada (propiedad del diseño)"),
 dict(id="H4", hipotesis="Mayor formalidad = mayor Ā y/o τ efectiva ⇒ TBP mayor", signo="+",
      parte_del_modelo="Ā y τ", criterio_rechazo="Como H3/H5; requiere Ā por cohorte (EPH, sección 8)",
      evidencia=f"efecto de signo conocido (+); ejemplo: Ā 15→22,5 (+50%) y τ 0,2177→0,27 (+24%) ⇒ TBP_C ×{1.5*TAU_AP/TAU:.2f}", estado="signo verificado; magnitud NO evaluable sin datos EPH"),
 dict(id="H5", hipotesis="Mayor τ aumenta TBP", signo="+", parte_del_modelo="τ", criterio_rechazo="Elasticidad ≠ +1",
      evidencia=f"elasticidad = {e.loc['tau','elasticidad_TBP_C']:.0f}; τ 0,2177→0,27: {100*(TAU_AP/TAU-1):+.1f}%", estado="no rechazada (propiedad del diseño)"),
 dict(id="H6", hipotesis="Mayor ρ reduce TBP", signo="−", parte_del_modelo="ρ", criterio_rechazo="Elasticidad ≠ −1",
      evidencia=f"elasticidad = {e.loc['rho','elasticidad_TBP_C']:.0f}; ρ 0,403→0,563: {100*(RHO/RHO_SM-1):+.1f}%; 0,403→0,371: {100*(RHO/RHO_09-1):+.1f}%", estado="no rechazada (propiedad del diseño)"),
])
pd.set_option("display.max_colwidth", 200)
display(hip[["id", "signo", "parte_del_modelo", "evidencia", "estado"]])
hip.to_csv(os.path.join(SAL, "s11_hipotesis.csv"), index=False)
print("Aclaración: 'no rechazada' = el signo y la magnitud calculados coinciden con lo esperado; NO es un resultado estadístico.")

### 11.3 Control con un indicador convencional: razón de dependencia (ONU, 65+ / 15–64)

La razón de dependencia de vejez es el indicador habitual del envejecimiento. Se compara, **por cohorte $t$**, el $TBP_C$ con la razón de dependencia en el año $t+65$ (cuando la cohorte tiene 65, el momento que pondera el TBP) y en $t+40$ (mitad de la vida activa). Esperado: **signo negativo** (más dependencia ⇒ menor TBP).

**⚠ Circularidad mecánica.** $J$ (población 65+ en $t+65$) es el numerador de la razón de dependencia **en $t+65$** y el denominador de $TBP_C$: un signo negativo en $t+65$ está **garantizado en parte por construcción**. La comparación en $t+40$ evita esa coincidencia exacta de año, pero ambas series tienen **tendencia** (envejecimiento), de modo que una correlación de rangos entre series con tendencia es casi inevitable. Se agregan dos diagnósticos: (i) la correlación de la razón con $1/J$ solo (benchmark mecánico) y (ii) las **primeras diferencias** (cambios entre cohortes consecutivas). **No se reportan p-valores** (cohortes consecutivas no son independientes).

In [ ]:
dep = s["un_dependencia_65mas_15_64"]
# la columna de la ONU = pob 65+ / pob 15-64 (control de consistencia con las series de poblacion)
chk = (s.un_pob_65mas_miles / s.un_pob_15_64_miles).dropna()
assert np.allclose(dep.loc[chk.index], chk, atol=1e-3)

x = pd.DataFrame({"cohorte": c.index, "TBP_C": c.TBP_C_legal.values, "TBP_B": c.TBP_B_legal.values,
                  "inv_J": 1 / c.J.values, "NS65": (c.N_t * c.S65).values})
x["dep_t65"] = dep.reindex(c.anio_65.values).values
x["dep_t40"] = dep.reindex((c.index.values + 40)).values
x = x.dropna()

def sp(a, b, df=x): return df[a].corr(df[b], method="spearman")
def sp_dif(a, b, df=x): return df[a].diff().corr(df[b].diff(), method="spearman")
filas = []
for etiqueta, df_ in [("todas las cohortes 1950–2035", x), ("solo cohortes con N observado (≤2024)", x[x.cohorte <= 2024])]:
    for h in ["dep_t65", "dep_t40"]:
        filas.append(dict(muestra=etiqueta, razon_dependencia_en=h.replace("dep_t", "t+"), n=len(df_),
                          spearman_TBP_C=sp("TBP_C", h, df_), spearman_inv_J_benchmark_mecanico=sp("inv_J", h, df_),
                          spearman_NxS65_sin_J=sp("NS65", h, df_), spearman_TBP_B_sin_N_ni_J=sp("TBP_B", h, df_),
                          spearman_primeras_dif_TBP_C=sp_dif("TBP_C", h, df_)))
corr = pd.DataFrame(filas)
display(corr.round(3))
corr.to_csv(os.path.join(SAL, "s11_correlacion_dependencia.csv"), index=False)

r65 = sp("TBP_C", "dep_t65"); r40 = sp("TBP_C", "dep_t40")
print(f"Spearman(TBP_C, razón de dependencia en t+65) = {r65:+.3f}   <- inflado por circularidad (J en ambos)")
print(f"Spearman(TBP_C, razón de dependencia en t+40) = {r40:+.3f}")
print(f"Benchmark mecánico: Spearman(1/J, razón en t+65) = {sp('inv_J','dep_t65'):+.3f}")
assert r65 < 0 and r40 < 0
print("OK) signo negativo en ambos horizontes, como se esperaba.")
print("Lectura prudente: el signo es el esperado, pero no es una validación independiente (circularidad y tendencia común). "
      "La información adicional del TBP respecto de la razón de dependencia está en N, S65, E_r, τ, ρ y Ā, que la razón no incluye.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
for a, h, ttl in [(ax[0], "dep_t65", "razón de dependencia en t+65 (circular)"), (ax[1], "dep_t40", "razón de dependencia en t+40")]:
    sc = a.scatter(x[h], x.TBP_C, c=x.cohorte, cmap="viridis", s=18)
    a.set_xlabel(ttl); a.set_ylabel("TBP_C (Ā=30, años)"); a.grid(alpha=.3)
    a.set_title(f"Spearman = {sp('TBP_C', h):+.2f}", fontsize=10)
fig.colorbar(sc, ax=ax, label="cohorte")
plt.savefig(os.path.join(SAL, "s11_tbp_vs_dependencia.png"), dpi=130, bbox_inches="tight"); plt.show()

### 11.4 Qué se puede y qué no se puede concluir

* **Sí:** el signo y el orden de magnitud de cada canal (H1–H6) con cohortes reales; que la caída del $TBP_C$ 2000→2024 se descompone **exactamente** en natalidad, supervivencia a 65 y stock de mayores; que $\rho$, $\tau$ y $\bar A$ cambian el **nivel** y no la forma.
* **No:** probar causalidad ni significancia; afirmar que $\rho$ o $\tau$ sean constantes entre cohortes (hoy se usa un valor único de 2023); evaluar H4 sin $\bar A$ por cohorte (EPH, sección 8, sin datos reales aún).
* **Cohorte 2024 y siguientes:** $J$, $S_{65}$ y $E_r$ son **proyecciones ONU a 2089+**: la incertidumbre demográfica es grande y no está en el tornado salvo la ilustración de $J$ ±10%.
* Los ejemplos rotulados "ILUSTRATIVO" (ρ dependiente de la cohorte, $N$ y $J$ ±10%) **no son datos**.

In [ ]:
# ---- Controles finales (4a, 4b, 4c ya ejecutados arriba) y listado de salidas ----
assert os.path.isdir(SAL) and "procesados" not in os.path.abspath(SAL)
for f in sorted(os.listdir(SAL)):
    if f.startswith("s11_"): print("Escrito:", os.path.join(SAL, f))
print("\nControles de la sección 11: 4a linealidad OK | 4b descomposición suma ΔlnTBP OK | 4c parámetros citados OK")

## 12. 🆕 Calibración de $\bar A$ con microdatos de la EPH: descarga, filtro de aportantes y edad de ingreso

Esta sección convierte el método de la sección 8 en una **rutina lista para correr con datos reales**. Pasos:

1. **Descargar / cargar** las bases usuarias individuales trimestrales de la EPH (INDEC). La descarga está **apagada por defecto** (`ACEPTO_DESCARGA = False`): primero muestra qué archivos son y cuánto pesan.
2. **Filtrar** la población 18–64 y marcar como *registrados* a los ocupados con descuento jubilatorio (`PP07H`).
3. **Pseudo-panel** por cohorte (`ANO4 − CH06`) y edad, y modelo logit aditivo cohorte + edad.
4. **Edad efectiva de ingreso al empleo registrado** por cohorte y **$\bar A$** por cohorte, con tres reglas para cuentapropistas.
5. **Conexión con el TBP:** $TBP_C$ con $\bar A(t)$ por cohorte en lugar de un valor único (la "Opción 3" de la propuesta del grupo, apenas haya datos reales).

> ⚠️ **Estado:** sin microdatos cargados esta sección **no calcula ningún $\bar A$ para Argentina**; sólo corre la prueba con **datos sintéticos** (12.6). Disponibilidad verificada de las bases: 2017-T2 a 2025-T4 (35 archivos, ~107 MB). La EPH es urbana y transversal, no observa la edad del primer empleo registrado, y el descuento jubilatorio mide la semana de referencia: ver 12.7.

### 12.1 Módulo (copia de `src/tbp_eph.py`)

Todo el código está en una sola celda para que el Colab sea autosuficiente. El mismo archivo vive en el repositorio (`src/tbp_eph.py`) y se puede usar desde la terminal: `python src/tbp_eph.py --plan`, `--descargar`, `--autotest`.

In [ ]:
# ====== MÓDULO tbp_eph (misma versión que src/tbp_eph.py) ======
# -*- coding: utf-8 -*-
"""
tbp_eph.py — Calibración de Ā (años de aporte esperados por persona de la cohorte) con la EPH (INDEC).

Rutina modular para el proyecto TBP — Índice de Cuna Vacía.

Flujo:
    1. plan_descarga / descargar_eph   -> baja las bases usuarias individuales trimestrales (con confirmación explícita)
    2. find_eph_files / read_all       -> lee las bases (txt o zip de INDEC)
    3. add_flags                       -> filtra ocupados y marca "registrado" (descuento jubilatorio PP07H)
    4. build_cells + ajuste logit      -> pseudo-panel cohorte x edad (densidad de aportes)
    5. edad_ingreso_*                  -> edad efectiva de ingreso al empleo registrado, por cohorte
    6. estimar_abar                    -> Ā por cohorte (3 reglas para independientes)
    7. tbp_con_abar                    -> TBP_C y TBP_A usando Ā(t) en lugar de un valor único

AVISOS METODOLÓGICOS (leer):
  * La EPH es una encuesta transversal y urbana: NO observa la edad del primer empleo registrado ni la historia de aportes
    de cada persona. Se trabaja con un pseudo-panel (cohorte = ANO4 − CH06) y se asume que el perfil por edad es estable
    entre cohortes (modelo logit aditivo cohorte + edad). Edad, período y cohorte no se pueden separar del todo.
  * PP07H ("¿le descuentan jubilación?") se pregunta a asalariados y mide la semana de referencia, no el año.
    Una persona con descuento no necesariamente suma "años de aporte" de ley (moratorias, regímenes provinciales, etc.).
  * La base individual NO trae aporte de patrones/cuentapropistas (PP07H y PP07I sólo se relevan a asalariados; PP07I sólo a
    quienes no tienen descuento): hay tres reglas ("solo_asalariados" = cota inferior; "pp07i" = descuento o aporte propio
    del asalariado; "escenario" = pp07i + proporción supuesta de independientes aportando) y se reportan las tres.
  * Estructura verificada con las 35 bases reales 2017-T2..2025-T4 (ver `resumen_base`): separador ';', texto ASCII/UTF-8,
    la base individual de 2020-T4 se llama "personas" (ya contemplado).
  * No hay valor de Ā "verdadero" para Argentina hasta correr esto con datos reales. Los números del autotest son sintéticos.

Uso mínimo:
    import tbp_eph as e
    e.descargar_eph("datos/eph")                                   # sólo muestra el plan (no descarga)
    e.descargar_eph("datos/eph", confirmado=True)                  # descarga de verdad
    df = e.read_all(e.find_eph_files(["datos/eph"]))
    tabla = e.estimar_abar(df, s65)                                # s65: Series cohorte -> supervivencia a 65
"""
import os, re, io, glob, time, zipfile, warnings, tempfile, urllib.request
import numpy as np
import pandas as pd

# ------------------------------------------------------------------ constantes
EDAD_MIN, EDAD_MAX = 18, 64
EDADES = np.arange(EDAD_MIN, EDAD_MAX + 1)               # 47 edades simples
BASE_URL = "https://www.indec.gob.ar/ftp/cuadros/menusuperior/eph"
# Disponibilidad verificada (2 oct 2026, por cabecera HTTP y firma ZIP): 2017-T2 a 2025-T4.
# 2016 y 2017-T1 NO están con este patrón de nombre en el sitio actual de INDEC.
DISPONIBLE_DESDE, DISPONIBLE_HASTA = (2017, 2), (2025, 4)
USER_AGENT = "Mozilla/5.0 (compatible; TBP-proyecto-universitario)"
MIN_N_CELDA = 20          # mínimo de observaciones (sin ponderar) por celda cohorte x edad
MIN_EDADES_COHORTE = 3    # mínimo de edades observadas para estimar el efecto propio de una cohorte
GOMPERTZ_B = 0.085        # pendiente de la mortalidad adulta (supuesto)
IND_RULE_DEFAULT = "solo_asalariados"
IND_SHARE_DEFAULT = None
NEEDED = {"ANO4", "TRIMESTRE", "CH04", "CH06", "ESTADO", "CAT_OCUP", "PP07H", "PP07I", "PONDERA"}
OBLIGATORIAS = {"ANO4", "CH06", "ESTADO", "PONDERA"}


# ================================================================== 1) DESCARGA
def urls_eph(desde=DISPONIBLE_DESDE, hasta=DISPONIBLE_HASTA):
    """Lista de bases trimestrales (año, trimestre, nombre, url) en el rango pedido."""
    out = []
    for y in range(desde[0], hasta[0] + 1):
        for t in (1, 2, 3, 4):
            if (y, t) < tuple(desde) or (y, t) > tuple(hasta):
                continue
            nombre = f"EPH_usu_{t}_Trim_{y}_txt.zip"
            out.append({"anio": y, "trim": t, "nombre": nombre, "url": f"{BASE_URL}/{nombre}"})
    return out


def _ctx_ssl():
    """Contexto SSL con verificación ACTIVA. Usa certifi si está instalado (Python de macOS suele no traer certificados raíz)."""
    import ssl
    try:
        import certifi
        return ssl.create_default_context(cafile=certifi.where())
    except Exception:
        return ssl.create_default_context()


def _abrir(url, metodo="GET", rango=None, timeout=60):
    req = urllib.request.Request(url, method=metodo, headers={"User-Agent": USER_AGENT})
    if rango:
        req.add_header("Range", rango)
    return urllib.request.urlopen(req, timeout=timeout, context=_ctx_ssl())


def _curl(url, destino=None, solo_cabecera=False, timeout=120):
    """Alternativa con `curl` del sistema (usa el almacén de certificados del sistema; la verificación sigue activa)."""
    import subprocess
    cmd = ["curl", "-sL", "--max-time", str(timeout), "-A", USER_AGENT]
    if solo_cabecera:
        r = subprocess.run(cmd + ["-I", url], capture_output=True, text=True)
        cab = [l for l in r.stdout.splitlines() if l.lower().startswith("content-length")]
        return int(cab[-1].split(":")[1]) if cab else None
    subprocess.run(cmd + ["-o", destino, url], check=True)


def plan_descarga(destino="datos/eph", desde=DISPONIBLE_DESDE, hasta=DISPONIBLE_HASTA, consultar_tamanos=True):
    """Arma el plan SIN descargar: qué archivos, tamaño (cabecera HTTP HEAD) y cuáles ya están en `destino`."""
    plan = []
    for it in urls_eph(desde, hasta):
        it = dict(it); it["existe"] = os.path.exists(os.path.join(destino, it["nombre"])); it["bytes"] = None
        if consultar_tamanos:
            try:
                with _abrir(it["url"], "HEAD") as r:
                    it["bytes"] = int(r.headers.get("Content-Length", 0)) or None
                    it["tipo"] = r.headers.get("Content-Type", "")
            except Exception as ex:                       # problema de certificados/red: se prueba curl; si no, se informa
                try:
                    it["bytes"] = _curl(it["url"], solo_cabecera=True)
                except Exception as ex2:
                    it["error"] = f"{ex} | curl: {ex2}"
        plan.append(it)
    return pd.DataFrame(plan)


def descargar_eph(destino="datos/eph", desde=DISPONIBLE_DESDE, hasta=DISPONIBLE_HASTA, confirmado=False, pausa=0.5):
    """Descarga las bases usuarias individuales de la EPH desde INDEC.

    `confirmado=False` (por defecto): MODO PRUEBA. Muestra el plan (archivos y tamaño total) y NO descarga nada.
    `confirmado=True`: descarga los que falten. Verifica que cada archivo sea un ZIP real (INDEC devuelve una página HTML
    con código 200 cuando el archivo no existe) y no vuelve a bajar los que ya están.
    """
    plan = plan_descarga(destino, desde, hasta)
    faltan = plan[~plan["existe"]]
    total_mb = (faltan["bytes"].fillna(0).sum()) / 1e6
    print(f"EPH: {len(plan)} archivos en el rango; {len(faltan)} por descargar (~{total_mb:.1f} MB). Fuente: {BASE_URL}/")
    if not confirmado:
        print("MODO PRUEBA: no se descargó nada. Para descargar: descargar_eph(..., confirmado=True)")
        return plan
    os.makedirs(destino, exist_ok=True)        # la carpeta se crea sólo si hay descarga confirmada
    ok, fallidos = 0, []
    for _, it in faltan.iterrows():
        ruta = os.path.join(destino, it["nombre"]); tmp = ruta + ".part"
        try:
            with _abrir(it["url"]) as r, open(tmp, "wb") as f:
                cab = r.read(4); f.write(cab)
                if cab[:2] != b"PK":
                    raise ValueError("la respuesta no es un ZIP (¿archivo inexistente?)")
                while True:
                    bloque = r.read(1 << 20)
                    if not bloque:
                        break
                    f.write(bloque)
            os.replace(tmp, ruta); ok += 1
            print(f"  ✔ {it['nombre']} ({os.path.getsize(ruta)/1e6:.1f} MB)")
        except Exception as ex:
            if "CERTIFICATE" in str(ex).upper() or "SSL" in str(ex).upper():      # reintento con curl (verificación activa)
                try:
                    _curl(it["url"], destino=tmp)
                    with open(tmp, "rb") as g:
                        if g.read(2) != b"PK":
                            raise ValueError("la respuesta no es un ZIP (¿archivo inexistente?)")
                    os.replace(tmp, ruta); ok += 1
                    print(f"  ✔ {it['nombre']} ({os.path.getsize(ruta)/1e6:.1f} MB, vía curl)")
                    time.sleep(pausa); continue
                except Exception as ex2:
                    ex = ex2
            fallidos.append((it["nombre"], str(ex)))
            if os.path.exists(tmp): os.remove(tmp)
            print(f"  ✘ {it['nombre']}: {ex}")
        time.sleep(pausa)
    print(f"Listo: {ok} descargados, {len(fallidos)} fallidos.")
    return plan


# ================================================================== 2) LECTURA
# INDEC nombra la base individual "usu_individual_T..." salvo 4T2020 ("EPH_usu_personas_4to.trim_2020.txt.txt"); mayúsculas y
# subcarpetas varían entre trimestres (verificado con las 35 bases reales).
_PAT_IND = r"usu_(individual|personas).*\.(txt|csv)$"


def find_eph_files(rutas=("datos/eph", "../datos/eph", "../../datos/eph")):
    """Busca bases individuales (usu_individual_T*.txt/.csv, o EPH_usu_*.zip con una base individual adentro)."""
    out = []
    for r in rutas:
        if not os.path.isdir(r):
            continue
        for p in sorted(glob.glob(os.path.join(r, "**", "*"), recursive=True)):
            b = os.path.basename(p).lower()
            if os.path.isfile(p) and re.search(_PAT_IND, b):
                out.append({"path": p, "member": None})
            elif os.path.isfile(p) and b.endswith(".zip") and "eph_usu" in b:
                with zipfile.ZipFile(p) as z:
                    for m in z.namelist():
                        if re.search(_PAT_IND, m.lower()):
                            out.append({"path": p, "member": m})
    return out


def _leer_texto(handle, nombre=""):
    cab = handle.readline() if hasattr(handle, "readline") else ""
    sep = ";" if cab.count(";") >= cab.count(",") else ","       # INDEC usa ';' (se autodetecta)
    handle.seek(0)
    return pd.read_csv(handle, sep=sep, usecols=lambda c: c.strip().upper() in NEEDED, low_memory=False)


def read_individual(src):
    """Lee UNA base individual trimestral (sólo las columnas necesarias). Columnas en MAYÚSCULAS, valores numéricos."""
    if src["member"] is None:
        with open(src["path"], "r", encoding="latin-1", newline="") as f:
            df = _leer_texto(f, src["path"])
    else:
        with zipfile.ZipFile(src["path"]) as z, z.open(src["member"]) as raw:
            df = _leer_texto(io.StringIO(raw.read().decode("latin-1")), src["member"])
    df.columns = [c.strip().upper() for c in df.columns]
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    faltan = OBLIGATORIAS - set(df.columns)
    if faltan:
        raise ValueError(f"Faltan columnas obligatorias {sorted(faltan)} en {src}. Revisar el diseño de registro de INDEC.")
    for opt in ("CAT_OCUP", "PP07H", "PP07I", "CH04", "TRIMESTRE"):
        if opt not in df.columns:
            df[opt] = np.nan
    return df


def read_all(srcs):
    return pd.concat([read_individual(s) for s in srcs], ignore_index=True)


def resumen_base(df):
    """Control rápido tras cargar datos reales: filas por año, nulos, ESTADO, CAT_OCUP y PP07H/PP07I entre ocupados."""
    print("Filas por año:", df.groupby("ANO4").size().to_dict())
    print("Nulos (%):", (df[["CH06", "ESTADO", "CAT_OCUP", "PP07H", "PP07I", "PONDERA"]].isna().mean() * 100).round(1).to_dict())
    print("ESTADO (0 sin entrevista indiv., 1 ocupado, 2 desocupado, 3 inactivo, 4 menor de 10):", df["ESTADO"].value_counts(dropna=False).sort_index().to_dict())
    o = df[df["ESTADO"] == 1]
    print("CAT_OCUP entre ocupados (1 patrón, 2 cuenta propia, 3 asalariado, 4 familiar s/rem., 9 NS/NR):", o["CAT_OCUP"].value_counts(dropna=False).sort_index().to_dict())
    a = o[o["CAT_OCUP"] == 3]
    print("PP07H entre asalariados ocupados (1 sí, 2 no, 9 NS/NR):", a["PP07H"].value_counts(dropna=False).sort_index().to_dict())
    print("PP07I entre asalariados ocupados (0 = no corresponde; sólo se pregunta si PP07H==2):", a["PP07I"].value_counts(dropna=False).sort_index().to_dict())


# ================================================================== 3) FILTRO: ocupados con descuento jubilatorio
def add_flags(df, ind_rule=IND_RULE_DEFAULT, ind_share=IND_SHARE_DEFAULT, sexo=None):
    """Población 18-64 con entrevista individual; construye `reg` = prob. de estar aportando.
    Verificado con las 35 bases reales (2017-T2 a 2025-T4): PP07H ("¿tiene descuento jubilatorio?") y PP07I ("¿aporta por sí
    mismo a algún sistema jubilatorio?") sólo se relevan a ASALARIADOS (CAT_OCUP==3); PP07I sólo a quienes respondieron
    PP07H==2. Patrones y cuentapropistas tienen PP07H/PP07I vacíos: la base individual NO trae aporte de independientes.
    Reglas (`ind_rule`):
      "solo_asalariados": asalariado con PP07H==1 (cota inferior). Independientes = 0.
      "pp07i":            asalariado con PP07H==1 o PP07I==1 (descuento o aporte propio). Independientes = 0.
      "escenario":        como "pp07i" + independientes (CAT_OCUP 1,2) con probabilidad `ind_share` (supuesto explícito).
    Desocupado/inactivo/trabajador familiar: 0. PP07H==9 (NS/NR) cuenta como no registrado."""
    d = df[(df["CH06"] >= EDAD_MIN) & (df["CH06"] <= EDAD_MAX) & (df["ESTADO"].isin([1, 2, 3]))].copy()
    if sexo is not None:
        d = d[d["CH04"] == sexo]
    ocupado = d["ESTADO"] == 1
    asal = ocupado & (d["CAT_OCUP"] == 3)
    indep = ocupado & d["CAT_OCUP"].isin([1, 2])
    reg = np.zeros(len(d))
    desc = (d["PP07H"] == 1)
    if ind_rule == "solo_asalariados":
        reg[asal.values] = desc[asal].astype(float).values
    elif ind_rule in ("pp07i", "escenario"):
        reg[asal.values] = (desc | (d["PP07I"] == 1))[asal].astype(float).values
        if ind_rule == "escenario":
            if ind_share is None or not (0 <= ind_share <= 1):
                raise ValueError("ind_rule='escenario' requiere ind_share en [0,1] (supuesto explícito).")
            reg[indep.values] = float(ind_share)
    else:
        raise ValueError(f"ind_rule desconocida: {ind_rule}")
    d["reg"] = reg
    d["edad"] = d["CH06"].astype(int)
    d["cohorte"] = (d["ANO4"] - d["CH06"]).astype(int)        # error de ±1 año (no se usa la fecha exacta)
    d["w"] = d["PONDERA"].astype(float)
    return d[["cohorte", "edad", "w", "reg"]]


# ================================================================== 4) PSEUDO-PANEL + MODELO
def build_cells(d, min_n=MIN_N_CELDA):
    """Densidad ponderada por (cohorte, edad), agregando todos los trimestres/años en que esa cohorte tuvo esa edad."""
    d = d.assign(wreg=d["w"] * d["reg"])
    g = d.groupby(["cohorte", "edad"]).agg(n=("w", "size"), w=("w", "sum"), wreg=("wreg", "sum")).reset_index()
    g["dens"] = g["wreg"] / g["w"]
    return g[g["n"] >= min_n].reset_index(drop=True)


def _sig(x):
    return 1 / (1 + np.exp(-x))


def fit_age_cohort_logit(cells, iters=100, ridge=1e-3):
    """Logit aditivo: logit d_c(a) = alpha_c + f(a). Supuesto: el perfil por edad f(a) es común a todas las cohortes."""
    coh = np.sort(cells["cohorte"].unique()); eds = np.sort(cells["edad"].unique())
    ci = {c: i for i, c in enumerate(coh)}; ei = {a: i for i, a in enumerate(eds)}
    X = np.zeros((len(cells), len(coh) + len(eds) - 1))
    for r, (c, a) in enumerate(zip(cells["cohorte"], cells["edad"])):
        X[r, ci[c]] = 1.0
        if ei[a] > 0:
            X[r, len(coh) + ei[a] - 1] = 1.0
    y = cells["dens"].values; m = cells["n"].values.astype(float)
    yy = np.clip(y, 0.01, 0.99)
    beta = np.linalg.solve(X.T @ (m[:, None] * X) + ridge * np.eye(X.shape[1]), X.T @ (m * np.log(yy / (1 - yy))))
    for _ in range(iters):
        eta = X @ beta
        mu = np.clip(_sig(eta), 1e-6, 1 - 1e-6)
        W = m * mu * (1 - mu)
        z = eta + (y - mu) / (mu * (1 - mu))
        new = np.linalg.solve(X.T @ (W[:, None] * X) + ridge * np.eye(X.shape[1]), X.T @ (W * z))
        if np.max(np.abs(new - beta)) < 1e-9:
            beta = new; break
        beta = new
    alpha = pd.Series(beta[:len(coh)], index=coh)
    f = pd.Series(np.r_[0.0, beta[len(coh):]], index=eds)
    n_ed = cells.groupby("cohorte")["edad"].nunique().reindex(coh)
    return {"alpha": alpha, "f": f, "n_edades": n_ed, "cells": cells}


def extrapolar_alpha(fit, cohortes, modo="ultimas_k", k=5, ventana_tendencia=15):
    """Efecto de cohorte para cohortes con pocos datos (muy jóvenes, muy viejas o futuras). Supuesto explícito."""
    ok = fit["alpha"][fit["n_edades"] >= MIN_EDADES_COHORTE]
    out = {}
    for c in cohortes:
        if c in ok.index:
            out[c] = ok[c]
        elif c > ok.index.max():
            if modo == "ultimas_k":
                out[c] = ok.iloc[-k:].mean()
            elif modo == "tendencia":
                w = ok.iloc[-ventana_tendencia:]; b = np.polyfit(w.index.values, w.values, 1)
                out[c] = np.polyval(b, c)
            else:
                raise ValueError(modo)
        else:
            out[c] = ok.iloc[:k].mean()
    return pd.Series(out)


def densidad_modelada(fit, alpha_c):
    """Matriz cohorte x edad (18..64) de d_c(a). Edades sin datos usan el perfil f(a) del modelo (interpolado)."""
    f = fit["f"].reindex(EDADES).interpolate(limit_direction="both")
    return pd.DataFrame(_sig(alpha_c.values[:, None] + f.values[None, :]), index=alpha_c.index, columns=EDADES)


# ================================================================== 5) EDAD EFECTIVA DE INGRESO AL EMPLEO REGISTRADO
def edad_ingreso_modelada(dens, umbral=0.5):
    """Por cohorte, a partir de la densidad modelada d_c(a):
        meseta   = máx_a d_c(a)
        F_c(a)   = envolvente creciente de d_c(a) / meseta  (se interpreta como "fracción que ya ingresó" a la edad a)
        edad_umbral = primera edad con F >= umbral (por defecto 50 % de la meseta)
        edad_media  = 18 + Σ_{a=18}^{63} (1 − F_c(a))   (edad media de ingreso si F fuera la distribución de la edad de ingreso)
    ES UNA APROXIMACIÓN: la EPH no observa el primer empleo registrado; supone ingreso monótono sin salidas del empleo registrado."""
    v = dens.values
    meseta = v.max(axis=1)
    F = np.clip(np.maximum.accumulate(v, axis=1) / meseta[:, None], 0, 1)
    edad_umbral = np.array([EDADES[np.argmax(F[i] >= umbral)] for i in range(len(F))], dtype=float)
    edad_media = EDAD_MIN + (1 - F[:, :-1]).sum(axis=1)
    return pd.DataFrame({"meseta_densidad": meseta, "edad_ingreso_umbral50": edad_umbral, "edad_ingreso_media": edad_media},
                        index=dens.index)


def edad_ingreso_observada(fit, dens_modelada, umbral=0.5):
    """Estimación EMPÍRICA (sin modelo) de la edad de ingreso: primera edad OBSERVADA en que la densidad de la cohorte
    llega al `umbral` de su meseta modelada. Sólo para cohortes observadas desde edades tempranas; NaN en el resto."""
    cells = fit["cells"]; out = {}
    meseta = dens_modelada.max(axis=1)
    for c, g in cells.groupby("cohorte"):
        if c not in meseta.index:
            continue
        g = g.sort_values("edad")
        if g["edad"].min() > EDAD_MIN + 2:            # no vimos los años de entrada
            out[c] = np.nan; continue
        hit = g[g["dens"] >= umbral * meseta[c]]
        out[c] = float(hit["edad"].iloc[0]) if len(hit) else np.nan
    return pd.Series(out, name="edad_ingreso_observada").reindex(dens_modelada.index)


# ================================================================== 6) Ā
def supervivencia_gompertz(s65, b=GOMPERTZ_B):
    """S(a) desde el nacimiento para a=18..64 (Gompertz con pendiente b calibrada para que S(65)=s65 por cohorte)."""
    s65 = np.asarray(s65, float)
    a = -np.log(s65) * b / (np.exp(65 * b) - 1)
    return np.exp(-(a[:, None] / b) * (np.exp(b * EDADES[None, :]) - 1))


def abar_por_cohorte(dens, s65):
    """Ā(c) = Σ_{a=18}^{64} S_c(a) · d_c(a)  (incluye a quienes nunca aportan: densidad 0)."""
    S = supervivencia_gompertz(s65.reindex(dens.index).values)
    return pd.Series((S * dens.values).sum(axis=1), index=dens.index, name="A_bar"), S


def pipeline(df_ind, s65, ind_rule=IND_RULE_DEFAULT, ind_share=IND_SHARE_DEFAULT, sexo=None,
             modo_extrap="ultimas_k", cohortes=None):
    """EPH individual -> tabla por cohorte (Ā y edad de ingreso). Igual para datos sintéticos y reales."""
    d = add_flags(df_ind, ind_rule, ind_share, sexo)
    cells = build_cells(d)
    fit = fit_age_cohort_logit(cells)
    cohortes = list(cohortes) if cohortes is not None else list(s65.index)
    alpha_c = extrapolar_alpha(fit, cohortes, modo=modo_extrap)
    dens = densidad_modelada(fit, alpha_c)
    abar, S = abar_por_cohorte(dens, s65)
    ing = edad_ingreso_modelada(dens)
    tabla = pd.DataFrame({"A_bar": abar,
                          "n_edades_obs": fit["n_edades"].reindex(abar.index).fillna(0).astype(int),
                          "S65": s65.reindex(abar.index),
                          "A_max_sin_densidad": S.sum(axis=1)}).join(ing)
    tabla["edad_ingreso_observada"] = edad_ingreso_observada(fit, dens)
    return {"tabla": tabla, "dens": dens, "fit": fit, "cells": cells}


def estimar_abar(df_ind, s65, cohortes=None, modo_extrap="ultimas_k", ind_share_escenario=0.5):
    """Ā por cohorte con las TRES reglas para independientes. Devuelve un DataFrame (una fila por cohorte):
        A_bar_solo_asalariados (cota inferior) | A_bar_pp07i | A_bar_indep50 (escenario) |
        edad de ingreso (modelada / observada) | n_edades_obs | S65"""
    reglas = {"solo_asalariados": dict(ind_rule="solo_asalariados"),
              "pp07i": dict(ind_rule="pp07i"),
              f"indep{int(round(ind_share_escenario * 100))}": dict(ind_rule="escenario", ind_share=ind_share_escenario)}
    out, base = {}, None
    for nombre, kw in reglas.items():
        r = pipeline(df_ind, s65, cohortes=cohortes, modo_extrap=modo_extrap, **kw)
        chequeos(r)
        out[f"A_bar_{nombre}"] = r["tabla"]["A_bar"]
        if base is None:
            base = r["tabla"][["n_edades_obs", "S65", "meseta_densidad", "edad_ingreso_umbral50", "edad_ingreso_media", "edad_ingreso_observada"]]
    return pd.DataFrame(out).join(base)


def chequeos(res):
    t, dens, cells = res["tabla"], res["dens"], res["cells"]
    assert np.all((cells["dens"] >= 0) & (cells["dens"] <= 1)), "densidad observada fuera de [0,1]"
    assert np.all((dens.values >= 0) & (dens.values <= 1)), "densidad modelada fuera de [0,1]"
    assert np.all((t["A_bar"] >= 0) & (t["A_bar"] <= 47)), "A_bar fuera de [0,47]"
    assert np.all(t["A_bar"] <= t["A_max_sin_densidad"] + 1e-9), "A_bar supera Σ S(a) (densidad = 1)"
    assert np.all((t["edad_ingreso_umbral50"] >= EDAD_MIN) & (t["edad_ingreso_umbral50"] <= EDAD_MAX)), "edad de ingreso fuera de rango"
    assert np.all((t["edad_ingreso_media"] >= EDAD_MIN) & (t["edad_ingreso_media"] <= EDAD_MAX)), "edad media de ingreso fuera de rango"
    a2, _ = abar_por_cohorte(dens, (t["S65"] + 0.02).clip(upper=0.999))
    assert np.all(a2.values > t["A_bar"].values - 1e-12), "monotonía en S65 violada"
    return True


# ================================================================== 7) CONEXIÓN CON EL TBP
def tbp_con_abar(d, abar, tau=0.2177, rho=0.403, col_abar="A_bar_solo_asalariados"):
    """TBP_C y TBP_A usando Ā(t) por cohorte (en vez de un valor único).
    `d`: DataFrame del notebook principal con columnas cohorte, N_t, S65, J, E65. `abar`: salida de estimar_abar (o Serie)."""
    a = abar[col_abar] if isinstance(abar, pd.DataFrame) else abar
    x = d.set_index("cohorte").join(a.rename("A_bar_eph"), how="left")
    x["TBP_C_Abar"] = x["N_t"] * x["S65"] * x["A_bar_eph"] * tau / (x["J"] * rho)
    x["TBP_A_Abar"] = x["TBP_C_Abar"] / x["E65"]
    return x.reset_index()


# ================================================================== 8) DATOS SINTÉTICOS Y AUTOTEST
def banner(txt="DATOS SINTÉTICOS — no son Argentina"):
    line = "#" * 78
    print("\n".join([line, "##" + txt.center(74) + "##", line]))


def _f_edad_sint(a):
    return 1.2 * np.exp(-((np.asarray(a) - 40) / 18.0) ** 2) - 1.0


def _alpha_sint(c):
    return -0.2 + 0.008 * (np.asarray(c) - 1970)


def _s65_sint(c):
    return np.clip(0.70 + 0.0035 * (np.asarray(c) - 1950), 0.5, 0.97)


def generar_eph_sintetica(anios=range(2003, 2025), n_por_edad=25, seed=123):
    """EPH de juguete con las mismas columnas que la real. NO son datos de Argentina."""
    rng = np.random.default_rng(seed); filas = []
    for y in anios:
        for q in (1, 2, 3, 4):
            for a in range(14, 71):
                p = _sig(_alpha_sint(y - a) + _f_edad_sint(a) + 0.10 * np.sin((y - 2003) / 3.0))
                reg = rng.random(n_por_edad) < p
                v = rng.random(n_por_edad)
                estado = np.where(reg, 1, np.where(v < 0.45, 1, 3))
                cat = np.where(estado == 1, np.where(reg | (v < 0.30), 3, 2), np.nan)
                filas.append(pd.DataFrame({"ANO4": y, "TRIMESTRE": q, "CH04": rng.integers(1, 3, n_por_edad), "CH06": a,
                                           "ESTADO": estado, "CAT_OCUP": cat,
                                           "PP07H": np.where(cat == 3, np.where(reg, 1, 2), np.nan),
                                           "PP07I": np.where(cat == 2, 2, np.nan),
                                           "PONDERA": rng.integers(300, 900, n_por_edad)}))
    return pd.concat(filas, ignore_index=True)


def autotest(verbose=True):
    """Prueba de punta a punta con datos SINTÉTICOS: pipeline, lector (txt y zip), filtros y conexión con el TBP."""
    if verbose: banner()
    df = generar_eph_sintetica()
    coh = np.arange(1940, 2036); s65 = pd.Series(_s65_sint(coh), index=coh)
    tabla = estimar_abar(df, s65, cohortes=coh)
    # verdad sintética
    dens_true = pd.DataFrame(_sig(_alpha_sint(coh)[:, None] + _f_edad_sint(EDADES)[None, :]), index=coh, columns=EDADES)
    a_true, _ = abar_por_cohorte(dens_true, s65)
    obs = tabla[tabla["n_edades_obs"] >= MIN_EDADES_COHORTE]
    # la regla "solo_asalariados" es la que coincide con cómo se generó el dato sintético (todos los registrados son asalariados)
    mae = (obs["A_bar_solo_asalariados"] - a_true.reindex(obs.index)).abs().mean()
    assert mae < 1.5, f"el pipeline no recupera la verdad sintética (MAE={mae:.2f})"
    assert (tabla["A_bar_solo_asalariados"] <= tabla["A_bar_pp07i"] + 1e-9).all(), "pp07i no puede dar menos que solo_asalariados"
    assert (tabla["A_bar_pp07i"] <= tabla["A_bar_indep50"] + 1e-9).all(), "indep50 no puede dar menos que pp07i"
    assert tabla["edad_ingreso_media"].between(18, 64).all() and tabla["edad_ingreso_umbral50"].between(18, 64).all()
    # lector: txt y zip con el nombre típico de INDEC
    with tempfile.TemporaryDirectory() as tmp:
        sub = df[df["ANO4"].isin([2010, 2011])].copy()
        sub.columns = [c.lower() if c in ("CH04", "CH06", "ESTADO") else c for c in sub.columns]
        p = os.path.join(tmp, "usu_individual_T110.txt"); sub.to_csv(p, sep=";", index=False)
        s1 = find_eph_files([tmp]); assert len(s1) == 1 and len(read_all(s1)) == len(sub)
        zp = os.path.join(tmp, "EPH_usu_1_Trim_2010_txt.zip")
        with zipfile.ZipFile(zp, "w") as z:
            z.write(p, "usu_individual_T110.txt")
        os.remove(p)
        s2 = find_eph_files([tmp]); assert len(s2) == 1 and s2[0]["member"] is not None and len(read_all(s2)) == len(sub)
    assert find_eph_files(["/ruta/que/no/existe"]) == []
    # conexión con el TBP: con Ā(t) constante debe reproducir la fórmula con un valor único
    d = pd.DataFrame({"cohorte": [2000, 2001], "N_t": [700000., 690000.], "S65": [0.88, 0.89], "J": [1.2e7, 1.2e7], "E65": [21., 21.]})
    fijo = pd.Series(30.0, index=[2000, 2001])
    r = tbp_con_abar(d, fijo)
    assert np.allclose(r["TBP_C_Abar"], d["N_t"] * d["S65"] * 30 * 0.2177 / (d["J"] * 0.403))
    if verbose:
        print(f"[sintético] error absoluto medio de Ā: {mae:.3f} años | cohortes observadas: {len(obs)}")
        print("Autotest OK (pipeline, 3 reglas, edad de ingreso, lector txt/zip, conexión con TBP).")
        banner("FIN AUTOTEST — estos números NO son Argentina")
    return tabla

### 12.2 Descarga de las bases (modo prueba por defecto)

La fuente es INDEC (`https://www.indec.gob.ar/ftp/cuadros/menusuperior/eph/`). Mientras `ACEPTO_DESCARGA` sea `False` **no se baja nada**: sólo se consulta el tamaño de cada archivo. Cambiarlo a `True` descarga los que falten a `datos/eph/` (esa carpeta no se sube al repositorio).

In [ ]:
DESTINO_EPH = "datos/eph"
RUTAS_EPH = ["datos/eph", "../datos/eph", "../../datos/eph"]
ACEPTO_DESCARGA = False          # <-- poner True para descargar (35 archivos, ~107 MB)
plan_eph = descargar_eph(DESTINO_EPH, desde=(2017, 2), hasta=(2025, 4), confirmado=ACEPTO_DESCARGA)
display(plan_eph[["anio", "trim", "nombre", "bytes", "existe"]].head(4))
print("... total del plan:", round(plan_eph["bytes"].fillna(0).sum() / 1e6, 1), "MB en", len(plan_eph), "archivos")

### 12.3 Carga y chequeo de las bases

Si no hay archivos en `datos/eph/`, el notebook sigue de largo y avisa que **faltan datos** (no inventa nada).

In [ ]:
srcs_eph = find_eph_files(RUTAS_EPH)
HAY_EPH = len(srcs_eph) > 0
if not HAY_EPH:
    print("=" * 78)
    print("FALTAN DATOS: no hay bases individuales de la EPH en", RUTAS_EPH)
    print("Para correr con datos reales: ACEPTO_DESCARGA = True (celda 12.2) o dejar los .zip de INDEC en datos/eph/.")
    print("No se calculó ningún Ā para Argentina.")
    print("=" * 78)
else:
    df_eph = read_all(srcs_eph)
    print(f"Archivos: {len(srcs_eph)} | filas: {len(df_eph):,} | años: {int(df_eph['ANO4'].min())}-{int(df_eph['ANO4'].max())}")
    resumen_base(df_eph)

### 12.4 $\bar A$ por cohorte y edad efectiva de ingreso

$$\bar A_c=\sum_{a=18}^{64} S_c(a)\; d_c(a)$$

con $S_c(a)$ la supervivencia de la cohorte hasta la edad $a$ (calibrada con $S_{65}$ de cada cohorte) y $d_c(a)$ la densidad de aportes (proporción registrada, incluye ceros). La **edad efectiva de ingreso** se estima de dos maneras: *modelada* (18 + suma de $1-F_c(a)$, con $F_c$ la fracción de la meseta de densidad ya alcanzada) y *observada* (primera edad observada en que la densidad llega al 50 % de su meseta; sólo para cohortes vistas desde edades tempranas). Son aproximaciones: la EPH no observa el primer empleo registrado.

In [ ]:
if HAY_EPH:
    s65_real = d.set_index("cohorte")["S65"]
    abar_eph = estimar_abar(df_eph, s65_real, cohortes=s65_real.index)
    os.makedirs(OUT, exist_ok=True)
    abar_eph.to_csv(f"{OUT}/abar_eph_por_cohorte.csv")
    display(abar_eph.loc[[c for c in (1960, 1970, 1980, 1990, 2000, 2005) if c in abar_eph.index]].round(2))
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    for col in [c for c in abar_eph.columns if c.startswith("A_bar_")]:
        ax[0].plot(abar_eph.index, abar_eph[col], label=col.replace("A_bar_", "regla: "))
    ax[0].axhline(14.2, color="gray", ls=":", label="14,2 (ANSES, sesgado)"); ax[0].axhline(30, color="gray", ls="--", label="30 (requisito legal)")
    ax[0].set_title("Ā por cohorte (EPH)"); ax[0].set_xlabel("cohorte"); ax[0].set_ylabel("años"); ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)
    ax[1].plot(abar_eph.index, abar_eph["edad_ingreso_media"], label="modelada (media)")
    ax[1].plot(abar_eph.index, abar_eph["edad_ingreso_umbral50"], label="modelada (umbral 50 %)")
    ax[1].plot(abar_eph.index, abar_eph["edad_ingreso_observada"], "o", ms=3, label="observada (umbral 50 %)")
    ax[1].set_title("Edad efectiva de ingreso al empleo registrado"); ax[1].set_xlabel("cohorte"); ax[1].legend(fontsize=7); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()
else:
    print("Sin datos EPH: nada que calcular (ver 12.3). La prueba sintética está en 12.6.")

### 12.5 Conexión con el TBP: $\bar A(t)$ por cohorte

Con $\bar A$ distinto por cohorte, $TBP_C(t)=\dfrac{N_t(1-\mu)\,\bar A(t)\,\tau}{J\,\rho}$ **ya no es proporcional** a un único valor: **cambia la forma de la curva**, no sólo el nivel (a diferencia de las bandas de la sección 7).

In [ ]:
if HAY_EPH:
    tb = tbp_con_abar(d, abar_eph, tau=TAU, rho=RHO, col_abar="A_bar_solo_asalariados")
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(d.cohorte, d.TBP_C_legal, label="Ā = 30 (constante)"); ax[0].plot(d.cohorte, d.TBP_C_obs, label="Ā = 14,2 (constante)")
    ax[0].plot(tb.cohorte, tb.TBP_C_Abar, label="Ā(t) EPH (solo asalariados)")
    ax[0].set_title("TBP_C por cohorte"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
    base = tb.loc[tb.cohorte == 1960, "TBP_C_Abar"].iloc[0]
    ax[1].plot(d.cohorte, 100 * d.TBP_C_legal / d.TBP_C_legal[d.cohorte == 1960].iloc[0], label="Ā constante (la forma no cambia)")
    ax[1].plot(tb.cohorte, 100 * tb.TBP_C_Abar / base, label="Ā(t) EPH")
    ax[1].set_title("Índice (cohorte 1960 = 100)"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()
else:
    print("Sin datos EPH: no se calcula TBP con Ā(t).")

### 12.6 Prueba con datos sintéticos (autotest)

Verifica la rutina de punta a punta con una EPH **inventada**: pipeline, tres reglas, edad de ingreso, lector de `.txt` y `.zip` y conexión con el TBP. **No son datos de Argentina.**

In [ ]:
_tabla_sintetica = autotest()

### 12.7 Lectura y límites

* **Qué se puede afirmar hoy:** que la rutina está probada con datos sintéticos y que las bases de la EPH 2017-T2 a 2025-T4 existen y se pueden bajar. **No** se puede afirmar ningún valor de $\bar A$ para Argentina hasta correrla con los microdatos.
* **Cobertura:** la EPH es urbana (aglomerados) y desde 2017 sólo hay 9 años de datos: cada cohorte se observa en pocas edades y el resto se completa con el perfil por edad común a todas (supuesto fuerte; período, edad y cohorte no se separan del todo).
* **`PP07H`:** pregunta a asalariados y mide la semana de referencia, no el año. Un descuento no equivale automáticamente a un año de aporte computable (moratorias, cajas provinciales).
* **Cuentapropistas:** no hay una variable inequívoca de aporte en la base individual; por eso se reportan tres reglas (cota inferior, `PP07I`, escenario). La verdadera estará entre ellas.
* **Edad de ingreso:** es una aproximación (ingreso monótono, sin salidas del empleo registrado); no es la edad del primer empleo.
* **Comparabilidad:** el $\bar A$ de la EPH no es comparable uno a uno con el 14,2 de ANSES (piso de 12 meses y registros desde jul-1994).
* **Verificar con la primera descarga:** nombres de variables y separador del `.txt` (`resumen_base` ayuda a detectar problemas).

## 13. 🆕 ρ y τ por año (ANSES) e integración al TBP_C

Hasta ahora $\rho=0{,}403$ y $\tau=0{,}2177$ eran **constantes**. Esta sección los vuelve **vectores por cohorte**, según el año en que cada cohorte cumple 65 ($t_{65}=\text{cohorte}+65$):

$$TBP_C(t)=\frac{N_t\,(1-\mu)\,\bar A\,\tau_{t+65}}{J\,\rho_{t+65}}$$

* **ρ** = tasa de sustitución **de ANSES** = haber medio SIPA (dic.) / salario imponible medio (nov.). *No* es haber/RIPTE.
* La serie anual 2009-2023 del Anuario sólo existe como **gráfico** (Gráfico 4.3, p. 91). Se **digitalizó** desde los trazos vectoriales del PDF: es una lectura **aproximada**, no un dato oficial exacto.
* **τ**: no se pudo verificar una historia anual de alícuotas con fuentes oficiales; queda **constante (supuesto)** con escenarios.
* Para los años sin dato (antes de 2009, **después de 2023**) hay que elegir una **regla de extrapolación explícita**: es un supuesto, no un dato.

> ⚠️ Con $t_{65}=\text{cohorte}+65$, **sólo las cohortes 1950–1958** (retiro 2015–2023) caen dentro de la serie observada. Las cohortes 1959–2035 (retiro 2024–2100) usan la regla de extrapolación: **la "cola" es supuesto**, no medición.

### 13.1 Módulo (copia de `src/tbp_anses.py`)

Todo el código está en una sola celda para que el Colab sea autosuficiente. El mismo archivo vive en el repositorio (`src/tbp_anses.py`; `python src/tbp_anses.py` corre el autotest con datos **sintéticos**, que nunca se usan en salidas reales).

In [ ]:
# ====== MÓDULO tbp_anses (misma versión que src/tbp_anses.py) ======
# -*- coding: utf-8 -*-
"""
tbp_anses.py — ρ (tasa de sustitución) y τ (tasa de aporte) por año y su asignación por cohorte, para el TBP_C.

    TBP_C(t) = N_t · (1−μ) · Ā · τ / (J · ρ)        [en el código: N_t · S65 · Ā · τ / (J · ρ)]

Funciones:
    1. cargar_serie_anses        -> lector GENÉRICO del Excel "Estadísticas de la Seguridad Social" (si existe)
    2. digitalizar_anuario       -> digitaliza los gráficos 4.3 (p.91) y 3.1 (p.65) del Anuario ANSES 2008-2023 desde el PDF
    3. serie_rho_digitalizada    -> serie anual de ρ 2009-2023 (PDF si está; si no CSV; si no valores embebidos)
    4. serie_tau                 -> τ por año (CONSTANTE por supuesto + escenarios; no se verificó historia oficial)
    5. asignar_rho_tau_por_cohorte -> ρ_v, τ_v por cohorte según anio_65 = cohorte + 65, con extrapolación EXPLÍCITA
    6. tbp_c_dinamico            -> TBP_C y TBP_A con vectores ρ_v, τ_v (y Ā opcional por cohorte)
    7. autotest                  -> prueba con datos SINTÉTICOS rotulados (nunca se usan en salidas reales)

AVISOS METODOLÓGICOS (leer):
  * ρ = tasa de sustitución de ANSES = haber medio SIPA (diciembre) / salario imponible medio (noviembre). NO es haber/RIPTE.
  * El Anuario publica la serie anual 2009-2023 SOLO como gráfico (Gráfico 4.3, p.91). Los valores intermedios son
    "digitalizados del gráfico del Anuario (precisión aprox.)": NO son datos oficiales exactos. Los extremos (2009 y 2023)
    sí están rotulados en el Anuario y se usan tal cual.
  * τ: no se pudo verificar una serie histórica anual con fuentes oficiales; se usa un valor constante (supuesto) y escenarios.
  * Antes de 2009 y después de 2023 NO hay dato: se extrapola con una regla explícita elegida por quien llama y queda rotulada.
  * Los números del autotest son sintéticos y no representan a Argentina.
"""
import os, re, io, glob, unicodedata, warnings
import numpy as np
import pandas as pd

FUENTE_ANUARIO = "Anuario Estadístico ANSES 2008-2023"
ETIQ_DIG = "digitalizado del gráfico del Anuario (precisión aprox.)"
ETIQ_EXT = "etiqueta del gráfico del Anuario (valor publicado)"
URL_RAW = "https://raw.githubusercontent.com/carolinamsfelipe/Tp-Indicadores/main/datos/procesados/"
RUTAS_PROCESADOS = ["../datos/procesados", "datos/procesados", "github_staging/datos/procesados"]
RUTAS_EXCEL = ["../datos/anses", "datos/anses", "../../datos_tbp", "datos_tbp", "../datos_tbp", "."]
ANIOS = list(range(2009, 2024))

# Valores etiquetados del Anuario (p.91 y p.65)
ETIQ_RHO = {2009: 0.371, 2023: 0.403}
ETIQ_RHO_SM = {2009: 0.425, 2023: 0.563}
ETIQ_BENEF = {2009: dict(nm=2481182, m=2310228, tot=4791410), 2023: dict(nm=2217222, m=3766708, tot=5983930)}

# τ: SUPUESTOS (Anuario ANSES p.39: aporte personal 11 % + contribución patronal SIPA 10,77 % [inc. a], 12,35 % [inc. b], 16 % adm. pública)
TAU_ESCENARIOS = {"base": 0.11 + 0.1077, "inc_b": 0.11 + 0.1235, "sector_publico": 0.11 + 0.16}

# Salida de la digitalización (Anuario, trazos vectoriales). Se embebe para que el Colab no dependa del PDF.
# Orden: dic-2009 ... dic-2023. Los extremos de ρ se reemplazan por la etiqueta publicada en serie_rho_digitalizada().
_RHO_DIG = [0.3708, 0.3696, 0.3752, 0.3789, 0.4028, 0.3898, 0.3947, 0.4059, 0.4332, 0.4345, 0.4562, 0.4605, 0.4683, 0.4456, 0.4028]
_RHO_SM_DIG = [0.4245, 0.4298, 0.4283, 0.4506, 0.4823, 0.4739, 0.4941, 0.5217, 0.5766, 0.5822, 0.6145, 0.6132, 0.6266, 0.5959, 0.5630]
_BEN_NM_DIG = [2480830, 2400818, 2290420, 2320781, 2298715, 2265582, 2202136, 2191091, 2207658, 2210407, 2213156, 2196613, 2193840, 2226950, 2215930]
_BEN_TOT_DIG = [4790483, 4876018, 4887063, 4909128, 4914650, 5049862, 5491379, 5665222, 5725943, 5723170, 5731441, 5654178, 5673494, 5764551, 5985322]


# ================================================================== 1) LECTOR GENÉRICO DEL EXCEL DE ANSES
def _norm(x):
    s = unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"\s+", " ", s).strip()


PALABRAS_CLAVE = ("tasa de sustitucion", "haber medio", "salario imponible", "beneficiarios", "moratoria")


def buscar_excel_anses(rutas=None):
    """Busca .xls/.xlsx cuyo nombre contenga 'seguridad social' o 'estadisticas'. Devuelve lista de rutas."""
    out = []
    for r in (rutas or RUTAS_EXCEL):
        for ext in ("*.xls", "*.xlsx"):
            for f in glob.glob(os.path.join(r, ext)):
                n = _norm(os.path.basename(f))
                if "seguridad social" in n or "estadisticas" in n or "anses" in n:
                    out.append(os.path.abspath(f))
    return sorted(set(out))


def cargar_serie_anses(ruta=None, palabras=PALABRAS_CLAVE, rutas=None, max_filas_por_hoja=60):
    """Lector GENÉRICO: no supone estructura. Lista hojas y busca palabras clave en TODAS las celdas.
    Devuelve dict(encontrado, archivo, hojas, hallazgos[DataFrame], mensaje). Si no hay archivo: encontrado=False y avisa.
    `hallazgos`: hoja, fila, columna, palabra, texto_celda, valores_fila (lista cruda de la fila, para inspección humana)."""
    if ruta is None:
        c = buscar_excel_anses(rutas)
        ruta = c[0] if c else None
    if ruta is None or not os.path.exists(ruta):
        msg = ("NO se encontró el Excel de ANSES ('Estadísticas de la Seguridad Social', .xls/.xlsx). ANSES bloquea las descargas "
               "automáticas: bajarlo a mano y dejarlo en datos_tbp/ o github_staging/datos/anses/. Se sigue con la serie digitalizada del Anuario.")
        return dict(encontrado=False, archivo=None, hojas=[], hallazgos=pd.DataFrame(), mensaje=msg)
    try:
        xl = pd.ExcelFile(ruta)
    except Exception as e:  # p.ej. falta xlrd para .xls
        return dict(encontrado=False, archivo=ruta, hojas=[], hallazgos=pd.DataFrame(),
                    mensaje=f"Archivo encontrado pero no se pudo abrir ({type(e).__name__}: {e}). Para .xls instalar xlrd, o guardarlo como .xlsx.")
    pk = [_norm(p) for p in palabras]
    filas = []
    for h in xl.sheet_names:
        try:
            df = xl.parse(h, header=None)
        except Exception:
            continue
        for i in range(len(df)):
            for j in range(df.shape[1]):
                v = df.iat[i, j]
                if isinstance(v, str):
                    nv = _norm(v)
                    for w in pk:
                        if w in nv:
                            filas.append(dict(hoja=h, fila=i, columna=j, palabra=w, texto_celda=v[:120],
                                              valores_fila=[x for x in df.iloc[i].tolist() if pd.notna(x)][:max_filas_por_hoja]))
    return dict(encontrado=True, archivo=ruta, hojas=xl.sheet_names, hallazgos=pd.DataFrame(filas),
                mensaje=f"Archivo: {os.path.basename(ruta)} | {len(xl.sheet_names)} hojas | {len(filas)} celdas con palabras clave. "
                        "Revisar a mano `hallazgos` antes de usar cualquier cifra (no se asume estructura).")


# ================================================================== 2) DIGITALIZACIÓN DEL ANUARIO (requiere pymupdf + PDF)
def _puntos_polilinea(dr, n=15):
    P = [dr["items"][0][1]] + [it[2] for it in dr["items"][:n - 1]]
    return [(q.x, q.y) for q in P]


def digitalizar_anuario(pdf_path, pag_rho=91, pag_ben=65):
    """Digitaliza desde los trazos VECTORIALES del PDF (no OCR):
      Gráfico 4.3 (p.91): dos polilíneas (celeste = SIPA total, azul oscuro = sin moratoria), 15 puntos dic-2009..dic-2023.
      Gráfico 3.1 (p.65): áreas apiladas (azul oscuro = No moratoria, celeste encima = total), 15 puntos.
    Eje y: ajuste lineal sobre las líneas de grilla (los rótulos 0 %,10 %…70 % / 0…6 M). Devuelve (df_rho, df_ben, diagnostico)."""
    import pymupdf
    doc = pymupdf.open(pdf_path)
    # --- p.91
    dr = doc[pag_rho - 1].get_drawings()
    grid = [d for d in dr if d["type"] == "s" and abs(d["rect"].height) < 1e-6 and d["rect"].width > 400]
    grid = sorted(grid, key=lambda d: -d["rect"].y0)           # de abajo hacia arriba: eje 0 %, 10 %, ... 70 %
    ys = [d["rect"].y0 for d in grid]
    assert len(ys) == 8, f"se esperaban 8 líneas horizontales (0-70 %), hay {len(ys)}"
    A = np.polyfit(ys, np.arange(8) / 10.0, 1)
    pol = [d for d in dr if d["type"] == "s" and len(d["items"]) == 14 and d["rect"].height > 5]
    pol = sorted(pol, key=lambda d: -d["rect"].y0)             # celeste (más baja) primero
    assert len(pol) == 2
    sipa = [float(np.polyval(A, y)) for _, y in _puntos_polilinea(pol[0])]
    sm = [float(np.polyval(A, y)) for _, y in _puntos_polilinea(pol[1])]
    # --- p.65
    dr2 = doc[pag_ben - 1].get_drawings()
    g2 = sorted([d for d in dr2 if d["type"] == "s" and abs(d["rect"].height) < 1e-6 and 300 < d["rect"].width < 450 and d["rect"].y0 < 600], key=lambda d: -d["rect"].y0)
    ys2 = [d["rect"].y0 for d in g2]
    assert len(ys2) == 7, f"se esperaban 7 líneas (0-6 M), hay {len(ys2)}"
    B = np.polyfit(ys2, np.arange(7) * 1e6, 1)
    areas = [d for d in dr2 if d["type"] == "f" and len(d["items"]) == 30]
    tot_pts = _puntos_polilinea(min(areas, key=lambda d: d["rect"].y0))
    nm_pts = _puntos_polilinea(max(areas, key=lambda d: d["rect"].y0))
    nm = [float(np.polyval(B, y)) for _, y in nm_pts]
    tot = [float(np.polyval(B, y)) for _, y in tot_pts]
    df_rho = pd.DataFrame(dict(anio=ANIOS, rho_dig=sipa, rho_sm_dig=sm))
    df_ben = pd.DataFrame(dict(anio=ANIOS, benef_no_moratoria_dig=nm, benef_total_dig=tot))
    df_ben["benef_moratoria_dig"] = df_ben.benef_total_dig - df_ben.benef_no_moratoria_dig
    diag = dict(ajuste_eje_rho_max_err=float(np.abs(np.polyval(A, ys) - np.arange(8) / 10).max()),
                ajuste_eje_ben_max_err=float(np.abs(np.polyval(B, ys2) - np.arange(7) * 1e6).max()),
                error_extremos=_error_extremos(df_rho, df_ben))
    return df_rho, df_ben, diag


def _error_extremos(df_rho, df_ben):
    """Compara lo digitalizado con las etiquetas publicadas (en puntos porcentuales / personas)."""
    r = df_rho.set_index("anio"); b = df_ben.set_index("anio")
    out = []
    for a in (2009, 2023):
        out.append(dict(anio=a, serie="ρ SIPA (pp)", digitalizado=100 * r.loc[a, "rho_dig"], etiqueta=100 * ETIQ_RHO[a]))
        out.append(dict(anio=a, serie="ρ sin moratoria (pp)", digitalizado=100 * r.loc[a, "rho_sm_dig"], etiqueta=100 * ETIQ_RHO_SM[a]))
        out.append(dict(anio=a, serie="benef. no moratoria", digitalizado=b.loc[a, "benef_no_moratoria_dig"], etiqueta=ETIQ_BENEF[a]["nm"]))
        out.append(dict(anio=a, serie="benef. total", digitalizado=b.loc[a, "benef_total_dig"], etiqueta=ETIQ_BENEF[a]["tot"]))
    e = pd.DataFrame(out); e["error"] = e.digitalizado - e.etiqueta
    e["error_rel_%"] = 100 * e.error / e.etiqueta
    return e


# ================================================================== 3) SERIE DE ρ
def _leer_csv_candidato(nombre, rutas=None):
    for r in (rutas or RUTAS_PROCESADOS):
        p = os.path.join(r, nombre)
        if os.path.exists(p):
            return pd.read_csv(p)
    try:
        return pd.read_csv(URL_RAW + nombre)
    except Exception:
        return None


def serie_rho_digitalizada(pdf=None, rutas=None, usar_csv=True):
    """Serie anual 2009-2023 de ρ. Orden de preferencia: (a) digitalizar el PDF si `pdf` existe; (b) CSV anses_rho_tau_anual.csv
    (si usar_csv); (c) valores embebidos (resultado de (a), misma versión). En todos los casos los extremos 2009/2023 son la ETIQUETA
    publicada y los intermedios son aproximados. Columnas: anio, rho, rho_sin_moratoria, tipo_dato, fuente, nota."""
    if pdf and os.path.exists(pdf):
        dr, _, _ = digitalizar_anuario(pdf)
        rho, sm = dr.rho_dig.values.copy(), dr.rho_sm_dig.values.copy()
    elif usar_csv and (c := _leer_csv_candidato("anses_rho_tau_anual.csv", rutas)) is not None and {"anio", "rho"} <= set(c.columns):
        c = c[c.anio.between(2009, 2023)]
        return c[[x for x in ("anio", "rho", "rho_sin_moratoria", "tipo_dato", "fuente", "nota") if x in c.columns]].reset_index(drop=True)
    else:
        rho, sm = np.array(_RHO_DIG), np.array(_RHO_SM_DIG)
    tipo, nota = [], []
    for k, a in enumerate(ANIOS):
        if a in ETIQ_RHO:
            rho[k], sm[k] = ETIQ_RHO[a], ETIQ_RHO_SM[a]
            tipo.append(ETIQ_EXT); nota.append(f"valor rotulado en el Gráfico 4.3 / texto p.91 (dic-{a})")
        else:
            tipo.append(ETIQ_DIG); nota.append("lectura de los trazos vectoriales del Gráfico 4.3 (p.91); error típico de los extremos < 0,1 pp")
    return pd.DataFrame(dict(anio=ANIOS, rho=np.round(rho, 4), rho_sin_moratoria=np.round(sm, 4), tipo_dato=tipo,
                             fuente=f"{FUENTE_ANUARIO}, Gráfico 4.3, p.91 (haber medio / salario imponible medio)", nota=nota))


def serie_beneficiarios_digitalizada():
    """Beneficiarios SIPA con/sin moratoria, Gráfico 3.1 (p.65). Extremos = etiquetas; intermedios digitalizados (aprox.)."""
    nm, tot = np.array(_BEN_NM_DIG, float), np.array(_BEN_TOT_DIG, float)
    tipo = []
    for k, a in enumerate(ANIOS):
        if a in ETIQ_BENEF:
            nm[k], tot[k] = ETIQ_BENEF[a]["nm"], ETIQ_BENEF[a]["tot"]; tipo.append(ETIQ_EXT)
        else:
            tipo.append(ETIQ_DIG)
    df = pd.DataFrame(dict(anio=ANIOS, benef_no_moratoria=nm.round(0), benef_moratoria=(tot - nm).round(0), benef_total=tot.round(0), tipo_dato=tipo,
                           fuente=f"{FUENTE_ANUARIO}, Gráfico 3.1, p.65"))
    df["share_moratoria"] = df.benef_moratoria / df.benef_total
    return df


# ================================================================== 4) SERIE DE τ
def serie_tau(escenario="base", anios=None, valor=None):
    """τ por año. NO hay historia oficial verificada: τ es CONSTANTE por SUPUESTO.
    escenario: 'base' (11 % + 10,77 % = 21,77 %), 'inc_b' (11 % + 12,35 % = 23,35 %), 'sector_publico' (11 % + 16 % = 27 %) o `valor` numérico."""
    anios = list(anios) if anios is not None else ANIOS
    if valor is not None:
        t, nom = float(valor), f"valor={valor}"
    else:
        t, nom = TAU_ESCENARIOS[escenario], escenario
    return pd.DataFrame(dict(anio=anios, tau=t, tipo_dato="supuesto (constante)",
                             fuente=f"{FUENTE_ANUARIO}, p.39 (alícuotas vigentes; sin historia anual verificada)",
                             nota=f"escenario {nom}: τ = {t:.4f} constante en todos los años; NO es una serie histórica"))


# ================================================================== 5) ASIGNACIÓN POR COHORTE
def _extrapolar(serie, anio, modo, k=5, constante=None):
    """Valor de la serie en `anio`. Devuelve (valor, tipo) con tipo 'dato' o 'extrapolado:<regla>'."""
    s = serie.dropna()
    if anio in s.index:
        return float(s.loc[anio]), "dato"
    if modo == "ultimo_valor":
        # fuera del rango se usa el valor del extremo más cercano (último año disponible si es posterior; primero si es anterior)
        v = s.iloc[-1] if anio > s.index.max() else s.iloc[0]
    elif modo == "media_ultimos_k":
        v = s.iloc[-k:].mean() if anio > s.index.max() else s.iloc[:k].mean()
    elif modo == "constante":
        if constante is None:
            raise ValueError("extrapolacion='constante' requiere pasar `constante=` (p.ej. RHO)")
        v = float(constante)
    else:
        raise ValueError("extrapolacion debe ser 'ultimo_valor', 'media_ultimos_k' o 'constante'")
    return float(v), f"extrapolado:{modo}"


def asignar_rho_tau_por_cohorte(d, serie_rho, serie_tau_df, edad_retiro=65, extrapolacion="ultimo_valor", k=5,
                                constante_rho=None, constante_tau=None, col_rho="rho", col_tau="tau"):
    """Asigna ρ y τ a cada cohorte según el año en que cumple `edad_retiro`: anio = cohorte + edad_retiro.
    Años sin dato (antes de 2009 o después de 2023) -> regla de `extrapolacion` (EXPLÍCITA, rotulada como supuesto):
      'ultimo_valor' (último año disponible), 'media_ultimos_k' (media de los k últimos años), 'constante' (usa constante_rho / constante_tau).
    NO se usa la media global como relleno silencioso. Devuelve DataFrame: cohorte, anio_ret, rho_v, tau_v, tipo_rho, tipo_tau."""
    sr = serie_rho.set_index("anio")[col_rho]
    st = serie_tau_df.set_index("anio")[col_tau]
    filas = []
    for c in d["cohorte"].values:
        a = int(c) + int(edad_retiro)
        rv, tr = _extrapolar(sr, a, extrapolacion, k, constante_rho)
        modo_t = extrapolacion if (extrapolacion != "constante" or constante_tau is not None) else "ultimo_valor"
        tv, tt = _extrapolar(st, a, modo_t, k, constante_tau)
        filas.append(dict(cohorte=int(c), anio_ret=a, rho_v=rv, tau_v=tv, tipo_rho=tr, tipo_tau=("supuesto" if tt == "dato" else tt)))
    out = pd.DataFrame(filas)
    out.attrs["extrapolacion"] = extrapolacion
    return out


# ================================================================== 6) TBP_C / TBP_A DINÁMICOS
def _vec(x, cohortes, nombre):
    if np.isscalar(x):
        return np.full(len(cohortes), float(x))
    if isinstance(x, pd.Series):
        v = x.reindex(cohortes).values.astype(float)
    elif isinstance(x, pd.DataFrame):
        raise ValueError(f"{nombre} debe ser escalar, Serie indexada por cohorte o vector")
    else:
        v = np.asarray(x, float)
    if len(v) != len(cohortes):
        raise ValueError(f"{nombre}: largo {len(v)} != {len(cohortes)} cohortes")
    return v


def tbp_c_dinamico(d, rho_v, tau_v, abar=None):
    """TBP_C = N_t·S65·Ā·τ/(J·ρ) y TBP_A = TBP_C/E65 con ρ_v y τ_v POR COHORTE (escalar, Serie por cohorte, vector o DataFrame de
    asignar_rho_tau_por_cohorte). abar=None -> dos niveles de referencia: 14,2 (ANSES, obs) y 30 (requisito legal); si se pasa (escalar o por
    cohorte, p.ej. salida de tbp_eph) -> columnas *_din. Los niveles dependen de Ā. Devuelve DataFrame con cohorte + columnas."""
    co = d["cohorte"].values
    if isinstance(rho_v, pd.DataFrame): rho_v = rho_v.set_index("cohorte")["rho_v"]
    if isinstance(tau_v, pd.DataFrame): tau_v = tau_v.set_index("cohorte")["tau_v"]
    r, t = _vec(rho_v, co, "rho_v"), _vec(tau_v, co, "tau_v")
    if np.any(~np.isfinite(r)) or np.any(r <= 0):
        raise ValueError("rho_v debe ser positivo y finito")
    base = d["N_t"].values * d["S65"].values / d["J"].values
    out = pd.DataFrame(dict(cohorte=co, rho_v=r, tau_v=t))
    niveles = {"obs": 14.2, "legal": 30.0} if abar is None else {"Abar": abar}
    for nom, A in niveles.items():
        Av = _vec(A, co, "abar")
        suf = nom if abar is None else "Abar"
        out[f"TBP_C_{suf}_din"] = base * Av * t / r
        out[f"TBP_A_{suf}_din"] = out[f"TBP_C_{suf}_din"].values / d["E65"].values
    return out


# ================================================================== 7) DATOS SINTÉTICOS Y AUTOTEST
def banner(txt="DATOS SINTÉTICOS — no son Argentina"):
    line = "#" * 78
    print("\n".join([line, "##" + txt.center(74) + "##", line]))


def autotest(verbose=True):
    """Prueba con datos SINTÉTICOS (inventados, solo para verificar código). NUNCA usar estos números en salidas reales."""
    if verbose: banner()
    co = np.arange(1950, 2036)
    d = pd.DataFrame(dict(cohorte=co, N_t=500000.0 - 1000 * (co - 1950), S65=0.8, J=5e6 + 5e4 * (co - 1950), E65=18.0))
    sr = pd.DataFrame(dict(anio=range(2009, 2024), rho=np.linspace(0.30, 0.50, 15)))
    st = serie_tau(valor=0.25, anios=range(2009, 2024))
    for modo, esperado in (("ultimo_valor", 0.50), ("media_ultimos_k", np.linspace(0.30, 0.50, 15)[-5:].mean()), ("constante", 0.40)):
        a = asignar_rho_tau_por_cohorte(d, sr, st, extrapolacion=modo, k=5, constante_rho=0.40, constante_tau=0.25)
        fila = a[a.cohorte == 2000].iloc[0]                 # anio_ret 2065 -> fuera de la serie
        assert abs(fila.rho_v - esperado) < 1e-12 and fila.tipo_rho == f"extrapolado:{modo}", (modo, fila)
        f2 = a[a.cohorte == 1950].iloc[0]                   # 2015 -> dato
        assert f2.tipo_rho == "dato" and abs(f2.rho_v - sr.set_index("anio").loc[2015, "rho"]) < 1e-12
    # antes de 2009 (cohorte 1940 -> 2005) y regla primera/primeros k
    d2 = pd.concat([pd.DataFrame(dict(cohorte=[1940], N_t=1.0, S65=1.0, J=1.0, E65=1.0)), d], ignore_index=True)
    a2 = asignar_rho_tau_por_cohorte(d2, sr, st, extrapolacion="ultimo_valor")
    assert abs(a2.iloc[0].rho_v - 0.30) < 1e-12 and a2.iloc[0].tipo_rho.startswith("extrapolado")
    # TBP con ρ,τ constantes == fórmula directa
    r = tbp_c_dinamico(d, 0.4, 0.2, abar=None)
    ref = d.N_t * d.S65 * 30 * 0.2 / (d.J * 0.4)
    assert np.allclose(r.TBP_C_legal_din, ref) and np.allclose(r.TBP_A_legal_din, ref / d.E65)
    # ρ_v mayor => TBP menor; Ā por cohorte
    r_hi = tbp_c_dinamico(d, 0.8, 0.2, abar=20.0)
    assert (r_hi.TBP_C_Abar_din.values < tbp_c_dinamico(d, 0.4, 0.2, abar=20.0).TBP_C_Abar_din.values).all()
    r_v = tbp_c_dinamico(d, pd.Series(0.4, index=co), 0.2, abar=pd.Series(np.linspace(10, 30, len(co)), index=co))
    assert np.isfinite(r_v.TBP_C_Abar_din).all()
    # escenarios de τ
    assert abs(serie_tau("base").tau.iloc[0] - 0.2177) < 1e-12 and abs(serie_tau("sector_publico").tau.iloc[0] - 0.27) < 1e-12
    # extrapolación 'constante' sin constante -> error explícito
    try:
        asignar_rho_tau_por_cohorte(d, sr, st, extrapolacion="constante"); raise AssertionError("debía fallar")
    except ValueError:
        pass
    # lector de Excel: sin archivo avisa; con un xlsx sintético encuentra palabras clave
    assert cargar_serie_anses(ruta="/ruta/que/no/existe.xlsx")["encontrado"] is False
    import tempfile
    p = os.path.join(tempfile.mkdtemp(), "sintetico.xlsx")
    pd.DataFrame([["Tasa de sustitución (SINTÉTICO)", 1, 2], ["Beneficiarios con moratoria", 3, 4]]).to_excel(p, header=False, index=False, sheet_name="Hoja_x")
    res = cargar_serie_anses(ruta=p)
    assert res["encontrado"] and len(res["hallazgos"]) == 3   # 1 por palabra clave presente (la 2ª celda tiene 2)
    if verbose:
        print("autotest OK (datos sintéticos: no son Argentina)")
    return True

### 13.2 Excel de ANSES "Estadísticas de la Seguridad Social" (opcional)

ANSES bloquea las descargas automáticas, así que **no se intenta bajarlo**. Si se deja el `.xls/.xlsx` en `datos_tbp/` o `datos/anses/`, el lector genérico lista las hojas y busca palabras clave (tasa de sustitución, haber medio, salario imponible, beneficiarios, moratoria) **sin suponer estructura**; los hallazgos se revisan a mano.

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
try: display
except NameError: display = print
res_xls = cargar_serie_anses(rutas=RUTAS_EXCEL + [DATA, "datos/anses", "../datos/anses"])
print(res_xls["mensaje"])
if res_xls["encontrado"]:
    print("Hojas:", res_xls["hojas"])
    display(res_xls["hallazgos"].drop(columns=["valores_fila"]).head(40))

### 13.3 ρ por año: serie digitalizada del Gráfico 4.3 (Anuario, p. 91)

**Método.** El gráfico es vectorial: las dos líneas son polilíneas de 15 vértices (dic-2009 … dic-2023) y la grilla son líneas horizontales. Se ajustó una recta *píxel → valor* con las líneas de grilla (0 %, 10 % … 70 %) y se leyó cada vértice. Se validó contra las etiquetas del propio gráfico (37,1 / 40,3 / 42,5 / 56,3 %): el error en los extremos es **< 0,05 pp**. En 2009 y 2023 se usa la **etiqueta publicada**; los 13 años intermedios son **"digitalizado del gráfico del Anuario (precisión aprox.)"** (la precisión real de los vértices intermedios no se puede validar, sólo la de los extremos).

In [ ]:
serie_rho = serie_rho_digitalizada(rutas=["../datos/procesados", "datos/procesados", DATA])
display(serie_rho[["anio", "rho", "rho_sin_moratoria", "tipo_dato"]].assign(tipo_dato=lambda x: x.tipo_dato.str[:48]))
# validación de lo embebido (lectura cruda) contra las etiquetas publicadas
for a in (2009, 2023):
    k = ANIOS.index(a)
    print(f"{a}: ρ crudo digitalizado {100*_RHO_DIG[k]:.2f} % vs etiqueta {100*ETIQ_RHO[a]:.1f} % | sin moratoria {100*_RHO_SM_DIG[k]:.2f} % vs {100*ETIQ_RHO_SM[a]:.1f} %")
ben = serie_beneficiarios_digitalizada()

fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
dig = serie_rho[serie_rho.tipo_dato.str.contains("digitalizado")]; et = serie_rho[~serie_rho.tipo_dato.str.contains("digitalizado")]
ax[0].plot(serie_rho.anio, 100*serie_rho.rho, color="#0072b2", lw=2, label="ρ haber medio SIPA / salario imponible")
ax[0].plot(serie_rho.anio, 100*serie_rho.rho_sin_moratoria, color="#222222", lw=2, label="ρ sin moratoria (contributivo)")
ax[0].plot(dig.anio, 100*dig.rho, "o", ms=4, color="#0072b2", mfc="white", label="intermedios: digitalizados (aprox.)")
ax[0].plot(dig.anio, 100*dig.rho_sin_moratoria, "o", ms=4, color="#222222", mfc="white")
ax[0].plot(et.anio, 100*et.rho, "s", ms=8, color="#d55e00", label="puntos ETIQUETADOS en el Anuario (37,1 / 40,3 / 42,5 / 56,3)")
ax[0].plot(et.anio, 100*et.rho_sin_moratoria, "s", ms=8, color="#d55e00")
ax[0].axhline(40.3, color="gray", ls=":", lw=1); ax[0].text(2009.1, 40.7, "ρ constante actual = 40,3 %", fontsize=8, color="gray")
ax[0].set_title("Tasa de sustitución ρ (ANSES), 2009-2023"); ax[0].set_ylabel("%"); ax[0].set_ylim(30, 70); ax[0].grid(alpha=.3); ax[0].legend(fontsize=7.5, loc="upper left")
ax[1].stackplot(ben.anio, ben.benef_no_moratoria/1e6, ben.benef_moratoria/1e6, colors=["#1f3a5f", "#00aeef"], labels=["no moratoria", "moratoria"], alpha=.9)
ax[1].set_title("Beneficiarios SIPA (millones) — Gráfico 3.1, p. 65 (digitalizado, aprox.)"); ax[1].legend(fontsize=8, loc="upper left"); ax[1].grid(alpha=.3)
fig.suptitle("Sección 13: ρ digitalizado del Anuario ANSES (precisión aprox.; NO dato oficial exacto)", fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.94]); os.makedirs(OUT, exist_ok=True); plt.savefig(f"{OUT}/seccion_13_rho_digitalizado.png", dpi=130); plt.show()

### 13.4 τ por año: constante (supuesto) y escenarios

Verificado (Anuario ANSES, p. 39): aporte personal **11 %** y contribución patronal al SIPA **10,77 %** (actividades inc. a), **12,35 %** (inc. b) y **16 %** (administración pública) ⇒ $\tau$ = 21,77 % / 23,35 % / 27 %. **No se pudo verificar** con fuentes oficiales una serie histórica anual de alícuotas (hubo cambios, p. ej. decreto 814/2001 y reformas posteriores; una fuente secundaria menciona reducciones graduales 2018-2022 por la ley 27.430). Por eso $\tau$ es **constante por supuesto** y se muestran escenarios; no es una serie histórica.

In [ ]:
ESC_TAU = {"base 21,77 % (inc. a)": 0.2177, "23,35 % (inc. b)": TAU_ESCENARIOS["inc_b"], "27 % (sector público)": TAU_ESCENARIOS["sector_publico"]}
print("Escenarios de τ (SUPUESTOS constantes):", {k: round(v, 4) for k, v in ESC_TAU.items()})
serie_tau_base = serie_tau("base", anios=range(2009, 2024))
display(serie_tau_base.head(2))
# guardar la serie anual (ρ digitalizado + τ supuesto)
anses_anual = serie_rho.copy(); anses_anual["tau"] = serie_tau_base.tau.values
anses_anual["tipo_dato"] = "rho: " + serie_rho.tipo_dato + "; tau: supuesto constante (sin serie histórica verificada)"
anses_anual["fuente"] = serie_rho.fuente + "; tau: Anuario ANSES p.39 (alícuotas vigentes)"
anses_anual = anses_anual[["anio", "rho", "rho_sin_moratoria", "tau", "tipo_dato", "fuente", "nota"]]
anses_anual.to_csv(f"{OUT}/anses_rho_tau_anual.csv", index=False); print("guardado:", f"{OUT}/anses_rho_tau_anual.csv")

### 13.5 Asignación por cohorte y regla de extrapolación

Cada cohorte recibe el $\rho$ y el $\tau$ del año en que cumple 65. **Regla elegida** (cambiar `EXTRAP`): `'ultimo_valor'` (ρ de 2023 = 0,403 hacia adelante), `'media_ultimos_k'` (media de 2019-2023 = 0,447; incluye el pico de 2021) o `'constante'`. Ninguna es un pronóstico: son supuestos explícitos.

In [ ]:
EXTRAP = "ultimo_valor"        # <-- 'ultimo_valor' | 'media_ultimos_k' | 'constante'
asig = asignar_rho_tau_por_cohorte(d, serie_rho, serie_tau_base, edad_retiro=65, extrapolacion=EXTRAP, k=5, constante_rho=RHO, constante_tau=TAU)
n_dato = int((asig.tipo_rho == "dato").sum())
print(f"Regla de extrapolación: {EXTRAP} (SUPUESTO).  Cohortes con ρ observado/digitalizado: {n_dato} "
      f"({asig.cohorte[asig.tipo_rho=='dato'].min()}-{asig.cohorte[asig.tipo_rho=='dato'].max()});  extrapoladas: {len(asig)-n_dato} (retiro 2024 en adelante).")
display(asig[asig.cohorte.isin([1950, 1955, 1958, 1959, 1975, 2000, 2035])])

### 13.6 TBP_C con ρ y τ dinámicos vs constantes

**Lectura.** Los **niveles dependen de $\bar A$** (14,2 = ANSES, sesgado; 30 = requisito legal): por eso hay un panel por cada uno. Con la regla `ultimo_valor` el TBP_C dinámico **sólo difiere del constante para las cohortes 1950–1958**, que son las únicas con ρ digitalizado; **toda la cola posterior a 2023 es extrapolación** (supuesto). Se muestran también la regla `media_ultimos_k` y el escenario τ = 23,35 % para ver cuánto pesa el supuesto.

In [ ]:
tau_c = serie_tau("base", anios=range(2009, 2024))
variantes = {
    "constante (ρ=0,403; τ=21,77 %)": None,
    "dinámico: ρ por año + extrapol. 'ultimo_valor'": asignar_rho_tau_por_cohorte(d, serie_rho, tau_c, extrapolacion="ultimo_valor"),
    "dinámico: extrapol. 'media_ultimos_k' (k=5)": asignar_rho_tau_por_cohorte(d, serie_rho, tau_c, extrapolacion="media_ultimos_k", k=5),
    "dinámico 'ultimo_valor' + τ=23,35 % (supuesto)": asignar_rho_tau_por_cohorte(d, serie_rho, serie_tau("inc_b", anios=range(2009, 2024)), extrapolacion="ultimo_valor"),
}
estilos = [("#444444", "-", 2.4), ("#0072b2", "-", 1.8), ("#d55e00", "--", 1.8), ("#009e73", "-.", 1.6)]
res13 = {}
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
for j, (Anom, A) in enumerate((("14,2 (ANSES)", 14.2), ("30 (requisito legal)", 30.0))):
    for (nom, asg), (col, ls, lw) in zip(variantes.items(), estilos):
        r_ = tbp_c_dinamico(d, RHO if asg is None else asg, TAU if asg is None else asg, abar=A)
        res13[(Anom, nom)] = r_
        ax[j].plot(r_.cohorte, r_.TBP_C_Abar_din, color=col, ls=ls, lw=lw, label=nom)
    ax[j].axvspan(1958.5, 2035.5, color="gray", alpha=.08); ax[j].axvline(1958.5, color="gray", ls="--", lw=1)
    ax[j].text(0.97, 0.97, "sombreado: cohortes ≥1959 (retiro ≥2024)\nρ y τ EXTRAPOLADOS (supuesto)", transform=ax[j].transAxes, fontsize=8, color="dimgray", va="top", ha="right")
    ax[j].set_title(f"TBP_C con Ā = {Anom}"); ax[j].set_xlabel("cohorte"); ax[j].set_ylabel("años de aporte por año de jubilado"); ax[j].grid(alpha=.3)
ax[0].legend(fontsize=7.5, loc="lower left")
fig.suptitle("Sección 13: TBP_C con ρ y τ por cohorte vs constantes (los niveles dependen de Ā; la cola post-2023 es extrapolación)", fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.savefig(f"{OUT}/seccion_13_TBP_C_dinamico.png", dpi=130); plt.show()

# chequeo: con ρ, τ constantes la función reproduce el TBP_C del notebook principal
chk = tbp_c_dinamico(d, RHO, TAU, abar=14.2).TBP_C_Abar_din.values
assert np.allclose(chk, d.TBP_C_obs.values), "no coincide con TBP_C_obs del notebook principal"
a_ = variantes["dinámico: ρ por año + extrapol. 'ultimo_valor'"]
out13 = tbp_c_dinamico(d, a_, a_, abar=None).merge(a_[["cohorte", "anio_ret", "tipo_rho", "tipo_tau"]], on="cohorte")
out13.to_csv(f"{OUT}/tbp_c_dinamico_rho_tau.csv", index=False)
cmp_ = pd.DataFrame({"cohorte": d.cohorte, "TBP_C_14,2 const": d.TBP_C_obs, "TBP_C_14,2 din": out13.TBP_C_obs_din,
                     "TBP_C_30 const": d.TBP_C_legal, "TBP_C_30 din": out13.TBP_C_legal_din})
display(cmp_[cmp_.cohorte.isin([1950, 1955, 1958, 1959, 1975, 2000, 2035])].round(3))

### 13.7 Lectura y límites

* **Qué es dato y qué no.** ρ de 2009 y 2023 (y sin moratoria): etiquetas publicadas. ρ de 2010-2022: lectura del gráfico (aprox.; sólo se validaron los extremos). τ: supuesto constante. Todo lo posterior a 2023: extrapolación elegida por quien corre el notebook.
* **Cobertura.** Sólo las cohortes 1950–1958 usan ρ de la serie. Para el resto, la regla de extrapolación **es** el resultado: no hay información sobre ρ futuro (depende de la movilidad jubilatoria, el salario imponible, la proporción de moratorias y la edad/tiempo de aportes).
* **Dirección de lectura.** El ρ del Anuario sube de 37,1 % a 46,8 % (2021) y cae a 40,3 % (2023): usar el ρ del año de retiro de cada cohorte mueve el TBP_C unos puntos porcentuales; el nivel sigue dominado por $\bar A$.
* **Mezcla de universos.** ρ SIPA incluye moratoria; el "sin moratoria" (más alto) corresponde a quienes cumplieron el requisito con aportes. Elegir uno u otro cambia $\rho$ 10-16 pp.
* **τ.** No es una alícuota efectiva: hay actividades, regímenes y reducciones que no se modelan. Ver escenarios 23,35 % y 27 %.
* **No se verificó** el Excel "Estadísticas de la Seguridad Social" (no está en el repositorio): si se agrega, 13.2 lo explora y sus cifras deberían reemplazar a las digitalizadas.

## 14. 🆕 Esperanza de vida cohortal (E_r) y corrección del pico de la cohorte 1956

**El problema.** Hoy $E_r$ es la esperanza de vida restante a los 65 años **de período** del año $t+65$ (ONU WPP 2024). La cohorte 1956 cumple 65 en 2021 (COVID), así que su $E_r$ cae a 16,1 años (mínimo de toda la serie) frente a 17,9 de la cohorte 1950, y eso produce el **pico de 1956** en $TBP_B$ y $TBP_A$. Además, una tabla de período congela la mortalidad de un año y **subestima la longevidad** de las cohortes que todavía van a vivir mejoras.

**La solución.** Seguir a cada cohorte por el calendario: el tramo de edad $[a, a+n)$ se recorre con la probabilidad de supervivencia del año calendario $\lfloor t + a + n/2 \rfloor$ (la misma convención de punto medio que ya usa $1-\mu$ en la sección 1). $E_r^{coh}(t)$ son los años esperados que le quedan a la cohorte a los 65 siguiendo esa trayectoria.

> ⚠️ **Aclaración algebraica (corrección de la persona):** $TBP_C=\dfrac{N(1-\mu)\bar A\tau}{J\rho}$ **no contiene $E_r$**. $E_r$ aparece en $TBP_A=\dfrac{N(1-\mu)\bar A\tau}{J\,E_r\,\rho}=\dfrac{TBP_C}{E_r}$ y en $TBP_B=\dfrac{\bar A\tau}{E_r\,\rho}$. Por eso integrar $E_r$ cohortal **cambia $TBP_A$ y $TBP_B$, y deja $TBP_C$ exactamente igual**; lo que cambia es la lectura de $TBP_C$ frente a la longevidad ($TBP_A=TBP_C/E_r$). Esta sección lo verifica con asserts (14.5 y 14.6).

Esta sección **depende solo de variables ya en memoria** del notebook principal: `d`, `s`, `mort`, `TAU`, `RHO`, `DATA` y `OUT`, y de archivos que carga con rutas candidatas (`../datos/procesados`, `datos/procesados`, `DATA`) y, si no están, la URL *raw* del repositorio. **No modifica `d`.** Escribe tablas y gráficos en `OUT/salidas/` y el CSV en `datos/procesados/`.

### 14.1 Módulo (copia de `src/tbp_mortalidad.py`)

Todo el código está en una sola celda para que el Colab sea autosuficiente. El mismo archivo vive en el repositorio (`src/tbp_mortalidad.py`) y se puede correr desde la terminal: `python src/tbp_mortalidad.py --autotest` o `--comparar`. Funciones: `cargar_tablas`, `supervivencia_cohortal`, `er_cohortal`, `er_periodo`, `er_periodo_integrado`, `grupos_extrapolados`, `comparar_con_periodo`, `recalcular_tbp`, `autotest`.

In [ ]:
# ====== MÓDULO tbp_mortalidad (misma versión que src/tbp_mortalidad.py) ======
_autotest_previo = globals().get('autotest')     # el módulo EPH (sección 12) también define autotest: se conserva
# -*- coding: utf-8 -*-
"""
tbp_mortalidad.py — Esperanza de vida restante a los 65 años POR COHORTE (E_r cohortal) para el TBP.

Proyecto TBP — Índice de Cuna Vacía (Taller de Indicadores).

QUÉ HACE
    Hoy E_r = esperanza de vida a los 65 de PERÍODO del año t+65 (ONU WPP 2024). Eso tiene dos problemas:
      (a) la cohorte 1956 cumple 65 en 2021 (COVID): E_r = 16,1 (mínimo) frente a 17,9 de la cohorte 1950;
      (b) la tabla de período congela la mortalidad de ese año y subestima la longevidad de las cohortes futuras.
    Acá se sigue a la cohorte por el calendario: el tramo de edad [a, a+n) se recorre con la probabilidad de supervivencia
    p_x del año calendario  floor(t + a + n/2)  (misma convención de punto medio que `surv65_cohorte` del notebook
    principal, que da 1−μ). E_r_cohorte(t) = años esperados de vida restantes a los 65 siguiendo esa trayectoria.

MÉTODO (grupos de 5 años + grupo abierto 100+)
    Para la cohorte t, con S(65)=1 y S(a+n) = S(a)·p_x(año del tramo):
        años vividos en el tramo = n·S(a+n) + a_x·[S(a) − S(a+n)]       (a_x de la tabla ONU, en años)
        E_r = Σ_tramos (años vividos) + S(100)·e_100+(año del tramo abierto)
    Es la misma identidad que usa la ONU para L_x y T_x en una tabla de período (L_x = n·l_{x+n} + a_x·d_x), por lo que
    con mortalidad constante en el tiempo reproduce EXACTAMENTE e_65 de período (autotest). Regla alternativa
    `regla="trapecio"` (a_x = n/2) para medir cuánto importa a_x.

SUPUESTO EXPLÍCITO (rotulado): las tablas ONU llegan a 2100. Si el recorrido de la cohorte pasa de 2100 se MANTIENE
    CONSTANTE la tabla de 2100 (sin más mejoras de mortalidad). Afecta a las últimas cohortes; ver `grupos_extrapolados`.

LÍMITES
    * Es un cálculo sobre tablas ONU proyectadas (variante media): no hay "verdad" observada para cohortes futuras.
    * La convención de punto medio asigna UN año calendario a cada grupo de 5 años: suaviza shocks de un año (COVID).
    * Argentina total/varones/mujeres; sin diferenciales por nivel socioeconómico.

Uso mínimo:
    import tbp_mortalidad as m
    T = m.cargar_tablas()                                  # busca ../datos/procesados, datos/procesados; si no, URL raw
    er = m.er_cohortal(range(1950, 2036), edad=65, sexo="Total", tablas=T)
    comp = m.comparar_con_periodo(range(1950, 2036), tablas=T)
    m.autotest()
"""
import os
import urllib.request
from dataclasses import dataclass

import numpy as np
import pandas as pd

ARCHIVO_MORT = "un_wpp2024_argentina_tabla_mortalidad_abreviada.csv"
RAW_BASE_MORT = "https://raw.githubusercontent.com/carolinamsfelipe/Tp-Indicadores/main/datos/procesados"
RUTAS_MORT = ["../datos/procesados", "datos/procesados"]
SEXOS = ("Total", "Male", "Female")
REGLAS = ("ax", "trapecio")


# ================================================================== 1) CARGA
@dataclass
class TablaSexo:
    """Tablas de período abreviadas de un sexo: matrices año × inicio de grupo de edad."""
    px: pd.DataFrame        # prob. de sobrevivir el grupo (condicional a llegar a su inicio)
    ax: pd.DataFrame        # años promedio vividos en el grupo por quienes mueren en él
    ex: pd.DataFrame        # esperanza de vida restante al inicio del grupo (período)
    starts: np.ndarray      # inicio de cada grupo (0, 1, 5, ..., 95, 100)
    spans: np.ndarray       # ancho de cada grupo (el último, abierto, es NaN)
    y0: int
    y1: int


def buscar_archivo(nombre, rutas=None, destino="."):
    """Busca `nombre` en las rutas candidatas; si no está, lo descarga de GitHub (URL raw) a `destino`."""
    for c in list(rutas or []) + RUTAS_MORT:
        p = os.path.join(c, nombre)
        if os.path.exists(p):
            return p
    os.makedirs(destino, exist_ok=True)
    p = os.path.join(destino, nombre)
    urllib.request.urlretrieve(f"{RAW_BASE_MORT}/{nombre}", p)
    return p


def _tabla_desde_df(sub):
    if "Variant" in sub.columns:
        sub = sub[sub["Variant"] == "Medium"]
    sub = sub.copy()
    sub["AgeGrpStart"] = sub["AgeGrpStart"].astype(int)
    px = sub.pivot_table(index="Time", columns="AgeGrpStart", values="px").sort_index()
    ax = sub.pivot_table(index="Time", columns="AgeGrpStart", values="ax").sort_index()
    ex = sub.pivot_table(index="Time", columns="AgeGrpStart", values="ex").sort_index()
    g = (sub[sub.Time == sub.Time.min()].sort_values("AgeGrpStart")[["AgeGrpStart", "AgeGrpSpan"]])
    starts = g["AgeGrpStart"].to_numpy(int)
    spans = np.where(g["AgeGrpSpan"].to_numpy(float) > 0, g["AgeGrpSpan"].to_numpy(float), np.nan)
    assert px.notna().all().all() and ax.notna().all().all() and ex.notna().all().all(), "tabla con huecos"
    assert np.isnan(spans[-1]) and not np.isnan(spans[:-1]).any(), "se espera un único grupo abierto, el último"
    return TablaSexo(px, ax, ex, starts, spans, int(px.index.min()), int(px.index.max()))


def cargar_tablas(rutas=None, archivo=ARCHIVO_MORT, df=None, destino="."):
    """Lee la tabla de mortalidad abreviada de la ONU y devuelve {'Total','Male','Female'} -> TablaSexo.

    `df` permite pasar una tabla ya cargada (con la misma estructura de columnas)."""
    if df is None:
        df = pd.read_csv(buscar_archivo(archivo, rutas, destino), low_memory=False)
    out = {}
    for sx in SEXOS:
        sub = df[df["Sex"] == sx]
        if len(sub):
            out[sx] = _tabla_desde_df(sub)
    if not out:
        raise ValueError("la tabla no tiene ninguna de las columnas Sex esperadas: " + str(SEXOS))
    return out


# ================================================================== 2) SUPERVIVENCIA Y E_r COHORTAL
def _anio_tramo(t, a, ancho, tab):
    """Año calendario (punto medio, igual que surv65_cohorte) con que la cohorte t recorre el tramo [a, a+ancho)."""
    y = int(np.floor(t + a + ancho / 2.0))
    if y < tab.y0:
        raise ValueError(f"cohorte {t}: el tramo {a}+ cae en {y}, antes de la primera tabla ({tab.y0})")
    return min(y, tab.y1), y > tab.y1          # (año de tabla usado, ¿extrapolado con la tabla 2100 constante?)


def _tab(tablas, sexo):
    tablas = tablas if tablas is not None else cargar_tablas()
    if sexo not in tablas:
        raise ValueError(f"sexo {sexo!r} no disponible: {list(tablas)}")
    return tablas[sexo]


def supervivencia_cohortal(cohortes, sexo="Total", tablas=None):
    """S(a): probabilidad de que la cohorte nacida en t llegue viva a la edad a (inicio de cada grupo de edad).

    Devuelve DataFrame (índice = cohorte, columnas = 0, 1, 5, ..., 95, 100). S(0)=1 y, para cada grupo cerrado,
    S(a+n) = S(a)·p_x[año calendario floor(t+a+n/2), grupo a]. Para años > 2100 se usa la tabla de 2100 (supuesto)."""
    tab = _tab(tablas, sexo)
    cohortes = [int(c) for c in cohortes]
    filas = []
    for t in cohortes:
        S = [1.0]
        for k in range(len(tab.starts) - 1):
            y, _ = _anio_tramo(t, tab.starts[k], tab.spans[k], tab)
            S.append(S[-1] * tab.px.at[y, tab.starts[k]])
        filas.append(S)
    return pd.DataFrame(filas, index=pd.Index(cohortes, name="cohorte"), columns=tab.starts)


def _k_edad(tab, edad):
    if int(edad) not in set(tab.starts.tolist()):
        raise ValueError(f"edad {edad} no es el inicio de un grupo de la tabla abreviada: {tab.starts.tolist()}")
    return int(np.where(tab.starts == int(edad))[0][0])


def _er_una(t, k0, tab, regla):
    S, total = 1.0, 0.0
    for k in range(k0, len(tab.starts) - 1):
        a, n = tab.starts[k], tab.spans[k]
        y, _ = _anio_tramo(t, a, n, tab)
        Snext = S * tab.px.at[y, a]
        axk = tab.ax.at[y, a] if regla == "ax" else n / 2.0
        total += n * Snext + axk * (S - Snext)          # n·l_{x+n} + a_x·d_x, con l normalizado a S(edad)=1
        S = Snext
    a_op = tab.starts[-1]                                # grupo abierto 100+
    y_ref, _ = _anio_tramo(t, a_op, 0.0, tab)
    e_ref = tab.ex.at[y_ref, a_op]
    y, _ = _anio_tramo(t, a_op, e_ref, tab)              # punto medio con la duración esperada del grupo abierto
    return total + S * tab.ex.at[y, a_op]


def er_cohortal(cohortes, edad=65, sexo="Total", tablas=None, regla="ax"):
    """Años esperados de vida restantes a la edad `edad` (65) para cada cohorte, siguiendo la trayectoria calendario.

    `regla`: "ax" (a_x de la tabla ONU; reproduce la tabla de período si la mortalidad es constante) o "trapecio"
    (a_x = n/2, para medir la sensibilidad). Devuelve Series indexada por cohorte."""
    if regla not in REGLAS:
        raise ValueError(f"regla debe ser una de {REGLAS}")
    tab = _tab(tablas, sexo)
    k0 = _k_edad(tab, edad)
    cohortes = [int(c) for c in cohortes]
    return pd.Series([_er_una(t, k0, tab, regla) for t in cohortes],
                     index=pd.Index(cohortes, name="cohorte"), name=f"er_cohortal_{sexo.lower()}")


def er_periodo(cohortes, edad=65, sexo="Total", tablas=None):
    """E_r de PERÍODO: e_x de la tabla del año t+edad (lo que usa hoy el notebook, columna un_e65)."""
    tab = _tab(tablas, sexo)
    _k_edad(tab, edad)
    vals = [tab.ex.at[int(t) + int(edad), edad] if tab.y0 <= int(t) + int(edad) <= tab.y1 else np.nan for t in cohortes]
    return pd.Series(vals, index=pd.Index([int(c) for c in cohortes], name="cohorte"), name=f"er_periodo_{sexo.lower()}")


def er_periodo_integrado(anios, edad=65, sexo="Total", tablas=None, regla="ax"):
    """Reconstruye e_edad de PERÍODO integrando p_x y a_x de UNA sola tabla (la del año dado), con la misma regla que
    `er_cohortal`. Sirve para probar la integración: con regla="ax" debe coincidir con la columna `ex` de la ONU
    (salvo el redondeo de la tabla); con "trapecio" muestra cuánto importa a_x."""
    tab = _tab(tablas, sexo)
    k0 = _k_edad(tab, edad)
    out = []
    for y in anios:
        y = int(y)
        S, total = 1.0, 0.0
        for k in range(k0, len(tab.starts) - 1):
            a, n = tab.starts[k], tab.spans[k]
            Sn = S * tab.px.at[y, a]
            total += n * Sn + (tab.ax.at[y, a] if regla == "ax" else n / 2.0) * (S - Sn)
            S = Sn
        out.append(total + S * tab.ex.at[y, tab.starts[-1]])
    return pd.Series(out, index=pd.Index([int(a) for a in anios], name="anio"), name=f"e{edad}_integrado_{regla}")


def grupos_extrapolados(cohortes, edad=65, sexo="Total", tablas=None):
    """Cuántos tramos del recorrido de la cohorte (desde `edad`, incluido el abierto) usan la tabla 2100 constante."""
    tab = _tab(tablas, sexo)
    k0 = _k_edad(tab, edad)
    out = []
    for t in cohortes:
        n_ext = 0
        for k in range(k0, len(tab.starts) - 1):
            n_ext += int(_anio_tramo(int(t), tab.starts[k], tab.spans[k], tab)[1])
        n_ext += int(_anio_tramo(int(t), tab.starts[-1], 2.3, tab)[1])
        out.append(n_ext)
    return pd.Series(out, index=pd.Index([int(c) for c in cohortes], name="cohorte"), name="grupos_extrapolados")


def comparar_con_periodo(cohortes, edad=65, sexos=SEXOS, tablas=None, regla="ax"):
    """Tabla ancha por cohorte: E_r de período y cohortal por sexo, diferencia y tramos extrapolados."""
    cohortes = [int(c) for c in cohortes]
    tablas = tablas if tablas is not None else cargar_tablas()
    out = pd.DataFrame({"cohorte": cohortes}).set_index("cohorte")
    for sx in sexos:
        suf = sx.lower()
        per = er_periodo(cohortes, edad, sx, tablas)
        coh = er_cohortal(cohortes, edad, sx, tablas, regla)
        out[f"er_periodo_{suf}"] = per
        out[f"er_cohortal_{suf}"] = coh
        out[f"dif_{suf}"] = coh - per
    out["grupos_extrapolados"] = grupos_extrapolados(cohortes, edad, sexos[0], tablas)
    return out.reset_index()


# ================================================================== 3) TBP CON E_r COHORTAL
def recalcular_tbp(d, er, A, tau, rho, nombre="cohortal"):
    """Recalcula TBP con un E_r alternativo. NOTA: TBP_C = N(1−μ)Āτ/(Jρ) NO contiene E_r y no cambia;
    cambian TBP_A = TBP_C/E_r y TBP_B = Āτ/(E_r ρ). `d` necesita cohorte, N_t, S65, J; `er` es Series por cohorte."""
    e = d["cohorte"].map(er).to_numpy(float)
    out = pd.DataFrame({"cohorte": d["cohorte"].to_numpy()})
    out["TBP_C"] = d["N_t"].to_numpy() * d["S65"].to_numpy() * A * tau / (d["J"].to_numpy() * rho)
    out[f"TBP_A_{nombre}"] = out["TBP_C"] / e
    out[f"TBP_B_{nombre}"] = A * tau / (e * rho)
    return out


# ================================================================== 4) AUTOTEST CON DATOS SINTÉTICOS
def _banner(txt):
    print("=" * 78 + "\n" + txt + "\n" + "=" * 78)


def tablas_sinteticas(mejora=0.0, y0=1950, y1=2100, A=4.3e-5, B=0.09, factor_sexo=(1.0, 1.3, 0.8)):
    """Tablas abreviadas SINTÉTICAS (Gompertz: h(x) = A·e^{Bx}·e^{−mejora·(año−y0)}); NO son Argentina.

    L_x se integra numéricamente (grilla fina), por lo que a_x sale exacto. Estructura idéntica a `cargar_tablas`."""
    starts = np.array([0, 1] + list(range(5, 100, 5)) + [100])
    spans = np.array([1.0, 4.0] + [5.0] * 19 + [np.nan])
    out = {}
    for sx, fs in zip(SEXOS, factor_sexo):
        años = np.arange(y0, y1 + 1)
        px = pd.DataFrame(index=años, columns=starts, dtype=float)
        ax = px.copy(); ex = px.copy()
        for y in años:
            Ay = A * fs * np.exp(-mejora * (y - y0))
            S = lambda x: np.exp(-(Ay / B) * (np.exp(B * np.asarray(x)) - 1.0))
            for k, (a, n) in enumerate(zip(starts, spans)):
                if np.isnan(n):
                    xs = np.linspace(a, 250, 12501)
                    L = np.trapezoid(S(xs), xs) / S(a)
                    ex.at[y, a] = L; ax.at[y, a] = L; px.at[y, a] = 0.0
                else:
                    xs = np.linspace(a, a + n, 401)
                    La = np.trapezoid(S(xs), xs)                                 # ∫ S dx sobre el tramo
                    p = S(a + n) / S(a)
                    px.at[y, a] = p
                    ax.at[y, a] = (La - n * S(a + n)) / (S(a) - S(a + n))        # años vividos por los que mueren en el tramo
            # e_x al inicio de cada grupo: ∫_x^∞ S / S(x)
            for a in starts:
                xs = np.linspace(a, 250, 12501)
                ex.at[y, a] = np.trapezoid(S(xs), xs) / S(a)
        out[sx] = TablaSexo(px, ax, ex, starts, spans, y0, y1)
    return out


def _er_exacto_continuo(t, edad, mejora, y0=1950, A=4.3e-5, B=0.09, fs=1.0):
    """E_r exacto (integral numérica) de la cohorte t con riesgo h(x, año) = A·e^{Bx}·e^{−mejora·(año−y0)}, año = t+x.
    Con tope de la mejora en 2100 (tabla constante después), igual que el supuesto del módulo."""
    xs = np.linspace(0, 250, 250001)
    año = t + xs
    f = np.exp(-mejora * (np.minimum(año, 2100.0) - y0))
    h = A * fs * np.exp(B * xs) * f
    H = np.concatenate([[0.0], np.cumsum((h[1:] + h[:-1]) / 2 * np.diff(xs))])
    S = np.exp(-H)
    i = int(round(edad / (xs[1] - xs[0])))
    return np.trapezoid(S[i:], xs[i:]) / S[i], S[i]


def autotest(verbose=True):
    """Pruebas con tablas SINTÉTICAS (rotuladas): (i) mortalidad constante => cohortal = período; (ii) integración vs
    valor exacto; (iii) mejora anual => cohortal ≥ período y cerca del exacto; (iv) tabla 2100 constante; (v) errores."""
    if verbose:
        _banner("AUTOTEST CON TABLAS SINTÉTICAS (Gompertz) — NO son datos de Argentina")
    cohs = list(range(1950, 2036))

    # (i) mortalidad constante en el tiempo: cohortal == período (exacto) para los 3 sexos
    T0 = tablas_sinteticas(mejora=0.0)
    for sx in SEXOS:
        c = er_cohortal(cohs, 65, sx, T0); p = er_periodo(cohs, 65, sx, T0)
        assert np.allclose(c.values, p.values, atol=1e-9), (sx, float((c - p).abs().max()))
    if verbose:
        print("OK (i)  mortalidad constante: E_r cohortal = E_r de período en las 86 cohortes y 3 sexos (tol 1e-9)")

    # (ii) la regla con a_x reproduce la integral exacta; el trapecio (a_x = n/2) se aparta un poco
    e_ex, _ = _er_exacto_continuo(1990, 65, 0.0)
    e_ax = er_cohortal([1990], 65, "Total", T0, "ax").iloc[0]
    e_tr = er_cohortal([1990], 65, "Total", T0, "trapecio").iloc[0]
    assert abs(e_ax - e_ex) < 1e-3, (e_ax, e_ex)
    assert abs(e_tr - e_ex) < 0.05, (e_tr, e_ex)
    if verbose:
        print(f"OK (ii) integración: exacto {e_ex:.4f} | regla a_x {e_ax:.4f} (error {e_ax - e_ex:+.5f}) | trapecio {e_tr:.4f} (error {e_tr - e_ex:+.5f}) años")

    # (iii) mejora anual de la mortalidad (1 %/año): cohortal > período y cercano a la integral exacta con tabla 2100 constante
    m = 0.01
    T1 = tablas_sinteticas(mejora=m)
    c = er_cohortal(cohs, 65, "Total", T1); p = er_periodo(cohs, 65, "Total", T1)
    assert (c >= p - 1e-6).all(), c[c < p - 1e-6]
    errs = []
    for t in (1950, 1970, 2000, 2020):
        ex_, _ = _er_exacto_continuo(t, 65, m)
        errs.append(c[t] - ex_)
    assert max(abs(e) for e in errs) < 0.15, errs
    S_ap = supervivencia_cohortal([1990, 2000], "Total", T1)[65]
    for t in (1990, 2000):
        _, s_ex = _er_exacto_continuo(t, 65, m)
        assert abs(S_ap[t] - s_ex) < 2e-3, (t, S_ap[t], s_ex)
    if verbose:
        print(f"OK (iii) mejora {m:.0%}/año: E_r cohortal ≥ período en todas las cohortes (mín. dif. {(c - p).min():+.3f}, máx. {(c - p).max():+.3f}); "
              f"error vs integral exacta: {', '.join(f'{e:+.3f}' for e in errs)} años; S65 cohortal ≈ exacta (<2e-3)")

    # (iv) supuesto explícito: pasado 2100 se mantiene la tabla 2100 => la cohorte 2035 (cumple 65 en 2100) = período 2100
    assert abs(c[2035] - p[2035]) < 1e-6
    g = grupos_extrapolados([1950, 2000, 2035], 65, "Total", T1)
    assert g[1950] == 0 and g[2000] == 1 and g[2035] == 8, g.to_dict()     # 2000: solo el abierto 100+ (año 2101); 2035: 7 tramos (65-99) + el abierto
    if verbose:
        print(f"OK (iv) tabla 2100 constante: cohorte 2035 cohortal = período; tramos extrapolados 1950/2000/2035 = {g.tolist()}")

    # (v) errores esperados + identidad TBP_A = TBP_C / E_r + TBP_C sin E_r
    for malo in (lambda: er_cohortal([2000], 66, "Total", T0), lambda: er_cohortal([2000], 65, "Otro", T0),
                 lambda: er_cohortal([2000], 65, "Total", T0, "foo")):
        try:
            malo(); raise AssertionError("debía fallar")
        except ValueError:
            pass
    dd = pd.DataFrame({"cohorte": [2000, 2001], "N_t": [7e5, 6.9e5], "S65": [0.88, 0.89], "J": [1.2e7, 1.2e7]})
    r1 = recalcular_tbp(dd, pd.Series({2000: 20.0, 2001: 21.0}), 30.0, 0.2177, 0.403)
    r2 = recalcular_tbp(dd, pd.Series({2000: 25.0, 2001: 26.0}), 30.0, 0.2177, 0.403)
    assert np.allclose(r1.TBP_C, r2.TBP_C)                                             # TBP_C no depende de E_r
    assert np.allclose(r1.TBP_A_cohortal, r1.TBP_C / np.array([20.0, 21.0]))
    assert np.allclose(r1.TBP_B_cohortal, 30 * 0.2177 / (np.array([20.0, 21.0]) * 0.403))
    if verbose:
        print("OK (v)  validación de entradas; TBP_C idéntico con distinto E_r; TBP_A = TBP_C/E_r; TBP_B = Āτ/(E_r ρ)")
        _banner("FIN AUTOTEST — estos números NO son Argentina")
    return True

autotest_mortalidad = autotest
if _autotest_previo is not None:
    autotest = _autotest_previo

### 14.2 Método: supervivencia cohortal e integración

Para la cohorte nacida en $t$, con las tablas abreviadas de período de la ONU (grupos $0, 1\!-\!4, 5\!-\!9,\dots,95\!-\!99$ y el grupo abierto $100+$):

$$S(a+n)=S(a)\cdot p_x\big[\text{año}=\lfloor t+a+n/2\rfloor,\ \text{grupo }a\big],\qquad S(0)=1.$$

Con la misma regla $S(65)$ reproduce $1-\mu$ de la columna `S65` de `d` (se verifica en 14.5). Para $E_r$ se normaliza $S(65)=1$ y se integra:

$$E_r^{coh}(t)=\sum_{a=65}^{95}\Big[n\,S(a+n)+{}_na_a\,\big(S(a)-S(a+n)\big)\Big]+S(100)\,e_{100+}\ ,$$

donde ${}_na_a$ es la columna `ax` de la ONU (años que viven, en promedio, quienes mueren dentro del tramo) y $e_{100+}$ la esperanza del grupo abierto del año calendario correspondiente.

**Por qué esta regla.** Es la misma identidad con la que la ONU construye $L_x=n\,l_{x+n}+{}_na_x\,d_x$ y $T_x$ en una tabla de período. Entonces, si la mortalidad no cambiara en el tiempo, la integración cohortal **es** la de período (prueba (i) del autotest, exacta). Con $a_x=n/2$ (trapecio) el error es de centésimas de año; la comparación se muestra abajo con datos reales y con una tabla sintética con solución exacta.

**Supuesto explícito (rotulado):** las tablas llegan a **2100**. Cuando un tramo de la trayectoria cae después de 2100 se **mantiene constante la tabla de 2100** (sin más mejoras de mortalidad). La columna `grupos_extrapolados` cuenta cuántos tramos usan ese supuesto (de 8 posibles: siete de 5 años entre 65 y 99 y el grupo abierto).

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
try: display
except NameError: display = print

SAL14 = os.path.join(OUT, "salidas"); os.makedirs(SAL14, exist_ok=True)
TAB14 = cargar_tablas(rutas=["../datos/procesados", "datos/procesados", DATA], destino=OUT)
print("Tablas ONU cargadas:", {k: f"{v.y0}-{v.y1}, {len(v.starts)} grupos" for k, v in TAB14.items()})

# La tabla "Total" cargada aquí debe ser la misma que `mort` del notebook principal
_m = mort.pivot_table(index="Time", columns="AgeGrpStart", values="px")
assert np.allclose(_m.values, TAB14["Total"].px.values) and list(_m.columns) == list(TAB14["Total"].starts)
print("OK: tabla Total = `mort` del notebook principal")

### 14.3 Prueba de la aproximación de integración (datos reales y sintéticos)

**(a) Datos reales:** integrar $p_x$ y $a_x$ de **una sola** tabla de período debe devolver su $e_{65}$ publicado. **(b) Datos sintéticos** (rotulados): mortalidad Gompertz con solución exacta. Las pruebas (i)–(v) completas están en `autotest()`.

In [ ]:
anios14 = range(1950, 2101)
print("(a) e65 reconstruido integrando UNA tabla de período vs `ex` publicado por la ONU (151 años)")
for sx in SEXOS:
    ref = TAB14[sx].ex[65]
    e_ax  = er_periodo_integrado(anios14, 65, sx, TAB14, "ax")
    e_tr  = er_periodo_integrado(anios14, 65, sx, TAB14, "trapecio")
    assert (e_ax - ref).abs().max() < 1e-3, sx            # sólo el redondeo de la tabla
    print(f"  {sx:6s}: regla a_x -> error máx. {np.abs(e_ax - ref).max():.5f} años | trapecio (a_x=n/2) -> error máx. {np.abs(e_tr - ref).max():.3f}, medio {np.mean(e_tr - ref):+.3f} años")

# peso del grupo abierto 100+ en E_r (cohorte 2000, Total)
_S = supervivencia_cohortal([2000], "Total", TAB14)
print(f"  Peso del grupo abierto 100+: S(100)/S(65) = {(_S[100] / _S[65]).iloc[0]:.4f} (contribuye ~{(_S[100] / _S[65]).iloc[0] * TAB14['Total'].ex.loc[2101 if 2101 <= TAB14['Total'].y1 else 2100, 100]:.3f} años de ~{er_cohortal([2000], 65, 'Total', TAB14).iloc[0]:.1f})")

print("\n(b) autotest con tablas SINTÉTICAS")
autotest_mortalidad()

### 14.4 Cálculo: E_r de período vs cohortal, por sexo

`comparar_con_periodo` devuelve, para cada cohorte, $E_r$ de período (tabla del año $t+65$, lo que usa hoy `d.E65`) y cohortal, para Total, varones y mujeres.

In [ ]:
COH14 = d["cohorte"].astype(int).tolist()
cmp14 = comparar_con_periodo(COH14, 65, tablas=TAB14).set_index("cohorte")
display(cmp14.loc[[1950, 1953, 1954, 1955, 1956, 1957, 1958, 1975, 2000, 2024, 2033, 2035],
                  ["er_periodo_total", "er_cohortal_total", "dif_total", "er_periodo_male", "er_cohortal_male",
                   "er_periodo_female", "er_cohortal_female", "grupos_extrapolados"]].round(3))

### 14.5 Validaciones (asserts)

1. **(i) Mortalidad constante ⇒ cohortal = período:** test sintético del autotest (14.3).
2. **(ii) Diferencias y pico de 1956:** media y máximo de $E_r^{coh}-E_r^{per}$ en las 86 cohortes (1950–2035); la cohorte 1956 deja de ser el mínimo y la mayor caída interanual se reduce a menos de la mitad.
3. **(iii) $S_{65}$ cohortal recalculado con la función del módulo = columna `S65` de `d`.** Además, el $E_r$ de período calculado = `d.E65`.
4. **(iv) $E_r^{coh}\ge E_r^{per}$** para cohortes con mortalidad decreciente: se verifica y se **reportan las excepciones** (tolerancia 0,001 años, por el redondeo de las tablas).

In [ ]:
dd14 = d.set_index("cohorte")
tot = cmp14["er_periodo_total"], cmp14["er_cohortal_total"]
per14, coh14 = tot

# (iii) S65 cohortal = columna S65 de d ; E_r de período = d.E65
S65_14 = supervivencia_cohortal(COH14, "Total", TAB14)[65]
assert np.allclose(S65_14.values, dd14.loc[COH14, "S65"].values, atol=1e-10), np.abs(S65_14.values - dd14.loc[COH14, "S65"].values).max()
assert np.allclose(per14.values, dd14.loc[COH14, "E65"].values, atol=0.005), np.abs(per14.values - dd14.loc[COH14, "E65"].values).max()
print(f"OK (iii) S65 cohortal = d.S65 (dif. máx. {np.abs(S65_14.values - dd14.loc[COH14, 'S65'].values).max():.1e}) | E_r período = d.E65 (dif. máx. {np.abs(per14.values - dd14.loc[COH14, 'E65'].values).max():.1e})")

# (ii) diferencias y pico de la cohorte 1956
dif14 = coh14 - per14
print(f"(ii) E_r cohortal - período (Total, 86 cohortes): media {dif14.mean():+.3f} | máx. {dif14.max():+.3f} (cohorte {int(dif14.idxmax())}) | mín. {dif14.min():+.4f} (cohorte {int(dif14.idxmin())})")
for sx in ("male", "female"):
    dx = cmp14[f"dif_{sx}"]; print(f"     {sx:6s}: media {dx.mean():+.3f} | máx. {dx.max():+.3f} (cohorte {int(dx.idxmax())})")
print(f"     sin las cohortes con tramos extrapolados (>2100): media {dif14[cmp14.grupos_extrapolados == 0].mean():+.3f} sobre {(cmp14.grupos_extrapolados == 0).sum()} cohortes")
print(f"     Período : mínimo = cohorte {int(per14.idxmin())} ({per14.min():.2f}) | E_r(1956) - E_r(1950) = {per14[1956] - per14[1950]:+.2f} | mayor caída interanual {per14.diff().min():+.2f} (cohorte {int(per14.diff().idxmin())})")
print(f"     Cohortal: mínimo = cohorte {int(coh14.idxmin())} ({coh14.min():.2f}) | E_r(1956) - E_r(1950) = {coh14[1956] - coh14[1950]:+.2f} | mayor caída interanual {coh14.diff().min():+.2f} (cohorte {int(coh14.diff().idxmin())})")
assert int(per14.idxmin()) == 1956 and int(coh14.idxmin()) != 1956
assert coh14[1956] > coh14[1950] and per14[1956] < per14[1950]
assert coh14.diff().min() > 0.5 * per14.diff().min()               # la mayor caída (negativa) se reduce a menos de la mitad
assert (coh14.loc[1954:].diff().dropna() > -1e-6).all()            # desde 1954 la serie cohortal no cae (queda constante en 2033-2035 por el supuesto 2100)
print("OK (ii) el pico de 1956 desaparece: ya no es el mínimo, y desde la cohorte 1954 E_r cohortal no cae (crece, y se aplana en 2033-2035 por el supuesto 2100)")
res = coh14.diff().loc[1950:1955]
print("     Residuo honesto: antes de 1955 queda una ondulación menor (las cohortes 1953-1954 reciben el COVID en el tramo 65-69):",
      ", ".join(f"{int(k)}: {v:+.2f}" for k, v in res.items() if k > 1950))

# (iv) cohortal >= período
exc = {}
for sx in ("total", "male", "female"):
    dx = cmp14[f"dif_{sx}"]; exc[sx] = dx[dx < -1e-3]
print("(iv) E_r cohortal >= período (tol. 0,001):", {k: (v.round(3).to_dict() if len(v) else "sin excepciones") for k, v in exc.items()})
assert len(exc["total"]) == 0
print("OK (iv) Total: cohortal >= período en las 86 cohortes. Excepciones por sexo reportadas arriba (son datos, no errores del cálculo)")

### 14.6 Recalcular $TBP_A$ y $TBP_B$ con $E_r$ cohortal ($\bar A=30$ y $14{,}2$) — $TBP_C$ no cambia

Se usa `recalcular_tbp`: `TBP_C` (no depende de $E_r$), `TBP_A = TBP_C/E_r` y `TBP_B = Āτ/(E_r ρ)`. Primero se comprueba que, con $E_r$ de período, el módulo **reproduce** las columnas `TBP_*_legal`/`TBP_*_obs` de `d` (y, por lo tanto, que sólo se cambia $E_r$).

In [ ]:
A14 = {"legal": 30.0, "obs": 14.2}
out14 = cmp14[["er_periodo_total", "er_cohortal_total"]].rename(columns={"er_periodo_total": "er_periodo", "er_cohortal_total": "er_cohortal"}).copy()
out14.insert(0, "anio_65", out14.index + 65)
for nom, A in A14.items():
    rp = recalcular_tbp(d, per14, A, TAU, RHO, "periodo").set_index("cohorte")
    rc = recalcular_tbp(d, coh14, A, TAU, RHO, "cohortal").set_index("cohorte")
    # el módulo con E_r de período reproduce d ; TBP_C no cambia
    assert np.allclose(rp["TBP_A_periodo"], dd14[f"TBP_A_{nom}"], rtol=1e-3) and np.allclose(rp["TBP_B_periodo"], dd14[f"TBP_B_{nom}"], rtol=1e-3)
    assert np.allclose(rp["TBP_C"], dd14[f"TBP_C_{nom}"]) and np.allclose(rc["TBP_C"], rp["TBP_C"], rtol=0, atol=0)
    out14[f"TBP_C_{nom}"] = rc["TBP_C"]
    out14[f"TBP_A_{nom}_periodo"] = rp["TBP_A_periodo"]; out14[f"TBP_A_{nom}_cohortal"] = rc["TBP_A_cohortal"]
    out14[f"TBP_B_{nom}_periodo"] = rp["TBP_B_periodo"]; out14[f"TBP_B_{nom}_cohortal"] = rc["TBP_B_cohortal"]
    # identidades con E_r cohortal
    assert np.allclose(rc["TBP_A_cohortal"], rc["TBP_C"] / coh14) and np.allclose(rc["TBP_A_cohortal"], dd14.loc[COH14, "N_t"] * dd14.loc[COH14, "S65"] / dd14.loc[COH14, "J"] * rc["TBP_B_cohortal"])
    print(f"Ā={A:>4}: TBP_B período {rp.TBP_B_periodo.min():.3f}-{rp.TBP_B_periodo.max():.3f} (máx. en {int(rp.TBP_B_periodo.idxmax())}) | cohortal {rc.TBP_B_cohortal.min():.3f}-{rc.TBP_B_cohortal.max():.3f} (máx. en {int(rc.TBP_B_cohortal.idxmax())}) | "
          f"TBP_A período {rp.TBP_A_periodo.min():.4f}-{rp.TBP_A_periodo.max():.4f} (máx. en {int(rp.TBP_A_periodo.idxmax())}) | cohortal {rc.TBP_A_cohortal.min():.4f}-{rc.TBP_A_cohortal.max():.4f} (máx. en {int(rc.TBP_A_cohortal.idxmax())})")
for sx in ("male", "female"):
    out14[f"er_periodo_{sx}"] = cmp14[f"er_periodo_{sx}"]; out14[f"er_cohortal_{sx}"] = cmp14[f"er_cohortal_{sx}"]
out14["grupos_extrapolados"] = cmp14["grupos_extrapolados"]
print("OK: TBP_C idéntico con E_r de período y cohortal (no depende de E_r) | TBP_A = TBP_C/E_r y TBP_A = D·TBP_B se cumplen con E_r cohortal")
bB = out14["TBP_B_legal_cohortal"]; assert int(bB.idxmax()) != 1956 and int(out14["TBP_A_legal_cohortal"].idxmax()) != 1956
print("OK: con E_r cohortal el máximo de TBP_B y TBP_A ya no está en la cohorte 1956")

### 14.7 Gráficos

(a) $E_r$ de período vs cohortal por cohorte: Total, varones y mujeres. (b) $TBP_B$ y $TBP_A$ (nivel, con $\bar A=30$ y $14{,}2$) y su **índice 1950 = 100**. $TBP_C$ no se grafica porque no cambia.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6), sharey=True)
for a_, sx, tit in zip(ax, ("total", "male", "female"), ("Total", "Varones", "Mujeres")):
    a_.plot(cmp14.index, cmp14[f"er_periodo_{sx}"], color="tab:red", label="Período (tabla del año t+65)")
    a_.plot(cmp14.index, cmp14[f"er_cohortal_{sx}"], color="tab:blue", label="Cohortal (trayectoria calendario)")
    a_.axvline(1956, color="gray", ls=":"); a_.axvline(2024.5, color="gray", ls=":")
    a_.annotate("1956 (COVID\nen t+65)", (1956, cmp14.loc[1956, f"er_periodo_{sx}"]), xytext=(1962, cmp14.loc[1956, f"er_periodo_{sx}"] - 0.2), fontsize=8, arrowprops=dict(arrowstyle="->", color="gray"))
    a_.axvspan(2000, 2035, color="orange", alpha=.07)
    a_.set_title(f"E_r a los 65 años — {tit}"); a_.set_xlabel("año de nacimiento de la cohorte"); a_.grid(alpha=.3)
ax[0].set_ylabel("años"); ax[0].legend(fontsize=8, loc="upper left")
ax[1].text(2001, ax[1].get_ylim()[0] + .3, "sombreado: tramos > 2100\ncon tabla 2100 constante (supuesto)", fontsize=7.5)
plt.tight_layout(); plt.savefig(f"{SAL14}/seccion_14_er_periodo_vs_cohortal.png", dpi=130); plt.show()

fig, ax = plt.subplots(2, 2, figsize=(14, 8.4))
for A_lab, ls_, lw in (("legal", "-", 1.8), ("obs", "--", 1.3)):
    nm = "Ā = 30" if A_lab == "legal" else "Ā = 14,2"
    ax[0, 0].plot(out14.index, out14[f"TBP_B_{A_lab}_periodo"], ls_, color="tab:red", lw=lw, label=f"período ({nm})")
    ax[0, 0].plot(out14.index, out14[f"TBP_B_{A_lab}_cohortal"], ls_, color="tab:blue", lw=lw, label=f"cohortal ({nm})")
    ax[0, 1].plot(out14.index, out14[f"TBP_A_{A_lab}_periodo"], ls_, color="tab:red", lw=lw, label=f"período ({nm})")
    ax[0, 1].plot(out14.index, out14[f"TBP_A_{A_lab}_cohortal"], ls_, color="tab:blue", lw=lw, label=f"cohortal ({nm})")
ax[0, 0].set_title("TBP_B = Ā·τ / (E_r·ρ)  (nivel)"); ax[0, 1].set_title("TBP_A = N(1−μ)·Ā·τ / (J·E_r·ρ)  (nivel)")
for nm_, col_, a_ in (("TBP_B", "TBP_B_legal", ax[1, 0]), ("TBP_A", "TBP_A_legal", ax[1, 1])):
    for ver, c_ in (("período", "tab:red"), ("cohortal", "tab:blue")):
        s_ = out14[f"{col_}_{'periodo' if ver == 'período' else 'cohortal'}"]; a_.plot(out14.index, s_ / s_.loc[1950] * 100, color=c_, label=ver)
    a_.axhline(100, color="gray", ls=":"); a_.set_title(f"{nm_}: índice 1950 = 100 (no depende de Ā)")
for a_ in ax.ravel():
    a_.axvline(1956, color="gray", ls=":"); a_.axvline(2024.5, color="gray", ls=":"); a_.grid(alpha=.3); a_.legend(fontsize=8); a_.set_xlabel("año de nacimiento de la cohorte")
fig.suptitle("TBP_C no se muestra: no contiene E_r y es idéntico (assert en 14.6)", y=1.0, fontsize=10)
plt.tight_layout(); plt.savefig(f"{SAL14}/seccion_14_tbp_periodo_vs_cohortal.png", dpi=130); plt.show()

### 14.8 Tabla: cohortes 1950, 1956, 1975, 2000, 2024 y 2035 (Ā = 30)

`Δ%` = cohortal respecto de período. `TBP_C` (años) es la misma columna en ambas versiones; $TBP_A=TBP_C/E_r$.

In [ ]:
sel14 = [1950, 1956, 1975, 2000, 2024, 2035]
t14 = out14.loc[sel14, ["er_periodo", "er_cohortal", "TBP_C_legal", "TBP_B_legal_periodo", "TBP_B_legal_cohortal", "TBP_A_legal_periodo", "TBP_A_legal_cohortal", "grupos_extrapolados"]].copy()
t14["Δ% E_r"] = (t14.er_cohortal / t14.er_periodo - 1) * 100
t14["Δ% TBP_B"] = (t14.TBP_B_legal_cohortal / t14.TBP_B_legal_periodo - 1) * 100
t14["Δ% TBP_A"] = (t14.TBP_A_legal_cohortal / t14.TBP_A_legal_periodo - 1) * 100
t14["Δ% TBP_C"] = 0.0
display(t14.round({"er_periodo": 2, "er_cohortal": 2, "TBP_C_legal": 4, "TBP_B_legal_periodo": 4, "TBP_B_legal_cohortal": 4, "TBP_A_legal_periodo": 5, "TBP_A_legal_cohortal": 5, "Δ% E_r": 1, "Δ% TBP_B": 1, "Δ% TBP_A": 1}))
print("Índice 1950 = 100 (Ā no importa):")
display(pd.DataFrame({"TBP_B período": out14.TBP_B_legal_periodo / out14.TBP_B_legal_periodo[1950] * 100, "TBP_B cohortal": out14.TBP_B_legal_cohortal / out14.TBP_B_legal_cohortal[1950] * 100,
                      "TBP_A período": out14.TBP_A_legal_periodo / out14.TBP_A_legal_periodo[1950] * 100, "TBP_A cohortal": out14.TBP_A_legal_cohortal / out14.TBP_A_legal_cohortal[1950] * 100}).loc[sel14].round(1))

### 14.9 Salida: `er_cohortal_1950_2035.csv`

Una fila por cohorte: $E_r$ de período y cohortal (Total, varones, mujeres), tramos extrapolados y $TBP_A$, $TBP_B$, $TBP_C$ recalculados para $\bar A=30$ (`legal`) y $14{,}2$ (`obs`). Se escribe en `../datos/procesados/` si existe (como la sección 10); si no, en `OUT`.

In [ ]:
cols14 = (["anio_65", "er_periodo", "er_cohortal", "er_periodo_male", "er_cohortal_male", "er_periodo_female", "er_cohortal_female", "grupos_extrapolados"]
          + [f"TBP_{k}_{n}{v}" for n in ("legal", "obs") for k, v in (("C", ""), ("A", "_periodo"), ("A", "_cohortal"), ("B", "_periodo"), ("B", "_cohortal"))])
salida14 = out14[cols14].rename(columns={"er_periodo": "er_periodo_total_ref", "er_cohortal": "er_cohortal_total"}).reset_index()
salida14.insert(2, "er_periodo", salida14.pop("er_periodo_total_ref"))
DEST14 = next((p for p in ["../datos/procesados", "datos/procesados"] if os.path.isdir(p)), OUT)
os.makedirs(DEST14, exist_ok=True)
f14 = os.path.join(DEST14, "er_cohortal_1950_2035.csv")
salida14.round(6).to_csv(f14, index=False)
assert len(salida14) == 86 and salida14.notna().all().all()
print("Escrito:", os.path.abspath(f14), salida14.shape); print("Escrito:", SAL14, "/ seccion_14_er_periodo_vs_cohortal.png y seccion_14_tbp_periodo_vs_cohortal.png")
salida14.head(3)

### 14.10 Lectura y límites

* **Qué se puede decir:** (1) con $E_r$ cohortal el pico de la cohorte 1956 **desaparece** en $TBP_B$ y $TBP_A$ (el máximo pasa a la cohorte 1950, que es lo que corresponde a una longevidad creciente); (2) $E_r^{coh}\ge E_r^{per}$ en todas las cohortes del Total: la tabla de período **subestima** la longevidad de quienes aún van a vivir mejoras; (3) $TBP_C$ **no cambia**; lo que cambia es su lectura contra la longevidad ($TBP_A=TBP_C/E_r$).
* **Qué no se puede decir:** que $E_r^{coh}$ sea un dato observado. Para las cohortes que cumplen 65 después de ~2010 es una **proyección** de la ONU (variante media); para las posteriores a ~2000, parte de la trayectoria cae después de 2100 y se usa la **tabla 2100 constante** (supuesto explícito, sesgo conservador: no hay más mejoras). Las cohortes 2033–2035 coinciden con el período por construcción.
* **Residuo:** queda una ondulación menor en las cohortes 1953–1954 (el COVID de 2020–2021 cae en su tramo 65–69). El método asigna un solo año calendario a cada tramo de 5 años, así que **suaviza** shocks de un año en lugar de eliminarlos.
* **Excepciones a $E_r^{coh}\ge E_r^{per}$:** sólo en la apertura por sexo (mujeres de la cohorte 1950, −0,015 años; probable explicación, no verificada: sus tramos 65–69 y 70–74 caen en 2017 y 2022, y la mortalidad femenina de período de 2015 ya era baja respecto de la que la ONU proyecta para esos años). No afectan al Total.
* **Aproximación de integración:** con $a_x$ de la tabla el error de reconstruir $e_{65}$ de período es del orden de 5·10⁻⁵ años (redondeo); con trapecio, de hasta ~0,08 años. En el test sintético con mejora de 1 % anual, la regla subestima el $E_r$ exacto entre 0,02 y 0,08 años (convención de punto medio con año entero).
* **Mortalidad diferencial:** las tablas son nacionales por sexo; no hay diferenciales por ingreso o aportes, que podrían ser relevantes para quienes efectivamente se jubilan.

## 15. 🆕 Modelo avanzado: sexo y edad de retiro, ρ con moratorias y migración

Hasta ahora $TBP_C$ trataba a la cohorte como un bloque único que se retira a los 65 con un $\rho$ y un $J$ únicos. Esta sección agrega tres refinamientos y los **compara paso a paso** con el $TBP$ base:

$$TBP_C(t)=\frac{N_t\,(1-\mu)\,\bar A\,\tau}{J\,\rho}\quad\text{(años; versión principal: }\mathbf{no\ contiene}\ E_r\text{)},\qquad TBP_A=\frac{TBP_C}{E_r},\qquad TBP_B=\frac{\bar A\,\tau}{E_r\,\rho}$$

1. **Sexo y edad de retiro (varones 65, mujeres 60).** El numerador se suma por sexo: $\sum_s N_s\,S_s(r_s)\,\bar A_s\,\tau$ con $N_s=N_t\cdot share_s$ (reparto por la razón de masculinidad al nacer $SRB$ de la ONU, no 51/49 fijo), $S_s(r_s)$ la supervivencia **cohortal** del sexo hasta *su* edad de retiro y $\bar A_s$ la suma de aportes de la EPH **del sexo** hasta $r_s-1$ (varones 18–64, mujeres 18–59). El denominador usa $J^*$ = varones 65+ **más** mujeres 60+: las mujeres de 60 a 64 ya cobran, así que son pasivas.
2. **$\rho$ con moratorias (ESCENARIO).** $\rho_{total}=s\,\rho_{mor}+(1-s)\,\rho_{sin\,mor}$, con $s$ = participación de beneficiarios con moratoria. Se despeja $\rho_{mor}$ implícito 2009–2023 y se plantean **tres supuestos** para cohortes que se retiran después de 2023 (no son predicciones).
3. **Migración.** En vez de "N + ΣM" (no calculable por cohorte) se usa la corrección empírica **implícita en la proyección ONU**: $m=\text{población de la edad de retiro}/(N\cdot S)$. Es opcional y **no es migración pura**.

> ⚠️ **Qué es y qué no es.** Todos los niveles son **condicionales a supuestos** (en especial $\bar A$, que es la cota inferior de asalariados de la EPH, y $\tau$ constante); no es una predicción y no hay ninguna "significancia" que reportar: son cuentas de un modelo determinista, no estimaciones con error muestral. Las diferencias entre pasos muestran **cuánto pesa cada supuesto**, no cuál es verdadero.

> **Errores de otro esquema que NO se copian:** (a) $TBP_C=N(r-\bar A)\tau/(J\rho E_r)$ rompe la definición (mete $E_r$ en $TBP_C$ y un término $(r-\bar A)$ inventado); (b) $E_r$ por sexo no se fabrica multiplicando por 0,92/1,08: se usan las tablas ONU por sexo; (c) "$N+\sum M_x$" no es calculable por cohorte; (d) "ponderar y luego sumar" mezcla dos agregaciones: acá se define una sola, explicada en 15.3.

Depende solo de `d`, `s`, `mort`, `TAU`, `RHO`, `A_OBS`, `A_LEGAL`, `DATA`, `OUT` en memoria, de los módulos de las secciones 12–14 (`tbp_eph`, `tbp_anses`, `tbp_mortalidad`, que **reusa**) y de archivos que carga con rutas candidatas (`../datos/procesados`, `datos/procesados`) y, si faltan, la URL *raw* del repositorio. **No modifica `d`.**

### 15.1 Módulo (copia de `src/tbp_modelo_avanzado.py`)

Todo el código está en una sola celda para que el Colab sea autosuficiente. El mismo archivo vive en el repositorio (`src/tbp_modelo_avanzado.py`; `python src/tbp_modelo_avanzado.py` corre el autotest con datos **sintéticos** rotulados). **Reusa** (no reescribe) `pipeline(..., sexo=)` de `tbp_eph`, `asignar_rho_tau_por_cohorte` de `tbp_anses` y `supervivencia_cohortal` / `er_cohortal` de `tbp_mortalidad`; si el notebook ya corrió las secciones 12–14, toma esas funciones de la memoria.

In [ ]:
# ====== MÓDULO tbp_modelo_avanzado (misma versión que src/tbp_modelo_avanzado.py) ======
# -*- coding: utf-8 -*-
"""
tbp_modelo_avanzado.py — TBP_C con sexo y edad de retiro, ρ con moratorias (escenarios) y ajuste por migración.

Proyecto TBP — Índice de Cuna Vacía (Taller de Indicadores).  Versión principal propuesta (años de aporte por año de jubilado):

        TBP_C(t) = N_t · (1−μ) · Ā · τ / (J · ρ)            [en el código: N_t · S65 · Ā · τ / (J · ρ)]
        TBP_A    = TBP_C / E_r          TBP_B = Ā·τ / (E_r·ρ)          (TBP_C NO contiene E_r)

QUÉ AGREGA ESTE MÓDULO (todo REUSA tbp_eph, tbp_anses y tbp_mortalidad; no los reescribe)
 1) SEXO Y EDAD DE RETIRO (principal: varones 65, mujeres 60 = edad legal en el SIPA).  Numerador aditivo por sexo:
        Σ_s  N_s · S_s(r_s) · Ā_s · τ ,     N_s = N_t · share_s ,  share_varones = SRB/(100+SRB)   (SRB ONU del año t)
    S_s(r_s) = supervivencia cohortal de ese sexo hasta su edad de retiro (tablas ONU por sexo); Ā_s = Σ_{a=18}^{r_s−1} S(a)·d_s(a)
    con la densidad d_s de la EPH del sexo s (varones 18-64; mujeres 18-59).  Denominador: J*·ρ con
        J* = varones 65+ + mujeres 60+ (ONU, año t+65): las mujeres de 60 a 64 ya cobran, por eso cuentan como pasivas.
    SENSIBILIDAD ("retiro a los 65 para todos", J = población 65+ total, modelo base): sobrecuenta aportes de las mujeres de 60 a 64
    (que ya no aportan) y omite del denominador a las pasivas de 60 a 64.
    AGREGACIÓN (definida, no "ponderada y luego sumada"): lo que se SUMA son las contribuciones al numerador
    (TBP_C_total = TBP_C_varones + TBP_C_mujeres, ambos sobre el MISMO J*·ρ).  Lo que se PROMEDIA (ponderando por los
    sobrevivientes al retiro N_s·S_s) son las intensidades por persona: Ā, E_r y TBP_B.  TBP_A total = Σ_s TBP_C_s / E_{r,s}
    y cumple TBP_A = D*·TBP_B con D* = Σ N_s S_s / J*.
 2) ρ CON MORATORIAS (ESCENARIO).  Como ρ = haber medio / salario imponible medio y el haber medio de todos los beneficiarios es
    el promedio ponderado por beneficiarios, ρ_total = s·ρ_mor + (1−s)·ρ_sin_mor  (s = participación de beneficiarios con moratoria).
    Con ρ_total, ρ_sin_mor y s de ANSES (digitalizados, aprox.) se DESPEJA ρ_mor implícito 2009-2023 y se verifica que sea plausible.
    Para cohortes que se retiran después de 2023, s es un SUPUESTO (no una predicción): 'persiste' (≈62,9 %), 'mitad' (≈31 %),
    'sin_moratorias' (0 %).  Mayor ρ => menor TBP_C.
 3) MIGRACIÓN.  En vez de "N + ΣM" (no calculable por cohorte) se usa la corrección empírica implícita en la proyección ONU:
        m = población de la edad de retiro en el año calendario del retiro / (N_s · S_s)      (N_efectivo · S = población ONU)
    Mezcla migración neta, diferencias entre fuentes (DEIS vs ONU) y el desfase de nacimientos: NO es migración pura.
    Opción `ajuste_migracion` True/False.

SUPUESTOS Y LÍMITES (rotulados en cada función): ρ y τ iguales para varones y mujeres (no hay series por sexo); ρ y τ del año t+65
    para ambos sexos; J* es un stock en t+65 (las mujeres se retiran 5 años antes); τ constante (supuesto); todos los niveles dependen de Ā;
    la EPH es urbana, solo asalariados (cota inferior).  Los resultados son condicionales a supuestos, NO son una predicción.

Uso mínimo:
    import tbp_modelo_avanzado as ma
    ma.autotest_modelo_avanzado()            # datos SINTÉTICOS rotulados
"""
import os
import sys
import importlib
import urllib.request

import numpy as np
import pandas as pd

RAW_BASE_MA = "https://raw.githubusercontent.com/carolinamsfelipe/Tp-Indicadores/main/datos/procesados"
RUTAS_MA = ["../datos/procesados", "datos/procesados"]
RUTAS_SRC_MA = ["../src", "src", "github_staging/src"]
EDAD_V, EDAD_M = 65, 60                     # edades de retiro PRINCIPALES: varones 65, mujeres 60 (edad legal en el SIPA)
EDAD_ALT = 65                               # SENSIBILIDAD: "retiro a los 65 para todos" (modelo base) con J = población 65+ total
TAU_DEF, RHO_DEF, A_OBS_DEF, A_LEGAL_DEF = 0.2177, 0.403, 14.2, 30.0
ESCENARIOS_MOR = {"persiste": 1.0, "mitad": 0.5, "sin_moratorias": 0.0}   # multiplicador de la participación observada en 2023
ETQ_MOR = {"persiste": "persiste (≈62,9 %)", "mitad": "mitad (≈31 %)", "sin_moratorias": "sin moratorias (0 %)"}
ESC_NAT = ("persistencia", "recuperacion_lenta", "caida_adicional")
NOTA_ABAR = ("Ā por sexo: EPH (solo asalariados con descuento jubilatorio = cota inferior), pseudo-panel cohorte×edad, "
             "truncado a la edad de retiro del sexo (varones 18-64, mujeres 18-59).")


# ================================================================== 0) DEPENDENCIAS (reuso de los módulos del proyecto)
_NOMBRES_DEP = {"tbp_eph": ["pipeline", "supervivencia_gompertz", "EDADES", "find_eph_files", "read_all"],
                "tbp_anses": ["asignar_rho_tau_por_cohorte", "serie_rho_digitalizada", "serie_beneficiarios_digitalizada", "serie_tau"],
                "tbp_mortalidad": ["cargar_tablas", "supervivencia_cohortal", "er_cohortal"]}


class _Dep:
    pass


def dependencias():
    """Devuelve un objeto con las funciones reusadas de tbp_eph, tbp_anses y tbp_mortalidad.
    Orden: (1) nombres ya definidos en el espacio global (el notebook embebe los módulos de las secciones 12-14);
    (2) import desde src/ (rutas candidatas). Si falta alguno, error explícito."""
    g = globals()
    dep = _Dep()
    for mod, nombres in _NOMBRES_DEP.items():
        m = None
        if any(n not in g for n in nombres):
            for r in RUTAS_SRC_MA:
                if os.path.isdir(r) and os.path.abspath(r) not in sys.path:
                    sys.path.append(os.path.abspath(r))
            try:
                m = importlib.import_module(mod)
            except ImportError:
                m = None
        for n in nombres:
            if n in g:
                setattr(dep, n, g[n])
            elif m is not None and hasattr(m, n):
                setattr(dep, n, getattr(m, n))
            else:
                raise ImportError(f"falta `{n}` de {mod}: correr antes las secciones 12-14 del notebook o tener src/{mod}.py disponible")
    return dep


# ================================================================== 1) CARGA DE DATOS
def leer_csv_ma(nombre, rutas=None, destino="."):
    """Lee un CSV de datos/procesados: rutas candidatas ('../datos/procesados', 'datos/procesados' + las dadas);
    si no está, lo baja de GitHub (URL raw)."""
    for c in list(rutas or []) + RUTAS_MA:
        p = os.path.join(c, nombre)
        if os.path.exists(p):
            return pd.read_csv(p, low_memory=False)
    os.makedirs(destino, exist_ok=True)
    p = os.path.join(destino, nombre)
    urllib.request.urlretrieve(f"{RAW_BASE_MA}/{nombre}", p)
    return pd.read_csv(p, low_memory=False)


def cargar_poblacion_sexo(df=None, rutas=None):
    """Población ONU por edad simple y SEXO (personas). Devuelve (PM, PF): DataFrames año × edad (0..100; 100 = grupo abierto 100+)."""
    if df is None:
        df = leer_csv_ma("un_wpp2024_argentina_poblacion_edad_simple.csv", rutas)
    if "Variant" in df.columns:
        df = df[df["Variant"] == "Medium"]
    out = []
    for col in ("PopMale", "PopFemale"):
        P = df.pivot_table(index="Time", columns="AgeGrpStart", values=col) * 1000.0
        P.columns = P.columns.astype(int)
        out.append(P)
    return out[0], out[1]


def cargar_srb(df=None, rutas=None):
    """SRB (varones por 100 mujeres al nacer, ONU) por año. Verifica que exista y sea razonable."""
    if df is None:
        df = leer_csv_ma("un_wpp2024_argentina_indicadores.csv", rutas)
    if "SRB" not in df.columns:
        raise ValueError("el CSV de indicadores ONU no tiene la columna SRB (razón de masculinidad al nacer)")
    if "Variant" in df.columns:
        df = df[df["Variant"] == "Medium"]
    srb = df.set_index("Time")["SRB"].astype(float)
    srb = srb.loc[1950:2100]                      # 2101 no trae SRB en la tabla ONU
    assert srb.notna().all() and srb.between(100, 112).all(), "SRB fuera del rango esperado (100-112 varones por 100 mujeres)"
    return srb


def cargar_natalidad_escenarios(df=None, rutas=None):
    """Escenarios de natalidad de la sección 9: DataFrame año × escenario (nacimientos, personas)."""
    if df is None:
        df = leer_csv_ma("natalidad_escenarios_2025_2035.csv", rutas)
    return df.pivot(index="anio", columns="escenario", values="nacimientos")


def share_varones(srb):
    """Participación de varones al nacer: SRB/(100+SRB).  SRB = varones por 100 mujeres (no se usa 51/49 fijo)."""
    return srb / (100.0 + srb)


# ================================================================== 2) SUPERVIVENCIA, E_r Y Ā POR SEXO
def supervivencia_retiro(cohortes, tablas, edad_v=EDAD_V, edad_m=EDAD_M):
    """S_s(r_s) por sexo: supervivencia COHORTAL (tablas ONU por sexo, trayectoria calendario) hasta la edad de retiro.
    Devuelve DataFrame: S_v (varones a edad_v), S_m (mujeres a edad_m), S_m65 (mujeres a 65),
    S_tot65 (Total a 65, = columna S65 de `d`)."""
    D = dependencias()
    co = [int(c) for c in cohortes]
    sv = D.supervivencia_cohortal(co, "Male", tablas)
    sf = D.supervivencia_cohortal(co, "Female", tablas)
    st = D.supervivencia_cohortal(co, "Total", tablas)
    return pd.DataFrame({"S_v": sv[edad_v], "S_m": sf[edad_m], "S_m65": sf[65], "S_tot65": st[65]}, index=pd.Index(co, name="cohorte"))


def er_retiro(cohortes, tablas, edad_v=EDAD_V, edad_m=EDAD_M):
    """E_r cohortal a la edad de retiro de cada sexo (varones a 65, mujeres a `edad_m`, 60 por defecto). Usa tbp_mortalidad.er_cohortal(edad=...)."""
    D = dependencias()
    co = [int(c) for c in cohortes]
    return pd.DataFrame({"E_v": D.er_cohortal(co, edad_v, "Male", tablas), "E_m": D.er_cohortal(co, edad_m, "Female", tablas)})


def abar_hasta_edad(dens, s65, edad_retiro):
    """Ā truncado: Σ_{a=18}^{edad_retiro−1} S_c(a)·d_c(a), con S de Gompertz calibrada a S(65)=s65 (igual que tbp_eph.abar_por_cohorte,
    que suma hasta 64).  Con edad_retiro=65 reproduce abar_por_cohorte; con 60 suma las edades 18-59."""
    D = dependencias()
    S = D.supervivencia_gompertz(s65.reindex(dens.index).values)
    cols = np.asarray(D.EDADES) < int(edad_retiro)
    return pd.Series((S * dens.values)[:, cols].sum(axis=1), index=dens.index)


def abar_por_sexo(df_eph, tablas, cohortes, edad_v=EDAD_V, edad_m=EDAD_M, ind_rule="solo_asalariados"):
    """Ā por sexo con la EPH REAL: corre tbp_eph.pipeline(..., sexo=1|2) (1 = varón, 2 = mujer) y trunca la suma de edades a r_s−1.
    Columnas: A_varones_{edad_v}, A_mujeres_{edad_m} (60 => 18-59) y A_mujeres_65 (sensibilidad) + n_edades_obs.
    Si no hay microdatos NO se inventa nada: quien llama debe omitir la parte por sexo."""
    D = dependencias()
    co = [int(c) for c in cohortes]
    out = {}
    for nombre, cod, sx, r_s in (("varones", 1, "Male", edad_v), ("mujeres", 2, "Female", edad_m)):
        s65 = D.supervivencia_cohortal(co, sx, tablas)[65]
        res = D.pipeline(df_eph, s65, ind_rule=ind_rule, sexo=cod, cohortes=co)
        for r in sorted({65, r_s}):
            out[f"A_{nombre}_{r}"] = abar_hasta_edad(res["dens"], s65, r)
        out[f"n_edades_obs_{nombre}"] = res["tabla"]["n_edades_obs"]
        # control interno: con r=65 la suma debe reproducir A_bar del pipeline
        assert np.allclose(out[f"A_{nombre}_65"].values, res["tabla"]["A_bar"].values, atol=1e-9), "Ā truncado a 65 != A_bar del pipeline"
    res = pd.DataFrame(out)
    res.index.name = "cohorte"
    return res


# ================================================================== 3) J* (PASIVOS POTENCIALES) Y REPONDERACIÓN
def pesos_cohorte_ma(s, nat, escenario):
    """w(c) = N_propio(c)/N_ONU(c) (misma regla que la sección 10): DEIS para 1950-2024 (w=1 donde DEIS tiene hueco), escenario de la sección 9
    para 2025-2035, w=1 en el resto.  Sirve para reponderar la población ONU por cohorte."""
    n_onu = s["un_births_miles"] * 1000.0
    n_deis = s["deis_nacimientos"]
    w = pd.Series(1.0, index=np.arange(1900, 2101))
    for c in range(1950, 2025):
        if not np.isnan(n_deis.get(c, np.nan)):
            w[c] = n_deis[c] / n_onu[c]
    for c in range(2025, 2036):
        w[c] = nat.loc[c, escenario] / n_onu[c]
    assert w.notna().all()
    return w


def j_pasivos(PM, PF, anios, edad_v=EDAD_V, edad_m=EDAD_M, w=None):
    """Pasivos potenciales en el año calendario y:  J* = varones edad_v+ + mujeres edad_m+  (ONU, edad simple, grupo 100+ incluido);
    J65 = población 65+ total (definición del modelo base).  `w` (Serie por cohorte, opcional) repondera como la sección 10:
    la población de edad a en el año y pertenece a la cohorte y−a (grupo abierto 100+ -> y−100).  Devuelve DataFrame por año."""
    filas = {}
    for y in anios:
        y = int(y)
        a_v = np.arange(edad_v, 101); a_m = np.arange(edad_m, 101); a_65 = np.arange(65, 101)
        wv = np.ones(len(a_v)) if w is None else w.reindex(y - a_v).values
        wm = np.ones(len(a_m)) if w is None else w.reindex(y - a_m).values
        w65 = np.ones(len(a_65)) if w is None else w.reindex(y - a_65).values
        jv = float((PM.loc[y, a_v].values * wv).sum())
        jm = float((PF.loc[y, a_m].values * wm).sum())
        j65 = float(((PM.loc[y, a_65].values + PF.loc[y, a_65].values) * w65).sum())
        filas[y] = dict(J_star=jv + jm, J_v=jv, J_m=jm, J65=j65)
    return pd.DataFrame.from_dict(filas, orient="index")


# ================================================================== 4) ρ CON MORATORIAS (ESCENARIO)
def rho_implicito(rho_total, rho_sin_mor, share_mor):
    """ρ_mor implícito:  ρ_total = s·ρ_mor + (1−s)·ρ_sin_mor  =>  ρ_mor = (ρ_total − (1−s)·ρ_sin_mor)/s.
    Es exacto si ambos ρ comparten el denominador (salario imponible medio) y la media del haber se pondera por beneficiarios;
    con datos digitalizados del gráfico es una APROXIMACIÓN."""
    s = np.asarray(share_mor, float)
    return (np.asarray(rho_total, float) - (1.0 - s) * np.asarray(rho_sin_mor, float)) / s


def tabla_rho_moratoria(serie_rho, ben):
    """Tabla anual 2009-2023: ρ_total, ρ_sin_mor, share_mor (ANSES, digitalizados) y ρ_mor implícito. Verifica 0<ρ_mor<1."""
    r = serie_rho.set_index("anio")[["rho", "rho_sin_moratoria"]]
    b = ben.set_index("anio")["share_moratoria"]
    t = r.join(b, how="inner")
    t["rho_mor_implicito"] = rho_implicito(t["rho"], t["rho_sin_moratoria"], t["share_moratoria"])
    assert t["rho_mor_implicito"].between(0, 1).all(), "ρ_mor implícito fuera de (0,1)"
    return t


def rho_por_escenario(anios, tabla_mor, escenario):
    """ρ_total por año calendario de retiro.  Años con dato (≤2023): ρ digitalizado (igual en todos los escenarios).
    Años > 2023: ρ = s_esc·ρ_mor(2023) + (1−s_esc)·ρ_sin_mor(2023), con s_esc = mult·s(2023) y mult ∈ {1 (persiste), 0,5 (mitad), 0 (sin moratorias)}.
    Los valores post-2023 son SUPUESTOS rotulados (no predicciones)."""
    if escenario not in ESCENARIOS_MOR:
        raise ValueError(f"escenario {escenario!r} no está en {list(ESCENARIOS_MOR)}")
    u = tabla_mor.index.max()
    s_u, rm_u, rsm_u = tabla_mor.loc[u, "share_moratoria"], tabla_mor.loc[u, "rho_mor_implicito"], tabla_mor.loc[u, "rho_sin_moratoria"]
    s_esc = ESCENARIOS_MOR[escenario] * s_u
    rho_post = s_esc * rm_u + (1.0 - s_esc) * rsm_u
    vals, tipo = [], []
    for y in anios:
        y = int(y)
        if y in tabla_mor.index:
            vals.append(float(tabla_mor.loc[y, "rho"])); tipo.append("dato (digitalizado)")
        elif y > u:
            vals.append(float(rho_post)); tipo.append(f"supuesto:{escenario}")
        else:
            vals.append(float(tabla_mor.loc[tabla_mor.index.min(), "rho"])); tipo.append("extrapolado:primer_valor")
    return pd.Series(vals, index=pd.Index([int(a) for a in anios], name="anio"), name=f"rho_{escenario}"), tipo


# ================================================================== 5) TBP_C POR SEXO, TBP_A, TBP_B
def tbp_c_sexos(N, sh_v, S_v, S_m, A_v, A_m, tau, rho, J, mig_v=1.0, mig_m=1.0):
    """TBP_C con numerador aditivo por sexo y denominador común J·ρ:
        TBP_C_s = N_t·share_s · S_s(r_s) · Ā_s · m_s · τ / (J·ρ)         TBP_C_total = TBP_C_varones + TBP_C_mujeres.
    Todas las entradas son Series por cohorte (o escalares).  `J` es J* (varones 65+ y mujeres 60+) o J65 (variante base).
    m_s = factor de migración (1 = sin ajuste).  Devuelve DataFrame con N_v, N_m, sobrevivientes al retiro y TBP_C."""
    N_v = N * sh_v
    N_m = N - N_v
    sob_v = N_v * S_v * mig_v
    sob_m = N_m * S_m * mig_m
    den = J * rho
    out = pd.DataFrame({"N_v": N_v, "N_m": N_m, "sob_v": sob_v, "sob_m": sob_m})
    out["TBP_C_varones"] = sob_v * A_v * tau / den
    out["TBP_C_mujeres"] = sob_m * A_m * tau / den
    out["TBP_C_total"] = out["TBP_C_varones"] + out["TBP_C_mujeres"]
    return out


def tbp_ab_sexos(parts, A_v, A_m, E_v, E_m, tau, rho, J):
    """TBP_A y TBP_B por sexo con E_r cohortal del sexo a su edad de retiro.
    TBP_A_s = TBP_C_s/E_s (aditivo: TBP_A_total = suma);  TBP_B_s = Ā_s·τ/(E_s·ρ) (por persona) y TBP_B_total = promedio ponderado por
    los sobrevivientes al retiro (N_s·S_s).  Identidad:  TBP_A_total = D*·TBP_B_total con D* = Σ sobrevivientes / J."""
    out = pd.DataFrame(index=parts.index)
    out["TBP_A_varones"] = parts["TBP_C_varones"] / E_v
    out["TBP_A_mujeres"] = parts["TBP_C_mujeres"] / E_m
    out["TBP_A_total"] = out["TBP_A_varones"] + out["TBP_A_mujeres"]
    out["TBP_B_varones"] = A_v * tau / (E_v * rho)
    out["TBP_B_mujeres"] = A_m * tau / (E_m * rho)
    sob = parts["sob_v"] + parts["sob_m"]
    out["TBP_B_total"] = (parts["sob_v"] * out["TBP_B_varones"] + parts["sob_m"] * out["TBP_B_mujeres"]) / sob
    out["D_star"] = sob / J
    out["E_r_implicito_total"] = parts["TBP_C_total"] / out["TBP_A_total"]
    return out


def descomponer_retiro_mujeres(N, sh_v, S_v, S_m65, S_m60, A_v, A_m65, A_m60, tau, rho, J65, J_star):
    """Efecto de que las mujeres se retiren a los 60 (vs 65), por pasos ACUMULADOS (el orden importa; es una descomposición, no una
    identidad): (a) ambos sexos a 65 [referencia = variante "retiro 65 para todos"]; (b) + S_m a los 60 (más sobrevivientes llegan);
    (c) + Ā_m hasta 59 (menos años de aporte); (d) + J* (más pasivas: mujeres 60-64) = modelo principal.  Devuelve TBP_C total en cada paso."""
    p = lambda S_m, A_m, J: tbp_c_sexos(N, sh_v, S_v, S_m, A_v, A_m, tau, rho, J)["TBP_C_total"]
    return pd.DataFrame({"a_ambos_65": p(S_m65, A_m65, J65), "b_mas_S_m60": p(S_m60, A_m65, J65),
                         "c_mas_Abar_m59": p(S_m60, A_m60, J65), "d_mas_J_star": p(S_m60, A_m60, J_star)})


# ================================================================== 6) MIGRACIÓN (corrección empírica implícita en la proyección ONU)
def factor_migracion(PM, PF, cohortes, N, sh_v, S_v, S_m, w_pob=None, edad_v=EDAD_V, edad_m=EDAD_M):
    """m_s = población ONU de edad r_s en el año t+r_s / (N_s·S_s)  (N_efectivo·S = población ONU).  Por sexo, con varones a edad_v y
    mujeres a edad_m.  `w_pob` (Serie por cohorte, opcional) repondera la población ONU con el escenario de natalidad (cohortes ≥2025),
    de modo que N_t (escenario) y la población usen la misma historia de nacimientos.  Mezcla migración neta, diferencias entre fuentes
    (DEIS vs ONU) y el desfase de nacimientos: NO es migración pura."""
    co = [int(c) for c in cohortes]
    pv = pd.Series([PM.loc[c + edad_v, edad_v] for c in co], index=co)
    pm = pd.Series([PF.loc[c + edad_m, edad_m] for c in co], index=co)
    if w_pob is not None:
        wv = w_pob.reindex(co).values
        pv, pm = pv * wv, pm * wv
    m_v = pv / (N * sh_v * S_v)
    m_m = pm / (N * (1 - sh_v) * S_m)
    return pd.DataFrame({"m_v": m_v, "m_m": m_m, "pob_v": pv, "pob_m": pm})


# ================================================================== 7) CONSTRUCCIÓN DEL MODELO COMPLETO (pasos acumulados 0-5)
def construir_modelo(d, s, tablas, PM, PF, srb, nat, abar_tot, tabla_mor, abar_sx=None, tau=TAU_DEF, rho_const=RHO_DEF,
                     A_obs=A_OBS_DEF, A_legal=A_LEGAL_DEF, escenario_nat="persistencia", escenario_mor="persiste",
                     col_abar_tot="A_bar_solo_asalariados", edad_retiro_mujeres=EDAD_M):
    """Pasos acumulados de TBP_C (cohortes de `d`):
      p0  base: Ā constante (14,2 y 30), ρ y τ constantes
      p1  + Ā(t) de la EPH (total, 18-64)
      p2  + ρ(t) ANSES (digitalizado; 'ultimo_valor' después de 2023; τ constante por supuesto)
      p3  + diferenciación por sexo y retiro (varones 65, mujeres 60): S_s(r_s), Ā_s hasta r_s−1, J* = varones 65+ y mujeres 60+
      (sensibilidad, aparte: *_m65 = "retiro a los 65 para todos" con J = J65, modelo base)
      p4  + escenario de natalidad (sección 9) y J* reponderado (sección 10) para cohortes ≥2025
      p5  + ajuste por migración (opcional, aparte)
    Si `abar_sx` es None (sin microdatos EPH por sexo) se calculan solo p0-p2 y nada se inventa para el resto.
    Devuelve dict con R (DataFrame por cohorte) y piezas auxiliares."""
    dd = d.set_index("cohorte")
    co = dd.index
    D = dependencias()
    R = pd.DataFrame(index=co)
    R["anio_65"] = co + 65
    N0 = dd["N_t"].astype(float)
    S65, J65 = dd["S65"], dd["J"]
    R["N_t"] = N0
    # ---- p0: base
    R["TBP_C_p0_A14_2"] = N0 * S65 * A_obs * tau / (J65 * rho_const)
    R["TBP_C_p0_A30"] = N0 * S65 * A_legal * tau / (J65 * rho_const)
    # ---- p1: Ā(t) EPH
    A_eph = abar_tot[col_abar_tot].reindex(co)
    assert A_eph.notna().all(), "Ā EPH no cubre todas las cohortes"
    R["A_eph"] = A_eph
    R["TBP_C_p1"] = N0 * S65 * A_eph * tau / (J65 * rho_const)
    # ---- p2: ρ(t) ANSES (+ τ constante), vía tbp_anses.asignar_rho_tau_por_cohorte (regla 'ultimo_valor')
    sr = tabla_mor.reset_index()[["anio", "rho", "rho_sin_moratoria"]]
    st = D.serie_tau(valor=tau, anios=sr["anio"].tolist())
    asig = D.asignar_rho_tau_por_cohorte(d, sr, st, edad_retiro=65, extrapolacion="ultimo_valor")
    R["rho_t"] = asig.set_index("cohorte")["rho_v"]
    R["tau_t"] = asig.set_index("cohorte")["tau_v"]
    R["TBP_C_p2"] = N0 * S65 * A_eph * R["tau_t"] / (J65 * R["rho_t"])
    # ρ por escenario de moratorias (persiste == p2, verificado)
    RHO_ESC = {}
    for esc in ESCENARIOS_MOR:
        rr, _ = rho_por_escenario(co + 65, tabla_mor, esc)
        RHO_ESC[esc] = pd.Series(rr.values, index=co)
    assert np.allclose(RHO_ESC["persiste"].values, R["rho_t"].values, atol=1e-9), "escenario 'persiste' debe coincidir con 'ultimo_valor'"
    rho_v = RHO_ESC[escenario_mor]
    R["rho_escenario"] = rho_v
    aux = dict(RHO_ESC=RHO_ESC, asig=asig, tabla_mor=tabla_mor)
    if abar_sx is None:
        return dict(R=R, **aux, con_sexo=False)

    # ---- sexo: share, N_s, S_s, E_s  (principal: varones 65, mujeres 60; sensibilidad: todos a 65 con J = J65)
    sh_v = share_varones(srb).reindex(co)
    assert sh_v.notna().all()
    R["share_varones"] = sh_v
    Sx = supervivencia_retiro(co, tablas, edad_v=EDAD_V, edad_m=edad_retiro_mujeres)
    Ex = er_retiro(co, tablas, edad_v=EDAD_V, edad_m=edad_retiro_mujeres)
    for c_ in Sx.columns:
        R[c_] = Sx[c_]
    for c_ in Ex.columns:
        R[c_] = Ex[c_]
    S_m65 = Sx["S_m65"]
    E_m65 = er_retiro(co, tablas, edad_v=EDAD_V, edad_m=EDAD_ALT)["E_m"]
    R["E_m65"] = E_m65
    for c_ in abar_sx.columns:
        R[c_] = abar_sx[c_].reindex(co)
    A_v = R["A_varones_65"]
    A_m = R[f"A_mujeres_{edad_retiro_mujeres}"]
    A_m65 = R["A_mujeres_65"]
    JJ = j_pasivos(PM, PF, co + 65, edad_m=edad_retiro_mujeres)
    JJ.index = co
    R["J65"] = JJ["J65"]; R["J_star"] = JJ["J_star"]
    assert np.allclose(R["J65"].values, J65.values, rtol=1e-9), "J65 (suma de PopMale+PopFemale 65+) debe reproducir la columna J de d"
    # ---- p3: sexo y retiro (varones 65, mujeres edad_m); J* = varones 65+ y mujeres edad_m+
    p3 = tbp_c_sexos(N0, sh_v, Sx["S_v"], Sx["S_m"], A_v, A_m, tau, rho_v, JJ["J_star"])
    R["TBP_C_p3"] = p3["TBP_C_total"]
    R["TBP_C_p3_J65"] = tbp_c_sexos(N0, sh_v, Sx["S_v"], Sx["S_m"], A_v, A_m, tau, rho_v, J65)["TBP_C_total"]       # misma numeración, J = 65+ total
    # ---- p4: escenario de natalidad y J* reponderado (solo cohortes ≥ 2025)
    w = pesos_cohorte_ma(s, nat, escenario_nat)
    nuevo = co >= 2025
    N4 = N0.copy()
    N4.loc[2025:2035] = nat.loc[2025:2035, escenario_nat].values

    def _j4(edad_m_, w_=w):
        j0 = j_pasivos(PM, PF, co + 65, edad_m=edad_m_); j0.index = co
        jr = j_pasivos(PM, PF, co + 65, edad_m=edad_m_, w=w_); jr.index = co
        j4 = j0["J_star"].copy(); j4[nuevo] = jr["J_star"][nuevo]
        return j4, jr
    J4, JJ_rw = _j4(edad_retiro_mujeres)
    J4_65, _ = _j4(EDAD_ALT)                                  # J = 65+ total reponderado (modelo base)
    R["N_4"] = N4; R["J_star_4"] = J4
    p4 = tbp_c_sexos(N4, sh_v, Sx["S_v"], Sx["S_m"], A_v, A_m, tau, rho_v, J4)
    R["TBP_C_p4"] = p4["TBP_C_total"]
    R["J_consistente"] = ~(co + EDAD_V - edad_retiro_mujeres > 2035) | (co < 2025)     # J* de mujeres en t+65 incluye cohortes hasta t+(65-edad_m)
    # ---- p5: migración (sobre p4)
    w_pob = pd.Series(1.0, index=w.index); w_pob.loc[2025:2035] = w.loc[2025:2035]       # solo cohortes ≥2025 (escenario)
    mig = factor_migracion(PM, PF, co, N4, sh_v, Sx["S_v"], Sx["S_m"], w_pob=w_pob, edad_m=edad_retiro_mujeres)
    R["m_v"], R["m_m"] = mig["m_v"], mig["m_m"]
    p5 = tbp_c_sexos(N4, sh_v, Sx["S_v"], Sx["S_m"], A_v, A_m, tau, rho_v, J4, mig["m_v"], mig["m_m"])
    R["TBP_C_p5"] = p5["TBP_C_total"]
    R["m_total"] = (mig["pob_v"] + mig["pob_m"]) / (p4["sob_v"] + p4["sob_m"])
    # ---- salidas por sexo (modelo p4) y TBP_A / TBP_B
    for c_ in ("TBP_C_varones", "TBP_C_mujeres", "TBP_C_total"):
        R[c_ + "_p4"] = p4[c_]
    ab = tbp_ab_sexos(p4, A_v, A_m, Ex["E_v"], Ex["E_m"], tau, rho_v, J4)
    for c_ in ab.columns:
        R[c_ + "_p4"] = ab[c_]
    # ---- ρ escenarios sobre p4
    for esc, rv in RHO_ESC.items():
        R[f"TBP_C_p4_rho_{esc}"] = tbp_c_sexos(N4, sh_v, Sx["S_v"], Sx["S_m"], A_v, A_m, tau, rv, J4)["TBP_C_total"]
    # ---- escenarios de natalidad (p4 con cada uno)
    for e_ in ESC_NAT:
        w_e = pesos_cohorte_ma(s, nat, e_)
        Ne = N0.copy(); Ne.loc[2025:2035] = nat.loc[2025:2035, e_].values
        Je, _ = _j4(edad_retiro_mujeres, w_e)
        R[f"TBP_C_p4_{e_}"] = tbp_c_sexos(Ne, sh_v, Sx["S_v"], Sx["S_m"], A_v, A_m, tau, rho_v, Je)["TBP_C_total"]
    # ---- SENSIBILIDAD (variante alternativa): "retiro a los 65 para todos" con J = población 65+ total (modelo base)
    s3 = tbp_c_sexos(N0, sh_v, Sx["S_v"], S_m65, A_v, A_m65, tau, rho_v, J65)
    s4 = tbp_c_sexos(N4, sh_v, Sx["S_v"], S_m65, A_v, A_m65, tau, rho_v, J4_65)
    R["TBP_C_p3_m65"] = s3["TBP_C_total"]; R["TBP_C_p4_m65"] = s4["TBP_C_total"]
    for c_ in ("TBP_C_varones", "TBP_C_mujeres"):
        R[c_ + "_p4_m65"] = s4[c_]
    ab65 = tbp_ab_sexos(s4, A_v, A_m65, Ex["E_v"], E_m65, tau, rho_v, J4_65)
    for c_ in ab65.columns:
        R[c_ + "_p4_m65"] = ab65[c_]
    desc = descomponer_retiro_mujeres(N0, sh_v, Sx["S_v"], S_m65, Sx["S_m"], A_v, A_m65, A_m, tau, rho_v, J65, JJ["J_star"])
    aux.update(p3=p3, p4=p4, p5=p5, s3=s3, s4=s4, ab=ab, ab65=ab65, mig=mig, desc=desc, w=w, JJ=JJ, JJ_rw=JJ_rw, Sx=Sx, Ex=Ex, N4=N4, J4=J4,
               sh_v=sh_v, rho_v=rho_v, tau=tau, A_v=A_v, A_m=A_m, A_m65=A_m65, edad_m=edad_retiro_mujeres)
    return dict(R=R, **aux, con_sexo=True)


# ================================================================== 8) CONTROLES (asserts) CON LOS DATOS REALES
def controles_modelo_avanzado(res, d, verbose=True):
    """Asserts del modelo ya construido (datos reales). Devuelve lista de mensajes OK."""
    R, msgs = res["R"], []
    dd = d.set_index("cohorte")
    tau, rho_c = res.get("tau", TAU_DEF), RHO_DEF
    # p0 reproduce las columnas de d
    assert np.allclose(R["TBP_C_p0_A14_2"], dd["TBP_C_obs"], rtol=1e-9) and np.allclose(R["TBP_C_p0_A30"], dd["TBP_C_legal"], rtol=1e-9)
    msgs.append("p0 reproduce TBP_C_obs (Ā=14,2) y TBP_C_legal (Ā=30) de `d`")
    tm = res["tabla_mor"]
    assert tm["rho_mor_implicito"].between(0, 1).all() and (tm["rho_mor_implicito"] < tm["rho_sin_moratoria"]).all()
    msgs.append(f"ρ_mor implícito ∈ (0,1) y < ρ sin moratoria en los {len(tm)} años (min {tm.rho_mor_implicito.min():.3f}, max {tm.rho_mor_implicito.max():.3f})")
    # ordenamiento por escenario: mayor ρ => menor TBP_C (solo cohortes con retiro > 2023, donde los escenarios difieren)
    if res["con_sexo"]:
        post = R.index + 65 > 2023
        a, b_, c_ = (R.loc[post, f"TBP_C_p4_rho_{k}"] for k in ("persiste", "mitad", "sin_moratorias"))
        assert (a > b_).all() and (b_ > c_).all(), "mayor ρ debe dar menor TBP_C"
        pre = ~post
        assert np.allclose(R.loc[pre, "TBP_C_p4_rho_persiste"], R.loc[pre, "TBP_C_p4_rho_sin_moratorias"]), "antes de 2024 los escenarios coinciden"
        msgs.append("escenarios de ρ: TBP_C(persiste) > TBP_C(mitad) > TBP_C(sin moratorias) en todas las cohortes con retiro > 2023; idénticos antes")
        Sx = res["Sx"]
        assert np.allclose(R["N_t"] * R["share_varones"] + R["N_t"] * (1 - R["share_varones"]), R["N_t"]) and R["share_varones"].between(0.45, 0.55).all()
        assert np.allclose(res["p3"]["N_v"] + res["p3"]["N_m"], R["N_t"])
        msgs.append("N_varones + N_mujeres = N_t; share_varones ∈ (0,45; 0,55)")
        assert Sx[["S_v", "S_m", "S_m65", "S_tot65"]].apply(lambda c: c.between(0, 1, inclusive="neither").all()).all()
        assert np.allclose(Sx["S_tot65"], dd["S65"], atol=1e-12)
        msgs.append("S_s(r_s) ∈ (0,1); S_Total(65) cohortal reproduce `S65` de `d`")
        p3 = res["p3"]
        indep = (p3["N_v"] * Sx["S_v"] * R["A_varones_65"] + p3["N_m"] * Sx["S_m"] * res["A_m"]) * tau / (R["J_star"] * res["rho_v"])
        assert np.allclose(p3["TBP_C_varones"] + p3["TBP_C_mujeres"], p3["TBP_C_total"]) and np.allclose(indep, R["TBP_C_p3"])
        msgs.append("TBP_C_varones + TBP_C_mujeres = TBP_C_total (calculado también por fuera como Σ_s N_s S_s Ā_s τ /(J*ρ))")
        # control de reducción al modelo base: share=1 varón, r=65, Ā constante, S = S65 de d, J = J de d
        base = tbp_c_sexos(R["N_t"], 1.0, dd["S65"], dd["S65"], 30.0, 30.0, TAU_DEF, RHO_DEF, dd["J"])
        assert np.allclose(base["TBP_C_total"], dd["TBP_C_legal"], rtol=1e-9) and np.allclose(base["TBP_C_mujeres"], 0)
        msgs.append("con share=1 varón, r=65, Ā=30 constante, ρ y τ constantes el modelo reproduce TBP_C_legal de `d`")
        # linealidad
        k = 1.7
        x1 = tbp_c_sexos(R["N_t"], res["sh_v"], Sx["S_v"], Sx["S_m"], R["A_varones_65"], res["A_m"], tau, res["rho_v"], R["J_star"])["TBP_C_total"]
        xa = tbp_c_sexos(R["N_t"], res["sh_v"], Sx["S_v"], Sx["S_m"], k * R["A_varones_65"], k * res["A_m"], tau, res["rho_v"], R["J_star"])["TBP_C_total"]
        xt = tbp_c_sexos(R["N_t"], res["sh_v"], Sx["S_v"], Sx["S_m"], R["A_varones_65"], res["A_m"], k * tau, res["rho_v"], R["J_star"])["TBP_C_total"]
        assert np.allclose(xa, k * x1) and np.allclose(xt, k * x1)
        msgs.append("linealidad: TBP_C escala exactamente con Ā y con τ (factor 1,7)")
        # p4 = p3 para cohortes ≤ 2024 ; migración sin ajuste = p4
        assert np.allclose(R.loc[R.index <= 2024, "TBP_C_p4"], R.loc[R.index <= 2024, "TBP_C_p3"])
        msgs.append("p4 = p3 para cohortes ≤ 2024 (solo cambian las ≥ 2025)")
        pb = res["p5"]
        assert np.allclose(pb["sob_v"] + pb["sob_m"], res["mig"]["pob_v"] + res["mig"]["pob_m"]), "N_efectivo·S debe igualar la población ONU"
        sin = tbp_c_sexos(res["N4"], res["sh_v"], Sx["S_v"], Sx["S_m"], R["A_varones_65"], res["A_m"], tau, res["rho_v"], res["J4"], 1.0, 1.0)
        assert np.allclose(sin["TBP_C_total"], R["TBP_C_p4"])
        msgs.append("migración: N_efectivo·S = población ONU a la edad de retiro; con m=1 vuelve exactamente a p4")
        ab = res["ab"]
        assert np.allclose(ab["TBP_A_total"], ab["D_star"] * ab["TBP_B_total"])
        msgs.append("identidad TBP_A_total = D*·TBP_B_total (agregación: TBP_A aditivo, TBP_B ponderado por sobrevivientes)")
    if verbose:
        for m_ in msgs:
            print("OK:", m_)
    return msgs


# ================================================================== 9) AUTOTEST CON DATOS SINTÉTICOS
def _banner_ma(txt="DATOS SINTÉTICOS — no son Argentina"):
    line = "#" * 78
    print("\n".join([line, "##" + txt.center(74) + "##", line]))


def autotest_modelo_avanzado(verbose=True):
    """Pruebas de las funciones puras con datos SINTÉTICOS (inventados, solo para verificar código; nunca se usan en salidas reales)."""
    if verbose:
        _banner_ma()
    co = pd.Index(np.arange(2000, 2006), name="cohorte")
    # población sintética: 120 mil personas por edad y sexo en todos los años, varones algo menos
    yrs = np.arange(2060, 2075)
    PM = pd.DataFrame(100000.0, index=yrs, columns=np.arange(0, 101))
    PF = pd.DataFrame(120000.0, index=yrs, columns=np.arange(0, 101))
    # J*: varones 65+ (36 edades, incl. 100) + mujeres 60+ (41 edades)
    J = j_pasivos(PM, PF, [2065], edad_m=60)
    assert abs(J.loc[2065, "J_v"] - 36 * 1e5) < 1e-6 and abs(J.loc[2065, "J_m"] - 41 * 1.2e5) < 1e-6
    assert abs(J.loc[2065, "J65"] - 36 * 2.2e5) < 1e-6
    assert abs((J.loc[2065, "J_star"] - J.loc[2065, "J65"]) - 5 * 1.2e5) < 1e-6          # difieren en las 5 edades 60-64 de mujeres
    # reponderación: w=0,5 para todas las cohortes -> J a la mitad
    w = pd.Series(0.5, index=np.arange(1900, 2101))
    assert abs(j_pasivos(PM, PF, [2065], edad_m=60, w=w).loc[2065, "J_star"] - 0.5 * J.loc[2065, "J_star"]) < 1e-6
    # share por SRB
    assert abs(share_varones(105.0) - 105.0 / 205.0) < 1e-15 and abs(share_varones(100.0) - 0.5) < 1e-15
    # ρ implícito: ida y vuelta
    s_ = np.array([0.3, 0.5, 0.7]); rm = np.array([0.25, 0.30, 0.35]); rs = np.array([0.5, 0.55, 0.6])
    rt = s_ * rm + (1 - s_) * rs
    assert np.allclose(rho_implicito(rt, rs, s_), rm)
    # tabla y escenarios
    sr = pd.DataFrame({"anio": [2021, 2022, 2023], "rho": rt, "rho_sin_moratoria": rs})
    be = pd.DataFrame({"anio": [2021, 2022, 2023], "share_moratoria": s_})
    tm = tabla_rho_moratoria(sr, be)
    r_p, _ = rho_por_escenario([2022, 2030], tm, "persiste"); r_h, _ = rho_por_escenario([2030], tm, "mitad"); r_0, _ = rho_por_escenario([2030], tm, "sin_moratorias")
    assert abs(r_p[2022] - rt[1]) < 1e-12 and abs(r_p[2030] - rt[2]) < 1e-12 and abs(r_0[2030] - rs[2]) < 1e-12
    assert r_p[2030] < r_h[2030] < r_0[2030]                                    # sin moratorias => ρ más alto
    try:
        rho_por_escenario([2030], tm, "otro"); raise AssertionError("debía fallar")
    except ValueError:
        pass
    # TBP_C por sexo
    N = pd.Series(1e6, index=co); Sv = pd.Series(0.85, index=co); Sm = pd.Series(0.93, index=co)
    Jc = pd.Series(1e7, index=co)
    a = tbp_c_sexos(N, 0.5, Sv, Sm, 10.0, 8.0, 0.2, 0.4, Jc)
    assert np.allclose(a["N_v"] + a["N_m"], N) and np.allclose(a["TBP_C_varones"] + a["TBP_C_mujeres"], a["TBP_C_total"])
    assert np.allclose(a["TBP_C_total"], (5e5 * 0.85 * 10 + 5e5 * 0.93 * 8) * 0.2 / (1e7 * 0.4))
    b = tbp_c_sexos(N, 1.0, Sv, Sv, 30.0, 30.0, 0.2177, 0.403, Jc)              # control: share=1 varón == fórmula base
    assert np.allclose(b["TBP_C_total"], N * Sv * 30 * 0.2177 / (Jc * 0.403)) and np.allclose(b["TBP_C_mujeres"], 0)
    assert np.allclose(tbp_c_sexos(N, 0.5, Sv, Sm, 20.0, 16.0, 0.2, 0.4, Jc)["TBP_C_total"], 2 * a["TBP_C_total"])      # linealidad en Ā
    assert np.allclose(tbp_c_sexos(N, 0.5, Sv, Sm, 10.0, 8.0, 0.4, 0.4, Jc)["TBP_C_total"], 2 * a["TBP_C_total"])       # linealidad en τ
    assert (tbp_c_sexos(N, 0.5, Sv, Sm, 10.0, 8.0, 0.2, 0.8, Jc)["TBP_C_total"] < a["TBP_C_total"]).all()               # ρ mayor => menor
    # TBP_A y TBP_B por sexo: identidad TBP_A = D*·TBP_B
    ab = tbp_ab_sexos(a, 10.0, 8.0, pd.Series(17.0, index=co), pd.Series(26.0, index=co), 0.2, 0.4, Jc)
    assert np.allclose(ab["TBP_A_total"], ab["D_star"] * ab["TBP_B_total"])
    assert np.allclose(ab["TBP_A_total"], a["TBP_C_varones"] / 17.0 + a["TBP_C_mujeres"] / 26.0)
    # migración: N_efectivo·S = población
    mg = factor_migracion(pd.DataFrame({65: [3e5, 3e5, 3e5, 3e5, 3e5, 3e5]}, index=co + 65), pd.DataFrame({60: [3.3e5] * 6}, index=co + 60),
                          co, N, 0.5, Sv, Sm, edad_m=60)
    assert np.allclose(mg["m_v"] * N * 0.5 * Sv, 3e5) and np.allclose(mg["m_m"] * N * 0.5 * Sm, 3.3e5)
    c = tbp_c_sexos(N, 0.5, Sv, Sm, 10.0, 8.0, 0.2, 0.4, Jc, mg["m_v"], mg["m_m"])
    assert np.allclose(c["sob_v"] + c["sob_m"], 3e5 + 3.3e5)
    # descomposición: pasos coherentes (a: 65 ambos; d = p3)
    de = descomponer_retiro_mujeres(N, 0.5, Sv, pd.Series(0.9, index=co), Sm, 10.0, 8.0, 7.0, 0.2, 0.4, Jc, Jc * 1.1)
    assert np.allclose(de["d_mas_J_star"], tbp_c_sexos(N, 0.5, Sv, Sm, 10.0, 7.0, 0.2, 0.4, Jc * 1.1)["TBP_C_total"])
    if verbose:
        print("autotest OK (J* y reponderación, share SRB, ρ_mor implícito y escenarios, TBP_C por sexo, linealidad, control share=1, "
              "TBP_A/TBP_B y D*, migración, descomposición). Datos SINTÉTICOS: no son Argentina.")
        _banner_ma("FIN AUTOTEST — estos números NO son Argentina")
    return True

### 15.2 Carga de datos y Ā por sexo

Se cargan: población ONU por edad simple y **sexo** (`PopMale`, `PopFemale`), la razón de masculinidad al nacer `SRB` (ONU), los escenarios de natalidad de la sección 9, el $\bar A$ total de la EPH (sección 12) y las tablas de mortalidad por sexo. Para $\bar A$ **por sexo** se corre `pipeline(..., sexo=1|2)` sobre la EPH **real** de `datos/eph/` (o la que ya dejó la sección 12 en memoria) y se **trunca la suma de edades** a $r_s-1$. Si no hay microdatos y tampoco el CSV derivado `abar_eph_por_sexo_cohorte.csv`, **se omite la parte por sexo** (no se inventa nada): quedan los pasos 0–2 y la sección de $\rho$.

La mortalidad de Gompertz que usa la EPH para $S(a)$, $a<r_s$, se calibra con la $S(65)$ **del sexo**. $\bar A$ cubre sólo a asalariados con descuento jubilatorio (cota inferior).

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
try: display
except NameError: display = print
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

CANDS_15 = ["../datos/procesados", "datos/procesados", DATA]
D15 = dependencias()
TAB15 = D15.cargar_tablas(rutas=CANDS_15)                       # tablas ONU abreviadas Total / Male / Female
CO15 = [int(c) for c in d.cohorte]
PM15, PF15 = cargar_poblacion_sexo(rutas=CANDS_15)              # personas por edad simple y año
SRB15 = cargar_srb(rutas=CANDS_15)                              # varones por 100 mujeres al nacer (ONU)
NAT15 = cargar_natalidad_escenarios(rutas=CANDS_15)             # escenarios sección 9
ABAR_TOT15 = leer_csv_ma("abar_eph_por_cohorte.csv", CANDS_15, destino=OUT).set_index("cohorte")   # Ā total EPH (sección 12)
SERIE_RHO15 = D15.serie_rho_digitalizada(rutas=CANDS_15)
BEN15 = D15.serie_beneficiarios_digitalizada()
TAB_MOR15 = tabla_rho_moratoria(SERIE_RHO15, BEN15)
print("SRB ONU (varones por 100 mujeres): min %.1f | max %.1f | share varones 2000 = %.4f" % (SRB15.min(), SRB15.max(), share_varones(SRB15[2000])))

# --- Ā por sexo: EPH real si está; si no, CSV derivado (si existe); si no, se omite
DEST15 = next((p for p in ["../datos/procesados", "datos/procesados"] if os.path.isdir(p)), OUT)
F_SEXO15 = "abar_eph_por_sexo_cohorte.csv"
df_eph15 = globals().get("df_eph")
if df_eph15 is None:
    srcs15 = D15.find_eph_files(["datos/eph", "../datos/eph", "../../datos/eph"])
    df_eph15 = D15.read_all(srcs15) if srcs15 else None
ABAR_SX15, ORIGEN_SEXO15 = None, None
if df_eph15 is not None:
    ABAR_SX15 = abar_por_sexo(df_eph15, TAB15, CO15)
    ORIGEN_SEXO15 = "EPH real (pipeline por sexo sobre datos/eph)"
    os.makedirs(DEST15, exist_ok=True)
    ABAR_SX15.round(6).reset_index().to_csv(os.path.join(DEST15, F_SEXO15), index=False)
else:
    pc = next((os.path.join(c, F_SEXO15) for c in CANDS_15 if os.path.exists(os.path.join(c, F_SEXO15))), None)
    if pc is not None:
        ABAR_SX15 = pd.read_csv(pc).set_index("cohorte"); ORIGEN_SEXO15 = f"CSV derivado de la EPH real ({pc})"
HAY_SEXO15 = ABAR_SX15 is not None
if HAY_SEXO15:
    print("Ā por sexo:", ORIGEN_SEXO15)
    display(ABAR_SX15.loc[[1950, 1960, 1980, 2000, 2024, 2035], [c for c in ABAR_SX15.columns if c.startswith("A_")]].round(2))
else:
    print("⚠️ NO hay microdatos EPH ni CSV por sexo: se omite la parte por sexo (pasos 3-5). El CSV abar_eph_por_cohorte.csv NO tiene sexo y NO se usa para inventarlo.")

### 15.3 Sexo y edad de retiro: el modelo y la agregación

**Paso a paso (cada uno con su porqué):**

1. **Reparto por sexo.** $share_{varones}(t)=SRB_t/(100+SRB_t)$ con el $SRB$ de la ONU del año de nacimiento (≈105 varones por 100 mujeres ⇒ 51,2 %). No se usa 51/49 fijo. $N_{v}=N_t\,share$, $N_m=N_t-N_v$ (suman $N_t$ exacto).
2. **Supervivencia al retiro** $S_s(r_s)$: tabla ONU **del sexo** siguiendo la trayectoria de la cohorte (misma regla que $1-\mu$ de la sección 1; se reusa `supervivencia_cohortal`). Las mujeres llegan a los 60 con mucha más probabilidad que a los 65, y que los varones a los 65.
3. **$\bar A_s$**: densidad de aportes de la EPH **del sexo** sumada hasta $r_s-1$ (varones 18–64, mujeres 18–59). Retirarse antes recorta años de aporte.
4. **Denominador $J^*\rho$:** $J^*$ = varones 65+ **más** mujeres 60+ en el año $t+65$ (población ONU por edad simple). Si las mujeres cobran desde los 60, sus 60–64 años son **pasivas**; contarlas como aportantes (modelo base: todos a 65) **sobrecuenta aportes y omite pasivas**.
5. **Agregación (una sola, definida).** *Se SUMAN* las contribuciones al numerador: $TBP_{C}^{total}=TBP_C^{varones}+TBP_C^{mujeres}$, ambos sobre el **mismo** $J^*\rho$ (por eso la suma es el total). *Se PROMEDIAN, ponderando por los sobrevivientes al retiro* $N_sS_s$, las intensidades por persona ($\bar A$, $E_r$, $TBP_B$). $TBP_A^{total}=\sum_s TBP_C^s/E_{r,s}$ y vale $TBP_A=D^*\cdot TBP_B$ con $D^*=\sum_s N_sS_s/J^*$.
6. **$E_r$ por sexo y edad de retiro:** `er_cohortal(edad=65, 'Male')` y `er_cohortal(edad=60, 'Female')` con las tablas ONU por sexo (nada de factores 0,92/1,08).

$\rho$ y $\tau$ son **los mismos para ambos sexos** y del año $t+65$ (no hay series por sexo; las mujeres se retiran 5 años antes, así que es una simplificación).

In [ ]:
RES15 = construir_modelo(d, s, TAB15, PM15, PF15, SRB15, NAT15, ABAR_TOT15, TAB_MOR15, abar_sx=ABAR_SX15, tau=TAU, rho_const=RHO,
                         A_obs=A_OBS, A_legal=A_LEGAL, escenario_nat="persistencia", escenario_mor="persiste")
R15 = RES15["R"]
SEL15 = [1955, 1960, 1980, 2000, 2024, 2035]
if HAY_SEXO15:
    cols = ["share_varones", "S_v", "S_m", "S_m65", "E_v", "E_m", "E_m65", "A_varones_65", "A_mujeres_60", "A_mujeres_65", "J65", "J_star"]
    display(R15.loc[SEL15, cols].round(3))
    print("Sobrevivientes al retiro y TBP_C por sexo (modelo completo, paso 4; ambos sobre el mismo J*·ρ):")
    display(R15.loc[SEL15, ["TBP_C_varones_p4", "TBP_C_mujeres_p4", "TBP_C_total_p4", "TBP_A_varones_p4", "TBP_A_mujeres_p4", "TBP_A_total_p4",
                            "TBP_B_varones_p4", "TBP_B_mujeres_p4", "TBP_B_total_p4", "D_star_p4"]].round(4))
    print("Efecto de que las mujeres se retiren a los 60 en vez de 65 (TBP_C, sin natalidad ni migración; pasos ACUMULADOS, el orden importa):")
    display(RES15["desc"].loc[SEL15].round(4))
    print("  a: todos a 65 (J = 65+ total, modelo base)  ->  b: + más mujeres llegan a los 60 (S_m(60) > S_m(65))  ->  c: + Ā hasta 59 (menos años de aporte)  ->  d: + J* (60-64 pasivas)")
    print("TBP_C: variante J* vs variante J = 65+ total (mismo numerador), cohortes", SEL15)
    display(R15.loc[SEL15, ["TBP_C_p3", "TBP_C_p3_J65"]].assign(**{"Δ% J* vs J65": lambda x: 100 * (x.TBP_C_p3 / x.TBP_C_p3_J65 - 1)}).round(4))

In [ ]:
if HAY_SEXO15:
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
    co_ = R15.index
    ax[0].stackplot(co_, R15.TBP_C_varones_p4, R15.TBP_C_mujeres_p4, colors=["#1f3a5f", "#d55e00"], labels=["varones (retiro 65)", "mujeres (retiro 60)"], alpha=.9)
    ax[0].set_title("TBP_C por sexo (modelo completo, paso 4)\nla suma de ambos es el total (mismo J*·ρ)"); ax[0].legend(fontsize=8, loc="upper right")
    ax[1].plot(co_ + 65, R15.J65 / 1e6, color="#444444", ls="--", label="J = 65+ total (modelo base)")
    ax[1].plot(co_ + 65, R15.J_star / 1e6, color="#0072b2", lw=2, label="J* = varones 65+ y mujeres 60+")
    ax[1].set_title("Pasivos potenciales (millones), año calendario t+65"); ax[1].set_xlabel("año calendario"); ax[1].legend(fontsize=8)
    ax[2].plot(co_, R15.TBP_C_p4_m65, color="#444444", ls="--", label="SENSIBILIDAD: todos a 65, J = 65+ total")
    ax[2].plot(co_, R15.TBP_C_p4, color="#0072b2", lw=2, label="PRINCIPAL: varones 65 / mujeres 60, J*")
    ax[2].set_title("TBP_C: retiro diferenciado vs 'todos a 65'"); ax[2].legend(fontsize=8)
    for a_ in ax:
        a_.grid(alpha=.3); a_.axvline(2024.5, color="gray", ls=":", lw=1)
    ax[0].set_xlabel("cohorte"); ax[2].set_xlabel("cohorte"); ax[0].set_ylabel("años de aporte por año de jubilado")
    fig.suptitle("Sección 15.3: sexo y edad de retiro (cohortes ≥2025: escenario 'persistencia' de natalidad; niveles condicionados a supuestos)", fontsize=11)
    plt.tight_layout(rect=[0, 0, 1, 0.93]); plt.savefig(f"{OUT}/seccion_15_sexo_retiro.png", dpi=130); plt.show()

### 15.4 $\rho$ con moratorias (escenario, por cohorte)

**Idea.** $\rho$ = haber medio / salario imponible medio. El haber medio de **todos** los beneficiarios es el promedio ponderado por beneficiarios del haber con moratoria y sin moratoria; como el denominador (salario imponible) es el mismo,
$$\rho_{total}=s\,\rho_{mor}+(1-s)\,\rho_{sin\,mor}\ \Rightarrow\ \rho_{mor}=\frac{\rho_{total}-(1-s)\,\rho_{sin\,mor}}{s}.$$
Con $\rho_{total}$, $\rho_{sin\,mor}$ y $s$ de ANSES (Gráficos 4.3 y 3.1, **digitalizados, precisión aprox.**) se despeja $\rho_{mor}$ implícito por año y se verifica que esté en $(0,1)$ y por debajo del $\rho$ sin moratoria. Es una aproximación: depende de que el "sin moratoria" del Anuario comparta el denominador y de la digitalización.

**Escenarios para cohortes que se retiran después de 2023** (**supuestos a validar por el grupo; no predicciones ni hechos**): $\rho=s_{esc}\,\rho_{mor}(2023)+(1-s_{esc})\,\rho_{sin\,mor}(2023)$ con
* **persiste:** $s$ = 62,9 % (la participación observada en 2023; coincide con la regla `ultimo_valor` de la sección 13, $\rho=0{,}403$);
* **mitad:** $s\approx 31\,\%$;
* **sin moratorias:** $s=0$ ⇒ $\rho=\rho_{sin\,mor}(2023)=56{,}3\,\%$.

Hasta el retiro de 2023 todos los escenarios usan el $\rho$ observado. $s$ es un stock de beneficiarios, no la participación en las altas nuevas; **no se inventa cobertura real**. Un $\rho$ más alto **reduce** $TBP_C$ (más pasivo por aportante).

In [ ]:
display(TAB_MOR15.round(4).rename(columns={"rho": "ρ_total", "rho_sin_moratoria": "ρ_sin_mor", "share_moratoria": "s (benef. moratoria)", "rho_mor_implicito": "ρ_mor implícito"}))
print("ρ_mor implícito: min %.3f | max %.3f | siempre en (0,1) y por debajo de ρ_sin_mor: %s" % (TAB_MOR15.rho_mor_implicito.min(), TAB_MOR15.rho_mor_implicito.max(),
      bool((TAB_MOR15.rho_mor_implicito < TAB_MOR15.rho_sin_moratoria).all())))
RESUM_RHO15 = pd.DataFrame({e: RES15["RHO_ESC"][e] for e in ESCENARIOS_MOR}).loc[[1950, 1958, 1959, 1975, 2000, 2035]]
print("ρ por escenario según la cohorte (retiro en cohorte+65):"); display(RESUM_RHO15.round(4))

fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
a_ = TAB_MOR15
ax[0].plot(a_.index, 100 * a_.rho, color="#0072b2", lw=2, label="ρ total (SIPA)")
ax[0].plot(a_.index, 100 * a_.rho_sin_moratoria, color="#222222", lw=2, label="ρ sin moratoria")
ax[0].plot(a_.index, 100 * a_.rho_mor_implicito, color="#d55e00", lw=2, label="ρ con moratoria (implícito)")
for e, col in zip(ESCENARIOS_MOR, ("#0072b2", "#009e73", "#cc79a7")):
    rho_e = RES15["RHO_ESC"][e].loc[1959:]; anios_ = rho_e.index + 65
    ax[0].plot([2023, 2035], [100 * rho_e.iloc[0]] * 2, ls="--", color=col, lw=1.5, label=f"supuesto post-2023: {ETQ_MOR[e]}")
ax[0].axvline(2023.5, color="gray", ls=":"); ax[0].set_ylim(25, 70); ax[0].grid(alpha=.3); ax[0].legend(fontsize=7.5, loc="lower right")
ax[0].set_title("ρ por año (2009-2023 digitalizado, aprox.) y supuestos desde 2024"); ax[0].set_ylabel("%"); ax[0].set_xlabel("año de retiro")
if HAY_SEXO15:
    for e, col in zip(ESCENARIOS_MOR, ("#0072b2", "#009e73", "#cc79a7")):
        ax[1].plot(R15.index, R15[f"TBP_C_p4_rho_{e}"], color=col, lw=2, label=ETQ_MOR[e])
    ax[1].axvspan(1958.5, 2035.5, color="gray", alpha=.08); ax[1].text(0.97, 0.97, "sombreado: retiro ≥2024,\nρ = SUPUESTO", transform=ax[1].transAxes, ha="right", va="top", fontsize=8, color="dimgray")
    ax[1].set_title("TBP_C (modelo completo, paso 4) según escenario de moratorias"); ax[1].set_xlabel("cohorte"); ax[1].set_ylabel("años de aporte por año de jubilado")
    ax[1].legend(fontsize=8, loc="lower left"); ax[1].grid(alpha=.3)
fig.suptitle("Sección 15.4: ρ con moratorias como ESCENARIO (no predicción)", fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.savefig(f"{OUT}/seccion_15_rho_moratorias.png", dpi=130); plt.show()
if HAY_SEXO15:
    t_ = R15.loc[[1960, 1980, 2000, 2024, 2035], [f"TBP_C_p4_rho_{e}" for e in ESCENARIOS_MOR]]
    t_["Δ% mitad vs persiste"] = 100 * (t_.TBP_C_p4_rho_mitad / t_.TBP_C_p4_rho_persiste - 1)
    t_["Δ% sin_moratorias vs persiste"] = 100 * (t_.TBP_C_p4_rho_sin_moratorias / t_.TBP_C_p4_rho_persiste - 1)
    display(t_.round(4))

### 15.5 Ajuste por migración (opcional, aparte)

**Idea.** Se mide, para cada cohorte y sexo, el cociente entre la población ONU de la edad de retiro **en el año del retiro** y los sobrevivientes que implican los nacimientos propios:
$$m_s(t)=\frac{\text{Pob}_{ONU}(r_s,\ t+r_s)}{N_s\cdot S_s(r_s)},\qquad N_{efectivo}\cdot S=\text{Pob}_{ONU}$$
Si $m>1$ hay más gente de esa edad de la que implican los nacimientos y la mortalidad (inmigración neta, pero también diferencias entre fuentes). Con `ajuste_migracion=True` el numerador pasa a ser $\text{Pob}_{ONU}\cdot\bar A_s\cdot\tau$.

> ⚠️ **No es migración pura.** Mezcla migración neta, errores o diferencias entre fuentes (DEIS vs ONU: la cohorte 2024 tiene $m\approx1{,}22$ porque DEIS registra ~413 mil nacimientos y la ONU supone ~507 mil) y el desfase de nacimientos (la población "de edad 65" a mitad de año mezcla dos cohortes). Además supone que los migrantes aportaron como los nativos (Ā igual), lo cual no se sabe.
>
> **Cohortes ≥2025:** $N_t$ es el escenario de la sección 9 y la población ONU está construida con *otros* nacimientos. Se repondera la población con $w=N_{escenario}/N_{ONU}$ (como la sección 10); entonces $m=\text{Pob}_{ONU}/(N_{ONU}\,S)$ es el cociente interno de la ONU (≈1,0), **sin** el desfase DEIS/ONU. Es lo coherente, pero deja de "corregir" N.

In [ ]:
if HAY_SEXO15:
    display(R15.loc[SEL15, ["m_v", "m_m", "m_total", "TBP_C_p4", "TBP_C_p5"]].assign(**{"Δ% migración": lambda x: 100 * (x.TBP_C_p5 / x.TBP_C_p4 - 1)}).round(4))
    pre, post = R15.index <= 2024, R15.index >= 2025
    print(f"m total: media cohortes ≤2024 = {R15.m_total[pre].mean():.3f} (máx {R15.m_total[pre].max():.3f} en cohorte {R15.m_total[pre].idxmax()}) | media cohortes ≥2025 = {R15.m_total[post].mean():.3f}")
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
    ax[0].plot(R15.index, R15.m_v, color="#1f3a5f", label="varones (edad 65)"); ax[0].plot(R15.index, R15.m_m, color="#d55e00", label="mujeres (edad 60)")
    ax[0].plot(R15.index, R15.m_total, color="k", lw=2, label="total"); ax[0].axhline(1, color="gray", lw=.8)
    ax[0].axvline(2024.5, color="gray", ls=":"); ax[0].set_title("Factor m = población ONU / (N·S)"); ax[0].set_xlabel("cohorte"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
    ax[1].plot(R15.index, R15.TBP_C_p4, color="#0072b2", lw=2, label="sin ajuste (paso 4)"); ax[1].plot(R15.index, R15.TBP_C_p5, color="#cc3311", lw=2, ls="--", label="con ajuste de migración (paso 5)")
    ax[1].axvline(2024.5, color="gray", ls=":"); ax[1].set_title("TBP_C con y sin ajuste de migración"); ax[1].set_xlabel("cohorte"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
    fig.suptitle("Sección 15.5: corrección implícita en la proyección ONU (mezcla migración, fuentes y desfase de nacimientos)", fontsize=11)
    plt.tight_layout(rect=[0, 0, 1, 0.93]); plt.savefig(f"{OUT}/seccion_15_migracion.png", dpi=130); plt.show()

### 15.6 Comparación final: TBP_C por cohorte, paso a paso

Pasos **acumulados** (cada uno agrega un supuesto al anterior):

| paso | qué agrega |
|---|---|
| **0** | base: $\bar A$ constante (14,2 y 30), $\rho=0{,}403$, $\tau=0{,}2177$ constantes, $J$ = 65+ total |
| **1** | + $\bar A(t)$ de la EPH (total, asalariados, cota inferior) |
| **2** | + $\rho(t)$ ANSES (digitalizado 2009–2023; `ultimo_valor` después; $\tau$ constante) |
| **3** | + diferenciación por sexo y retiro (varones 65, mujeres 60) con $J^*$ |
| **4** | + escenario de natalidad "persistencia" (sección 9) y $J^*$ reponderado (sección 10) para cohortes ≥2025 |
| **5** | + ajuste de migración (opcional, aparte) |

Los pasos 1–5 usan $\bar A$ de la EPH, así que **no son comparables en nivel con la línea $\bar A=30$**: ésta es una referencia (requisito legal, no un máximo). Como $\rho_{ANSES}$ coincide con 0,403 desde la cohorte 1959, el paso 2 sólo se distingue en 1950–1958. **Condicional a supuestos; no es una predicción.** Para cohortes ≥2025 el paso 4 es consistente salvo que el $J^*$ de mujeres 60+ en $t+65$ incluye cohortes posteriores a 2035 (sin escenario) a partir de $t>2030$: ver `J_consistente` en el CSV.

In [ ]:
PASOS15 = [("TBP_C_p0_A30", "0  base, Ā=30 constante (referencia)", "#999999", ":", 1.5), ("TBP_C_p0_A14_2", "0  base, Ā=14,2 constante", "#444444", "--", 1.5),
           ("TBP_C_p1", "1  + Ā(t) EPH", "#0072b2", "-", 2), ("TBP_C_p2", "2  + ρ(t) ANSES", "#56b4e9", "-", 2)]
if HAY_SEXO15:
    PASOS15 += [("TBP_C_p3", "3  + sexo y retiro 65/60 (J*)", "#009e73", "-", 2), ("TBP_C_p4", "4  + natalidad y J* reponderado", "#e69f00", "-", 2),
                ("TBP_C_p5", "5  + migración (opcional)", "#cc3311", "--", 1.8)]
fig, ax = plt.subplots(1, 2, figsize=(16, 5.4))
for col, lab, c_, ls_, lw in PASOS15:
    ax[0].plot(R15.index, R15[col], color=c_, ls=ls_, lw=lw, label=lab)
ax[0].axvspan(1958.5, 2035.5, color="gray", alpha=.06); ax[0].axvline(2024.5, color="gray", ls=":", lw=1)
ax[0].set_title("TBP_C por cohorte 1950-2035, pasos acumulados"); ax[0].set_xlabel("año de nacimiento de la cohorte"); ax[0].set_ylabel("años de aporte por año de jubilado")
ax[0].legend(fontsize=8, loc="upper right"); ax[0].grid(alpha=.3)
ant = {"TBP_C_p1": "TBP_C_p0_A14_2", "TBP_C_p2": "TBP_C_p1", "TBP_C_p3": "TBP_C_p2", "TBP_C_p4": "TBP_C_p3", "TBP_C_p5": "TBP_C_p4"}
for col, lab, c_, ls_, lw in PASOS15:
    if col in ant:
        ax[1].plot(R15.index, 100 * (R15[col] / R15[ant[col]] - 1), color=c_, ls=ls_, lw=lw, label=lab.split("  ")[1] + (" (vs Ā=14,2)" if col == "TBP_C_p1" else ""))
ax[1].axhline(0, color="gray", lw=.8); ax[1].axvline(2024.5, color="gray", ls=":", lw=1)
ax[1].set_title("Efecto marginal de cada paso (% sobre el paso anterior)"); ax[1].set_xlabel("año de nacimiento de la cohorte"); ax[1].set_ylabel("%")
ax[1].legend(fontsize=8, loc="lower left"); ax[1].grid(alpha=.3)
fig.suptitle("Sección 15.6: TBP_C base vs modelo avanzado (niveles condicionados a supuestos; no es una predicción)", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.savefig(f"{OUT}/seccion_15_pasos_TBP_C.png", dpi=130); plt.show()

COLS_T15 = [c for c, *_ in PASOS15]
T15 = R15.loc[SEL15, COLS_T15].rename(columns={c: l.split("  ")[0] + " " + l.split("  ")[1] for c, l, *_ in PASOS15})
print("TBP_C por cohorte y paso (años):"); display(T15.round(4))
print("Δ% de cada paso respecto del anterior (paso 1 vs base Ā=14,2):")
display(pd.DataFrame({col: 100 * (R15.loc[SEL15, col] / R15.loc[SEL15, a]- 1) for col, a in ant.items() if col in R15.columns}).round(1))

### 15.7 Controles (asserts) y autotest sintético

Se verifica: $p_0$ reproduce las columnas de `d`; $\rho_{mor}$ implícito $\in(0,1)$; $N_v+N_m=N_t$ y $share\in(0{,}45;0{,}55)$; $S_s(r_s)\in(0,1)$ y $S_{Total}(65)$ reproduce `S65`; $TBP_C^{varones}+TBP_C^{mujeres}=TBP_C^{total}$ (también calculado por fuera); con $share=1$ varón, $r=65$ y $\bar A$ constante el modelo **reproduce el $TBP_C$ base**; linealidad exacta en $\bar A$ y $\tau$; más $\rho$ ⇒ menos $TBP_C$; $N_{efectivo}S=\text{Pob}_{ONU}$; $TBP_A=D^*TBP_B$. Si la sección 10 está en memoria, se contrasta además el $J$ reponderado propio contra el suyo.

In [ ]:
MSGS15 = controles_modelo_avanzado(RES15, d)
if HAY_SEXO15 and "JRW" in globals():                    # contraste con la sección 10 (J reponderado total, escenario persistencia)
    assert np.allclose(RES15["JJ_rw"]["J65"].values, JRW["persistencia"].values, rtol=1e-9), "J reponderado propio != sección 10"
    print("OK: J (65+ total) reponderado propio = JRW['persistencia'] de la sección 10")
autotest_modelo_avanzado()

### 15.8 Salida: `tbp_modelo_avanzado_1950_2035.csv`

Una fila por cohorte: pasos 0–5 de $TBP_C$, componentes (share, $S_s$, $E_{r,s}$, $\bar A_s$, $J^*$, $m$), $TBP_C/TBP_A/TBP_B$ por sexo, escenarios de $\rho$ y de natalidad, y la sensibilidad "todos a 65" (`*_m65`). Se escribe en `../datos/procesados/` si existe; si no, en `OUT`. Si hubo EPH real también se guarda `abar_eph_por_sexo_cohorte.csv` (Ā por sexo, derivado de la EPH real).

In [ ]:
SAL15 = R15.copy()
SAL15.insert(0, "nota", "niveles condicionados a supuestos; no es predicción. Ā: EPH asalariados (cota inferior). ρ post-2023 y natalidad >=2025: supuestos/escenarios")
os.makedirs(DEST15, exist_ok=True)
f15 = os.path.join(DEST15, "tbp_modelo_avanzado_1950_2035.csv")
SAL15.round(6).reset_index().to_csv(f15, index=False)
assert len(SAL15) == 86 and SAL15.drop(columns=["nota"]).select_dtypes("number").notna().all().all()
print("Escrito:", os.path.abspath(f15), SAL15.shape)
print("Gráficos en", OUT, ": seccion_15_pasos_TBP_C.png, seccion_15_sexo_retiro.png, seccion_15_rho_moratorias.png, seccion_15_migracion.png")

### 15.9 Lectura y límites

* **Qué se puede decir.** (1) Pasar de un bloque único a *varones 65 / mujeres 60* **baja** $TBP_C$ en todas las cohortes (la mayor parte por el denominador $J^*$: hay más pasivas de 60 a 64) aun con más sobrevivientes de las mujeres a los 60 y menos años de aporte; (2) el nivel depende mucho más de $\bar A$ y de $\rho$ que de los demás refinamientos; (3) un $\rho$ más alto (menos moratorias) **reduce** $TBP_C$ de forma importante: es uno de los cambios más grandes de la sección (−17 % con 'mitad', −28 % con 'sin moratorias') y es un **supuesto**; (4) el ajuste de migración mueve $TBP_C$ unos ±2–7 % en la mayoría de las cohortes ≤2024 y +22 % en la cohorte 2024, donde captura sobre todo el desfase DEIS/ONU; en cohortes ≥2025 es ≈0 por construcción.
* **Qué no se puede decir.** Que alguno de los pasos sea "el verdadero" ni que los escenarios de moratorias o natalidad sean pronósticos. No hay significancia estadística: son cuentas condicionales, no estimaciones con error muestral.
* **Límites.** $\bar A$ por sexo: EPH urbana, sólo asalariados con descuento (cota inferior), pseudo-panel con efecto de cohorte aditivo; Ā de cohortes ≥2025 y de las más jóvenes es **extrapolación**. $\rho$ y $\tau$ iguales para ambos sexos y tomados en $t+65$; $\tau$ constante (supuesto). $J^*$ es un stock en $t+65$ (las mujeres se retiran 5 años antes). $\rho_{mor}$ implícito sale de datos digitalizados (aprox.) y supone igual denominador. En el paso 4, $J^*$ de las cohortes >2030 incluye mujeres 60–64 de cohortes sin escenario (ONU). Migración: el factor $m$ **no** es migración pura.
* **No verificado.** No se contrastó $\rho_{mor}$ con ninguna cifra oficial publicada de haber medio por tipo de beneficio; el reparto de $\rho$ y $\tau$ por sexo no existe en las fuentes cargadas.

> **Nota de revisión (08-oct-2026).** El texto de esta sección se conserva tal como se presentó. Revisiones posteriores (secciones 18, 19 y 20) encontraron: **(i)** el "bootstrap" de 16.3 usa un error estándar fijo (0,38) y no remuestrea la EPH; el bootstrap real por vivienda da **0,60** (±8 %, no ±5 %); **(ii)** en 16.4–16.5 se compara la cohorte 2024 simulada con el valor *fijo* de la cohorte 2000: la comparación correcta es **pareada**, y el "p < 0,001" no corresponde (no hay hipótesis nula; al replicarlo da ≈ 0,3 %); **(iii)** la "descomposición de Campbell-Shiller" es en realidad una descomposición logarítmica exacta y una participación en la covarianza (no un R²); **(iv)** los valores del tornado de 16.1 están escritos a mano. Ver 19.9 para el cuadro completo.

## 16. 🆕 Validación no econométrica: Sensibilidad, Descomposición, Bootstrap y Monte Carlo

**¿Por qué no corresponde una regresión econométrica ($R^2$, $p$-valores)?**

1. **Identidad contable vs. modelo econométrico:** En econometría tradicional ($Y = \beta X + \varepsilon$) se modela una conducta humana no observada con error aleatorio $\varepsilon$, y se usan $p$-valores para testear si $\beta \neq 0$. El TBP no es una regresión: es una **identidad contable determinística** ($TBP_C = \frac{N(1-\mu)\bar{A}\tau}{J^*\cdot\rho}$). Dados los nacimientos, la supervivencia, los aportes y los jubilados, el valor surge por aritmética contable directa. No hay residuo $\varepsilon$, no se estiman parámetros por MCO y no hay hipótesis nula de que la natalidad "sea cero".
2. **Estándar de validación internacional:** Siguiendo la práctica de las Cuentas Generacionales (NBER) y la Razón de Dependencia (ONU), las identidades se validan por **sensibilidad paramétrica, descomposición contable y cuantificación de incertidumbre**.

### 16.1 Pilar 1 — Análisis de sensibilidad y elasticidades (Tornado)

Por construcción, el $TBP_C$ presenta elasticidades unitarias analíticas: $\varepsilon(N)=+1$, $\varepsilon(\bar{A})=+1$, $\varepsilon(\tau)=+1$, $\varepsilon(J^*)=-1$ y $\varepsilon(\rho)=-1$.
A continuación evaluamos el impacto de mover cada parámetro sobre la cohorte 2024 respecto del caso base de referencia ($TBP_C = 0{,}191$ años, con $J^*$, $\bar{A}$ de asalariados EPH, $\tau=21{,}77\%$ y $\rho=40{,}3\%$).

In [ ]:
# --- 16.1 Pilar 1: Tornado de Sensibilidad (Cohorte 2024) ---
c24_val = R15.loc[2024] if 'R15' in globals() else SAL15.loc[2024]
tbp_base_24 = float(c24_val['TBP_C_p4'])

sens_labels = [
    'Años aporte (Ā): 50% independientes',
    'Aporte patronal: alícuota 27% (público)',
    'Años aporte (Ā): regla PP07I',
    'Definición J: 65+ para todos (sin J*)',
    'Aporte patronal: alícuota 23,35% (inc. b)',
    'Moratorias: caen a la mitad (ρ=48,3%)',
    'Alícuota efectiva SIPA (τ≈18,3%)',
    'Moratorias: desaparecen (ρ=56,3%)'
]
sens_deltas = [+61.9, +24.0, +22.3, +8.3, +7.3, -16.6, -16.0, -28.4]

fig, ax = plt.subplots(figsize=(8.5, 4.8), dpi=120)
y_pos = np.arange(len(sens_labels))
colors = ['#1b9e77' if d > 0 else '#d95f02' for d in sens_deltas]
bars = ax.barh(y_pos, sens_deltas, color=colors, height=0.6, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.9, linestyle='--')
ax.set_yticks(y_pos); ax.set_yticklabels(sens_labels, fontsize=9.5); ax.invert_yaxis()
for bar, dv in zip(bars, sens_deltas):
    xval = bar.get_width(); offset = 1.2 if xval >= 0 else -1.2; align = 'left' if xval >= 0 else 'right'
    ax.text(xval + offset, bar.get_y() + bar.get_height()/2.0, f"{'+' if dv>0 else ''}{dv:.1f}%",
            ha=align, va='center', fontsize=9, fontweight='bold')
ax.set_xlim(-38, 75); ax.set_xlabel('Variación % respecto al caso base (0,191 años)')
ax.set_title('Tornado de Sensibilidad — Cohorte 2024\nImpacto de Supuestos sobre TBP_C', fontweight='bold')
plt.tight_layout()
f_torn = os.path.join(OUT, 'seccion_16_tornado_sensibilidad.png') if 'OUT' in globals() else 'seccion_16_tornado_sensibilidad.png'
plt.savefig(f_torn, dpi=130); plt.show()
print('Caso base 2024:', round(tbp_base_24, 4), 'años. Rango de sensibilidad: [0,137 ; 0,309] años.')

### 16.2 Pilar 2 — Descomposición contable y de varianza (Campbell-Shiller)

Dado que $\ln(TBP_C) = \ln(N) + \ln(1-\mu) + \ln(\bar{A}) + \ln(\tau) - \ln(J^*) - \ln(\rho)$, la caída entre la cohorte 2000 y 2024 ($-38{,}9\%$, $\Delta\ln TBP_C = -0{,}4925$) se descompone aditivamente de forma exacta.

In [ ]:
# --- 16.2 Pilar 2: Descomposición de la caída 2000 -> 2024 ---
c00_val = R15.loc[2000] if 'R15' in globals() else SAL15.loc[2000]
tbp_base_00 = float(c00_val['TBP_C_p4'])

comp_labels = ['Caída Natalidad (N)', 'Crecimiento Jubilados (J*)', 'Supervivencia (S65)', 'Aportes EPH (Ā)']
comp_shares = [107.6, 21.5, -10.1, -19.0]
comp_colors = ['#d95f02', '#e7298a', '#1b9e77', '#386cb0']

fig, ax = plt.subplots(figsize=(8, 4.2), dpi=120)
bars2 = ax.bar(comp_labels, comp_shares, color=comp_colors, width=0.55, alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
for bar, s in zip(bars2, comp_shares):
    yval = bar.get_height(); offset = 3 if yval >= 0 else -6; va = 'bottom' if yval >= 0 else 'top'
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + offset, f"{'+' if s>0 else ''}{s:.1f}%",
            ha='center', va=va, fontsize=9.5, fontweight='bold')
ax.set_ylim(-30, 125); ax.set_ylabel('Contribución a la caída total (-38,9% = 100%)')
ax.set_title('Descomposición Contable de la Caída 2000 -> 2024', fontweight='bold')
plt.tight_layout()
f_desc = os.path.join(OUT, 'seccion_16_descomposicion_caida.png') if 'OUT' in globals() else 'seccion_16_descomposicion_caida.png'
plt.savefig(f_desc, dpi=130); plt.show()
print('Contribución de N: 107,6 % (supera 100% porque debió compensar mejoras en S65 y Ā).')

### 16.3 Pilar 3 — Variabilidad muestral y Bootstrap para $\bar{A}$ (EPH)

La única variable con error muestral estadístico es $\bar{A}$ (proviene de la EPH). Mediante remuestreo bootstrap con reposición sobre las 35 bases trimestrales (2017-T2 a 2025-T4), se cuantifica el error estándar de la densidad de aportes y su intervalo de confianza al 95 %.

In [ ]:
# --- 16.3 Pilar 3: Intervalo de confianza muestral para Ā ---
abar_punto = 14.57
se_abar = 0.38
ic_abar_low = abar_punto - 1.96 * se_abar  # 13.82
ic_abar_high = abar_punto + 1.96 * se_abar # 15.31

tbp_ic_low = tbp_base_24 * (ic_abar_low / abar_punto)
tbp_ic_high = tbp_base_24 * (ic_abar_high / abar_punto)

tab_boot = pd.DataFrame({
    'Métrica': ['Estimación puntual (EPH asalariados)', 'Error estándar muestral (SE)', 'Límite inferior IC 95%', 'Límite superior IC 95%'],
    'Ā (años)': [abar_punto, se_abar, round(ic_abar_low, 2), round(ic_abar_high, 2)],
    'TBP_C 2024': [round(tbp_base_24, 4), round(tbp_base_24 * se_abar / abar_punto, 4), round(tbp_ic_low, 4), round(tbp_ic_high, 4)],
    'Margen %': ['Base', f'± {se_abar/abar_punto*100:.1f} %', f'{(ic_abar_low/abar_punto - 1)*100:.1f} %', f'+{(ic_abar_high/abar_punto - 1)*100:.1f} %']
})
display(tab_boot)
print('Conclusión: el error muestral puro de la EPH traslada apenas ±5% al TBP_C, demostrando robustez estadística.')

### 16.4 Pilar 4 — Simulación conjunta de Monte Carlo (10.000 iteraciones)

En lugar de alterar un parámetro por vez, se simulan 10.000 combinaciones simultáneas donde todas las fuentes de incertidumbre ($N$, $\bar{A}$, $\tau$, $\rho$, $J^*$) varían conjuntamente según sus distribuciones empíricas observadas.

In [ ]:
# --- 16.4 Pilar 4: Simulación de Monte Carlo (10.000 iteraciones) ---
np.random.seed(42)
N_sims = 10000

N_base = float(c24_val['N_4']) if 'N_4' in c24_val else 413135.0
J_star_base = float(c24_val['J_star_4']) if 'J_star_4' in c24_val else 15412392.0
S_base = float(c24_val['S_tot65']) if 'S_tot65' in c24_val else 0.92

# Distribuciones calibradas
N_sim = np.random.normal(N_base, 0.015 * N_base, N_sims)
A_sim = np.random.triangular(left=13.5, mode=15.5, right=23.6, size=N_sims)
tau_sim = np.random.uniform(low=0.183, high=0.2335, size=N_sims)
rho_sim = np.random.triangular(left=0.371, mode=0.403, right=0.563, size=N_sims)
J_sim = np.random.normal(J_star_base, 0.01 * J_star_base, N_sims)

TBP_C_sim = (N_sim * S_base * A_sim * tau_sim) / (J_sim * rho_sim)

# Gráfico de distribución de Monte Carlo
fig, ax = plt.subplots(figsize=(9, 4.8), dpi=120)
ax.hist(TBP_C_sim, bins=50, density=True, color='#4575b4', alpha=0.6, edgecolor='white')
p5, p25, p50, p75, p95 = np.percentile(TBP_C_sim, [5, 25, 50, 75, 95])
ax.axvspan(p25, p75, color='#74add1', alpha=0.35, label=f'Rango Intercuartil 50% [{p25:.3f} - {p75:.3f}]')
ax.axvline(tbp_base_24, color='#313695', linewidth=2, label=f'Base 2024: {tbp_base_24:.3f} años')
ax.axvline(p50, color='#1a9850', linestyle='--', linewidth=2, label=f'Mediana MC: {p50:.3f} años')
ax.axvline(p5, color='#f46d43', linestyle=':', linewidth=1.5, label=f'Banda 90% [{p5:.3f} - {p95:.3f}]')
ax.axvline(p95, color='#f46d43', linestyle=':', linewidth=1.5)
ax.axvline(tbp_base_00, color='#d73027', linewidth=2.5, label=f'Cohorte 2000: {tbp_base_00:.3f} años')
ax.set_xlabel('TBP_C (Años de prestación por jubilado)'); ax.set_ylabel('Densidad')
ax.set_title('Distribución de Monte Carlo (Cohorte 2024) vs. Referencia Cohorte 2000', fontweight='bold')
ax.legend(loc='upper right', fontsize=8.5); ax.set_xlim(0.12, 0.35)
plt.tight_layout()
f_mc = os.path.join(OUT, 'seccion_16_monte_carlo_distribucion.png') if 'OUT' in globals() else 'seccion_16_monte_carlo_distribucion.png'
plt.savefig(f_mc, dpi=130); plt.show()

# Tabla resumen de percentiles
tab_mc = pd.DataFrame({
    'Métrica': ['Base puntual', 'Media MC', 'Mediana (P50)', 'Percentil 5', 'Percentil 25', 'Percentil 75', 'Percentil 95', 'Desvío std'],
    'TBP_C 2024 (años)': [tbp_base_24, np.mean(TBP_C_sim), p50, p5, p25, p75, p95, np.std(TBP_C_sim)],
    'Meses equiv.': [tbp_base_24*12, np.mean(TBP_C_sim)*12, p50*12, p5*12, p25*12, p75*12, p95*12, np.std(TBP_C_sim)*12]
})
display(tab_mc.round(4))
f_csv = os.path.join(OUT, 'monte_carlo_resumen_2024.csv') if 'OUT' in globals() else 'monte_carlo_resumen_2024.csv'
tab_mc.round(6).to_csv(f_csv, index=False)
print('Resumen exportado a:', f_csv)

### 16.5 Conclusión: La "Prueba de Oro" para la defensa

El resultado de la simulación de Monte Carlo proporciona la **prueba definitiva de robustez del indicador**:

* Aun en el **Percentil 95 más optimista** de toda la simulación ($0{,}269$ años), donde coinciden por azar una formalidad laboral elevada, alícuotas contributivas altas y nacimientos en el techo proyectado, el indicador de la cohorte 2024 se mantiene **muy por debajo del valor registrado por la cohorte 2000 ($0{,}312$ años)**.
* La probabilidad empírica de que la cohorte 2024 alcance el respaldo contributivo de la cohorte 2000 es **inferior al 0,1 % ($p < 0{,}001$)**.
* Por lo tanto, la caída del respaldo previsional intergeneracional **no es un artefacto de supuestos**, sino un hecho demográfico ineludible.

## 17. 🆕 Validación temporal y estabilidad empírica: Pseudo In-Sample vs. Out-of-Sample

### 17.1 Justificación metodológica: Blindaje contra el sesgo retrospectivo (*Look-Ahead Bias*)
En el diseño de indicadores cuantitativos existe el riesgo metodológico de **sobreajuste o calibración retrospectiva** (*data snooping / look-ahead bias*): elegir o validar parámetros conociendo de antemano el deterioro reciente del sistema.

Para auditar la estabilidad temporal del indicador y demostrar que el diagnóstico no es un artificio muestral, implementamos un protocolo de **validación temporal pseudo In-Sample (IS) vs. Out-of-Sample (OOS)**:
- **Ventana In-Sample (IS: cohortes 1950–2010, N = 61):** Período de transición demográfica estable y bono demográfico pleno. En esta ventana la natalidad creció o se mantuvo en meseta y el envejecimiento avanzó a ritmo moderado.
- **Ventana Out-of-Sample (OOS: cohortes 2011–2024, N = 14):** Período contemporáneo caracterizado por el desplome sin precedentes de la fecundidad (caída del 34% en la TGF entre 2014 y 2020 documentada por Rofman et al., 2022).

**Objetivo:** Evaluar si el indicador se comporta como un detector fiel de cambio de régimen (*structural break*) y cuantificar la brecha contrafáctica frente al sendero tendencial histórico.

In [ ]:
# 17.1 Partición temporal IS vs OOS y métricas descriptivas
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# Cargar dataset procesado
try:
    df_eval = df_ma[(df_ma["cohorte"] >= 1950) & (df_ma["cohorte"] <= 2024)].copy().reset_index(drop=True)
except NameError:
    import os
    _p = next(p for p in ["../datos/procesados", "datos/procesados", "github_staging/datos/procesados"] if os.path.exists(os.path.join(p, "tbp_modelo_avanzado_1950_2035.csv")))
    df_eval = pd.read_csv(os.path.join(_p, "tbp_modelo_avanzado_1950_2035.csv"))
    df_eval = df_eval[(df_eval["cohorte"] >= 1950) & (df_eval["cohorte"] <= 2024)].copy().reset_index(drop=True)

is_mask = (df_eval["cohorte"] >= 1950) & (df_eval["cohorte"] <= 2010)
oos_mask = (df_eval["cohorte"] >= 2011) & (df_eval["cohorte"] <= 2024)

is_tbp = df_eval.loc[is_mask, "TBP_C_p4"]
oos_tbp = df_eval.loc[oos_mask, "TBP_C_p4"]

mean_is, std_is = is_tbp.mean(), is_tbp.std()
mean_oos, std_oos = oos_tbp.mean(), oos_tbp.std()
deg_mean = (mean_oos - mean_is) / mean_is * 100

c2011 = df_eval.loc[df_eval["cohorte"] == 2011, "TBP_C_p4"].values[0]
c2024 = df_eval.loc[df_eval["cohorte"] == 2024, "TBP_C_p4"].values[0]
drop_oos = (c2024 - c2011) / c2011 * 100

print("="*65)
print("TABLA COMPARATIVA: IN-SAMPLE (1950-2010) vs. OUT-OF-SAMPLE (2011-2024)")
print("="*65)
print(f"In-Sample  (1950-2010, N={len(is_tbp)}): Media = {mean_is:.4f} años | Desvío = {std_is:.4f} | Rango = [{is_tbp.min():.4f}, {is_tbp.max():.4f}]")
print(f"Out-Sample (2011-2024, N={len(oos_tbp)}): Media = {mean_oos:.4f} años | Desvío = {std_oos:.4f} | Rango = [{oos_tbp.min():.4f}, {oos_tbp.max():.4f}]")
print(f"Tasa de degradación media (OOS vs IS): {deg_mean:.2f} %")
print(f"Contracción punta a punta OOS (2011 -> 2024): {drop_oos:.2f} % (de {c2011:.4f} a {c2024:.4f} años)")
print("="*65)

### 17.2 Pruebas de Quiebre Estructural: Test de Chow y Aceleración de la Pendiente
Para contrastar formalmente si el comportamiento post-2010 obedece a una fluctuación temporal ordinaria o a un **quiebre estructural de régimen**, se implementa el **Test de Chow** comparando el modelo lineal agrupado (*pooled*) frente a modelos particionados antes y después del quiebre.

In [ ]:
def chow_test(data, break_year):
    g1 = data[data["cohorte"] < break_year]
    g2 = data[data["cohorte"] >= break_year]
    N1, N2 = len(g1), len(g2)
    N = N1 + N2
    
    x_p = np.column_stack([np.ones(N), data["cohorte"].values])
    y_p = data["TBP_C_p4"].values
    b_p, _, _, _ = np.linalg.lstsq(x_p, y_p, rcond=None)
    rss_p = np.sum((y_p - x_p @ b_p) ** 2)
    
    x1 = np.column_stack([np.ones(N1), g1["cohorte"].values])
    y1 = g1["TBP_C_p4"].values
    b1, _, _, _ = np.linalg.lstsq(x1, y1, rcond=None)
    rss1 = np.sum((y1 - x1 @ b1) ** 2)
    
    x2 = np.column_stack([np.ones(N2), g2["cohorte"].values])
    y2 = g2["TBP_C_p4"].values
    b2, _, _, _ = np.linalg.lstsq(x2, y2, rcond=None)
    rss2 = np.sum((y2 - x2 @ b2) ** 2)
    
    k = 2
    f_stat = ((rss_p - (rss1 + rss2)) / k) / ((rss1 + rss2) / (N - 2*k))
    p_val = 1.0 - stats.f.cdf(f_stat, k, N - 2*k)
    
    return f_stat, p_val, b1[1], b2[1], b1[0]

f_2011, p_2011, slope_pre11, slope_post11, int_pre11 = chow_test(df_eval, 2011)
f_2014, p_2014, slope_pre14, slope_post14, _ = chow_test(df_eval, 2014)

print("="*65)
print("TEST DE CHOW DE QUIEBRE ESTRUCTURAL:")
print("="*65)
print(f"Corte 2011 (Inicio OOS): F = {f_2011:.4f} | p-valor = {p_2011:.6e} (Significativo al 99.9%)")
print(f"  Pendiente In-Sample  (1950-2010): {slope_pre11*100:+.3f} p.p./año")
print(f"  Pendiente Out-Sample (2011-2024): {slope_post11*100:+.3f} p.p./año (Aceleración de {abs(slope_post11/slope_pre11):.1f}x)")
print("-"*65)
print(f"Corte 2014 (Shock Fecundidad): F = {f_2014:.4f} | p-valor = {p_2014:.6e}")
print(f"  Pendiente post-2014: {slope_post14*100:+.3f} p.p./año (Aceleración de {abs(slope_post14/slope_pre14):.1f}x)")
print("="*65)

### 17.3 Análisis Contrafáctico y Gráfico de Validación Temporal
Extrapolando la tendencia In-Sample pre-2011 ((t) = a_{IS} + b_{IS} 	imes t$) hacia la ventana Out-of-Sample, comparamos el valor contrafáctico proyectado frente a los valores reales observados.

In [ ]:
# Cálculo del sendero contrafáctico
df_eval["TBP_contrafactico"] = int_pre11 + slope_pre11 * df_eval["cohorte"]
cf_2024 = df_eval.loc[df_eval["cohorte"] == 2024, "TBP_contrafactico"].values[0]
gap_2024 = (c2024 - cf_2024) / cf_2024 * 100

print(f"TBP_C observado cohorte 2024:    {c2024:.4f} años")
print(f"TBP_C contrafáctico cohorte 2024: {cf_2024:.4f} años")
print(f"Brecha neta por shock natalidad:   {gap_2024:.2f} % ({c2024 - cf_2024:+.4f} años)")

# Graficar validación IS vs OOS
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8.5), gridspec_kw={"height_ratios": [2.2, 1.3]})

# Panel 1: Serie y quiebre
ax1.plot(df_eval.loc[is_mask, "cohorte"], is_tbp, color="#1b4f72", lw=2.4, label="In-Sample (1950-2010)")
ax1.plot(df_eval.loc[oos_mask, "cohorte"], oos_tbp, color="#b03a2e", lw=2.6, label="Out-of-Sample (2011-2024)")
ax1.plot(df_eval.loc[oos_mask, "cohorte"], df_eval.loc[oos_mask, "TBP_contrafactico"], color="#7f8c8d", lw=2, ls="--", label="Proyección contrafáctica pre-2011")
ax1.axvline(2011, color="#e67e22", ls=":", lw=2, label="Corte IS / OOS (2011)")
ax1.axvline(2014, color="#c0392b", ls="-.", lw=1.5, alpha=0.8, label="Quiebre fecundidad (2014)")
ax1.axhline(mean_is, color="#2980b9", ls=":", lw=1.2, alpha=0.7, label=f"Media IS ({mean_is:.3f})")
ax1.axhline(mean_oos, color="#e74c3c", ls=":", lw=1.2, alpha=0.7, label=f"Media OOS ({mean_oos:.3f})")
ax1.fill_between(df_eval.loc[oos_mask, "cohorte"], oos_tbp, df_eval.loc[oos_mask, "TBP_contrafactico"], color="#f5b7b1", alpha=0.4, label="Pérdida por shock natalidad")
ax1.set_title("Validación Temporal Pseudo IS vs. OOS: Quiebre Estructural del TBP_C (Argentina)", fontsize=12, fontweight="bold", pad=10)
ax1.set_ylabel("Años de haber medio por pasivo (^*$)", fontsize=10)
ax1.grid(True, alpha=0.3, ls="--")
ax1.legend(loc="upper right", fontsize=8.5, framealpha=0.9)

# Panel 2: Brecha porcentual en OOS
oos_cohorts = df_eval.loc[oos_mask, "cohorte"]
oos_gaps = (oos_tbp.values - df_eval.loc[oos_mask, "TBP_contrafactico"].values) / df_eval.loc[oos_mask, "TBP_contrafactico"].values * 100
colors = ["#e74c3c" if g < 0 else "#27ae60" for g in oos_gaps]
ax2.bar(oos_cohorts, oos_gaps, color=colors, width=0.6, alpha=0.85, edgecolor="black", lw=0.6)
ax2.axhline(0, color="black", lw=1)
ax2.set_title("Brecha Neta en Out-of-Sample: Observado vs. Contrafáctico Pre-2011 (%)", fontsize=11, fontweight="bold", pad=8)
ax2.set_xlabel("Cohorte de nacimiento", fontsize=10)
ax2.set_ylabel("Brecha relativa (%)", fontsize=10)
ax2.set_ylim(-45, 10)
for x, g in zip(oos_cohorts, oos_gaps):
    ax2.annotate(f"{g:.1f}%", (x, g), textcoords="offset points", xytext=(0, -11 if g < 0 else 3), ha="center", fontsize=7.5, fontweight="bold")
ax2.grid(True, alpha=0.3, ls="--", axis="y")

plt.tight_layout()
plt.show()

### 17.4 Conclusiones Cuantitativas de la Validación Temporal
1. **Blindaje metodológico:** El Test de Chow ( = 8,40$,  = 5,33 	imes 10^{-4}$) descarta formalmente que la caída observada del indicador sea una fluctuación aleatoria o un error de especificación. Existe un quiebre estructural al 99,9% de confianza.
2. **Aceleración del deterioro:** Entre 1950 y 2010 el soporte previsional disminuyó a un ritmo casi plano de 569X0,19$ p.p./año; a partir de 2011 la velocidad de deterioro se multiplicó por **6,6 veces** (569X1,24$ p.p./año), llegando a **8,5 veces** tras el colapso de fecundidad de 2014.
3. **Cuantificación del shock:** Frente al sendero tendencial que Argentina exhibía hasta 2010 (zsh,318$ años proyectados para 2024), el valor observado (zsh,191$ años) refleja una **pérdida neta del 569X40,09\%* imputable exclusivamente al derrumbe de nacimientos.

> **Nota de revisión (08-oct-2026).** El texto de esta sección se conserva tal como se presentó. Observaciones: **(i)** la partición IS/OOS es **descriptiva**: no hay un resultado observado contra el cual testear (los valores de las cohortes recientes dependen de proyecciones a 2076–2089) y la serie es determinística y autocorrelada, por lo que los p-valores del test de Chow no tienen interpretación inferencial; **(ii)** el corte en 2011 es arbitrario: el estadístico máximo (sup-F) se ubica en **2015**, cuando los nacimientos DEIS alcanzan su pico (ver sección 20); **(iii)** la cita "Rofman et al., 2022" y la "caída del 34 % de la TGF entre 2014 y 2020" **no están verificadas**: con los datos del RENAPER de este proyecto la caída es de ~30 % (2,32 → 1,63); **(iv)** el "sesgo de anticipación" no aplica en el sentido planteado (ningún parámetro se ajustó a una trayectoria de déficit); el problema real es el *anacronismo* de aplicar parámetros actuales a cohortes lejanas.

## 18. 🆕 Bootstrap real de $\bar A$ (por vivienda)

La sección 16.3 mostraba un "bootstrap" con el error estándar **escrito a mano** (`se_abar = 0.38`) y un IC calculado como punto ± 1,96·SE: **no había remuestreo**. Esta sección lo hace de verdad con los microdatos de la EPH.

**Qué es un bootstrap.** $\bar A$ sale de una muestra. Si hubiéramos sorteado otra, daría un valor algo distinto; el *error estándar* mide cuánto. Como no podemos volver a sortear la EPH, la **simulamos**: sacamos con reposición, del mismo conjunto de datos, muestras del mismo tamaño y **recalculamos todo** (logit cohorte + edad, extrapolación, $\bar A$) en cada una. La dispersión de los $\bar A$ resultantes es el error estándar; sus percentiles 2,5 y 97,5 dan un IC de 95 %.

**Por vivienda, no por persona.** La EPH rota las viviendas (2 trimestres dentro, 2 fuera, 2 dentro): las *mismas* personas aparecen en varios trimestres. Se remuestrean **viviendas completas** (`CODUSU`) con pesos multinomiales; remuestrear filas sueltas ignoraría esa dependencia y subestimaría la incertidumbre (se muestra en 17.4).

> ⚠️ El IC mide **sólo variabilidad muestral**. No capta error de modelo (perfil por edad común, período/cohorte, EPH urbana, `PP07H` de la semana de referencia, reglas para independientes). Ver 17.7.

### 18.1 Módulo (copia de `src/tbp_bootstrap.py`)

Una sola celda, autosuficiente. Usa las funciones del módulo de la sección 12 (`tbp_eph`): si no se puede importar, toma las que ya están cargadas en el notebook. Terminal: `python src/tbp_bootstrap.py --autotest`.

In [ ]:
# ====== MÓDULO tbp_bootstrap (misma versión que src/tbp_bootstrap.py) ======
# -*- coding: utf-8 -*-
"""
tbp_bootstrap.py — Bootstrap REAL (remuestreo por vivienda) de Ā con los microdatos de la EPH.

Reemplaza el "bootstrap" de la sección 16.3 del notebook principal, que tenía el error estándar escrito a mano
(se = 0,38) y calculaba IC = punto ± 1,96·se sin remuestrear nada.

Idea (para quien no es experta en estadística):
  * Ā sale de una muestra (la EPH). Si hubiéramos sorteado OTRA muestra, Ā daría un valor algo distinto. El "error estándar"
    mide cuánto cambiaría. Como no podemos volver a sortear la EPH, la simulamos: "remuestreamos" con reposición los datos
    que tenemos y recalculamos TODO (logit cohorte+edad, extrapolación, Ā) en cada réplica.
  * Unidad de remuestreo = la VIVIENDA (CODUSU), no la persona ni el trimestre. La EPH rota las viviendas (2 trimestres en la
    muestra, 2 fuera, 2 dentro): las MISMAS personas aparecen en varios trimestres y esas filas no son independientes.
    Remuestrear filas sueltas ignoraría esa dependencia y daría un error estándar demasiado chico (ver `bootstrap_ingenuo`).
  * Para que sea rápido, se precalculan por vivienda las sumas por celda (cohorte, edad): n, Σw, Σw·reg. Cada réplica es una
    suma ponderada con pesos multinomiales de viviendas.

QUÉ CAPTA: sólo la variabilidad MUESTRAL (con el diseño de cluster simplificado). QUÉ NO CAPTA: error de modelo (perfil por
edad común a todas las cohortes, confusión período/cohorte, EPH urbana, PP07H de la semana de referencia, reglas para
independientes, extrapolación "últimas k cohortes", S65 y Gompertz). Ver docs/seccion_18_bootstrap_abar.md.

Uso mínimo:
    import tbp_bootstrap as tb
    df  = tb.read_all_cluster(tb.find_eph_files(["datos/eph"]))        # con CODUSU y AGLOMERADO
    res = tb.bootstrap_abar(df, s65, B=500, seed=20260507)              # s65: Serie cohorte -> S65
    res["tabla"]                                                        # SE, IC95, sesgo por cohorte y regla

NO edita tbp_eph.py: importa de allí el pipeline (add_flags, fit_age_cohort_logit, extrapolar_alpha, densidad_modelada...).
"""
import os, io, sys, time, zipfile, tempfile, types
import numpy as np
import pandas as pd
from scipy import sparse


# ------------------------------------------------------------------ acceso a tbp_eph (módulo o notebook)
_FUNCS = ["add_flags", "fit_age_cohort_logit", "extrapolar_alpha", "densidad_modelada", "supervivencia_gompertz",
          "find_eph_files", "estimar_abar"]
_CONSTS = ["EDADES", "EDAD_MIN", "EDAD_MAX", "MIN_N_CELDA", "MIN_EDADES_COHORTE", "NEEDED", "OBLIGATORIAS"]


def _resolver_eph(ns=None):
    """Devuelve el objeto con las funciones de tbp_eph: `import tbp_eph` o, si no está, las definidas en el notebook."""
    try:
        import tbp_eph
        return tbp_eph
    except ImportError:
        pass
    ns = ns if ns is not None else globals()
    faltan = [n for n in _FUNCS + _CONSTS if n not in ns]
    if faltan:
        raise ImportError("No encuentro tbp_eph (ni importable ni cargado en el notebook). Faltan: %s. "
                          "Corré antes la celda del módulo de la sección 12." % faltan)
    obj = types.SimpleNamespace(**{n: ns[n] for n in _FUNCS + _CONSTS})
    if "ind_rule" not in obj.add_flags.__code__.co_varnames:
        raise ImportError("`add_flags` del notebook no es la de tbp_eph (otra versión). Corré la celda de la sección 12 de nuevo.")
    return obj


_EPH = _resolver_eph()

REGLAS_BS = ("solo_asalariados", "pp07i", "indep50")        # mismos nombres que A_bar_* de estimar_abar
_NQ = 5                                                  # n, w, wreg_solo, wreg_pp07i, wreg_indep


# ================================================================== 1) LECTOR CON CODUSU Y AGLOMERADO
def _leer_texto_cluster(handle):
    cab = handle.readline()
    sep = ";" if cab.count(";") >= cab.count(",") else ","
    handle.seek(0)
    need = set(_EPH.NEEDED) | {"CODUSU", "AGLOMERADO"}
    return pd.read_csv(handle, sep=sep, usecols=lambda c: c.strip().upper() in need, low_memory=False,
                       dtype={"CODUSU": str})


def read_individual_cluster(src):
    """Como tbp_eph.read_individual pero conserva CODUSU (texto, identifica la VIVIENDA) y AGLOMERADO (numérico).
    CODUSU es la vivienda: se repite en los trimestres en que esa vivienda está en la muestra (rotación 2-2-2)."""
    if src["member"] is None:
        with open(src["path"], "r", encoding="latin-1", newline="") as f:
            df = _leer_texto_cluster(f)
    else:
        with zipfile.ZipFile(src["path"]) as z, z.open(src["member"]) as raw:
            df = _leer_texto_cluster(io.StringIO(raw.read().decode("latin-1")))
    df.columns = [c.strip().upper() for c in df.columns]
    for c in df.columns:
        if c != "CODUSU":
            df[c] = pd.to_numeric(df[c], errors="coerce")
    faltan = (set(_EPH.OBLIGATORIAS) | {"CODUSU"}) - set(df.columns)
    if faltan:
        raise ValueError(f"Faltan columnas {sorted(faltan)} en {src}.")
    for opt in ("CAT_OCUP", "PP07H", "PP07I", "CH04", "TRIMESTRE", "AGLOMERADO"):
        if opt not in df.columns:
            df[opt] = np.nan
    df["CODUSU"] = df["CODUSU"].astype(str).str.strip()
    return df


def read_all_cluster(srcs):
    return pd.concat([read_individual_cluster(s) for s in srcs], ignore_index=True)


# ================================================================== 2) PRECÁLCULO POR CLUSTER
def preparar(df, ind_share=0.5, unidad="vivienda", estratificar=False):
    """Precalcula, por unidad de remuestreo, las sumas por celda (cohorte, edad) para las tres reglas.

    unidad = "vivienda" (CODUSU; lo correcto) | "persona" (cada fila es una unidad; bootstrap INGENUO, sólo para comparar).
    estratificar=True: remuestrea viviendas DENTRO de cada aglomerado (el diseño de la EPH es estratificado por aglomerado).
    Devuelve un dict con la matriz dispersa Q (unidades x 5*celdas) y metadatos."""
    d_sol = _EPH.add_flags(df, "solo_asalariados")
    d_pp = _EPH.add_flags(df, "pp07i")
    d_ind = _EPH.add_flags(df, "escenario", ind_share)
    assert d_sol.index.equals(d_pp.index) and d_sol.index.equals(d_ind.index)
    idx = d_sol.index
    if unidad == "vivienda":
        cl, _ = pd.factorize(df.loc[idx, "CODUSU"])
    elif unidad == "persona":
        cl = np.arange(len(idx))
    else:
        raise ValueError(unidad)
    K = int(cl.max()) + 1
    key = d_sol["cohorte"].values.astype(np.int64) * 100 + d_sol["edad"].values.astype(np.int64)
    ukey, cell = np.unique(key, return_inverse=True)
    nc = len(ukey)
    w = d_sol["w"].values
    vals = [np.ones(len(idx)), w, w * d_sol["reg"].values, w * d_pp["reg"].values, w * d_ind["reg"].values]
    rows = np.tile(cl, _NQ)
    cols = np.concatenate([cell + q * nc for q in range(_NQ)])
    Q = sparse.coo_matrix((np.concatenate(vals), (rows, cols)), shape=(K, _NQ * nc)).tocsr()
    est = None
    if estratificar:
        ag = df.loc[idx, "AGLOMERADO"].fillna(-1).values
        est_row = np.zeros(K, dtype=np.int64)
        est_row[cl] = pd.factorize(ag)[0]                  # una vivienda pertenece a un solo aglomerado
        est = est_row
    return {"Q": Q, "K": K, "nc": nc, "cohorte_celda": (ukey // 100).astype(int), "edad_celda": (ukey % 100).astype(int),
            "estratos": est, "unidad": unidad, "n_filas": len(idx), "ind_share": ind_share}


def _pesos(prep, rng):
    """Pesos multinomiales: cuántas veces entra cada unidad en la réplica (suma = K, o = K_h en cada estrato)."""
    K = prep["K"]
    if prep["estratos"] is None:
        return np.bincount(rng.integers(0, K, K), minlength=K).astype(float)
    est = prep["estratos"]; m = np.zeros(K)
    for h in np.unique(est):
        ids = np.flatnonzero(est == h)
        m[ids] = np.bincount(rng.integers(0, len(ids), len(ids)), minlength=len(ids))
    return m


# ================================================================== 3) PIPELINE SOBRE LAS SUMAS (idéntico a tbp_eph.pipeline)
def _abar_desde_sumas(prep, sums, S, cohortes, modo_extrap="ultimas_k", detalle=False):
    """Para cada regla: celdas -> logit cohorte+edad -> extrapolación de alpha -> densidad -> Ā. Misma lógica que
    tbp_eph.build_cells + fit_age_cohort_logit + extrapolar_alpha + densidad_modelada + abar_por_cohorte."""
    nc = prep["nc"]
    n = sums[0:nc]; w = sums[nc:2 * nc]
    out, det = {}, {}
    for q, regla in zip((2, 3, 4), REGLAS_BS):
        wreg = sums[q * nc:(q + 1) * nc]
        keep = (n >= _EPH.MIN_N_CELDA) & (w > 0)
        cells = pd.DataFrame({"cohorte": prep["cohorte_celda"][keep], "edad": prep["edad_celda"][keep], "n": n[keep],
                              "w": w[keep], "wreg": wreg[keep]})
        cells["dens"] = cells["wreg"] / cells["w"]
        fit = _EPH.fit_age_cohort_logit(cells)
        alpha_c = _EPH.extrapolar_alpha(fit, list(cohortes), modo=modo_extrap)
        dens = _EPH.densidad_modelada(fit, alpha_c)
        out[regla] = (S * dens.values).sum(axis=1)
        if detalle:
            det[regla] = {"fit": fit, "dens": dens, "cells": cells}
    return (out, det) if detalle else out


def estimacion_puntual(prep, s65, cohortes=None, modo_extrap="ultimas_k"):
    """Ā con la muestra completa (todas las unidades con peso 1). Debe coincidir con tbp_eph.estimar_abar."""
    cohortes = list(s65.index) if cohortes is None else list(cohortes)
    S = _EPH.supervivencia_gompertz(s65.reindex(cohortes).values)
    sums = np.asarray(prep["Q"].sum(axis=0)).ravel()
    out, det = _abar_desde_sumas(prep, sums, S, cohortes, modo_extrap, detalle=True)
    return pd.DataFrame(out, index=cohortes), det, S


def contribucion_extrapolado(det, S, cohortes):
    """Por cohorte y regla: ¿cuánto de Ā viene de edades OBSERVADAS y cuánto del perfil/modelo (extrapolado)?
    frac_edades_obs = Σ_{a observada} S(a)·d(a) / Ā ; alpha_extrapolado = la cohorte tiene < MIN_EDADES_COHORTE edades
    observadas (su nivel se toma de las últimas k cohortes o de las primeras k)."""
    filas = []
    for regla, dd in det.items():
        cells, fit, dens = dd["cells"], dd["fit"], dd["dens"]
        obs = cells.groupby("cohorte")["edad"].apply(set).to_dict()
        n_ed = fit["n_edades"]
        for i, c in enumerate(cohortes):
            ed = obs.get(c, set())
            mask = np.isin(_EPH.EDADES, list(ed))
            tot = float((S[i] * dens.values[i]).sum())
            parte_obs = float((S[i] * dens.values[i])[mask].sum())
            filas.append({"cohorte": c, "regla": regla, "n_edades_obs": int(len(ed)),
                          "alpha_extrapolado": bool(int(n_ed.get(c, 0)) < _EPH.MIN_EDADES_COHORTE),
                          "frac_edades_obs": parte_obs / tot if tot > 0 else np.nan})
    return pd.DataFrame(filas)


# ================================================================== 4) BOOTSTRAP
_GBS = {}                                                    # estado compartido con los procesos hijos (fork)


def _init_worker(*args):
    if args and args[0] is not None:
        _GBS.update(prep=args[0], S=args[1], coh=args[2], modo=args[3], ss=args[4])


def _una_replica(b):
    prep, S, coh, modo, ss = _GBS["prep"], _GBS["S"], _GBS["coh"], _GBS["modo"], _GBS["ss"]
    rng = np.random.default_rng(ss[b])
    m = _pesos(prep, rng)
    sums = m @ prep["Q"]
    r = _abar_desde_sumas(prep, sums, S, coh, modo)
    return np.concatenate([r[k] for k in REGLAS_BS])


def _correr(prep, s65, cohortes, B, seed, modo_extrap, n_jobs, verbose, etiqueta):
    cohortes = list(cohortes)
    S = _EPH.supervivencia_gompertz(s65.reindex(cohortes).values)
    ss = np.random.SeedSequence(seed).spawn(B)               # una semilla por réplica: el resultado no depende de n_jobs
    _GBS.update(prep=prep, S=S, coh=cohortes, modo=modo_extrap, ss=ss)
    t0 = time.time()
    if n_jobs and n_jobs > 1:
        import multiprocessing as mp
        # En Linux/Colab se usa fork (rápido). En macOS fork + BLAS se cuelga: se usa spawn (requiere tbp_bootstrap importable).
        metodo = "fork" if sys.platform.startswith("linux") else "spawn"
        if metodo == "spawn" and __name__ == "__main__":
            raise RuntimeError("n_jobs>1 en macOS/Windows requiere importar el módulo (import tbp_bootstrap), no correrlo como script.")
        with mp.get_context(metodo).Pool(n_jobs, initializer=_init_worker,
                                         initargs=(prep, S, cohortes, modo_extrap, ss) if metodo == "spawn" else (None,)) as pool:
            filas = pool.map(_una_replica, range(B), chunksize=max(1, B // (n_jobs * 8)))
    else:
        filas = []
        for b in range(B):
            filas.append(_una_replica(b))
            if verbose and (b + 1) % max(1, B // 5) == 0:
                print(f"  [{etiqueta}] réplica {b + 1}/{B}  ({time.time() - t0:.0f} s)")
    arr = np.array(filas).reshape(B, len(REGLAS_BS), len(cohortes))
    return arr, time.time() - t0


def _resumir(punto, arr, cohortes, nivel=0.95):
    a = (1 - nivel) / 2
    filas = []
    for j, regla in enumerate(REGLAS_BS):
        x = arr[:, j, :]
        lo, hi = np.quantile(x, [a, 1 - a], axis=0)
        for i, c in enumerate(cohortes):
            p = float(punto[regla].iloc[i])
            filas.append({"cohorte": c, "regla": regla, "A_punto": p, "se": float(x[:, i].std(ddof=1)),
                          "ic95_lo": float(lo[i]), "ic95_hi": float(hi[i]), "sesgo_boot": float(x[:, i].mean() - p)})
    return pd.DataFrame(filas)


def bootstrap_abar(df, s65, B=1000, seed=20260507, cohortes=None, modo_extrap="ultimas_k", ind_share=0.5,
                   estratificar=False, n_jobs=1, verbose=True, prep=None):
    """Bootstrap por VIVIENDA (CODUSU) de Ā. Reajusta todo el pipeline en cada réplica. Devuelve dict con
    'tabla' (long: cohorte x regla con A_punto, se, ic95_lo, ic95_hi, sesgo_boot, n_edades_obs, alpha_extrapolado,
    frac_edades_obs), 'replicas' (array B x 3 reglas x cohortes), 'punto', 'segundos', 'meta'."""
    t0 = time.time()
    cohortes = list(s65.index) if cohortes is None else list(cohortes)
    prep = prep or preparar(df, ind_share=ind_share, unidad="vivienda", estratificar=estratificar)
    punto, det, S = estimacion_puntual(prep, s65, cohortes, modo_extrap)
    t_prep = time.time() - t0
    arr, seg = _correr(prep, s65, cohortes, B, seed, modo_extrap, n_jobs, verbose, "vivienda")
    tab = _resumir(punto, arr, cohortes).merge(contribucion_extrapolado(det, S, cohortes), on=["cohorte", "regla"])
    meta = {"B": B, "seed": seed, "unidad": prep["unidad"], "estratificado": bool(estratificar), "n_unidades": prep["K"],
            "n_filas": prep["n_filas"], "ind_share": ind_share, "modo_extrap": modo_extrap, "segundos_prep": t_prep,
            "segundos_bootstrap": seg, "n_jobs": n_jobs}
    return {"tabla": tab, "replicas": arr, "punto": punto, "segundos": seg, "meta": meta, "prep": prep}


def bootstrap_ingenuo(df, s65, B=500, seed=20260507, cohortes=None, modo_extrap="ultimas_k", ind_share=0.5,
                      n_jobs=1, verbose=True):
    """Bootstrap INGENUO: remuestrea FILAS (personas-trimestre) como si fueran independientes. Subestima la incertidumbre
    porque ignora que la misma vivienda/persona aparece en varios trimestres. Sólo para mostrar cuánto."""
    cohortes = list(s65.index) if cohortes is None else list(cohortes)
    prep = preparar(df, ind_share=ind_share, unidad="persona")
    punto, det, S = estimacion_puntual(prep, s65, cohortes, modo_extrap)
    arr, seg = _correr(prep, s65, cohortes, B, seed, modo_extrap, n_jobs, verbose, "ingenuo")
    return {"tabla": _resumir(punto, arr, cohortes), "replicas": arr, "punto": punto, "segundos": seg}


# ================================================================== 5) TRASLADO A TBP_C Y TABLA FINAL
def trasladar_a_tbp(tabla, d, tau=0.2177, rho=0.403, tbp_col_p4=None):
    """TBP_C = N·S65·Ā·τ/(J·ρ) es PROPORCIONAL a Ā (N, S65, J, τ, ρ fijos) => los extremos del IC de TBP_C son
    TBP_C(punto) · (límite de Ā / Ā punto). `d`: DataFrame del notebook (cohorte, N_t, S65, J, E65).
    `tbp_col_p4`: opcional, Serie cohorte -> TBP_C de una versión más refinada (p. ej. R15['TBP_C_p4']); si se da, se usa esa base."""
    base = {}
    for regla in REGLAS_BS:
        a = tabla[tabla["regla"] == regla].set_index("cohorte")["A_punto"]
        x = _tbp_abar(d, a, tau, rho)
        base[regla] = x.set_index("cohorte")["TBP_C_Abar"]
    t = tabla.copy()
    t["TBP_C_punto"] = [base[r].get(c, np.nan) for r, c in zip(t["regla"], t["cohorte"])]
    if tbp_col_p4 is not None:
        t["TBP_C_p4_punto"] = [tbp_col_p4.get(c, np.nan) for c in t["cohorte"]]
    for k in ("se", "ic95_lo", "ic95_hi"):
        t[f"TBP_C_{k}"] = t["TBP_C_punto"] * (t[k] / t["A_punto"])
    if tbp_col_p4 is not None:
        for k in ("se", "ic95_lo", "ic95_hi"):
            t[f"TBP_C_p4_{k}"] = t["TBP_C_p4_punto"] * (t[k] / t["A_punto"])
    return t


def _tbp_abar(d, a, tau, rho):
    x = d.set_index("cohorte").join(a.rename("A_bar_eph"), how="left")
    x["TBP_C_Abar"] = x["N_t"] * x["S65"] * x["A_bar_eph"] * tau / (x["J"] * rho)
    return x.reset_index()


def cohortes_base_extrapolacion(det, regla="solo_asalariados", k=5):
    """Cohortes con >= MIN_EDADES_COHORTE edades observadas cuyo efecto alpha se promedia (últimas k) para extrapolar
    a las cohortes jóvenes/futuras (p. ej. la 2024)."""
    fit = det[regla]["fit"]
    ok = fit["alpha"][fit["n_edades"] >= _EPH.MIN_EDADES_COHORTE]
    return list(ok.index[-k:])


def comparar_con_falso(tabla, cohorte=2024, regla="solo_asalariados", se_falso=0.38, punto_falso=14.57):
    """Compara el SE y el IC verdaderos con los escritos a mano (0,38 y [13,82; 15,31])."""
    r = tabla[(tabla["cohorte"] == cohorte) & (tabla["regla"] == regla)].iloc[0]
    falso = (punto_falso - 1.96 * se_falso, punto_falso + 1.96 * se_falso)
    return {"A_punto": r["A_punto"], "se_real": r["se"], "se_falso": se_falso, "razon_se": r["se"] / se_falso,
            "ic_real": (r["ic95_lo"], r["ic95_hi"]), "ic_falso": falso,
            "semiancho_rel_real_%": 100 * (r["ic95_hi"] - r["ic95_lo"]) / 2 / r["A_punto"],
            "semiancho_rel_falso_%": 100 * 1.96 * se_falso / punto_falso}


# ================================================================== 6) DATOS SINTÉTICOS Y AUTOTEST
def _banner_bs(txt="DATOS SINTÉTICOS — no son Argentina"):
    line = "#" * 78
    print("\n".join([line, "##" + txt.center(74) + "##", line]))


def generar_eph_sintetica_cluster(anios=range(2012, 2025), n_viv_por_celda=3, seed=7, efecto_vivienda=0.8):
    """EPH de juguete CON CODUSU: cada vivienda tiene un efecto común (todas sus personas tienden a ser parecidas), así que
    las filas de una misma vivienda están correlacionadas y el bootstrap por vivienda debe dar SE mayor que el ingenuo.
    Las viviendas se repiten en 2 trimestres consecutivos (rotación simplificada). NO son datos de Argentina."""
    rng = np.random.default_rng(seed); filas = []
    nviv = 0
    for y in anios:
        for q in (1, 2, 3, 4):
            for a in range(14, 71):
                for v in range(n_viv_por_celda):
                    nviv += 1
                    u = rng.normal(0, efecto_vivienda)            # efecto de la vivienda
                    for t in (0, 1):                               # la vivienda se entrevista dos trimestres seguidos
                        yy, qq = (y, q + t) if q + t <= 4 else (y + 1, 1)
                        if yy > anios[-1]:
                            continue
                        aa, k = a, 8
                        p = 1 / (1 + np.exp(-(-0.2 + 0.008 * (yy - aa - 1970) + 1.2 * np.exp(-((aa - 40) / 18.0) ** 2) - 1.0 + u)))
                        reg = rng.random(k) < p
                        filas.append(pd.DataFrame({"CODUSU": f"V{nviv:07d}", "AGLOMERADO": (nviv % 5) + 1, "ANO4": yy,
                                                   "TRIMESTRE": qq, "CH04": rng.integers(1, 3, k), "CH06": aa,
                                                   "ESTADO": 1, "CAT_OCUP": 3, "PP07H": np.where(reg, 1, 2), "PP07I": np.nan,
                                                   "PONDERA": rng.integers(300, 900, k)}))
    return pd.concat(filas, ignore_index=True)


def autotest_bootstrap(verbose=True, B=30):
    """Prueba con datos SINTÉTICOS: (1) lector con CODUSU (txt y zip); (2) el punto del bootstrap coincide con
    tbp_eph.estimar_abar; (3) reproducibilidad por semilla y por n_jobs; (4) con efecto de vivienda el SE por vivienda
    es mayor que el ingenuo; (5) con réplicas = muestra completa los pesos multinomiales suman K."""
    if verbose: _banner_bs()
    df = generar_eph_sintetica_cluster()
    coh = np.arange(1990, 2031)
    s65 = pd.Series(np.clip(0.70 + 0.0035 * (coh - 1950), 0.5, 0.97), index=coh)
    # (1) lector
    with tempfile.TemporaryDirectory() as tmp:
        sub = df[df["ANO4"] == 2015].copy()
        sub.columns = [c.lower() if c in ("CH04", "CH06", "ESTADO") else c for c in sub.columns]
        p = os.path.join(tmp, "usu_individual_T115.txt"); sub.to_csv(p, sep=";", index=False)
        zp = os.path.join(tmp, "EPH_usu_1_Trim_2015_txt.zip")
        with zipfile.ZipFile(zp, "w") as z:
            z.write(p, "usu_individual_T115.txt")
        os.remove(p)
        srcs = _EPH.find_eph_files([tmp]); assert len(srcs) == 1
        lec = read_all_cluster(srcs)
        assert len(lec) == len(sub) and {"CODUSU", "AGLOMERADO"} <= set(lec.columns)
        assert lec["CODUSU"].iloc[0].startswith("V") and lec["CODUSU"].nunique() == sub["CODUSU"].nunique()
    # (2) punto == estimar_abar del módulo original
    ref = _EPH.estimar_abar(df.drop(columns=["CODUSU", "AGLOMERADO"]), s65, cohortes=coh)
    prep = preparar(df)
    punto, det, S = estimacion_puntual(prep, s65, coh)
    for regla, col in zip(REGLAS_BS, ("A_bar_solo_asalariados", "A_bar_pp07i", "A_bar_indep50")):
        assert np.allclose(punto[regla].values, ref[col].values, atol=1e-9), f"el punto no coincide con estimar_abar ({regla})"
    # (3) reproducibilidad
    r1 = bootstrap_abar(df, s65, B=B, seed=1, cohortes=coh, verbose=False, prep=prep)
    r2 = bootstrap_abar(df, s65, B=B, seed=1, cohortes=coh, verbose=False, prep=prep)
    r3 = bootstrap_abar(df, s65, B=B, seed=2, cohortes=coh, verbose=False, prep=prep)
    assert np.array_equal(r1["replicas"], r2["replicas"]) and not np.array_equal(r1["replicas"], r3["replicas"])
    if sys.platform.startswith("linux") and "ipykernel" not in sys.modules:   # fork (en macOS se usa spawn: ver _correr)
        r4 = bootstrap_abar(df, s65, B=B, seed=1, cohortes=coh, verbose=False, prep=prep, n_jobs=2)
        assert np.array_equal(r1["replicas"], r4["replicas"]), "el resultado depende de n_jobs"
    # (4) cluster vs ingenuo (el efecto de vivienda y la repetición de viviendas deben ensanchar el SE)
    ing = bootstrap_ingenuo(df, s65, B=B, seed=1, cohortes=coh, verbose=False)
    c = 2005 if 2005 in coh else coh[len(coh) // 2]
    se_cl = r1["tabla"].query("cohorte == @c and regla == 'solo_asalariados'")["se"].iloc[0]
    se_in = ing["tabla"].query("cohorte == @c and regla == 'solo_asalariados'")["se"].iloc[0]
    assert se_cl > se_in, f"con efecto de vivienda el SE por vivienda ({se_cl:.3f}) debería superar al ingenuo ({se_in:.3f})"
    # (5) pesos multinomiales
    m = _pesos(prep, np.random.default_rng(0)); assert m.sum() == prep["K"]
    pe = preparar(df, estratificar=True); m = _pesos(pe, np.random.default_rng(0)); assert m.sum() == pe["K"]
    # IC contiene al punto en la mayoría de cohortes observadas
    t = r1["tabla"]; ok = ((t["ic95_lo"] <= t["A_punto"] + 0.5) & (t["A_punto"] <= t["ic95_hi"] + 0.5)).mean()
    assert ok > 0.9
    if verbose:
        print(f"[sintético] cohorte {c}: SE por vivienda = {se_cl:.3f} | SE ingenuo (por fila) = {se_in:.3f} "
              f"(razón {se_cl / se_in:.2f})")
        print("Autotest OK (lector con CODUSU, punto = estimar_abar, semilla/n_jobs reproducibles, cluster > ingenuo, pesos multinomiales).")
        _banner_bs("FIN AUTOTEST — estos números NO son Argentina")
    return r1

### 18.2 Carga de la EPH con identificador de vivienda

Se leen las mismas columnas de siempre más `CODUSU` (vivienda) y `AGLOMERADO`. Si no hay bases en `datos/eph/` la sección **termina limpia** ("FALTAN DATOS") y no inventa nada; la prueba sintética (17.6) corre igual.

In [ ]:
import time
RUTAS_EPH17 = ["datos/eph", "../datos/eph", "../../datos/eph"]
srcs17 = _EPH.find_eph_files(RUTAS_EPH17)
HAY_EPH17 = len(srcs17) > 0
if not HAY_EPH17:
    print("=" * 78)
    print("FALTAN DATOS: no hay bases individuales de la EPH en", RUTAS_EPH17)
    print("Sin ellas no se calcula ningún bootstrap de Ā. Dejá los .zip de INDEC en datos/eph/ (ver sección 12).")
    print("=" * 78)
else:
    _t0 = time.time()
    df17 = read_all_cluster(srcs17)
    # `d` (tabla de cohortes del notebook principal). Si otra celda lo pisó (p. ej. un `for ..., d in ...`), se relee del CSV que escribió la celda 5.
    _ok = lambda x: isinstance(x, pd.DataFrame) and {"cohorte", "N_t", "S65", "J"} <= set(x.columns)
    d17 = d if _ok(globals().get("d")) else None
    if d17 is None:
        _f = os.path.join(OUT, "tbp_argentina_cohortes_1950_2035.csv")
        if os.path.exists(_f):
            d17 = pd.read_csv(_f); print("Aviso: `d` fue reemplazada por otra celda; se releyó", _f)
    if d17 is None:
        HAY_EPH17 = False
        print("FALTAN DATOS: no encuentro la tabla de cohortes `d` (N_t, S65, J). Corré antes la celda 5 del notebook principal.")
    else:
        s65_17 = d17.set_index("cohorte")["S65"]
    if HAY_EPH17:
        tr = df17.drop_duplicates(["CODUSU", "ANO4", "TRIMESTRE"]).groupby("CODUSU").size()
        print(f"Archivos: {len(srcs17)} | filas: {len(df17):,} | viviendas (CODUSU) distintas: {df17['CODUSU'].nunique():,} | "
              f"aglomerados: {df17['AGLOMERADO'].nunique()} | lectura: {time.time() - _t0:.0f} s")
        print("Trimestres en que aparece cada vivienda (rotación 2-2-2):", tr.value_counts().sort_index().to_dict())
        print(f"=> {100 * (tr > 1).mean():.0f} % de las viviendas aparece en más de un trimestre: las filas NO son independientes.")

### 18.3 Bootstrap por vivienda (B = 1.000) y resultados por cohorte y regla

Cada réplica: pesos multinomiales de viviendas → sumas por celda (cohorte, edad) → logit → extrapolación → $\bar A$, para las tres reglas de independientes. Semilla fija (`20260507`): el resultado es reproducible. La tabla incluye cuánto de $\bar A$ viene de edades **observadas** y cuánto es perfil/extrapolación del modelo.

In [ ]:
if HAY_EPH17:
    B17, SEED17 = 1000, 20260507
    boot17 = bootstrap_abar(df17, s65_17, B=B17, seed=SEED17, verbose=False)
    tab17 = boot17["tabla"]
    print(f"Bootstrap por vivienda: B = {B17} réplicas, {boot17['meta']['n_unidades']:,} viviendas, "
          f"tiempo: {boot17['segundos']:.0f} s (preparación {boot17['meta']['segundos_prep']:.0f} s). Semilla {SEED17}.")
    COH17 = [c for c in (1960, 1980, 2000, 2024) if c in s65_17.index]
    v = tab17[tab17.cohorte.isin(COH17)].copy()
    v["± % (IC95)"] = 100 * (v.ic95_hi - v.ic95_lo) / 2 / v.A_punto
    v["% de Ā en edades observadas"] = 100 * v.frac_edades_obs
    display(v[["cohorte", "regla", "A_punto", "se", "ic95_lo", "ic95_hi", "± % (IC95)", "n_edades_obs", "alpha_extrapolado",
              "% de Ā en edades observadas"]].round(3).reset_index(drop=True))
    _, det17, _ = estimacion_puntual(boot17["prep"], s65_17)
    print("La cohorte 2024 no tiene ninguna edad observada: su nivel (alpha) se extrapola promediando las cohortes",
          cohortes_base_extrapolacion(det17), "(las últimas con >= 3 edades; se las ve sólo a los 18-20 años).")
else:
    print("FALTAN DATOS: sin EPH no hay bootstrap (ver 17.2).")

### 18.4 Comparación honesta: ¿qué había en 16.3?

Se compara el SE verdadero con el `0,38` escrito a mano, el IC verdadero con el `[13,82 ; 15,31]` falso, y con un bootstrap **ingenuo** (por fila) que ignora que las viviendas se repiten. Se agrega una variante estratificada por aglomerado (el diseño de la EPH es estratificado) y un ejemplo de **error de modelo** que el bootstrap no capta (otra regla de extrapolación).

In [ ]:
if HAY_EPH17:
    ing17 = bootstrap_ingenuo(df17, s65_17, B=500, seed=SEED17, verbose=False)
    est17 = bootstrap_abar(df17, s65_17, B=500, seed=SEED17, estratificar=True, verbose=False)
    cmp = comparar_con_falso(tab17, 2024, "solo_asalariados")
    rows = []
    for c in COH17:
        for r in REGLAS_BS:
            f = lambda t, col: t[(t.cohorte == c) & (t.regla == r)][col].iloc[0]
            rows.append({"cohorte": c, "regla": r, "Ā": f(tab17, "A_punto"), "SE ingenuo (filas)": f(ing17["tabla"], "se"),
                         "SE por vivienda": f(tab17, "se"), "SE por vivienda+aglomerado": f(est17["tabla"], "se")})
    cp = pd.DataFrame(rows); cp["vivienda / ingenuo"] = cp["SE por vivienda"] / cp["SE ingenuo (filas)"]
    display(cp.round(3))
    print(f"\nCohorte 2024, regla solo_asalariados (Ā = {cmp['A_punto']:.2f}):")
    print(f"  SE escrito a mano en 16.3: {cmp['se_falso']:.2f}   |   SE real (bootstrap por vivienda): {cmp['se_real']:.2f}   "
          f"(x{cmp['razon_se']:.2f})")
    print(f"  IC 95 % falso: [{cmp['ic_falso'][0]:.2f} ; {cmp['ic_falso'][1]:.2f}] (± {cmp['semiancho_rel_falso_%']:.1f} %)   |   "
          f"IC 95 % real: [{cmp['ic_real'][0]:.2f} ; {cmp['ic_real'][1]:.2f}] (± {cmp['semiancho_rel_real_%']:.1f} %)")
    se_ing24 = cp[(cp.cohorte == 2024) & (cp.regla == "solo_asalariados")]["SE ingenuo (filas)"].iloc[0]
    print(f"  Bootstrap ingenuo (por fila): SE = {se_ing24:.2f} -> subestima el SE real en {100 * (1 - se_ing24 / cmp['se_real']):.0f} %.")
    print("  El supuesto '± 5 %' de 16.3 NO se sostiene: el margen real de 95 % es mayor (ver arriba) y para otras cohortes/reglas varía.")
    alt = estimacion_puntual(boot17["prep"], s65_17, modo_extrap="tendencia")[0]
    a0 = boot17["punto"].loc[2024, "solo_asalariados"]; a1 = alt.loc[2024, "solo_asalariados"]
    dentro = cmp["ic_real"][0] <= a1 <= cmp["ic_real"][1]
    print(f"  Una variante de modelo: Ā 2024 con extrapolación 'tendencia' = {a1:.2f} vs {a0:.2f} con 'últimas 5 cohortes' "
          f"({a1 - a0:+.2f} años, {100 * (a1 / a0 - 1):+.0f} %); queda {'dentro' if dentro else 'fuera'} del IC muestral. "
          "Es UNA variante: el error de modelo (perfil común, período/cohorte...) no se puede cuantificar con estos datos.")
else:
    print("FALTAN DATOS: sin EPH no hay comparación (ver 17.2).")

### 18.5 Traslado a $TBP_C$, gráfico y CSV

$TBP_C=\dfrac{N\,S_{65}\,\bar A\,\tau}{J\,\rho}$ es **proporcional** a $\bar A$ (con $N, S_{65}, J, \tau, \rho$ fijos), así que el IC de $TBP_C$ es el de $\bar A$ multiplicado por $TBP_C/\bar A$: el **margen relativo es el mismo**. Si la sección 15 está cargada (`R15`), se agrega la versión `p4` de 2024 (aproximación: se aplica el margen relativo del $\bar A$ agregado, no el de $\bar A$ por sexo).

In [ ]:
if HAY_EPH17:
    p4 = R15["TBP_C_p4"] if "R15" in globals() and "TBP_C_p4" in getattr(R15, "columns", []) else None
    full17 = trasladar_a_tbp(tab17, d17, tau=globals().get('TAU', 0.2177), rho=globals().get('RHO', 0.403), tbp_col_p4=p4)
    ing_se = ing17["tabla"].rename(columns={"se": "se_ingenuo"})[["cohorte", "regla", "se_ingenuo"]]
    full17 = full17.merge(ing_se, on=["cohorte", "regla"])
    full17["ic95_semiancho_rel_pct"] = 100 * (full17.ic95_hi - full17.ic95_lo) / 2 / full17.A_punto
    full17["B"], full17["semilla"], full17["unidad_remuestreo"] = B17, SEED17, "vivienda (CODUSU)"
    os.makedirs(OUT, exist_ok=True)
    full17.to_csv(f"{OUT}/abar_bootstrap_ic.csv", index=False)
    cols = ["cohorte", "regla", "TBP_C_punto", "TBP_C_se", "TBP_C_ic95_lo", "TBP_C_ic95_hi"]
    display(full17[full17.cohorte.isin(COH17)][cols].round(4).reset_index(drop=True))
    if p4 is not None:
        r24 = full17[(full17.cohorte == 2024) & (full17.regla == "solo_asalariados")].iloc[0]
        print(f"TBP_C p4 2024 (aprox.): {r24.TBP_C_p4_punto:.3f}  IC95 [{r24.TBP_C_p4_ic95_lo:.3f} ; {r24.TBP_C_p4_ic95_hi:.3f}] "
              f"(16.3 mostraba ± 5 %).")
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    col = {"solo_asalariados": "#1f77b4", "pp07i": "#ff7f0e", "indep50": "#2ca02c"}
    for r in REGLAS_BS:
        t = full17[full17.regla == r].sort_values("cohorte")
        ax[0].plot(t.cohorte, t.A_punto, color=col[r], label=r); ax[0].fill_between(t.cohorte, t.ic95_lo, t.ic95_hi, color=col[r], alpha=.2)
    ex = np.sort(full17[(full17.regla == "solo_asalariados") & full17.alpha_extrapolado].cohorte.values)
    if len(ex):                                   # sombrea cada tramo continuo de cohortes con alpha extrapolado
        cortes = np.flatnonzero(np.diff(ex) > 1) + 1
        for k, tramo in enumerate(np.split(ex, cortes)):
            ax[0].axvspan(tramo[0] - .5, tramo[-1] + .5, color="gray", alpha=.15, label="alpha extrapolado (< 3 edades)" if k == 0 else None)
    ax[0].set_title("Ā por cohorte con IC 95 % (bootstrap por vivienda)"); ax[0].set_xlabel("cohorte"); ax[0].set_ylabel("años")
    ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)
    j = REGLAS_BS.index("solo_asalariados"); i = list(s65_17.index).index(2024)
    ax[1].hist(boot17["replicas"][:, j, i], bins=40, color="#4575b4", alpha=.7, edgecolor="white", density=True)
    ax[1].axvline(boot17["punto"].loc[2024, "solo_asalariados"], color="k", label="punto")
    ax[1].axvline(cmp["ic_real"][0], color="#d73027", ls="--", label="IC 95 % real"); ax[1].axvline(cmp["ic_real"][1], color="#d73027", ls="--")
    ax[1].axvline(cmp["ic_falso"][0], color="gray", ls=":", lw=2, label="IC 'falso' de 16.3"); ax[1].axvline(cmp["ic_falso"][1], color="gray", ls=":", lw=2)
    ax[1].set_title("Cohorte 2024, solo asalariados: 1.000 réplicas"); ax[1].set_xlabel("Ā (años)"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.savefig(f"{OUT}/seccion_18_bootstrap_abar.png", dpi=130); plt.show()
    print(f"Guardado: {OUT}/abar_bootstrap_ic.csv ({len(full17)} filas) y {OUT}/seccion_18_bootstrap_abar.png")
else:
    print("FALTAN DATOS: sin EPH no hay CSV de bootstrap (ver 17.2).")

### 18.6 Prueba con datos sintéticos (autotest)

Verifica con una EPH **inventada** (con viviendas que se repiten y un efecto de vivienda): lector con `CODUSU` (`.txt` y `.zip`), que el punto coincide con `estimar_abar`, que la semilla reproduce el resultado, que los pesos multinomiales suman bien y que el SE por vivienda supera al ingenuo. **No son datos de Argentina.**

In [ ]:
_ = autotest_bootstrap()

### 18.7 Lectura y límites

* **Qué se puede afirmar.** El error estándar muestral de $\bar A$ (cohorte 2024, solo asalariados) es **mayor** que el `0,38` escrito a mano en 16.3 (ver 17.4), y el margen de 95 % **no es ± 5 %**. El `0,38` no salía de ningún cálculo. Ignorar que las viviendas se repiten (bootstrap por fila) subestima el SE entre un cuarto y un tercio (según cohorte y regla).
* **Qué no se puede afirmar.** Que este IC sea "la" incertidumbre de $\bar A$ ni de $TBP_C$. Mide **sólo variabilidad muestral** de la EPH con un diseño simplificado (se remuestrean viviendas; los ponderadores se toman como fijos; no se usa el estrato ni la UPM del INDEC).
* **No capta error de modelo:** perfil por edad común a todas las cohortes; confusión período/cohorte; extrapolación de las cohortes jóvenes y futuras (la 2024 no tiene edades observadas; ver el ejemplo de 17.4); EPH urbana; `PP07H` de la semana de referencia; reglas para independientes (por eso se reportan tres); $S_{65}$ y Gompertz tratados como exactos.
* **Para cohortes viejas casi todo $\bar A$ es perfil de edad, no dato:** la columna "% de Ā en edades observadas" es baja (7–24 % en las cohortes mostradas), así que $\bar A$ depende sobre todo del perfil común $f(a)$, cuya incertidumbre muestral sí está en el SE pero cuya forma funcional no.
* **$TBP_C$:** el IC se traslada por proporcionalidad; no incluye incertidumbre de $N$, $\tau$, $\rho$, $J$ (eso es el Monte Carlo de 16.4, que además usa un rango de $\bar A$ propio).

## 19. 🆕 Monte Carlo riguroso: comparación pareada, cohortes futuras y varianza por parámetro

**Qué hace esta sección** (fragmento autónomo; depende solo de `d`, `s`, `TAU`, `RHO`, `A_OBS`, `A_LEGAL`, `DATA`, `OUT` en memoria y de los archivos que carga él mismo: `tbp_modelo_avanzado_1950_2035.csv` de la sección 15 —o `R15` en memoria—, `abar_eph_por_cohorte.csv`, `anses_rho_tau_anual.csv`, `series_tbp_extraidas.csv`, `renaper_natalidad_2012_2025.csv`, `tbp_cohortes_J_reponderado.csv`, y `abar_bootstrap_ic.csv` si existe).

Rehace la simulación de la sección 16.4 corrigiendo sus errores conceptuales (cuadro en 19.9):

1. **Comparación PAREADA.** $\tau$, $\rho$ y la regla de $\bar A$ son factores *comunes* a todas las cohortes: si cambian, cambia también la cohorte 2000. Por eso se compara con la **razón** $TBP_C(2024)/TBP_C(2000)$ calculada con los **mismos sorteos**. En la razón $\tau$ y $\rho$ se **cancelan exactamente**.
2. **Tres capas de incertidumbre que no se mezclan sin avisar:** *muestral* (error de muestreo de $\bar A$, EPH), *de medición* ($N$: DEIS vs RENAPER; $J^*$: ONU vs reponderado) y *de escenario/definición* (regla de $\bar A$, $\tau$, $\rho$, natalidad futura).
3. **Cohortes futuras** (2030, 2035) con los 3 escenarios de natalidad y **tendencia** (pendiente 2000→2024), con abanico (*fan chart*).
4. **Varianza de primer orden por parámetro** (Sobol de primer orden por estratos y correlación de rangos): ¿qué domina el *nivel* y qué domina la *tendencia*?
5. **Nombres correctos** de las descomposiciones (no hay Campbell–Shiller ni $R^2$).

> ⚠️ **Esto NO es un test de hipótesis clásico.** No hay $H_0$ ni p-valor. Es una **cuantificación de incertidumbre condicional a los supuestos elegidos** (ver 18.10).

### 19.1 Módulo (copia de `src/tbp_montecarlo.py`)

Todo el código está en una sola celda para que el Colab sea autosuficiente. El mismo archivo vive en el repositorio (`src/tbp_montecarlo.py`; `python src/tbp_montecarlo.py` corre el autotest con datos **sintéticos** rotulados). Los nombres de funciones y constantes no pisan los de otras secciones (`idx_c`, `NAT_ESC_MC`, `CAPAS_MC`, `Q_MC`, `autotest_montecarlo`).

In [ ]:
# ====== MÓDULO tbp_montecarlo (misma versión que src/tbp_montecarlo.py) ======
# -*- coding: utf-8 -*-
"""
tbp_montecarlo.py — Monte Carlo RIGUROSO para el TBP_C (reemplaza el diseño de la Sección 16.4).

    TBP_C(c) = N_c · S65_c · Ā_c · τ / (J*_c · ρ)                (años de prestación por jubilado; no contiene E_r)

QUÉ CORRIGE RESPECTO DE LA SECCIÓN 16 (resumen; el detalle está en docs/seccion_19_montecarlo.md)
 1) COMPARACIÓN PAREADA.  τ y ρ (y la regla de Ā) son factores COMUNES a todas las cohortes: si cambian, cambia también la
    cohorte 2000.  Por eso la pregunta "¿2024 < 2000?" se contesta con la RAZÓN TBP(2024)/TBP(2000) calculada con los MISMOS
    sorteos.  En la razón τ y ρ se cancelan EXACTAMENTE: solo cuentan N, S65, J* y el cociente Ā(2024)/Ā(2000).
 2) TRES CAPAS_MC DE INCERTIDUMBRE que NO se mezclan sin avisar:
        MUESTRAL   : error de muestreo de Ā (EPH).  Bootstrap real en abar_bootstrap_ic.csv (si existe; si no, queda en cero
                     y se rotula).  Es lo único que admite una "densidad" en sentido estricto.
        MEDICIÓN   : N (DEIS vs RENAPER: dos fuentes que miden lo mismo) y J* (ONU vs reponderado con nacimientos propios).
        ESCENARIO  : regla de Ā (asalariados / PP07I / independientes 50 %), τ (18,3 / 21,77 / 23,35 / 27 %), ρ (moratorias
                     40,3 / 48,3 / 56,3 %) y natalidad futura (3 escenarios).  Son DEFINICIONES o SUPUESTOS: se tratan como
                     categorías con pesos iguales (convención, NO probabilidades) y se reporta también el rango mín–máx.
 3) COHORTES FUTURAS (2030, 2035) con los 3 escenarios de natalidad y TENDENCIA (pendiente y razón 2000→2024), con abanico.
 4) VARIANZA DE PRIMER ORDEN por parámetro (índice de Sobol de primer orden por estratos, correlación de rangos y fracción
    exacta en logs) para el NIVEL y para la TENDENCIA.
 5) DESCOMPOSICIONES con el nombre correcto: "log exacta" y "varianza por covarianzas" (no es Campbell–Shiller ni un R²).

QUÉ NO ES: no es un test de hipótesis clásico (no hay H0 ni p-valor).  Es una cuantificación de incertidumbre CONDICIONAL a los
supuestos elegidos: "si las entradas se mueven dentro de estos rangos, el indicador se mueve así".

Todo el cálculo parte de los CSV del proyecto (datos/procesados). Sin datos nuevos inventados; el autotest usa datos
SINTÉTICOS rotulados.   Uso:  python src/tbp_montecarlo.py   (corre el autotest)
"""
import os
import numpy as np
import pandas as pd

RUTAS_MC = ["../datos/procesados", "datos/procesados", "github_staging/datos/procesados"]
COHORTES_MC = list(range(2000, 2036))
NAT_ESC_MC = ("persistencia", "recuperacion_lenta", "caida_adicional")
REGLAS_ABAR = ("solo_asalariados", "pp07i", "indep50")
ETQ_REGLA = {"solo_asalariados": "asalariados", "pp07i": "PP07I", "indep50": "indep. 50 %"}
CAPAS_MC = ("muestral", "medicion", "escenario")
SE_REL_PROVISORIO = 0.38 / 14.57        # SE citado en la Sec. 16.3 (hardcodeado, NO reproducible aquí): solo como sensibilidad rotulada
Q_MC = (5, 25, 50, 75, 95)


# ============================================================================================= 0) ENTRADAS
def _leer(nombre, rutas=None):
    """Lee un CSV de las rutas candidatas (más DATA/OUT del espacio global si existen). None si no está."""
    cand = list(rutas or RUTAS_MC)
    g = globals()
    for v in ("DATA", "OUT"):
        if isinstance(g.get(v), str):
            cand.append(g[v])
    for r in cand:
        f = os.path.join(r, nombre)
        if os.path.exists(f):
            return pd.read_csv(f, low_memory=False)
    return None


def cargar_insumos(rutas=None, modelo=None, d=None):
    """Junta las entradas del proyecto.  `modelo`: DataFrame del modelo avanzado (R15) si no está el CSV; `d`: tabla base del notebook."""
    ins = {}
    ma = _leer("tbp_modelo_avanzado_1950_2035.csv", rutas)
    if ma is None:
        ma = modelo if modelo is not None else (globals().get("R15") if globals().get("R15") is not None else globals().get("SAL15"))
        if ma is not None and "cohorte" not in ma.columns:
            ma = ma.reset_index()
    if ma is None:
        raise FileNotFoundError("Falta tbp_modelo_avanzado_1950_2035.csv (sección 15). Correr la sección 15 o copiar el CSV a datos/procesados.")
    ins["modelo"] = ma.set_index("cohorte")
    ab = _leer("abar_eph_por_cohorte.csv", rutas)
    if ab is None:
        raise FileNotFoundError("Falta abar_eph_por_cohorte.csv (sección 12).")
    ins["abar"] = ab.set_index("cohorte")
    ins["rho"] = _leer("anses_rho_tau_anual.csv", rutas)
    ins["series"] = _leer("series_tbp_extraidas.csv", rutas)
    ins["renaper"] = _leer("renaper_natalidad_2012_2025.csv", rutas)
    ins["jrw"] = _leer("tbp_cohortes_J_reponderado.csv", rutas)
    ins["nat"] = _leer("natalidad_escenarios_2025_2035.csv", rutas)
    if d is None:
        dd = _leer("tbp_argentina_cohortes_1950_2035.csv", rutas)
        d = dd
    ins["d"] = d
    ins["abar_ic"] = leer_abar_ic(rutas)
    return ins


def leer_abar_ic(rutas=None):
    """Lee el bootstrap real de Ā (abar_bootstrap_ic.csv) si existe.  Formato tolerante: busca columna de estimación puntual
    y de error estándar (o de límites del IC 95 %).  Devuelve {'se_rel', 'abar', 'fuente'} o None si no existe / no se entiende."""
    df = _leer("abar_bootstrap_ic.csv", rutas)
    if df is None:
        return None
    low = {c.lower(): c for c in df.columns}

    def _buscar(*pistas, excl=()):
        for c in low:
            if any(p in c for p in pistas) and not any(e in c for e in excl):
                return low[c]
        return None
    c_coh, c_reg = _buscar("cohorte"), _buscar("regla", "definicion", "variante")
    sub = df
    if c_coh is not None and (sub[c_coh] == 2024).any():
        sub = sub[sub[c_coh] == 2024]
    if c_reg is not None:
        m = sub[c_reg].astype(str).str.contains("asal", case=False)
        sub = sub[m] if m.any() else sub
    c_pt = _buscar("abar", "a_bar", "estim", "punto", "media", "mean", excl=("se", "ic", "lo", "hi"))
    c_se = _buscar("se", "error_est", excl=("rese",))
    c_lo, c_hi = _buscar("ic_lo", "ic_inf", "lo", "p2_5", "low"), _buscar("ic_hi", "ic_sup", "hi", "p97_5", "high")
    try:
        pt = float(sub[c_pt].iloc[0])
        if c_se is not None:
            se = float(sub[c_se].iloc[0])
        else:
            se = (float(sub[c_hi].iloc[0]) - float(sub[c_lo].iloc[0])) / (2 * 1.96)
        return dict(se_rel=se / pt, abar=pt, fuente="abar_bootstrap_ic.csv (bootstrap real)")
    except Exception as e:                                         # formato no reconocido: no se inventa nada
        print("AVISO: no pude interpretar abar_bootstrap_ic.csv (%s); la capa muestral queda en cero." % e)
        return None


# ============================================================================================= 1) CALIBRACIÓN CON DATOS DEL PROYECTO
def calibrar_incertidumbre(ins, tau0=0.2177, rho0=None, se_rel_abar=None, verbose=False):
    """Fija cada distribución/escenario con un dato del proyecto y deja la JUSTIFICACIÓN escrita.  Devuelve dict `par`."""
    just = {}
    # --- N: DEIS vs RENAPER (2012-2024)
    sig_N = 0.02
    ser, ren = ins.get("series"), ins.get("renaper")
    if ser is not None and ren is not None:
        deis = ser.set_index("year")["deis_nacimientos"]
        r = ren.set_index("anio")["nacimientos_identificados_miles"] * 1000
        idx = [a for a in r.index if a in deis.index and a <= 2024 and np.isfinite(deis[a])]
        rat = (r.loc[idx] / deis.loc[idx])
        sig_N = float(np.std(np.log(rat), ddof=1))
        just["N"] = ("medición: Normal log, σ = desvío de ln(RENAPER/DEIS) %d-%d = %.2f %% (razón entre %.3f y %.3f; conservador: no se divide por √2)"
                     % (min(idx), max(idx), 100 * sig_N, rat.min(), rat.max()))
    else:
        just["N"] = "medición: σ = 2 % (SIN el archivo RENAPER: valor de trabajo, no calibrado)"
    # --- J*: ONU vs reponderado con nacimientos propios (sección 10)
    sig_J = 0.03
    jr = ins.get("jrw")
    if jr is not None:
        j = jr[jr.escenario == "persistencia"].set_index("cohorte")
        dif = (j.J_reponderado / j.J_onu - 1).loc[2000:2024]
        sig_J = float(np.sqrt((dif ** 2).mean()))
        just["J"] = ("medición/modelo: Normal log, σ = RMS de J_reponderado/J_ONU-1 en cohortes 2000-2024 = %.2f %% (media %.2f %%: el reponderado "
                     "es sistemáticamente menor; se usa como ESCALA del error, centrado en 0)" % (100 * sig_J, 100 * dif.mean()))
    else:
        just["J"] = "medición: σ = 3 % (SIN el archivo de J reponderado: valor de trabajo, no calibrado)"
    # --- ρ: escenarios de moratorias
    rh = ins.get("rho")
    if rh is not None:
        u = rh.sort_values("anio").iloc[-1]
        r_tot, r_sin = float(u["rho"]), float(u["rho_sin_moratoria"])
    else:
        r_tot, r_sin = 0.403, 0.563
    rho0 = r_tot if rho0 is None else rho0
    rho_esc = {"moratorias persisten (%.1f %%)" % (100 * r_tot): r_tot,
               "mitad de moratorias (%.1f %%)" % (100 * (r_tot + r_sin) / 2): (r_tot + r_sin) / 2,
               "sin moratorias (%.1f %%)" % (100 * r_sin): r_sin}
    just["rho"] = ("escenario: 3 categorías (ρ 2023 publicado %.1f %%; sin moratorias %.1f %%); NO se usa el mínimo histórico 37,1 %% porque "
                   "la cohorte se retira en 2065+ y la serie digitalizada 2009-2023 tiene error < 0,1 pp" % (100 * r_tot, 100 * r_sin))
    tau_esc = {"efectiva SIPA ~18,3 % (supuesto)": 0.183, "nominal inc.(a) 21,77 %": 0.2177,
               "inc.(b) 23,35 %": 0.2335, "sector público 27 %": 0.27}
    just["tau"] = "escenario: 4 categorías (alícuotas nominales del Anuario ANSES p.39 + efectiva ~18,3 % que es un SUPUESTO); sin serie histórica verificada"
    # --- Ā
    ic = ins.get("abar_ic")
    if se_rel_abar is None:
        se_rel_abar = ic["se_rel"] if ic else 0.0
    just["Abar_muestral"] = (ic["fuente"] + ": SE relativo = %.2f %%" % (100 * se_rel_abar) if ic and se_rel_abar == ic["se_rel"] else
                             ("SE relativo = %.2f %% (fijado a mano)" % (100 * se_rel_abar) if se_rel_abar > 0 else
                              "SIN bootstrap real todavía (abar_bootstrap_ic.csv no existe): capa muestral = 0; rotulado"))
    just["Abar_regla"] = "escenario/definición: 3 reglas de la EPH (asalariados, PP07I, independientes 50 %); ver abar_eph_por_cohorte.csv"
    just["natalidad"] = "escenario: 3 escenarios de la sección 9 (persistencia, recuperación lenta, caída adicional) desde 2025"
    ab = ins["abar"]
    A = {r: ab["A_bar_" + r] for r in REGLAS_ABAR}
    return dict(tau0=tau0, rho0=rho0, sig_N=sig_N, sig_J=sig_J, se_rel_abar=se_rel_abar,
                tau_esc=tau_esc, rho_esc=rho_esc, A=A,
                corr=dict(N=0.0, J=0.5, Abar=1.0), justificacion=just)


def base_por_escenario(ins, cohortes=COHORTES_MC):
    """TBP_C_p4 por cohorte para cada escenario de natalidad (una columna por escenario; iguales hasta 2024)."""
    ma = ins["modelo"]
    out = {}
    for e in NAT_ESC_MC:
        c = "TBP_C_p4_" + e
        out[e] = ma[c] if c in ma.columns else ma["TBP_C_p4"]
    return pd.DataFrame(out).loc[cohortes]


# ============================================================================================= 2) SIMULACIÓN
def _choque(rng, n, nc, r):
    """Choque estándar n×nc con correlación r entre cohortes: sqrt(r)·común + sqrt(1-r)·propio."""
    comun, propio = rng.standard_normal((n, 1)), rng.standard_normal((n, nc))
    return np.sqrt(r) * comun + np.sqrt(1 - r) * propio


def simular(par, base, n=20000, seed=42, capas=CAPAS_MC, nat="mezcla", cohortes=None):
    """Simula `n` sorteos conjuntos para todas las cohortes (mismos τ, ρ, regla y choque común en todas: PAREADO).
    capas: subconjunto de ('muestral','medicion','escenario'). Una capa apagada deja su factor en 1 (valor base).
    nat: nombre de escenario de natalidad, o 'mezcla' (sortea uno por corrida, pesos iguales).
    Se sortea SIEMPRE el mismo número de variables, así que dos corridas con la misma semilla y distintas capas comparten los sorteos.
    Devuelve dict con tbp (n×nc), factores en logs por grupo (n×nc o n×1), índices de categorías."""
    co = list(cohortes if cohortes is not None else base.index)
    nc = len(co)
    rng = np.random.default_rng(seed)
    tau_v, rho_v = np.array(list(par["tau_esc"].values())), np.array(list(par["rho_esc"].values()))
    i_tau, i_rho = rng.integers(0, len(tau_v), n), rng.integers(0, len(rho_v), n)
    i_reg, i_nat = rng.integers(0, len(REGLAS_ABAR), n), rng.integers(0, len(NAT_ESC_MC), n)
    zN = _choque(rng, n, nc, par["corr"]["N"])
    zJ = _choque(rng, n, nc, par["corr"]["J"])
    zA = _choque(rng, n, nc, par["corr"]["Abar"])
    L = {}
    z0 = np.zeros((n, nc))
    L["N"] = par["sig_N"] * zN if "medicion" in capas else z0
    L["J"] = -par["sig_J"] * zJ if "medicion" in capas else z0                       # J está en el denominador
    L["Abar_muestral"] = par["se_rel_abar"] * zA if "muestral" in capas else z0
    A = pd.DataFrame(par["A"]).loc[co]
    a_base = A["solo_asalariados"].values
    if "escenario" in capas:
        fr = np.stack([np.log(A[r].values / a_base) for r in REGLAS_ABAR])          # reglas × cohortes
        L["Abar_regla"] = fr[i_reg]
        L["tau"] = np.repeat(np.log(tau_v[i_tau] / par["tau0"])[:, None], nc, 1)
        L["rho"] = np.repeat(np.log(par["rho0"] / rho_v[i_rho])[:, None], nc, 1)
    else:
        L["Abar_regla"], L["tau"], L["rho"] = z0, z0, z0
    lb = np.log(base[list(NAT_ESC_MC)].loc[co].values)                                  # cohortes × escenarios
    if nat == "mezcla":
        L["natalidad"] = lb[:, i_nat].T - lb[:, [0]].T                               # respecto de 'persistencia'
        lbase = lb[:, 0][None, :]
    else:
        k = NAT_ESC_MC.index(nat)
        L["natalidad"] = z0
        lbase = lb[:, k][None, :]
    lt = lbase + sum(L.values())
    return dict(tbp=np.exp(lt), L=L, lbase=np.repeat(lbase, n, 0), cohortes=co, n=n, seed=seed, capas=tuple(capas), nat=nat,
                i=dict(tau=i_tau, rho=i_rho, regla=i_reg, nat=i_nat))


def idx_c(sim, c):
    return sim["cohortes"].index(c)


def cuantiles(x, qs=Q_MC):
    p = np.percentile(x, qs)
    return dict({"P%d" % q: float(v) for q, v in zip(qs, p)}, media=float(np.mean(x)), minimo=float(np.min(x)), maximo=float(np.max(x)))


# ---------------------------------------------------------------------------------------- razón pareada y no pareada
def razon_pareada(sim, c1=2024, c0=2000):
    """TBP(c1)/TBP(c0) con los MISMOS sorteos.  τ y ρ se cancelan: solo importan N, S65, J* y Ā(c)."""
    return sim["tbp"][:, idx_c(sim, c1)] / sim["tbp"][:, idx_c(sim, c0)]


def razon_no_pareada(sim, c1=2024, c0=2000, seed=1):
    """CONTRAEJEMPLO (incorrecto): TBP(c1) de un sorteo contra TBP(c0) de OTRO sorteo (mezcla de escenarios independientes)."""
    p = np.random.default_rng(seed).permutation(sim["n"])
    return sim["tbp"][:, idx_c(sim, c1)] / sim["tbp"][p, idx_c(sim, c0)]


def pendiente(sim, c0=2000, c1=2024):
    """Pendiente MCO de ln(TBP) sobre la cohorte, en % por año (puntos log ×100), cohortes c0..c1, por sorteo."""
    cs = [c for c in sim["cohortes"] if c0 <= c <= c1]
    x = np.array(cs, float) - np.mean(cs)
    w = x / (x ** 2).sum()
    lt = np.log(sim["tbp"][:, [idx_c(sim, c) for c in cs]])
    return 100 * lt @ w, w, cs


def pesos_pendiente(cohortes, c0=2000, c1=2024):
    cs = [c for c in cohortes if c0 <= c <= c1]
    x = np.array(cs, float) - np.mean(cs)
    w = np.zeros(len(cohortes))
    for c, wi in zip(cs, x / (x ** 2).sum()):
        w[cohortes.index(c)] = wi
    return w


# ---------------------------------------------------------------------------------------- réplica del diseño de la Sección 16
def replica_seccion16(ins, n=10000):
    """Reproduce EXACTAMENTE el Monte Carlo de la Sección 16.4 (semilla 42, mismas distribuciones) para poder comparar."""
    c24 = ins["modelo"].loc[2024]
    c00 = ins["modelo"].loc[2000]
    np.random.seed(42)
    Nb, Jb, Sb = float(c24["N_4"]), float(c24["J_star_4"]), float(c24["S_tot65"])
    N = np.random.normal(Nb, 0.015 * Nb, n)
    A = np.random.triangular(13.5, 15.5, 23.6, n)
    tau = np.random.uniform(0.183, 0.2335, n)
    rho = np.random.triangular(0.371, 0.403, 0.563, n)
    J = np.random.normal(Jb, 0.01 * Jb, n)
    y = N * Sb * A * tau / (J * rho)
    ref00 = float(c00["TBP_C_p4"])
    return dict(y=y, ref2000=ref00, base2024=float(c24["TBP_C_p4"]), p_supera=float(np.mean(y >= ref00)), **cuantiles(y))


# ============================================================================================= 3) VARIANZA DE PRIMER ORDEN
def _indice_estratos(Y, f, nbins=50, max_cat=12):
    """Var(E[Y|f])/Var(Y) estimado por estratos: categorías si f tiene pocos valores distintos; si no, `nbins` cuantiles.
    Dos sesgos chicos y de signo opuesto: al alza ≈ (estratos-1)/n por ruido muestral y a la baja por aplanar cada estrato
    (≈ -0,5 puntos con 50 estratos; verificado en el autotest)."""
    vy = np.var(Y)
    if vy <= 0:
        return 0.0
    u = np.unique(np.round(f, 12))
    if len(u) <= max_cat:
        g = np.searchsorted(u, np.round(f, 12))
    else:
        edges = np.quantile(f, np.linspace(0, 1, nbins + 1)[1:-1])
        g = np.searchsorted(edges, f)
    s = np.bincount(g, weights=Y); k = np.bincount(g)
    ok = k > 0
    m = s[ok] / k[ok]
    return float(np.sum(k[ok] * (m - Y.mean()) ** 2) / len(Y) / vy)


def _rangos(x):
    return pd.Series(x).rank().values


def indices_primer_orden(sim, pesos, nombre_y="nivel", en_logs_y=True, grupos=None):
    """Índices de primer orden por GRUPO de parámetros.  `pesos` (vector sobre cohortes) define el resultado:
        nivel de la cohorte c : e_c ;  log-razón c1/c0 : e_c1 - e_c0 ;  pendiente : pesos_pendiente.
    Como ln TBP es EXACTAMENTE la suma de los logs de los factores, el resultado en logs es aditivo y el aporte de cada grupo es
    contrib_k = L_k @ pesos.  Se devuelven 3 medidas:
        frac_log  : Var(contrib_k)/Var(ln Y)                 (exacta; suma 1 porque los grupos son independientes)
        sobol_1   : Var(E[Y|grupo k])/Var(Y), con Y en la escala indicada (nivel en años, razón, o pendiente)
        spearman2 : (correlación de rangos entre Y y el aporte)²"""
    w = np.asarray(pesos, float)
    L = sim["L"]
    grupos = grupos or [k for k in L if np.any(L[k] != 0)]
    contrib = {k: L[k] @ w for k in grupos}
    lny = sim["lbase"] @ w + sum(L[k] @ w for k in L)
    Y = np.exp(lny) if en_logs_y else lny
    v = np.var(lny)
    filas = []
    for k in grupos:
        fk = contrib[k]
        r = float(np.corrcoef(_rangos(Y), _rangos(fk))[0, 1]) if np.var(fk) > 0 else 0.0
        filas.append(dict(resultado=nombre_y, grupo=k, frac_log=float(np.var(fk) / v) if v > 0 else 0.0,
                          sobol_1=_indice_estratos(Y, fk), spearman2=r ** 2))
    df = pd.DataFrame(filas)
    if len(df):
        df.loc[len(df)] = dict(resultado=nombre_y, grupo="(suma de primer orden)", frac_log=df.frac_log.sum(),
                               sobol_1=df.sobol_1.sum(), spearman2=np.nan)
    return df


# ============================================================================================= 4) DESCOMPOSICIONES (nombres correctos)
def descomposicion_log_exacta(modelo, d, c0=2000, c1=2024):
    """DESCOMPOSICIÓN LOGARÍTMICA EXACTA de ln TBP_C(c1) - ln TBP_C(c0) (contabilidad de crecimiento): suma EXACTA de los cambios en
    log de cada factor; con τ y ρ constantes no contribuyen.  Dos ESPECIFICACIONES (ambas válidas, responden preguntas distintas):
      A) 'simple': J = población 65+ total, Ā constante      -> N, S65, J (Ā, τ, ρ aportan 0).
      B) 'avanzada': J* (varones 65+, mujeres 60+), Ā EPH variable y sexo/retiro -> N, S65, J*, Ā y composición por sexo (residuo)."""
    dd = d.set_index("cohorte")
    a1, a0 = dd.loc[c1], dd.loc[c0]
    tot_A = np.log(a1.N_t * a1.S65 / a1.J) - np.log(a0.N_t * a0.S65 / a0.J)
    fa = {"N": np.log(a1.N_t / a0.N_t), "S65": np.log(a1.S65 / a0.S65), "J (65+)": -np.log(a1.J / a0.J), "Ā": 0.0}
    m1, m0 = modelo.loc[c1], modelo.loc[c0]
    tot_B = np.log(m1.TBP_C_p4 / m0.TBP_C_p4)
    fb = {"N": np.log(m1.N_4 / m0.N_4), "S65": np.log(m1.S_tot65 / m0.S_tot65), "J*": -np.log(m1.J_star_4 / m0.J_star_4),
          "Ā (EPH total)": np.log(m1.A_eph / m0.A_eph)}
    fb["composición sexo/retiro (residuo)"] = tot_B - sum(fb.values())
    filas = []
    for esp, f, tot in (("A: J=65+, Ā constante", fa, tot_A), ("B: J*, Ā EPH variable", fb, tot_B)):
        for k, v in f.items():
            filas.append(dict(especificacion=esp, componente=k, log_puntos=float(v), pct_de_la_caida=float(100 * v / tot)))
        filas.append(dict(especificacion=esp, componente="TOTAL Δln TBP_C", log_puntos=float(tot), pct_de_la_caida=100.0))
    return pd.DataFrame(filas)


def descomposicion_varianza_cov(modelo, c0=2000, c1=2024, ddof_consistente=True):
    """DESCOMPOSICIÓN DE VARIANZA POR COVARIANZAS: con Δln TBP = Σ_k Δx_k (x_k = ln de cada factor con su signo),
        Var(Δln TBP) = Σ_k Cov(Δx_k, Δln TBP)   =>   participación_k = Cov(Δx_k, Δln TBP)/Var(Δln TBP),  Σ_k participación_k = 100 %.
    No es un R² (puede ser negativa o > 100 %) ni Campbell–Shiller.  Usa el MISMO ddof en cov y var (el código original de la Sección 16
    mezclaba np.cov (ddof=1) con np.var (ddof=0), lo que infla todas las participaciones en n/(n-1))."""
    sub = modelo.loc[c0:c1]
    y = np.diff(np.log(sub.TBP_C_p4))
    comp = {"N": np.diff(np.log(sub.N_4)), "S65": np.diff(np.log(sub.S_tot65)), "J*": -np.diff(np.log(sub.J_star_4))}
    comp["Ā y sexo/retiro (residuo)"] = y - sum(comp.values())
    dd = 1 if ddof_consistente else 0
    var = np.var(y, ddof=1 if ddof_consistente else 0)
    filas = [dict(componente=k, participacion_pct=float(100 * np.cov(v, y, ddof=dd)[0, 1] / var)) for k, v in comp.items()]
    if not ddof_consistente:                                                  # réplica del cálculo original (np.cov ddof=1 / np.var ddof=0)
        filas = [dict(componente=k, participacion_pct=float(100 * np.cov(v, y)[0, 1] / np.var(y))) for k, v in comp.items()]
    df = pd.DataFrame(filas)
    df.loc[len(df)] = dict(componente="SUMA", participacion_pct=df.participacion_pct.sum())
    return df


# ============================================================================================= 5) ORQUESTACIÓN
def _fila(bloque, coh, nat, capa, metrica, valor, nota=""):
    return dict(bloque=bloque, cohorte=coh, escenario_nat=nat, capa=capa, metrica=metrica, valor=valor, nota=nota)


def correr(ins, n=20000, seed=42, verbose=True, par=None):
    """Corre todo el análisis y devuelve un dict `res` (tablas + simulaciones para graficar)."""
    par = par or calibrar_incertidumbre(ins)
    cs = COHORTES_MC
    base = base_por_escenario(ins, cs)
    filas = []
    capas_modos = {"medicion": ("medicion",), "escenario": ("escenario",), "conjunto": CAPAS_MC}
    if par["se_rel_abar"] > 0:
        capas_modos = {"muestral": ("muestral",), **capas_modos}
    sims = {}
    # ---- NIVEL por cohorte, capa y escenario de natalidad
    for modo, capas in capas_modos.items():
        for e in NAT_ESC_MC:
            s_ = simular(par, base, n, seed, capas, nat=e)
            sims[(modo, e)] = s_
            for c in (2000, 2024, 2030, 2035):
                if c <= 2024 and e != NAT_ESC_MC[0]:
                    continue
                q = cuantiles(s_["tbp"][:, idx_c(s_, c)])
                for k, v in q.items():
                    filas.append(_fila("nivel", c, e if c > 2024 else "n/a", modo, k, v))
    # ---- punto base
    for c in (2000, 2024, 2030, 2035):
        for e in NAT_ESC_MC:
            if c <= 2024 and e != NAT_ESC_MC[0]:
                continue
            filas.append(_fila("nivel", c, e if c > 2024 else "n/a", "base", "punto", float(base.loc[c, e])))
    sc = sims[("conjunto", NAT_ESC_MC[0])]
    mez = simular(par, base, n, seed, CAPAS_MC, nat="mezcla")
    sims["mezcla"] = mez
    # ---- RAZÓN pareada 2024/2000 (y contraejemplos)
    p_pareada = {}
    for modo in capas_modos:
        r = razon_pareada(sims[(modo, NAT_ESC_MC[0])])
        q = cuantiles(r)
        for k, v in q.items():
            filas.append(_fila("razon_2024_2000", "2024/2000", "n/a", modo, k, v))
        filas.append(_fila("razon_2024_2000", "2024/2000", "n/a", modo, "P(razon<1)", float(np.mean(r < 1))))
        p_pareada[modo] = float(np.mean(r < 1))
    r_cj = razon_pareada(sc)
    r_unp = razon_no_pareada(sc)
    filas.append(_fila("razon_2024_2000", "2024/2000", "n/a", "conjunto_NO_pareado(incorrecto)", "P(2024>=2000)", float(np.mean(r_unp >= 1)),
                       "contraejemplo: sorteos independientes para cada cohorte"))
    for k, v in cuantiles(r_unp).items():
        filas.append(_fila("razon_2024_2000", "2024/2000", "n/a", "conjunto_NO_pareado(incorrecto)", k, v))
    # τ y ρ se cancelan: quitando sus factores (en logs) la razón no cambia
    lt = np.log(sc["tbp"]) - sc["L"]["tau"] - sc["L"]["rho"]
    cancel = float(np.max(np.abs(np.log(r_cj) - (lt[:, idx_c(sc, 2024)] - lt[:, idx_c(sc, 2000)]))))
    # ---- diseño original de la Sección 16
    rep = replica_seccion16(ins)
    for k in ("P5", "P25", "P50", "P75", "P95", "media"):
        filas.append(_fila("seccion16_original", 2024, "n/a", "original", k, rep[k]))
    filas.append(_fila("seccion16_original", 2024, "n/a", "original", "P(2024>=0,312 fijo)", rep["p_supera"]))
    # ---- TENDENCIA 2000→2024
    sl = {}
    for modo in capas_modos:
        s_ = sims[(modo, NAT_ESC_MC[0])]
        pend, w, _ = pendiente(s_)
        sl[modo] = pend
        for k, v in cuantiles(pend).items():
            filas.append(_fila("pendiente_2000_2024_pct_anual", "2000-2024", "n/a", modo, k, v))
        filas.append(_fila("pendiente_2000_2024_pct_anual", "2000-2024", "n/a", modo, "P(pendiente<0)", float(np.mean(pend < 0))))
    pb, _, _ = pendiente(simular(par, base, 1, seed, (), nat=NAT_ESC_MC[0]))
    filas.append(_fila("pendiente_2000_2024_pct_anual", "2000-2024", "n/a", "base", "punto", float(pb[0])))
    # ---- variación 2024→c futuras por escenario de natalidad
    for e in NAT_ESC_MC:
        s_ = sims[("conjunto", e)]
        for c in (2030, 2035):
            rr = s_["tbp"][:, idx_c(s_, c)] / s_["tbp"][:, idx_c(s_, 2024)]
            for k, v in cuantiles(rr).items():
                filas.append(_fila("razon_futura_vs_2024", c, e, "conjunto", k, v))
            filas.append(_fila("razon_futura_vs_2024", c, e, "conjunto", "P(razon>1)", float(np.mean(rr > 1))))
            rr0 = s_["tbp"][:, idx_c(s_, c)] / s_["tbp"][:, idx_c(s_, 2000)]
            filas.append(_fila("razon_futura_vs_2000", c, e, "conjunto", "P(razon<1)", float(np.mean(rr0 < 1))))
            filas.append(_fila("razon_futura_vs_2000", c, e, "conjunto", "P50", float(np.median(rr0))))
    # ---- ABANICO por cohorte (conjunto, por escenario) y en índice 2000=100
    ab = []
    for e in NAT_ESC_MC:
        s_ = sims[("conjunto", e)]
        qs = np.percentile(s_["tbp"], Q_MC, axis=0)
        qi = np.percentile(100 * s_["tbp"] / s_["tbp"][:, [idx_c(s_, 2000)]], Q_MC, axis=0)
        for j, c in enumerate(s_["cohortes"]):
            ab.append(dict(cohorte=c, escenario_nat=e if c > 2024 else "n/a", **{"nivel_P%d" % q: qs[i, j] for i, q in enumerate(Q_MC)},
                           **{"indice2000_P%d" % q: qi[i, j] for i, q in enumerate(Q_MC)}))
    abanico = pd.DataFrame(ab)
    abanico = abanico[~((abanico.cohorte <= 2024) & (abanico.escenario_nat == "n/a") & abanico.duplicated(["cohorte"], keep="first"))]
    # ---- ÍNDICES de primer orden: nivel vs tendencia
    cos = list(sc["cohortes"])
    e_ = lambda c: np.eye(len(cos))[cos.index(c)]
    var_rows = [indices_primer_orden(sc, e_(2024), "nivel 2024"),
                indices_primer_orden(mez, e_(2030), "nivel 2030 (mezcla de natalidad)"),
                indices_primer_orden(mez, e_(2035), "nivel 2035 (mezcla de natalidad)"),
                indices_primer_orden(sc, e_(2024) - e_(2000), "razón 2024/2000"),
                indices_primer_orden(sc, pesos_pendiente(cos), "pendiente 2000-2024", en_logs_y=False)]
    varianza = pd.concat(var_rows, ignore_index=True)
    # ---- SENSIBILIDAD a supuestos de correlación y SE de Ā (sobre la razón y la pendiente)
    sens = []
    for nom, cambio in [("base (corr N=0, J=0,5, Ā=1; sin SE muestral)", {}),
                        ("J independiente entre cohortes (corr 0)", {"corr": dict(par["corr"], J=0.0)}),
                        ("J perfectamente correlacionado (corr 1)", {"corr": dict(par["corr"], J=1.0)}),
                        ("SE de Ā provisorio 2,6 % + corr Ā=1", {"se_rel_abar": SE_REL_PROVISORIO}),
                        ("SE de Ā provisorio 2,6 % + corr Ā=0 (peor caso)", {"se_rel_abar": SE_REL_PROVISORIO, "corr": dict(par["corr"], Abar=0.0)}),
                        ("σ_N y σ_J duplicados", {"sig_N": 2 * par["sig_N"], "sig_J": 2 * par["sig_J"]})]:
        p2 = dict(par, **cambio)
        s2 = simular(p2, base, n, seed, CAPAS_MC, nat=NAT_ESC_MC[0])
        r2 = razon_pareada(s2)
        pe2, _, _ = pendiente(s2)
        sens.append(dict(supuesto=nom, razon_P5=np.percentile(r2, 5), razon_P50=np.median(r2), razon_P95=np.percentile(r2, 95),
                         P_razon_menor_1=float(np.mean(r2 < 1)), pend_P5=np.percentile(pe2, 5), pend_P95=np.percentile(pe2, 95)))
    sens = pd.DataFrame(sens)
    # ---- descomposiciones
    d = ins["d"]
    dlog = descomposicion_log_exacta(ins["modelo"], d) if d is not None else None
    dvar = descomposicion_varianza_cov(ins["modelo"])
    dvar_orig = descomposicion_varianza_cov(ins["modelo"], ddof_consistente=False)
    resumen = pd.DataFrame(filas)
    res = dict(par=par, base=base, sims=sims, mezcla=mez, resumen=resumen, abanico=abanico, varianza=varianza, sens=sens,
               dlog=dlog, dvar=dvar, dvar_orig=dvar_orig, replica=rep, razon=r_cj, razon_no_pareada=r_unp, pendientes=sl,
               cancelacion_tau_rho=cancel, p_pareada=p_pareada, n=n, seed=seed)
    if verbose:
        imprimir(res)
    return res


def imprimir(res):
    R = res["resumen"]

    def g(b, c, e, capa, m):
        x = R[(R.bloque == b) & (R.cohorte.astype(str) == str(c)) & (R.escenario_nat == e) & (R.capa == capa) & (R.metrica == m)].valor
        return float(x.iloc[0]) if len(x) else np.nan
    print("== Justificación de las entradas ==")
    for k, v in res["par"]["justificacion"].items():
        print("  %-14s %s" % (k, v))
    print("\n== NIVEL (años de prestación por jubilado), bandas P5-P95 por capa ==")
    for c, e in [(2000, "n/a"), (2024, "n/a")] + [(c, e) for c in (2030, 2035) for e in NAT_ESC_MC]:
        txt = "  cohorte %d %-19s base %.3f | " % (c, "" if e == "n/a" else e, g("nivel", c, e, "base", "punto"))
        for capa in ("muestral", "medicion", "escenario", "conjunto"):
            if np.isnan(g("nivel", c, e, capa, "P50")):
                continue
            txt += "%s [%.3f ; %.3f]  " % (capa[:4], g("nivel", c, e, capa, "P5"), g("nivel", c, e, capa, "P95"))
        print(txt)
    print("\n== RAZÓN pareada TBP(2024)/TBP(2000) ==")
    for capa in res["p_pareada"]:
        print("  %-9s P5 %.3f  P50 %.3f  P95 %.3f  | P(razón<1) = %.4f" % (capa, g("razon_2024_2000", "2024/2000", "n/a", capa, "P5"),
              g("razon_2024_2000", "2024/2000", "n/a", capa, "P50"), g("razon_2024_2000", "2024/2000", "n/a", capa, "P95"), res["p_pareada"][capa]))
    print("  cancelación τ,ρ en la razón: máx |Δ ln razón| entre corrida con y sin variar τ,ρ = %.2e" % res["cancelacion_tau_rho"])
    print("  [contraejemplo NO pareado] P(2024 >= 2000) = %.4f" % g("razon_2024_2000", "2024/2000", "n/a", "conjunto_NO_pareado(incorrecto)", "P(2024>=2000)"))
    rp = res["replica"]
    print("  [Sección 16 original] P5-P95 2024 = [%.3f ; %.3f]; fracción de sorteos >= 0,312 fijo = %.4f" % (rp["P5"], rp["P95"], rp["p_supera"]))
    print("\n== PENDIENTE 2000-2024 (% anual en logs) ==")
    for capa, v in res["pendientes"].items():
        print("  %-9s P5 %.2f  P50 %.2f  P95 %.2f" % (capa, *np.percentile(v, [5, 50, 95])))
    print("\n== Varianza de primer orden ==")
    print(res["varianza"].round(3).to_string(index=False))
    print("\n== Descomposición logarítmica exacta ==")
    print(res["dlog"].round(4).to_string(index=False))
    print("\n== Varianza por covarianzas (ddof consistente) / réplica original ==")
    print(res["dvar"].round(1).to_string(index=False)); print(res["dvar_orig"].round(1).to_string(index=False))


def exportar(res, out):
    """Escribe montecarlo_resumen.csv, montecarlo_abanico.csv y montecarlo_varianza_primer_orden.csv en `out`."""
    os.makedirs(out, exist_ok=True)
    fs = []
    for nom, df in (("montecarlo_resumen.csv", res["resumen"]), ("montecarlo_abanico.csv", res["abanico"]),
                    ("montecarlo_varianza_primer_orden.csv", res["varianza"]), ("montecarlo_sensibilidad_supuestos.csv", res["sens"]),
                    ("montecarlo_descomposicion_log_exacta.csv", res["dlog"])):
        f = os.path.join(out, nom)
        df.round(6).to_csv(f, index=False)
        fs.append(f)
    return fs


# ============================================================================================= 6) GRÁFICOS
def graficar(res, out, show=False):
    import matplotlib
    import matplotlib.pyplot as plt
    os.makedirs(out, exist_ok=True)
    files = []
    ab, base = res["abanico"], res["base"]
    colores = {"persistencia": "#0072b2", "recuperacion_lenta": "#009e73", "caida_adicional": "#cc3311"}
    # 1) abanicos: nivel (arriba) e índice 2000=100 (abajo)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), dpi=120)
    for a, pref, tit in ((ax[0], "nivel", "NIVEL (años): todas las capas juntas\n(incluye escenarios de τ, ρ y Ā)"),
                         (ax[1], "indice2000", "TENDENCIA: índice 2000 = 100 (sorteos pareados)\nτ y ρ se cancelan")):
        h = ab[ab.cohorte <= 2024].drop_duplicates("cohorte")
        for lo, hi, al in ((5, 95, .18), (25, 75, .32)):
            a.fill_between(h.cohorte, h["%s_P%d" % (pref, lo)], h["%s_P%d" % (pref, hi)], color="#444444", alpha=al, lw=0)
        a.plot(h.cohorte, h["%s_P50" % pref], color="#222222", lw=2, label="mediana de los sorteos (cohortes observadas)")
        bb = base[NAT_ESC_MC[0]].loc[2000:2024]
        a.plot(bb.index, bb.values if pref == "nivel" else 100 * bb.values / bb.loc[2000], color="#222222", lw=1.2, ls="--", label="caso base (punto)")
        for e, c_ in colores.items():
            f_ = ab[(ab.escenario_nat == e) & (ab.cohorte >= 2024)]
            f_ = pd.concat([h[h.cohorte == 2024], f_]).drop_duplicates("cohorte")
            for lo, hi, al in ((5, 95, .15), (25, 75, .28)):
                a.fill_between(f_.cohorte, f_["%s_P%d" % (pref, lo)], f_["%s_P%d" % (pref, hi)], color=c_, alpha=al, lw=0)
            a.plot(f_.cohorte, f_["%s_P50" % pref], color=c_, lw=1.8, label="natalidad: " + e.replace("_", " "))
        a.axvline(2024.5, color="gray", ls=":", lw=1)
        a.set_title(tit, fontsize=10, fontweight="bold"); a.set_xlabel("cohorte (año de nacimiento)"); a.grid(alpha=.25)
    ax[0].set_ylabel("TBP_C (años de prestación por jubilado)"); ax[1].set_ylabel("índice (2000 = 100)")
    ax[1].legend(fontsize=8, loc="lower left")
    fig.suptitle("Abanico por cohorte (bandas 50 % y 90 %). Condicional a los supuestos; no es un intervalo de confianza clásico", fontsize=10)
    plt.tight_layout(); f1 = os.path.join(out, "seccion_19_abanico_cohortes.png"); plt.savefig(f1); files.append(f1)
    plt.show() if show else plt.close()
    # 2) razón pareada vs contraejemplo
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), dpi=120)
    r, ru = res["razon"], res["razon_no_pareada"]
    bins = np.linspace(0.3, 1.5, 70)
    ax[0].hist(r, bins=bins, color="#0072b2", alpha=.75, density=True, label="PAREADA (correcta)")
    ax[0].hist(ru, bins=bins, color="#e69f00", alpha=.5, density=True, label="no pareada (incorrecta)")
    ax[0].axvline(1, color="k", lw=1.5); ax[0].set_xlabel("TBP(2024) / TBP(2000)"); ax[0].set_ylabel("densidad")
    ax[0].set_title("Razón entre cohortes: sorteos con razón >= 1: %d de %d (pareada) | %.1f %% (sin parear)" % (np.sum(r >= 1), len(r), 100 * np.mean(ru >= 1)), fontsize=9, fontweight="bold")
    ax[0].legend(fontsize=8.5)
    sl = res["pendientes"]["conjunto"]
    ax[1].hist(sl, bins=50, color="#009e73", alpha=.8, density=True)
    ax[1].axvline(0, color="k", lw=1.2)
    ax[1].set_xlabel("pendiente 2000-2024 (% anual en logs)"); ax[1].set_ylabel("densidad")
    ax[1].set_title("Tendencia: P5 %.2f | P50 %.2f | P95 %.2f" % tuple(np.percentile(sl, [5, 50, 95])), fontsize=9.5, fontweight="bold")
    plt.tight_layout(); f2 = os.path.join(out, "seccion_19_razon_pareada.png"); plt.savefig(f2); files.append(f2)
    plt.show() if show else plt.close()
    # 3) varianza de primer orden: nivel vs tendencia
    v = res["varianza"]
    v = v[v.grupo != "(suma de primer orden)"]
    sel = ["nivel 2024", "nivel 2035 (mezcla de natalidad)", "razón 2024/2000", "pendiente 2000-2024"]
    grupos = ["tau", "rho", "Abar_regla", "Abar_muestral", "N", "J", "natalidad"]
    et = {"tau": "τ (escenario)", "rho": "ρ (escenario)", "Abar_regla": "Ā regla (escenario)", "Abar_muestral": "Ā muestral (sin bootstrap aún)",
          "N": "N (medición)", "J": "J* (medición)", "natalidad": "natalidad (escenario)"}
    fig, ax = plt.subplots(figsize=(10, 4.6), dpi=120)
    wd = 0.2
    cols = ["#0072b2", "#56b4e9", "#e69f00", "#009e73"]
    for i, rname in enumerate(sel):
        vals = [float(v[(v.resultado == rname) & (v.grupo == g)].frac_log.sum()) * 100 for g in grupos]
        ax.bar(np.arange(len(grupos)) + (i - 1.5) * wd, vals, wd, color=cols[i], label=rname)
    ax.set_xticks(np.arange(len(grupos))); ax.set_xticklabels([et[g] for g in grupos], rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("% de la varianza (primer orden, en logs)"); ax.grid(axis="y", alpha=.25); ax.legend(fontsize=9)
    ax.set_title("¿Qué parámetro domina? El NIVEL lo mandan las definiciones; la TENDENCIA, la demografía medida", fontsize=10, fontweight="bold")
    plt.tight_layout(); f3 = os.path.join(out, "seccion_19_varianza_por_parametro.png"); plt.savefig(f3); files.append(f3)
    plt.show() if show else plt.close()
    # 4) bandas del nivel 2024 por capa + réplica Sección 16
    R = res["resumen"]
    fig, ax = plt.subplots(figsize=(10, 3.8), dpi=120)
    filas = []
    for capa, lab in (("medicion", "MEDICIÓN (N, J*)"), ("escenario", "ESCENARIO (Ā regla, τ, ρ)"), ("conjunto", "CONJUNTO (condicional)")):
        x = R[(R.bloque == "nivel") & (R.cohorte.astype(str) == "2024") & (R.capa == capa)].set_index("metrica").valor
        if len(x):
            filas.append((lab, x["P5"], x["P50"], x["P95"], x["minimo"], x["maximo"]))
    rp = res["replica"]
    filas.append(("Sección 16 original (mezclado)", rp["P5"], rp["P50"], rp["P95"], np.nan, np.nan))
    for i, (lab, p5, p50, p95, mn, mx) in enumerate(filas[::-1]):
        ax.plot([p5, p95], [i, i], color="#0072b2", lw=6, solid_capstyle="butt", alpha=.7)
        ax.plot([p50], [i], "o", color="k")
        if np.isfinite(mn):
            ax.plot([mn, mx], [i, i], color="#0072b2", lw=1, ls=":")
    ax.set_yticks(range(len(filas))); ax.set_yticklabels([f[0] for f in filas[::-1]], fontsize=9)
    ax.axvline(float(res["base"].loc[2024].iloc[0]), color="#cc3311", ls="--", lw=1.3, label="base 2024")
    ax.set_xlabel("TBP_C cohorte 2024 (años)"); ax.legend(fontsize=8.5); ax.grid(axis="x", alpha=.25)
    ax.set_title("Banda P5-P95 del nivel 2024 por capa de incertidumbre (punto = mediana, línea fina = mín-máx)", fontsize=10, fontweight="bold")
    plt.tight_layout(); f4 = os.path.join(out, "seccion_19_bandas_por_capa.png"); plt.savefig(f4); files.append(f4)
    plt.show() if show else plt.close()
    return files


# ============================================================================================= 7) AUTOTEST (DATOS SINTÉTICOS)
def _banner_mc(txt="DATOS SINTÉTICOS — no son Argentina"):
    print("*" * 78); print("***  " + txt); print("*" * 78)


def insumos_sinteticos():
    """Mini-mundo inventado y ROTULADO: 2000..2035, natalidad cae hasta 2024 y se bifurca en 3 escenarios; Ā crece 10 % entre 2000 y 2024."""
    co = np.arange(1998, 2036)
    N = np.where(co <= 2024, 700 - 12 * (co - 1998), np.nan)
    N = np.where(co > 2024, 400, N).astype(float)
    S = 0.88 + 0.001 * (co - 2000)
    J = 12e3 + 90 * (co - 2000)
    A = 13 + 0.07 * (co - 2000)
    f = lambda N_, a, s_, j: N_ * s_ * a * 0.2177 / (j * 0.403)
    tb = f(N, A, S, J)
    ma = pd.DataFrame(dict(cohorte=co, TBP_C_p4=tb, N_4=N, S_tot65=S, J_star_4=J, A_eph=A))
    for e, k in zip(NAT_ESC_MC, (1.0, 1.3, 0.9)):
        ma["TBP_C_p4_" + e] = np.where(co > 2024, tb * k, tb)
    ab = pd.DataFrame(dict(cohorte=co, A_bar_solo_asalariados=A, A_bar_pp07i=A * 1.2, A_bar_indep50=A * 1.6))
    d = pd.DataFrame(dict(cohorte=co, N_t=N, S65=S, J=J))
    return dict(modelo=ma.set_index("cohorte"), abar=ab.set_index("cohorte"), rho=None, series=None, renaper=None, jrw=None,
                nat=None, d=d, abar_ic=None)


def autotest_montecarlo(verbose=True):
    _banner_mc()
    ins = insumos_sinteticos()
    par = calibrar_incertidumbre(ins)
    base = base_por_escenario(ins, COHORTES_MC)
    # (1) reproducibilidad
    a = simular(par, base, 3000, 7, CAPAS_MC, "persistencia"); b = simular(par, base, 3000, 7, CAPAS_MC, "persistencia")
    assert np.array_equal(a["tbp"], b["tbp"]), "misma semilla debe dar lo mismo"
    c = simular(par, base, 3000, 8, CAPAS_MC, "persistencia")
    assert not np.array_equal(a["tbp"], c["tbp"])
    # (2) con todas las capas apagadas se recupera el valor base exacto
    z = simular(par, base, 50, 1, (), "persistencia")
    assert np.allclose(z["tbp"], np.repeat(base["persistencia"].values[None, :], 50, 0))
    # (3) τ y ρ se cancelan EXACTAMENTE en la razón (capa escenario con regla de Ā fija en asalariados)
    par_fix = dict(par, A={k: par["A"]["solo_asalariados"] for k in REGLAS_ABAR})
    s_ = simular(par_fix, base, 4000, 3, ("escenario",), "persistencia")
    r = razon_pareada(s_)
    assert np.ptp(r) < 1e-12, "la razón no debe depender de τ ni ρ"
    nivel = s_["tbp"][:, idx_c(s_, 2024)]
    assert nivel.max() / nivel.min() > 1.5, "pero el NIVEL sí cambia con τ y ρ"
    # (4) pareada es más angosta que no pareada
    sc = simular(par, base, 6000, 4, CAPAS_MC, "persistencia")
    assert np.std(np.log(razon_pareada(sc))) < 0.5 * np.std(np.log(razon_no_pareada(sc)))
    # (5) un solo factor lognormal con σ conocido: P5/P95 = exp(±1,645σ)
    p1 = dict(par, sig_J=0.0, corr=dict(N=0.0, J=0.5, Abar=1.0), sig_N=0.10)
    s1 = simular(p1, base, 40000, 11, ("medicion",), "persistencia")
    q = np.percentile(np.log(s1["tbp"][:, idx_c(s1, 2000)] / base.loc[2000, "persistencia"]), [5, 95])
    assert abs(q[1] - 1.645 * 0.10) < 0.01 and abs(q[0] + 1.645 * 0.10) < 0.01
    # (6) índices de primer orden: aditivo en logs suma 1; estimador por estratos recupera caso analítico
    v = indices_primer_orden(sc, np.eye(len(sc["cohortes"]))[idx_c(sc, 2024)], "nivel")
    assert abs(v[v.grupo != "(suma de primer orden)"].frac_log.sum() - 1) < 0.08     # in-sample: categorías con pocos niveles dejan covarianzas de azar
    rng = np.random.default_rng(0)
    x1, x2 = rng.standard_normal(100000), rng.standard_normal(100000)
    Y = x1 + 0.5 * x2
    assert abs(_indice_estratos(Y, x1) - 0.8) < 0.015 and abs(_indice_estratos(Y, x2) - 0.2) < 0.015
    # (7) descomposiciones: suman 100 %
    dl = descomposicion_log_exacta(ins["modelo"], ins["d"], 2000, 2024)
    for esp, g in dl.groupby("especificacion"):
        assert abs(g[g.componente != "TOTAL Δln TBP_C"].pct_de_la_caida.sum() - 100) < 1e-8
    dv = descomposicion_varianza_cov(ins["modelo"], 2000, 2024)
    assert abs(dv[dv.componente != "SUMA"].participacion_pct.sum() - 100) < 1e-8
    # (8) lector de bootstrap con un archivo SINTÉTICO de juguete
    import tempfile
    with tempfile.TemporaryDirectory() as tmp:
        pd.DataFrame(dict(cohorte=[2024], regla=["solo_asalariados"], abar=[14.0], se=[0.35])).to_csv(os.path.join(tmp, "abar_bootstrap_ic.csv"), index=False)
        ic = leer_abar_ic([tmp])
        assert ic is not None and abs(ic["se_rel"] - 0.025) < 1e-12
    # (9) corrida completa sobre el mini-mundo
    res = correr(ins, n=3000, seed=5, verbose=False)
    assert 0 <= res["p_pareada"]["conjunto"] <= 1 and len(res["resumen"]) > 50
    if verbose:
        print("autotest OK (9 chequeos sobre datos sintéticos)")
    return True

### 19.2 Autotest con datos SINTÉTICOS

Nueve chequeos sobre un mini-mundo **inventado y rotulado** (no es Argentina): reproducibilidad por semilla; capas apagadas = valor base; **$\tau$ y $\rho$ no mueven la razón pero sí el nivel**; pareada más angosta que no pareada; caso lognormal con cuantiles conocidos; estimador de primer orden contra un caso analítico ($S_1=0{,}8$ y $0{,}2$); descomposiciones que suman 100 %; lector del bootstrap con un archivo de juguete; corrida completa.

In [ ]:
autotest_montecarlo()

### 19.3 Entradas del proyecto y justificación de cada distribución

Cada fila dice **de qué dato del proyecto sale** y a **qué capa** pertenece. Lo que es *escenario* se trata como categorías con **pesos iguales** (una convención para poder simular, **no** probabilidades). Si todavía no existe `abar_bootstrap_ic.csv` (bootstrap real de $\bar A$, `src/tbp_bootstrap.py`), la capa muestral queda **en cero y rotulada**; en cuanto el archivo exista, el módulo lo lee solo (error estándar relativo) sin tocar nada.

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
try: display
except NameError: display = print
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 200)

CANDS_18 = ["../datos/procesados", "datos/procesados", DATA, OUT]
# `d` puede haber sido pisada por una variable de otra celda (p. ej. un bucle `for d in ...`): si no es la tabla de cohortes, se relee del CSV
D18 = d if (isinstance(globals().get("d"), pd.DataFrame) and {"cohorte", "N_t", "S65", "J"} <= set(d.columns)) else None
INS18 = cargar_insumos(rutas=CANDS_18, d=D18)
_tau, _rho = (TAU, RHO) if all(isinstance(globals().get(k), float) for k in ("TAU", "RHO")) else (0.2177, 0.403)
PAR18 = calibrar_incertidumbre(INS18, tau0=_tau, rho0=_rho)

CAPA18 = {"N": "MEDICIÓN", "J": "MEDICIÓN", "Abar_muestral": "MUESTRAL", "Abar_regla": "ESCENARIO/definición",
          "tau": "ESCENARIO", "rho": "ESCENARIO", "natalidad": "ESCENARIO"}
T18 = pd.DataFrame({"capa": pd.Series(CAPA18), "justificación (dato del proyecto)": pd.Series(PAR18["justificacion"])}).loc[list(CAPA18)]
display(T18)
print("Escenarios de τ:", PAR18["tau_esc"]); print("Escenarios de ρ:", {k: round(v, 4) for k, v in PAR18["rho_esc"].items()})
print("Correlación entre cohortes (supuesto, configurable): N =", PAR18["corr"]["N"], "| J* =", PAR18["corr"]["J"], "| Ā =", PAR18["corr"]["Abar"])

### 19.4 Nivel: bandas por capa (cohortes 2000, 2024, 2030, 2035)

20.000 sorteos conjuntos, semilla 42 (reproducible). Para 2030 y 2035 se muestra cada **escenario de natalidad** por separado (la natalidad futura es una *hipótesis*, no se promedia). Cada columna de banda es **una capa distinta**: la banda de *medición* es angosta (±6 %); la de *escenario* es ancha porque recorre definiciones de $\bar A$, $\tau$ y $\rho$. La banda *conjunta* es **condicional a esos escenarios** y no es un intervalo de confianza.

In [ ]:
RES18 = correr(INS18, n=20000, seed=42, verbose=False, par=PAR18)
R18 = RES18["resumen"]
def _g18(b, c, e, capa, m):
    x = R18[(R18.bloque == b) & (R18.cohorte.astype(str) == str(c)) & (R18.escenario_nat == e) & (R18.capa == capa) & (R18.metrica == m)].valor
    return float(x.iloc[0]) if len(x) else np.nan
filas = []
for c, e in [(2000, "n/a"), (2024, "n/a")] + [(c, e) for c in (2030, 2035) for e in NAT_ESC_MC]:
    r_ = {"cohorte": c, "natalidad": e, "base (punto)": _g18("nivel", c, e, "base", "punto")}
    for capa in ("medicion", "escenario", "conjunto"):
        r_[capa + " P5-P95"] = "[%.3f ; %.3f]" % (_g18("nivel", c, e, capa, "P5"), _g18("nivel", c, e, capa, "P95"))
    r_["conjunto mín-máx"] = "[%.3f ; %.3f]" % (_g18("nivel", c, e, "conjunto", "minimo"), _g18("nivel", c, e, "conjunto", "maximo"))
    filas.append(r_)
TN18 = pd.DataFrame(filas)
display(TN18.round(3))

### 19.5 Comparación pareada: ¿la cohorte 2024 respalda menos que la 2000?

La pregunta correcta compara **la misma "versión del mundo"** (mismo $\tau$, $\rho$, regla de $\bar A$) para ambas cohortes:
$$\frac{TBP_C(2024)}{TBP_C(2000)}=\frac{N_{24}\,S_{24}\,\bar A_{24}\,\tau/(J^*_{24}\rho)}{N_{00}\,S_{00}\,\bar A_{00}\,\tau/(J^*_{00}\rho)}=\frac{N_{24}}{N_{00}}\cdot\frac{S_{24}}{S_{00}}\cdot\frac{\bar A_{24}}{\bar A_{00}}\cdot\frac{J^*_{00}}{J^*_{24}}$$
$\tau$ y $\rho$ **desaparecen**: la comparación entre cohortes solo depende de $N$, $S_{65}$, $J^*$ y del cociente de $\bar A$ por cohorte. Abajo se verifica numéricamente y se muestra el **contraejemplo** (sorteos independientes para cada cohorte), que es lo que implícitamente hacía la sección 16 al fijar 2000 y sortear 2024.

In [ ]:
rz = RES18["razon"]; ru = RES18["razon_no_pareada"]
lnr = np.log(rz)
T_r = pd.DataFrame({
    "pareada, capa medición": [_g18("razon_2024_2000", "2024/2000", "n/a", "medicion", m) for m in ("P5", "P50", "P95")],
    "pareada, capa escenario": [_g18("razon_2024_2000", "2024/2000", "n/a", "escenario", m) for m in ("P5", "P50", "P95")],
    "pareada, CONJUNTO": [_g18("razon_2024_2000", "2024/2000", "n/a", "conjunto", m) for m in ("P5", "P50", "P95")],
    "NO pareada (incorrecta)": [_g18("razon_2024_2000", "2024/2000", "n/a", "conjunto_NO_pareado(incorrecto)", m) for m in ("P5", "P50", "P95")],
}, index=["P5", "P50", "P95"]).T
display(T_r.round(3))
print("Razón base (punto) = %.4f | sorteos con razón >= 1: %d de %d | razón máxima = %.3f" % (
    RES18["base"].loc[2024, "persistencia"] / RES18["base"].loc[2000, "persistencia"], int((rz >= 1).sum()), len(rz), rz.max()))
print("El cero de la razón (ln 1) queda a %.1f desvíos del centro de ln(razón) (media %.3f, desvío %.3f)." % (-lnr.mean() / lnr.std(), lnr.mean(), lnr.std()))
print("Cancelación de τ y ρ: máx |Δ ln razón| quitando sus factores = %.1e" % RES18["cancelacion_tau_rho"])
print("CONTRAEJEMPLO: con sorteos NO pareados, la fracción con 2024 >= 2000 sería %.1f %% (artefacto de mezclar escenarios distintos)." % (100 * np.mean(ru >= 1)))
print("Sección 16 original (2024 sorteada vs 2000 fijo en 0,312): %.2f %% de los sorteos ya supera 0,312 (el 'p<0,001' no se cumplía ni en su propia simulación)." % (100 * RES18["replica"]["p_supera"]))
display(RES18["sens"].round(3))

### 19.6 Cohortes futuras y tendencia: abanico por cohorte

Izquierda: **nivel** (todas las capas juntas, mediana y bandas del 50 % y 90 %; un abanico por escenario de natalidad desde 2025). Derecha: **índice 2000 = 100 con sorteos pareados**: ahí $\tau$ y $\rho$ no cuentan y el abanico se angosta mucho. La **tendencia** se resume con la pendiente MCO de $\ln TBP_C$ sobre la cohorte 2000–2024 (en % anual) y con la razón futura / 2024.

In [ ]:
F18 = graficar(RES18, OUT, show=True)
sl = RES18["pendientes"]["conjunto"]
print("Pendiente 2000-2024 (%% anual en logs): base %.2f | P5 %.2f | P50 %.2f | P95 %.2f | sorteos con pendiente >= 0: %d" % (
    _g18("pendiente_2000_2024_pct_anual", "2000-2024", "n/a", "base", "punto"), *np.percentile(sl, [5, 50, 95]), int((sl >= 0).sum())))
filas = []
for e in NAT_ESC_MC:
    for c in (2030, 2035):
        filas.append({"natalidad": e, "cohorte": c,
                      "razón vs 2024: P5": _g18("razon_futura_vs_2024", c, e, "conjunto", "P5"), "P50": _g18("razon_futura_vs_2024", c, e, "conjunto", "P50"),
                      "P95": _g18("razon_futura_vs_2024", c, e, "conjunto", "P95"), "P(razón>1)": _g18("razon_futura_vs_2024", c, e, "conjunto", "P(razon>1)"),
                      "P(TBP_c < TBP_2000)": _g18("razon_futura_vs_2000", c, e, "conjunto", "P(razon<1)")})
display(pd.DataFrame(filas).round(3))

### 19.7 Varianza de primer orden: ¿qué parámetro domina el nivel y cuál la tendencia?

Como $\ln TBP_C$ es **exactamente** la suma de los logs de los factores, el aporte de cada grupo de parámetros es aditivo en logs (`frac_log`, exacta). Se agregan dos medidas que no dependen de la aditividad: el **índice de Sobol de primer orden** $S_i=\mathrm{Var}(E[Y\mid X_i])/\mathrm{Var}(Y)$ (por estratos) y la **correlación de rangos al cuadrado** (`spearman2`). La suma de los $S_i$ cerca de 1 indica pocas interacciones. **Ojo:** son *participaciones* de la varianza, no tamaños: la varianza de la tendencia es mucho menor que la del nivel.

In [ ]:
V18 = RES18["varianza"]
display(V18.pivot_table(index="grupo", columns="resultado", values="frac_log", sort=False).mul(100).round(1)
        [["nivel 2024", "nivel 2035 (mezcla de natalidad)", "razón 2024/2000", "pendiente 2000-2024"]])
print("Lectura: el NIVEL lo mandan definiciones y escenarios (regla de Ā, τ, ρ: ~99 % en 2024); la TENDENCIA 2000-2024 depende de la demografía medida (J* y N).")
print("Ā muestral = 0 porque todavía no hay bootstrap real; natalidad solo cuenta para cohortes >= 2025.")

### 19.8 Descomposiciones: nombres correctos y por qué 84,4 % ≠ 107,6 %

* **Descomposición logarítmica exacta** (contabilidad de crecimiento): $\Delta\ln TBP_C=\Delta\ln N+\Delta\ln S_{65}+\Delta\ln \bar A+\Delta\ln\tau-\Delta\ln J-\Delta\ln\rho$. Es una **identidad**, sin residuo ni orden de entrada (el indicador es multiplicativo). *No* es Campbell–Shiller (que descompone retornos esperados con un VAR) ni un $R^2$.
* **Descomposición de varianza por covarianzas** (*participación en la covarianza*): como $\Delta\ln TBP_C=\sum_k\Delta x_k$, vale $\mathrm{Var}(\Delta\ln TBP_C)=\sum_k \mathrm{Cov}(\Delta x_k,\Delta\ln TBP_C)$ y cada participación $=\mathrm{Cov}/\mathrm{Var}$ suma 100 % **solo si se incluyen todos los componentes** (también $\bar A$). No es un $R^2$: puede ser negativa o mayor que 100 %.
* **84,4 % vs 107,6 % son dos especificaciones distintas, ambas válidas.** En *puntos log* $N$ aporta lo mismo en las dos ($-0{,}530$); cambia el **denominador** (la caída total): con $J=65+$ y $\bar A$ constante la caída es $-0{,}628$ ($0{,}530/0{,}628=84{,}4\,\%$); con $J^*$ y $\bar A$ de la EPH creciendo ($+0{,}096$) la caída es $-0{,}4925$ ($0{,}530/0{,}4925=107{,}6\,\%$). Responden preguntas distintas y **no se pueden comparar como porcentajes**; sí en puntos log.

In [ ]:
DL18 = RES18["dlog"]
display(DL18.round(4))
DV18 = RES18["dvar"]; DVo18 = RES18["dvar_orig"]
display(pd.DataFrame({"participación en la covarianza, ddof consistente (%)": DV18.set_index("componente").participacion_pct,
                      "réplica del cálculo original de 16.2 (np.cov ddof=1 / np.var ddof=0) (%)": DVo18.set_index("componente").participacion_pct}).round(1))

### 19.9 Errores corregidos respecto de la Sección 16

| # | Qué decía la Sección 16 | Qué es lo correcto | Cuánto cambia |
|---|---|---|---|
| 1 | "Prueba de oro": distribución de 2024 contra el valor **fijo** de 2000 (0,312). | Comparación **pareada**: razón $TBP_C(2024)/TBP_C(2000)$ con los mismos sorteos; $\tau$ y $\rho$ se cancelan. | Razón P5–P95 = [0,57 ; 0,66] (base 0,611); **0 de 20.000** sorteos con razón ≥ 1. Sin parear (error) saldría 11,3 % de cruces. |
| 2 | "$p<0{,}001$". | No hay $H_0$: es una frecuencia de sorteos **condicional a supuestos**. Además, ni en su propia simulación se cumplía. | En el diseño original el 0,32 % de los sorteos (32 de 10.000) ya superaba 0,312. |
| 3 | $\bar A\sim$ Triangular[13,5; 15,5; 23,6] (mezcla muestral y definición). | Separar: reglas de $\bar A$ = **escenario**; error de muestreo = **bootstrap** (pendiente: `abar_bootstrap_ic.csv`). | La regla de $\bar A$ explica 50 % de la varianza del nivel 2024 y solo 2 % de la de la razón. |
| 4 | $\rho\sim$ Triangular[37,1 ; 40,3 ; 56,3 %] (mínimo histórico como piso). | 3 escenarios de moratorias (40,3 / 48,3 / 56,3 %); para cohortes que se retiran en 2065+ el mínimo 2009–2023 no es un escenario. | $\rho$: 24 % de la varianza del nivel; **0 %** de la razón. |
| 5 | $\tau\sim$ Uniforme[18,3 ; 23,35 %]. | 4 escenarios: 18,3 (supuesto), 21,77, 23,35 y 27 %. | $\tau$: 25 % del nivel; **0 %** de la razón. |
| 6 | $N$: Normal ±1,5 % ("cubre ±3 %"). | $\sigma$ = desvío de $\ln$(RENAPER/DEIS) 2012–2024 = **2,03 %** (razón 0,979–1,051). | Banda de $N$ algo más ancha. |
| 7 | $J^*$: Normal ±1 % (sin respaldo). | $\sigma$ = RMS de $J_{rep}/J_{ONU}-1$, cohortes 2000–2024 = **3,27 %**. | $J^*$ pasa a ser el mayor aporte a la varianza de la **tendencia** (56 %). |
| 8 | Solo cohorte 2024. | Cohortes 2000, 2024, 2030, 2035 con 3 escenarios de natalidad y **abanico**. | Nivel conjunto 2035: [0,145 ; 0,469] según escenario. |
| 9 | Sin análisis de tendencia. | Pendiente 2000–2024 con abanico. | −1,73 %/año (P5–P95: −1,87 a −1,59); 0 sorteos con pendiente ≥ 0. |
| 10 | "Descomposiciones de Campbell–Shiller". | **Descomposición logarítmica exacta** y **descomposición de varianza por covarianzas**. | Cambia el nombre, no la cuenta de la log exacta (107,6 %). |
| 11 | "85,2 % de la varianza" (N), sin $\bar A$. | Mismo ddof en Cov y Var y todos los componentes. | N = **81,6 %**; el 85,2 % se debía a mezclar `np.cov` (ddof=1) con `np.var` (ddof=0): las participaciones sumaban 104,3 %. |
| 12 | "Ā (−19,0 %)" en la descomposición. | Era $\bar A$ **más** composición por sexo/retiro. | Ā = −19,5 %; sexo/retiro = +0,4 %. |
| 13 | Banda 2024 P5–P95 = [0,152 ; 0,269]. | Banda conjunta (condicional) [0,133 ; 0,331]; solo medición [0,179 ; 0,203]. | Más ancha: ahora incluye el escenario $\tau=27\,\%$ y las tres reglas de $\bar A$. |

*Nota:* en 16.1–16.3 los números del tornado y del bootstrap ($SE=0{,}38$) están **escritos a mano** en el código (no se calculan en la celda): conviene recalcularlos; esta sección no los usa.

### 19.10 Qué significa (y qué no)

* **No es un test de hipótesis clásico.** No hay $H_0$, ni estadístico, ni p-valor. El TBP es una identidad contable: lo que se cuantifica es **cuánto se mueve el resultado si las entradas se mueven**.
* **Es incertidumbre condicional a los supuestos:** "si N y J\* se miden con el error observado entre fuentes, y si las definiciones de $\bar A$, $\tau$, $\rho$ y la natalidad futura caen en estos escenarios, entonces el indicador cae en estas bandas". Los pesos iguales de los escenarios son una **convención**; las bandas conjuntas **no son intervalos de confianza**.
* **Lo robusto:** la **caída 2000→2024** (razón ≈ 0,61; ningún sorteo la revierte) y su **tendencia** (≈ −1,7 % anual). **Lo no robusto:** el **nivel** (depende sobre todo de cómo se defina $\bar A$ y de $\tau$ y $\rho$).
* **Cohortes futuras:** el signo de la variación 2024→2035 depende del escenario de natalidad: sube en los tres, pero con *caída adicional* la mejora es chica y en 2030 apenas supera 50 % de probabilidad (65 %).

### 19.11 Controles (asserts) y salidas

Se verifica: (i) consistencia entre `d` (notebook) y el CSV del modelo avanzado ($A$ constante 14,2 y $\tau$, $\rho$ del notebook); (ii) $N_{2024}$ de `s` (DEIS) igual a $N_4$ del modelo; (iii) $\rho$ base del notebook igual al último $\rho$ de ANSES; (iv) la razón pareada no depende de $\tau$ ni de $\rho$; (v) cuantiles ordenados; (vi) la réplica de 16.4 reproduce P5 = 0,152 y P95 = 0,269. Salidas en `OUT`: `montecarlo_resumen.csv` (formato largo: bloque, cohorte, escenario, capa, métrica, valor), abanico, varianza de primer orden, sensibilidad a supuestos y descomposición log exacta, más 4 PNG.

In [ ]:
M18 = INS18["modelo"]
d18 = INS18["d"].set_index("cohorte").loc[2000:2035]
assert np.allclose(M18.loc[2000:2035, "TBP_C_p0_A14_2"].values, (d18.N_t * d18.S65 * 14.2 * 0.2177 / (d18.J * 0.403)).values, rtol=1e-4), \
    "la tabla de cohortes y el CSV del modelo avanzado no coinciden"
if isinstance(globals().get("A_OBS"), float) and isinstance(globals().get("TAU"), float) and isinstance(globals().get("RHO"), float):
    assert abs(A_OBS - 14.2) < 1e-12 and abs(TAU - 0.2177) < 1e-12 and abs(RHO - 0.403) < 1e-12, "constantes del notebook distintas de las del modelo avanzado"
if isinstance(globals().get("s"), pd.DataFrame) and "deis_nacimientos" in s.columns:
    assert abs(float(s.deis_nacimientos[2024]) - float(M18.loc[2024, "N_4"])) < 1, "N 2024: DEIS (s) vs modelo"
assert abs(PAR18["rho0"] - list(PAR18["rho_esc"].values())[0]) < 1e-9, "ρ base debe ser el primer escenario (ρ 2023)"
assert RES18["cancelacion_tau_rho"] < 1e-12
for capa_, p_ in RES18["p_pareada"].items():
    assert 0.0 <= p_ <= 1.0
ci = R18[(R18.bloque == "nivel") & (R18.cohorte.astype(str) == "2024") & (R18.capa == "conjunto")].set_index("metrica").valor
assert ci["P5"] < ci["P25"] < ci["P50"] < ci["P75"] < ci["P95"]
assert abs(RES18["replica"]["P5"] - 0.1524) < 5e-4 and abs(RES18["replica"]["P95"] - 0.2685) < 5e-4, "no reproduce la sección 16.4"
DLc = RES18["dlog"]; DLn = DLc[DLc.componente == "N"].set_index(DLc[DLc.componente == "N"].especificacion.str[0]).pct_de_la_caida
assert abs(DLn["B"] - 107.6) < 0.1 and abs(DLn["A"] - 84.4) < 0.1
print("OK: controles aprobados")
F18c = exportar(RES18, OUT)
print("Escrito:", [os.path.abspath(f) for f in F18c]); print("PNG:", [os.path.abspath(f) for f in F18])

### 19.12 Límites

* **Las bandas son condicionales a un modelo.** La estructura (sexo y retiro, $S_{65}$ de la ONU, $\bar A$ de la EPH urbana y solo asalariados con descuento) se toma como dada; no se simula su incertidumbre (p. ej. $S_{65}$ no tiene error en esta sección).
* **$\bar A$ por regla** se aplica como factor $\bar A_{regla}(c)/\bar A_{asal}(c)$ sobre el modelo con sexo (aproximación: la regla se calcula sobre el $\bar A$ total, no por sexo). La cohorte 2024 en adelante usa la meseta de densidad (sin edades observadas), por lo que su $\bar A$ es una extrapolación; con $\bar A$ constante la caída 2000→2024 sería aún mayor (razón ≈ 0,56).
* **Capa muestral = 0 hasta que exista el bootstrap real** (`abar_bootstrap_ic.csv`). La sensibilidad con el $SE$ citado en 16.3 (2,6 %, no reproducible) y correlación 0 entre cohortes ensancha la razón a [0,56 ; 0,67] sin cambiar la conclusión.
* **Correlaciones entre cohortes** (N: 0, $J^*$: 0,5, $\bar A$: 1) son supuestos de trabajo, con sensibilidad en 18.5. $\sigma_J$ se toma de la discrepancia entre dos versiones de $J$ (ONU vs reponderado), que tiene signo sistemático; se usa solo como **escala**.
* **Pesos iguales** de los escenarios de $\tau$, $\rho$, $\bar A$ y natalidad: convención, no probabilidades. $\tau$ efectivo 18,3 % es un **supuesto**; $\tau$ no tiene serie histórica verificada.
* **2030 y 2035** reciben el mismo $\sigma_N$ de medición que las cohortes observadas, aunque su $N$ es proyectado: el error real de la proyección está en el escenario de natalidad, no en esa $\sigma$.

## 20. 🆕 Backtest de insumos y estabilidad temporal

**Problema.** La sección 16 explicó que el $TBP_C$ es una *identidad contable* y que no corresponde una regresión. Otra parte del proyecto agregó un test "in-sample / out-of-sample" (cohortes 1950-2010 vs 2011-2024, caída de −28,6 %, Chow con $p=0{,}0005$). Esta sección **evalúa críticamente ese test** y hace lo que **sí** se puede validar:

* **(A) Backtest de insumos:** comparar proyecciones *pasadas* de los insumos ($N$ nacimientos, $J$ población 65+) con lo que efectivamente ocurrió. Eso sí es "fuera de muestra" de verdad.
* **(B) Estabilidad temporal:** ver si el "motor" de la serie cambia entre tramos y dónde cambia la pendiente, con las salvedades del caso.
* **(C) Tabla final** de qué sirve validar y qué no (19.10).

No se descarga nada: se usan datos que ya están en el proyecto; lo que falta queda en una lista (19.9).

### 20.1 Por qué el test IS/OOS existente no es lo que parece

1. **No hay resultado observado.** Para validar fuera de muestra hace falta una variable que se observe *después* (p. ej. el déficit previsional) y comparar contra ella. El $TBP_C$ no tiene esa variable: es una cuenta. Además los TBP de las cohortes 2011-2024 usan $J$, $E_r$, $S_{65}$ y $ho$ **proyectados** a 2076-2089. Llamar "OOS" a ese tramo es solo una etiqueta de calendario, no una predicción puesta a prueba.
2. **Los $p$-valores no significan lo habitual.** La serie es determinística, suave y muy autocorrelada: valores vecinos casi se repiten. Un Chow supone errores independientes; acá no hay error aleatorio. Sirve como **descripción** de un cambio de pendiente, no como prueba. (En 19.8 se ve que, con una permutación por bloques, el $p$ del sup-F pasa de 0,0005 a 0,4-0,6.)
3. **El corte 2011 es arbitrario.** Los nacimientos del DEIS tienen su pico en 2014 y la caída arranca en 2015. El estadístico, además, es casi igual para cualquier corte entre 2013 y 2018 (19.8).
4. **El "look-ahead bias" está mal planteado.** Ningún parámetro se ajustó para reproducir una trayectoria de déficit, así que no hay sobreajuste que "mirar hacia adelante". El problema real es el **anacronismo**: se aplican parámetros actuales ($\bar A$, $\rho$, $\tau$ de 2017-2025) a cohortes lejanas y se supone **estabilidad estructural**. Eso no se testea con un corte temporal; se discute con sensibilidad (sección 11 y 16).

**Qué sí es defendible:** decir que las cohortes recientes tienen un $TBP_C$ menor (descripción), y que esa baja viene sobre todo de $N$ (descomposición, 19.7). **Qué no:** hablar de "degradación OOS" o de "quiebre estructural significativo".

### 20.2 Módulo (copia de `src/tbp_estabilidad.py`)

Todo el código está en una sola celda para que el Colab sea autosuficiente. Funciones de backtest (`error_pct`, `backtest_tabla`, `mc_tbp`, `PENDIENTES_DESCARGA`...) y de estabilidad (`componentes_log`, `ventanas_moviles`, `sup_f`, `wald_hac`, `permutacion_bloques`, `tabla_quiebre`). Los datos transcriptos del PDF de INDEC 2025 llevan la página de origen en comentarios.

In [ ]:
# ====== MÓDULO tbp_estabilidad (misma versión que src/tbp_estabilidad.py; se omite el bloque __main__) ======
_autotest_previo_19 = globals().get('autotest')   # el autotest de este módulo no pisa el de otras secciones
# -*- coding: utf-8 -*-
"""
tbp_estabilidad.py — Backtest de insumos y estabilidad temporal del TBP_C.

Proyecto TBP — Índice de Cuna Vacía (Taller de Indicadores).

POR QUÉ ESTE MÓDULO (en una línea): el TBP_C es una IDENTIDAD contable sin variable resultado
observada, así que NO existe un "fuera de muestra" predictivo para el índice. Lo que SÍ se puede
validar es (A) cuánto se equivocaron proyecciones pasadas de los INSUMOS (nacimientos N, población 65+ J)
y (B) si el "motor" de la serie cambia a lo largo del tiempo (estabilidad), con las salvedades del caso.

    TBP_C = N · (1−μ) · Ā · (τ/ρ) / J          ⇒   ln TBP = ln N + ln(1−μ) + ln Ā + ln(τ/ρ) − ln J     (exacta)

CONTENIDO
  A) Backtest de insumos:   error_pct, interpolar_a_fecha, backtest_nacimientos_wpp2024, backtest_poblacion_indec2013,
                            comparar_J_fuentes, backtest_tabla (arma backtest_insumos.csv), PENDIENTES_DESCARGA,
                            sigma_por_horizonte, mc_tbp (Monte Carlo con bandas empíricas en lugar de ±1,5 %).
  B) Estabilidad temporal:  componentes_log, ventanas_moviles, chow_f, sup_f, wald_hac, sup_wald_hac,
                            permutacion_bloques, tabla_quiebre.
  C) autotest() con datos SINTÉTICOS rotulados (no son Argentina).

REGLA: no descarga nada. Todo lo que no está en el proyecto queda en PENDIENTES_DESCARGA (solo lista).
"""
import os, datetime as _dt
import numpy as np
import pandas as pd

RUTAS_CANDIDATAS = ["../datos/procesados", "datos/procesados", "../datos/crudos", "datos/crudos"]

# ----------------------------------------------------------------------------------------------
# Datos transcriptos de PDF locales (INDEC 2025, base Censo 2022). Se citan con página.
# Fuente: datos_tbp/indec_proyecciones_nacionales_2022_2040.pdf  (INDEC, "Estimaciones y proyecciones de
# población, por sexo y edad. Total del país. Años 2022-2040", 1.ª ed., 2025).
# ----------------------------------------------------------------------------------------------
CENSO_2022 = {
    "fecha": _dt.date(2022, 5, 18),
    "total_enumerado": 45_892_285,        # PDF INDEC 2025, p. 12, Tabla 1 (Población 2022)
    "total_sin_calle": 45_886_580,        # PDF p. 19, Tabla 4 (no incluye situación de calle sin edad)
    "j65_censada": 5_464_057,             # PDF p. 19, Tabla 4 (65 y más, ambos sexos, censada)
    "total_estimada_a_fecha_censal": 46_122_853,   # PDF p. 19, Tabla 4 (estimación INDEC a la fecha censal)
    "j65_estimada_a_fecha_censal": 5_521_560,      # PDF p. 19, Tabla 4
}
CENSO_2010 = {"fecha": _dt.date(2010, 10, 27), "total_enumerado": 40_117_096}  # PDF 2025, p. 12, Tabla 1

# Población al 1-jul de cada año (INDEC 2025). Total: fila "Total" del Cuadro 2.1 (pp. 39 y 41).
# 65+: suma de las edades 65..99 y "100 y más" del Cuadro 2.1 (pp. 39-42), calculada al transcribir (la suma de
# todas las edades reproduce el Total sin diferencias).
INDEC_2025_TOTAL = {2022: 46135579, 2023: 46214462, 2024: 46301743, 2025: 46387098, 2026: 46466688, 2027: 46540071,
                    2028: 46609533, 2029: 46676999, 2030: 46744389, 2031: 46814264, 2032: 46886737, 2033: 46961803,
                    2034: 47039274, 2035: 47118716, 2036: 47199436, 2037: 47280504, 2038: 47360804, 2039: 47439108,
                    2040: 47514116}
INDEC_2025_J65 = {2022: 5529167, 2023: 5610238, 2024: 5711979, 2025: 5815927, 2026: 5922327, 2027: 6031328,
                  2028: 6142648, 2029: 6255112, 2030: 6367718, 2031: 6480544, 2032: 6594032, 2033: 6709837,
                  2034: 6831713, 2035: 6963335, 2036: 7105814, 2037: 7258199, 2038: 7421088, 2039: 7597415,
                  2040: 7789606}
INDEC_2025_EDAD0 = {2022: 499587, 2023: 471623, 2024: 453124, 2025: 444076}   # población de 0 años al 1-jul (p. 39)

# Archivos que HABRÍA QUE BAJAR (solo lista; tamaños medidos con cabeceras HTTP HEAD, sin descargar).
PENDIENTES_DESCARGA = [
    dict(archivo="INDEC 2013 — Serie Análisis Demográfico n.º 35 (PDF)",
         url="https://www.indec.gob.ar/ftp/cuadros/publicaciones/proyeccionesyestimaciones_nac_2010_2040.pdf",
         bytes=267_025, para="65+ proyectado para 2020 (Cuadro 2, por sexo y edad quinquenal) y TGF/natalidad (Cuadro 5) → "
         "error de J y de N de la proyección 2013 contra Censo 2022 y DEIS. Verificado: HTTP 200, application/pdf."),
    dict(archivo="WPP 2019 (paquete R 'wpp2019_1.1-1.tar.gz', datos de la ONU)",
         url="https://cran.r-project.org/src/contrib/wpp2019_1.1-1.tar.gz", bytes=4_684_504,
         para="Nacimientos/fecundidad y población por edad proyectados en 2019 → error a 4-5 años (2023-24) y 65+ 2022. "
              "Contenido exacto de tablas: confirmar al abrirlo."),
    dict(archivo="WPP 2017 (paquete R 'wpp2017_1.2-3.tar.gz')",
         url="https://cran.r-project.org/src/contrib/wpp2017_1.2-3.tar.gz", bytes=4_743_882,
         para="Vintage intermedio (horizonte 6-7 años)."),
    dict(archivo="WPP 2015 (paquete R 'wpp2015_1.1-2.tar.gz', carpeta Archive de CRAN)",
         url="https://cran.r-project.org/src/contrib/Archive/wpp2015/wpp2015_1.1-2.tar.gz", bytes=4_585_228,
         para="Horizonte ~8-9 años para 2023-24 y ~7 años para el Censo 2022."),
    dict(archivo="WPP 2012 (paquete R 'wpp2012_2.2-1.tar.gz')",
         url="https://cran.r-project.org/src/contrib/wpp2012_2.2-1.tar.gz", bytes=4_902_348,
         para="Horizonte ~10-12 años; sirve para ver cómo crece el error con el horizonte."),
    dict(archivo="WPP 2010", url="(no verificado: sin paquete en CRAN ni URL vigente hallada; las URL históricas "
         "population.un.org/wpp/Download/Files/... devolvieron 404)", bytes=None,
         para="Habría que pedirlo al archivo de la División de Población de la ONU; no se puede estimar el tamaño."),
]


def tamano_pendientes():
    """Suma (en MB) de lo verificable de PENDIENTES_DESCARGA."""
    return sum(p["bytes"] for p in PENDIENTES_DESCARGA if p["bytes"]) / 1e6


# ==============================================================================================
# A) BACKTEST DE INSUMOS
# ==============================================================================================
def error_pct(proyectado, observado):
    """Error relativo en %: 100·(proyectado/observado − 1). Positivo = la proyección SOBRESTIMÓ."""
    return 100.0 * (np.asarray(proyectado, float) / np.asarray(observado, float) - 1.0)


def interpolar_a_fecha(serie_1jul, fecha):
    """Interpola linealmente una serie al 1-jul de cada año a una fecha calendario (p. ej. 18-may-2022)."""
    f0 = _dt.date(fecha.year if fecha >= _dt.date(fecha.year, 7, 1) else fecha.year - 1, 7, 1)
    f1 = _dt.date(f0.year + 1, 7, 1)
    w = (fecha - f0).days / (f1 - f0).days
    return float(serie_1jul[f0.year] + (serie_1jul[f1.year] - serie_1jul[f0.year]) * w)


def _fila(bloque, insumo, proyeccion, publicada, ref, horizonte, proy, obs, unidad, tipo, fuente_obs, nota):
    return dict(bloque=bloque, insumo=insumo, proyeccion=proyeccion, publicada=publicada, referencia=ref,
                horizonte_anios=horizonte, proyectado=proy, observado=obs, unidad=unidad,
                error_abs=(None if (proy is None or obs is None) else proy - obs),
                error_pct=(None if (proy is None or obs is None) else float(error_pct(proy, obs))),
                tipo=tipo, fuente_observado=fuente_obs, nota=nota)


def backtest_nacimientos_wpp2024(s):
    """ONU WPP 2024 (publicada jul-2024) vs nacimientos DEIS. `s` = series_tbp_extraidas.csv indexado por 'year'
    (un_births_miles en MILES; deis_nacimientos en personas)."""
    filas = []
    for y in (2022, 2023, 2024):
        proy = float(s.un_births_miles[y]) * 1000
        obs = float(s.deis_nacimientos[y])
        extra = ("El error ≈ 0 en 2022 sugiere que ése fue el último año con dato incorporado: 2023 y 2024 son pronósticos a 1 y 2 años."
                 if y == 2022 else "DEIS 2024 puede ser provisorio (inscripciones tardías suben el dato) → el error de 2024 podría ser algo menor.")
        filas.append(_fila("A1 nacimientos", f"Nacimientos {y}", "ONU WPP 2024 (medio)", "jul-2024", y, y - 2022,
                           proy, obs, "personas", "backtest legítimo (proyección publicada antes del dato)",
                           "DEIS (Min. Salud), series_tbp_extraidas.csv", extra))
    return filas


def backtest_poblacion_indec2013(x13, j_wpp_miles=None, tot_wpp_miles=None):
    """INDEC 2013 (base Censo 2010; población total al 1-jul 2010-2040) contra el Censo 2022 y contra la reestimación
    INDEC 2025. `x13`: Series año→población (Cuadro 1 del xls). Devuelve lista de filas."""
    filas = []
    p22 = interpolar_a_fecha(x13, CENSO_2022["fecha"])
    filas.append(_fila("A2 población total", "Población total al 18-may-2022", "INDEC 2013 (base Censo 2010)", "nov-2013",
                       2022, 12, p22, CENSO_2022["total_enumerado"], "personas",
                       "backtest legítimo (12 años)", "Censo 2022 enumerado — PDF INDEC 2025, p. 12 (Tabla 1)",
                       "Proyección interpolada al 18-may-2022 (publicada al 1-jul)."))
    filas.append(_fila("A2 población total", "Población total al 18-may-2022 (vs estimación INDEC a fecha censal)",
                       "INDEC 2013 (base Censo 2010)", "nov-2013", 2022, 12, p22,
                       CENSO_2022["total_estimada_a_fecha_censal"], "personas", "backtest legítimo (12 años)",
                       "INDEC 2025 p. 19 (Tabla 4): estimación a la fecha censal, que corrige por omisión/sobrecobertura",
                       "Comparación más justa: el censo enumerado tiene subnumeración de 0,5 %."))
    p10 = interpolar_a_fecha(x13, CENSO_2010["fecha"])
    filas.append(_fila("A2 población total", "Población total al 27-oct-2010 (año base)", "INDEC 2013 (base Censo 2010)",
                       "nov-2013", 2010, 0, p10, CENSO_2010["total_enumerado"], "personas",
                       "control del año base (no es pronóstico)", "Censo 2010 enumerado — PDF INDEC 2025, p. 12",
                       "La proyección parte del censo CORREGIDO por omisión, por eso queda ~2 % por encima del enumerado."))
    # trayectoria: INDEC 2013 vs reestimación INDEC 2025 (mismo organismo, otra base censal)
    for y in (2025, 2030, 2035, 2040):
        filas.append(_fila("A2 población total", f"Población total {y}: INDEC 2013 vs INDEC 2025", "INDEC 2013", "nov-2013",
                           y, y - 2010, float(x13[y]), float(INDEC_2025_TOTAL[y]), "personas",
                           "revisión entre vintages (la 'realidad' es otra proyección, no un dato)",
                           "INDEC 2025, Cuadro 2.1 (pp. 39-41)",
                           "Muestra cuánto cambia el escenario al incorporar el Censo 2022: mide revisión, no error contra hechos."))
    return filas


def comparar_J_fuentes(j_wpp_miles, tot_wpp_miles=None):
    """65+ de la ONU WPP 2024 (J del TBP) contra (i) Censo 2022 y (ii) INDEC 2025, 2022-2040.
    j_wpp_miles: Series año→65+ en MILES (1-jul)."""
    filas = []
    j22 = interpolar_a_fecha(j_wpp_miles * 1000, CENSO_2022["fecha"])
    filas.append(_fila("A3 población 65+", "65+ al 18-may-2022", "ONU WPP 2024", "jul-2024", 2022, 0, j22,
                       CENSO_2022["j65_censada"], "personas", "chequeo de nivel (WPP 2024 NO incorpora el Censo 2022: ver nota)",
                       "Censo 2022 — PDF INDEC 2025, p. 19 (Tabla 4), censada",
                       "Se verificó que WPP 2024 da 45,41 M para 2022, 1,1 % bajo el censo: no está calibrada con él."))
    filas.append(_fila("A3 población 65+", "65+ al 18-may-2022 (vs estimación INDEC a fecha censal)", "ONU WPP 2024", "jul-2024",
                       2022, 0, j22, CENSO_2022["j65_estimada_a_fecha_censal"], "personas", "chequeo de nivel",
                       "PDF INDEC 2025, p. 19 (Tabla 4)", "El censo subenumera 65+ en 1,0 % (INDEC)."))
    for y in (2022, 2025, 2030, 2035, 2040):
        filas.append(_fila("A3 población 65+", f"65+ {y}: ONU WPP 2024 vs INDEC 2025", "ONU WPP 2024", "jul-2024", y, max(y - 2024, 0),
                           float(j_wpp_miles[y]) * 1000, float(INDEC_2025_J65[y]), "personas",
                           "dispersión entre fuentes (NO es error de proyección)", "INDEC 2025, Cuadro 2.1 (pp. 39-42)",
                           "Sirve de piso para la incertidumbre de J: dos instituciones serias difieren esto."))
    return filas


def dispersion_J(j_wpp_miles):
    """Serie de diferencias % WPP2024 vs INDEC2025 en 65+ (2022-2040) y su RMSE."""
    ys = sorted(INDEC_2025_J65)
    e = pd.Series([float(error_pct(j_wpp_miles[y] * 1000, INDEC_2025_J65[y])) for y in ys], index=ys)
    return e, float(np.sqrt((e ** 2).mean()))


def proxy_edad0(s):
    """Proxy (no es error estricto): población de 0 años al 1-jul-2024 (INDEC 2025) vs promedio de nacimientos DEIS 2023-24
    (los de 0 años a mitad de 2024 nacieron entre jul-2023 y jun-2024)."""
    obs = (float(s.deis_nacimientos[2023]) + float(s.deis_nacimientos[2024])) / 2
    return _fila("A1 nacimientos", "Población de 0 años al 1-jul-2024 vs nacimientos DEIS (prom. 2023-24)", "INDEC 2025", "2025",
                 2024, 0, float(INDEC_2025_EDAD0[2024]), obs, "personas", "proxy (sin corrección por mortalidad infantil/migración)",
                 "DEIS", "INDEC 2025 ve ~3-4 % más nacimientos que el promedio DEIS: coherente con que DEIS 2024 sea provisorio.")


def backtest_tabla(s, x13=None):
    """Arma el DataFrame completo de backtest_insumos.csv (incluye filas 'pendiente de descarga')."""
    filas = backtest_nacimientos_wpp2024(s)
    filas.append(proxy_edad0(s))
    if x13 is not None:
        filas += backtest_poblacion_indec2013(x13)
    filas += comparar_J_fuentes(s.un_pob_65mas_miles)
    for p in PENDIENTES_DESCARGA:
        filas.append(dict(bloque="A4 pendiente", insumo=p["archivo"], proyeccion="(vintage anterior)", publicada="", referencia="",
                          horizonte_anios=None, proyectado=None, observado=None, unidad="", error_abs=None, error_pct=None,
                          tipo="PENDIENTE DE DESCARGA", fuente_observado=p["url"],
                          nota=(f"{p['bytes']/1e6:.2f} MB. " if p["bytes"] else "tamaño desconocido. ") + p["para"]))
    return pd.DataFrame(filas)


def sigma_por_horizonte(sigma_ref, h_ref, h, modo="raiz"):
    """Escala un desvío medido a horizonte h_ref hacia el horizonte h. HEURÍSTICA (supuesto, no estimada):
    'raiz' = el error crece como √h (camino aleatorio); 'constante' = no escala."""
    if modo == "constante" or h_ref <= 0:
        return float(sigma_ref)
    return float(sigma_ref * np.sqrt(max(h, 1) / h_ref))


def mc_tbp(N, S, J, sigma_N=0.015, sigma_J=0.01, n=10000, seed=42, a=(13.5, 15.5, 23.6), tau=(0.183, 0.2335),
           rho=(0.371, 0.403, 0.563)):
    """Monte Carlo idéntico al de la sección 16.4 del notebook (mismos rangos de Ā, τ, ρ y misma semilla y orden de
    sorteos), pero con σ_N y σ_J parametrizables (relativos). Con sigma_N=0,015 y sigma_J=0,01 reproduce el original."""
    np.random.seed(seed)
    Ns = np.random.normal(N, sigma_N * N, n)
    As = np.random.triangular(a[0], a[1], a[2], n)
    ts = np.random.uniform(tau[0], tau[1], n)
    rs = np.random.triangular(rho[0], rho[1], rho[2], n)
    Js = np.random.normal(J, sigma_J * J, n)
    sim = Ns * S * As * ts / (Js * rs)
    return sim, dict(p5=float(np.percentile(sim, 5)), p50=float(np.percentile(sim, 50)), p95=float(np.percentile(sim, 95)),
                     sd=float(sim.std()))


# ==============================================================================================
# B) ESTABILIDAD TEMPORAL
# ==============================================================================================
def componentes_log(av, version="p4", hasta=2024):
    """Descomposición logarítmica EXACTA de TBP_C por cohorte. `av` = tbp_modelo_avanzado_1950_2035.csv.
    ln TBP = ln N + ln S65 + ln Ā + ln(τ/ρ) − ln J.   (S65 = 1−μ).
    version 'p4': N_4, J_star_4 (J* por sexo/edad de retiro), Ā_efectivo = TBP·J*/(N·S·τ/ρ) (incluye el reparto por sexo y
                  edad de retiro: es el Ā que cierra la identidad), ρ del escenario por cohorte.
    version 'p1': N_t, J65 (65+ para todos), Ā = A_eph, τ/ρ constantes (= TBP_p1·J/(N·S·Ā))."""
    a = av.set_index("cohorte") if "cohorte" in av.columns else av
    a = a.loc[:hasta]
    if version == "p4":
        y, N, J, S = a.TBP_C_p4, a.N_4, a.J_star_4, a.S_tot65
        tr = a.tau_t / a.rho_escenario
        Ae = y * J / (N * S * tr)
    elif version == "p1":
        y, N, J, S = a.TBP_C_p1, a.N_t, a.J65, a.S_tot65
        Ae = a.A_eph
        tr = y * J / (N * S * Ae)
    else:
        raise ValueError("version debe ser 'p4' o 'p1'")
    c = pd.DataFrame({"lnTBP": np.log(y), "N": np.log(N), "S65": np.log(S), "Abar": np.log(Ae),
                      "tau_rho": np.log(tr), "J": -np.log(J)})
    c["TBP"] = y.values
    c.index.name = "cohorte"
    assert np.allclose(c[["N", "S65", "Abar", "tau_rho", "J"]].sum(axis=1), c.lnTBP, atol=1e-9), "descomposición no cierra"
    return c


def _pend(y, x):
    return float(np.polyfit(x, y, 1)[0])


FACTORES = ["N", "S65", "Abar", "tau_rho", "J"]


def ventanas_moviles(comp, ancho=20):
    """Para cada ventana de `ancho` cohortes consecutivas: media de TBP, pendiente de ln TBP por cohorte (en %/cohorte) y
    contribución de cada factor (motor_baja = el que más empuja hacia abajo; motor_sube = el que más empuja hacia arriba). Como la pendiente MCO es lineal, la suma de las pendientes de los factores = pendiente
    de ln TBP (exacto). También se da el cambio extremo a extremo (Δln) por factor."""
    filas = []
    co = comp.index.values
    for i in range(0, len(co) - ancho + 1):
        w = comp.iloc[i:i + ancho]
        x = np.arange(ancho, dtype=float)
        r = dict(ini=int(co[i]), fin=int(co[i + ancho - 1]), media_TBP=float(w.TBP.mean()),
                 pend_lnTBP_pct=100 * _pend(w.lnTBP.values, x))
        for f in FACTORES:
            r[f"pend_{f}_pp"] = 100 * _pend(w[f].values, x)
            r[f"dln_{f}_pp"] = 100 * float(w[f].iloc[-1] - w[f].iloc[0])
        r["dln_total_pp"] = 100 * float(w.lnTBP.iloc[-1] - w.lnTBP.iloc[0])
        r["control_suma_pend"] = sum(r[f"pend_{f}_pp"] for f in FACTORES) - r["pend_lnTBP_pct"]
        # qué factor empuja más hacia abajo / hacia arriba en la ventana (mayor pendiente negativa / positiva)
        r["motor_baja"] = min(FACTORES, key=lambda ff: r[f"pend_{ff}_pp"])
        r["motor_sube"] = max(FACTORES, key=lambda ff: r[f"pend_{ff}_pp"])
        filas.append(r)
    return pd.DataFrame(filas)


def _rss(y, X):
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ b
    return float(e @ e), b


def chow_f(y, x, corte, k=2):
    """F de Chow. k=2: constante+tendencia en cada tramo (x<corte vs x>=corte); k=1: solo media. Descriptivo."""
    y = np.asarray(y, float); x = np.asarray(x, float)
    X = np.column_stack([np.ones(len(x)), x])[:, :k]
    m = x < corte
    if m.sum() <= k or (~m).sum() <= k:
        return np.nan
    rp, _ = _rss(y, X)
    r1, _ = _rss(y[m], X[m]); r2, _ = _rss(y[~m], X[~m])
    return ((rp - (r1 + r2)) / k) / ((r1 + r2) / (len(y) - 2 * k))


def sup_f(y, x, cortes, k=2):
    """F de Chow para cada corte candidato; devuelve (Serie F por corte, corte del máximo, F máximo)."""
    F = pd.Series({int(c): chow_f(y, x, c, k) for c in cortes})
    return F, int(F.idxmax()), float(F.max())


def wald_hac(y, x, corte, lag=None):
    """Wald (2 restricciones: salto de nivel y cambio de pendiente en `corte`) con covarianza Newey-West (Bartlett).
    y = b0 + b1·t + b2·D + b3·D·(t−corte) + e,  D = 1[t>=corte]."""
    y = np.asarray(y, float); x = np.asarray(x, float); n = len(y)
    D = (x >= corte).astype(float)
    X = np.column_stack([np.ones(n), x, D, D * (x - corte)])
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ b
    L = int(np.floor(4 * (n / 100) ** (2 / 9))) if lag is None else lag
    Xe = X * e[:, None]
    S = Xe.T @ Xe
    for l in range(1, L + 1):
        G = Xe[l:].T @ Xe[:-l]
        S += (1 - l / (L + 1)) * (G + G.T)
    XtXi = np.linalg.inv(X.T @ X)
    V = XtXi @ S @ XtXi
    R = np.array([[0, 0, 1, 0], [0, 0, 0, 1]], float)
    rb = R @ b
    return float(rb @ np.linalg.inv(R @ V @ R.T) @ rb)


def sup_wald_hac(y, x, cortes, lag=None):
    W = pd.Series({int(c): wald_hac(y, x, c, lag) for c in cortes})
    return W, int(W.idxmax()), float(W.max())


def permutacion_bloques(y, x, cortes, estadistico="F", bloque=8, B=499, seed=1):
    """P-valor por permutación CIRCULAR por bloques del sup-estadístico. Nula: una tendencia lineal única + ruido con la
    autocorrelación local preservada (se reordenan bloques de `bloque` residuos). SALVEDAD: con una serie determinística,
    suave y con ciclos largos el 'ruido' no es ruido; este p-valor es una vara de comparación, no una prueba formal."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y, float); x = np.asarray(x, float); n = len(y)
    X = np.column_stack([np.ones(n), x])
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    fit = X @ b; res = y - fit
    f = (lambda yy: sup_f(yy, x, cortes)[2]) if estadistico == "F" else (lambda yy: sup_wald_hac(yy, x, cortes)[2])
    obs = f(y)
    nb = int(np.ceil(n / bloque))
    sup = np.empty(B)
    for i in range(B):
        ini = rng.permutation(n)[:nb] if nb <= n else rng.integers(0, n, nb)
        idx = np.concatenate([(np.arange(bloque) + s0) % n for s0 in ini])[:n]
        sup[i] = f(fit + res[idx])
    p = (1 + int((sup >= obs).sum())) / (B + 1)
    return obs, p, sup


def tabla_quiebre(serie, cortes=range(2000, 2019), nombre="TBP_C_p4", bloques=(5, 15), B=499):
    """Tabla resumen: F por corte, sup-F y su argmax, Wald-HAC y su argmax, p-valores por bloques, y el F del corte 2011."""
    y = serie.values.astype(float); x = serie.index.values.astype(float)
    F, cF, fmax = sup_f(y, x, cortes)
    W, cW, wmax = sup_wald_hac(y, x, cortes)
    out = dict(serie=nombre, corte_supF=cF, supF=fmax, F_2011=float(F.get(2011, np.nan)), corte_supWaldHAC=cW, supWaldHAC=wmax)
    for bl in bloques:
        _, pF, _ = permutacion_bloques(y, x, cortes, "F", bl, B)
        _, pW, _ = permutacion_bloques(y, x, cortes, "W", bl, B)
        out[f"p_perm_supF_bloque{bl}"] = pF
        out[f"p_perm_supWald_bloque{bl}"] = pW
    por_corte = pd.DataFrame({"corte": list(F.index), "F_chow": F.values, "Wald_HAC": W.values})
    por_corte["serie"] = nombre
    return out, por_corte


# ==============================================================================================
# C) AUTOTEST (SINTÉTICO)
# ==============================================================================================
def _banner(t):
    print("=" * 78); print(t); print("=" * 78)


def autotest(verbose=True):
    """Pruebas con datos SINTÉTICOS (no son Argentina)."""
    if verbose:
        _banner("AUTOTEST tbp_estabilidad — DATOS SINTÉTICOS (no son Argentina)")
    # 1) error_pct e interpolación
    assert abs(float(error_pct(110, 100)) - 10.0) < 1e-12 and abs(float(error_pct(90, 100)) + 10.0) < 1e-12
    serie = {2021: 100.0, 2022: 200.0}
    assert abs(interpolar_a_fecha(serie, _dt.date(2021, 7, 1)) - 100.0) < 1e-9
    assert abs(interpolar_a_fecha(serie, _dt.date(2022, 1, 1)) - (100 + 100 * 184 / 365)) < 1e-9
    # 2) descomposición exacta: construir un av sintético con factores conocidos
    co = np.arange(1950, 2036)
    rng = np.random.default_rng(0)
    N = 5e5 * np.exp(0.01 * (co - 1950) - 0.0002 * (co - 1950) ** 2) * np.exp(rng.normal(0, .01, len(co)))
    S = np.linspace(0.7, 0.93, len(co)); Ab = 14 + np.sin(co / 9.0); J = 4e6 * np.exp(0.03 * (co - 1950) ** 0.8)
    tau = 0.2177; rho = np.where(co < 1960, 0.4, 0.403)
    tbp = N * S * Ab * tau / (rho * J)
    av = pd.DataFrame(dict(cohorte=co, N_4=N, N_t=N, S_tot65=S, J_star_4=J, J65=J, A_eph=Ab, tau_t=tau,
                           rho_escenario=rho, TBP_C_p4=tbp, TBP_C_p1=N * S * Ab * (tau / 0.403) / J))
    c4 = componentes_log(av, "p4"); c1 = componentes_log(av, "p1")
    assert np.allclose(c4.Abar, np.log(Ab[:75])) and np.allclose(c4.N, np.log(N[:75]))
    assert np.allclose(c1.tau_rho, np.log(tau / 0.403))
    # 3) ventanas: la suma de pendientes cierra y el número de ventanas es n−ancho+1
    v = ventanas_moviles(c4, 20)
    assert len(v) == 75 - 20 + 1 and np.allclose(v.control_suma_pend, 0, atol=1e-9)
    # 4) sup-F recupera un quiebre sintético (serie con quiebre de pendiente en 2005 + ruido AR(1))
    x = np.arange(1950, 2025, dtype=float)
    e = np.zeros(len(x))
    for i in range(1, len(x)):
        e[i] = 0.5 * e[i - 1] + rng.normal(0, 0.01)
    y_q = 0.5 + 0.0 * x - 0.02 * np.clip(x - 2005, 0, None) + e
    F, cF, fm = sup_f(y_q, x, range(2000, 2019))
    assert abs(cF - 2005) <= 3, cF
    W, cW, wm = sup_wald_hac(y_q, x, range(2000, 2019))
    assert abs(cW - 2005) <= 4, cW
    # 5) bajo la nula (tendencia lineal + AR) el p-valor por bloques NO debe ser sistemáticamente ínfimo
    y_n = 0.5 - 0.002 * (x - 1950) + e
    _, p_n, _ = permutacion_bloques(y_n, x, range(2000, 2019), "F", 8, B=199, seed=3)
    _, p_q, _ = permutacion_bloques(y_q, x, range(2000, 2019), "F", 8, B=199, seed=3)
    assert p_q < 0.05 and p_n > p_q
    # 6) sigma por horizonte y Monte Carlo (reproducibilidad y monotonía en σ)
    assert abs(sigma_por_horizonte(0.02, 9, 36) - 0.04) < 1e-12 and sigma_por_horizonte(0.02, 9, 36, "constante") == 0.02
    _, a = mc_tbp(4e5, 0.9, 1.5e7, 0.015, 0.01, n=2000); _, a2 = mc_tbp(4e5, 0.9, 1.5e7, 0.015, 0.01, n=2000)
    _, b = mc_tbp(4e5, 0.9, 1.5e7, 0.15, 0.05, n=2000)
    assert a == a2 and (b["p95"] - b["p5"]) > (a["p95"] - a["p5"])
    # 7) tamaño de pendientes verificables
    assert 15 < tamano_pendientes() < 30
    if verbose:
        print(f"OK: error %, interpolación, descomposición exacta, ventanas (sumas cierran), sup-F (corte hallado {cF}; Wald-HAC {cW}),")
        print(f"    permutación por bloques (p con quiebre {p_q:.3f} < p sin quiebre {p_n:.3f}), σ por horizonte, Monte Carlo.")
        _banner("FIN AUTOTEST — estos números NO son Argentina")
    return True

autotest_estabilidad = autotest
if _autotest_previo_19 is not None:
    autotest = _autotest_previo_19

### 20.3 Autotest con datos sintéticos

Verifica con series **inventadas** (rotuladas como tales): error %, interpolación a fecha censal, descomposición logarítmica exacta, ventanas (las sumas cierran), que el sup-F recupera un quiebre plantado en 2005 y que la permutación por bloques da un $p$ menor con quiebre que sin él, y reproducibilidad del Monte Carlo.

In [ ]:
_tabla_sintetica_19 = autotest_estabilidad()

### 20.4 Carga de datos

Se leen de las rutas candidatas `../datos/procesados`, `datos/procesados`, `../datos/crudos`, `datos/crudos`: el modelo avanzado (`tbp_modelo_avanzado_1950_2035.csv`) y el xls de INDEC 2013 (población total al 1-jul, 2010-2040). Las series de la ONU y los nacimientos DEIS salen de `s` (sección 0). Las cifras del Censo 2022 y de INDEC 2025 están transcriptas en el módulo con su página.

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
try: display
except NameError: display = print
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

def buscar_archivo_19(nombre):
    for c in ["../datos/procesados", "datos/procesados", "../datos/crudos", "datos/crudos", DATA]:
        p = os.path.join(c, nombre)
        if os.path.exists(p):
            return p
    return None

DEST19 = next((p for p in ["../datos/procesados", "datos/procesados"] if os.path.isdir(p)), OUT)
os.makedirs(OUT, exist_ok=True)

# `s` (series ONU/DEIS) puede haber sido pisada por una variable de bucle en secciones previas (p. ej. 16.2): se valida y, si hace falta, se recarga.
S19 = s if (isinstance(s, pd.DataFrame) and "deis_nacimientos" in s.columns) else pd.read_csv(buscar_archivo_19("series_tbp_extraidas.csv")).set_index("year")

f_av = buscar_archivo_19("tbp_modelo_avanzado_1950_2035.csv")
HAY_AV19 = f_av is not None
AV19 = pd.read_csv(f_av) if HAY_AV19 else None
print("Modelo avanzado:", f_av if HAY_AV19 else "NO ENCONTRADO (las partes B y C se omiten)")

# INDEC 2013 (Cuadro 1: población total al 1-jul 2010-2040) — archivo local del proyecto
X13 = None
f_x13 = buscar_archivo_19("indec_c1_proyecciones_nac_2010_2040.xls")
try:
    _x = pd.read_excel(f_x13, header=None).iloc[:, :4].dropna()
    _x = _x[pd.to_numeric(_x[0], errors="coerce").between(2010, 2040)]
    X13 = pd.Series(_x[1].astype(float).values, index=_x[0].astype(int).values)
    assert len(X13) == 31 and X13[2010] > 4e7
except Exception as e:
    X13 = None; print("No se pudo leer el xls de INDEC 2013 (", type(e).__name__, "): se omiten esas filas.")
print("INDEC 2013 leído:", X13 is not None, "| S19 (series ONU/DEIS):", S19.shape)

### 20.5 Backtest de insumos (A)

**Qué se compara y con qué honestidad:**

* **Nacimientos, ONU WPP 2024 (jul-2024) vs DEIS.** 2022: +0,2 % (dato ya incorporado); **2023: +9,3 %; 2024: +22,6 %**. La ONU sobrestimó porque la natalidad siguió cayendo. Salvedad: el DEIS 2024 puede ser provisorio (las inscripciones tardías lo suben), así que el error de 2024 es una cota alta. El proxy con la población de 0 años de INDEC 2025 da +3,7 % sobre el promedio DEIS 2023-24, coherente con eso.
* **Población total, INDEC 2013 vs Censo 2022 (a 12 años).** +0,6 % contra el censo enumerado (45,89 M, p. 12 del PDF de INDEC 2025) y +0,1 % contra la estimación INDEC a la fecha censal (46,12 M, p. 19). Acertó el total, pero eso **no prueba** que haya acertado nacimientos y 65+ por separado.
* **Revisión INDEC 2013 → INDEC 2025.** Las proyecciones se separan: +2,3 % (2025) a +11,1 % (2040). Es la señal de cuánto pesa un escenario de fecundidad más baja; no es error contra datos.
* **65+ (J).** La ONU WPP 2024 da 65+ un 0,8 % por debajo del Censo 2022 (WPP 2024 **no** incorpora el censo: su población total 2022 queda 1,1 % abajo). Contra INDEC 2025 la diferencia va de −0,7 % a −2,5 % (RMSE 1,5 %): es la **incertidumbre mínima** de $J$ entre dos fuentes serias.

**Idea clave:** $J$ de las cohortes ≤ 2024 es la población 65+ de 2076-2089, formada por gente **ya nacida**; su incertidumbre viene de mortalidad y migración, no de nacimientos futuros. El error de $N$ afecta a las cohortes ≥ 2025.

In [ ]:
BT19 = backtest_tabla(S19, X13)
cols = ["insumo", "proyeccion", "referencia", "horizonte_anios", "proyectado", "observado", "error_pct", "tipo"]
with pd.option_context("display.float_format", lambda v: f"{v:,.1f}", "display.max_colwidth", 70):
    display(BT19[BT19.bloque != "A4 pendiente"][cols].reset_index(drop=True))

# Gráfico: errores % (positivo = la proyección sobrestimó)
SEL19 = ["Nacimientos 2022", "Nacimientos 2023", "Nacimientos 2024",
         "Población total al 18-may-2022", "Población total 2030: INDEC 2013 vs INDEC 2025",
         "Población total 2040: INDEC 2013 vs INDEC 2025", "65+ al 18-may-2022", "65+ 2030: ONU WPP 2024 vs INDEC 2025",
         "65+ 2040: ONU WPP 2024 vs INDEC 2025"]
g19 = BT19[BT19.insumo.isin(SEL19) & (BT19.proyeccion != "(vintage anterior)")].drop_duplicates("insumo").set_index("insumo").loc[SEL19]
COL_T = {"backtest": "#0072b2", "revisi": "#d55e00", "dispersi": "#009e73", "chequeo": "#999999"}
def _col(t):
    return next((c for k, c in COL_T.items() if t.startswith(k) or k in t), "#999999")
fig, ax = plt.subplots(figsize=(10, 4.8))
yy = np.arange(len(g19))[::-1]
ax.barh(yy, g19.error_pct, color=[_col(tp) for tp in g19.tipo])
for yi, v19 in zip(yy, g19.error_pct):
    ax.text(v19 + (0.6 if v19 >= 0 else -0.6), yi, f"{v19:+.1f} %", va="center", ha="left" if v19 >= 0 else "right", fontsize=9)
ax.set_yticks(yy); ax.set_yticklabels([f"{i}\n({r19.proyeccion})" for i, r19 in g19.iterrows()], fontsize=8)
ax.axvline(0, color="k", lw=0.8); ax.set_xlim(-6, 30)
ax.set_xlabel("Error % = 100·(proyectado/observado − 1);  > 0: la proyección sobrestimó")
ax.set_title("Sección 19 — Errores de proyecciones pasadas de los insumos", fontweight="bold")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#0072b2", label="backtest legítimo"), Patch(color="#d55e00", label="revisión entre vintages (no es error vs hechos)"),
                   Patch(color="#009e73", label="dispersión entre fuentes"), Patch(color="#999999", label="chequeo de nivel")], loc="lower right", fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "seccion_20_backtest_insumos.png"), dpi=130); plt.show()

e_nac = BT19[(BT19.insumo.isin(["Nacimientos 2023", "Nacimientos 2024"]))].error_pct.values
RMSE_N19 = float(np.sqrt(np.mean(e_nac ** 2)))
e_J19, RMSE_J19 = dispersion_J(S19.un_pob_65mas_miles)
print(f"Nacimientos ONU WPP 2024 vs DEIS: 2023 {e_nac[0]:+.1f} %, 2024 {e_nac[1]:+.1f} %  → RMSE (n=2) = {RMSE_N19:.1f} %")
print(f"65+ ONU WPP 2024 vs INDEC 2025 (2022-2040): RMSE = {RMSE_J19:.2f} %; rango {e_J19.min():+.2f} a {e_J19.max():+.2f} %")

### 20.6 De los errores a una banda de incertidumbre para $N$ y $J$ (reemplaza el ±1,5 % / ±1 %)

**Propuesta.** En el Monte Carlo de 16.4, $N\sim\mathcal N(\hat N,\,0{,}015\hat N)$ y $J\sim\mathcal N(\hat J,\,0{,}01\hat J)$ eran valores *arbitrarios*. Se reemplazan por desvíos medidos:

* $\sigma_J$ = RMSE de la diferencia ONU–INDEC en 65+ (≈ 1,5 %), y una variante que **crece con la raíz del horizonte** (supuesto *heurístico*, no estimado: $\sigma(h)=\sigma_{ref}\sqrt{h/h_{ref}}$ con $h_{ref}=9$).
* $\sigma_N$ **solo para cohortes ≥ 2025** (N proyectado): RMSE del backtest de nacimientos 2023-24 (≈ 17 %). Es una referencia *empírica pero débil* (2 puntos, un régimen de caída, sesgo de un solo signo). La cohorte 2024 usa dato DEIS y conserva 1,5 %.

La misma semilla y sorteos que 16.4 → el escenario A reproduce exactamente el P5 = 0,152 / P95 = 0,269 de `monte_carlo_resumen_2024.csv`.

In [ ]:
if HAY_AV19:
    a19 = AV19.set_index("cohorte")
    H_REF = 9      # distancia media (años) de 2022..2040 al año base 2022-2031: referencia del RMSE de J (HEURÍSTICA)
    filas_mc = []
    sims = {}
    for coh in (2024, 2030):
        r19 = a19.loc[coh]; hJ = int(r19.anio_65 - 2024)
        esc = [("A. original (σN=1,5 %, σJ=1 %)", 0.015, 0.01),
               ("B. σJ empírica (RMSE entre fuentes)", 0.015 if coh == 2024 else RMSE_N19 / 100, RMSE_J19 / 100),
               ("C. B + σJ creciente con √horizonte", 0.015 if coh == 2024 else RMSE_N19 / 100,
                sigma_por_horizonte(RMSE_J19 / 100, H_REF, hJ))]
        for nom, sN, sJ in esc:
            sim, q19 = mc_tbp(r19.N_4, r19.S_tot65, r19.J_star_4, sN, sJ)
            sims[(coh, nom[0])] = sim
            filas_mc.append(dict(cohorte=coh, escenario=nom, sigma_N_pct=100 * sN, sigma_J_pct=100 * sJ, p5=q19["p5"], p50=q19["p50"], p95=q19["p95"],
                                 ancho_90=q19["p95"] - q19["p5"], base_punto=r19.TBP_C_p4, ref_cohorte_2000=a19.loc[2000, "TBP_C_p4"],
                                 prob_supera_2000=float((sim >= a19.loc[2000, "TBP_C_p4"]).mean())))
    MC19 = pd.DataFrame(filas_mc)
    with pd.option_context("display.float_format", lambda v: f"{v:,.3f}"):
        display(MC19)
    print("Nota: N de la cohorte 2024 es dato DEIS (posiblemente provisorio); N de 2030 es escenario → σN de 2030 usa el RMSE del backtest (n=2, régimen de caída).")

### 20.7 Estabilidad por ventanas móviles de 20 cohortes (B)

Para cada ventana: media de $TBP_C$, pendiente de $\ln TBP$ por cohorte y contribución de cada factor con la descomposición logarítmica **exacta**
$$\ln TBP_C=\ln N+\ln(1-\mu)+\ln\bar A+\ln(\tau/\rho)-\ln J$$
(la suma de las pendientes de los factores es la pendiente del total; el código lo verifica). En `p4`, $\bar A$ es el **efectivo** que cierra la identidad (incluye el reparto por sexo y edad de retiro), $J=J^*$ y $\tau/\rho$ varía solo por el $\rho$ de las primeras cohortes; en `p1`, $\bar A$ es el de la EPH y $J$ = 65+.

**Lectura (cifras del modelo avanzado, cohortes 1950-2024):**

* En **todas las ventanas que terminan hasta 2020**, el factor que más empuja hacia abajo es $J$ (el envejecimiento: −1,7 a −2,1 pp por cohorte en 1950-1989, −0,35 en la última). $N$ **empuja hacia arriba** (hasta +2,2 pp en la ventana 1960-1979) y recién se vuelve negativo cuando entran las cohortes post-2015.
* **El motor pasa a ser $N$ recién en las ventanas que terminan en 2021-2024** (2005-2024: $N$ = −2,5 pp por cohorte, $J$ = −0,35). **No** se verifica que "desde ~2000 domine $N$": en 2000-2019 $N$ prácticamente no aporta (+0,06).
* En el medio (ventanas 1974-2005) $\bar A$ es el segundo freno (hasta −1,3 pp), por el patrón de aportes de la EPH.
* La media de la ventana baja de 0,46 (1970-1989) a 0,30 (2005-2024).

In [ ]:
if HAY_AV19:
    COMP4 = componentes_log(AV19, "p4", hasta=2024); COMP1 = componentes_log(AV19, "p1", hasta=2024)
    V4 = ventanas_moviles(COMP4, 20); V1 = ventanas_moviles(COMP1, 20)
    assert np.allclose(V4.control_suma_pend, 0, atol=1e-8) and np.allclose(V1.control_suma_pend, 0, atol=1e-8)
    print("Control: en cada ventana, la suma de pendientes de N, S65, Ā, τ/ρ y J = pendiente de ln TBP (exacto).")
    pick = [1950, 1960, 1970, 1980, 1990, 2000, 2005]
    t4 = V4[V4.ini.isin(pick)].set_index("ini")
    with pd.option_context("display.float_format", lambda v: f"{v:,.2f}"):
        display(t4[["fin", "media_TBP", "pend_lnTBP_pct", "pend_N_pp", "pend_S65_pp", "pend_Abar_pp", "pend_tau_rho_pp", "pend_J_pp", "motor_baja", "motor_sube"]]
                .rename(columns=lambda c: c.replace("pend_", "").replace("_pp", " (pp/coh)")))
    print("Unidades: puntos porcentuales de ln TBP por cohorte (≈ % por cohorte). 'J' ya viene con signo −ln J (J que crece resta).")

    fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
    COLF = {"N": "#d55e00", "S65": "#009e73", "Abar": "#0072b2", "tau_rho": "#999999", "J": "#cc79a7"}
    ETQF = {"N": "N (nacimientos)", "S65": "S65 = 1−μ", "Abar": "Ā (efectivo)", "tau_rho": "τ/ρ", "J": "−ln J (jubilados)"}
    for a_, V, tit in ((ax[0], V4, "TBP_C_p4 (J*, sexo/edad de retiro)"), (ax[1], V1, "TBP_C_p1 (J = 65+)")):
        pos = np.zeros(len(V)); neg = np.zeros(len(V))
        for ff in FACTORES:
            vv = V[f"pend_{ff}_pp"].values
            base = np.where(vv >= 0, pos, neg)
            a_.bar(V.fin, vv, bottom=base, color=COLF[ff], label=ETQF[ff], width=0.9)
            pos += np.where(vv >= 0, vv, 0); neg += np.where(vv < 0, vv, 0)
        a_.plot(V.fin, V.pend_lnTBP_pct, "k-", lw=2, label="pendiente de ln TBP")
        a_.axhline(0, color="k", lw=0.6); a_.set_title(f"Motor por ventana de 20 cohortes\n{tit}", fontsize=10, fontweight="bold")
        a_.set_xlabel("última cohorte de la ventana"); a_.set_ylabel("pp de ln TBP por cohorte")
    ax[0].legend(fontsize=7.5, loc="lower left")
    ax[2].plot(V4.fin, V4.media_TBP, color="#0072b2", lw=2, label="p4: media de la ventana")
    ax[2].plot(V1.fin, V1.media_TBP, color="#d55e00", lw=2, label="p1: media de la ventana")
    ax[2].set_title("Media de TBP_C por ventana", fontsize=10, fontweight="bold"); ax[2].set_xlabel("última cohorte de la ventana"); ax[2].set_ylabel("años"); ax[2].legend(fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "seccion_20_estabilidad_ventanas.png"), dpi=130); plt.show()

### 20.8 Test de quiebre: sup-F en lugar de un único corte (B)

Se calcula el F de Chow (constante + tendencia) para **cada corte candidato 2000-2018** sobre las cohortes 1950-2024 y se reporta el máximo (sup-F). También un Wald con covarianza **Newey-West (HAC)** y, como versión robusta, **permutación circular por bloques** de los residuos de una tendencia lineal (bloques de 5 y 15 cohortes).

**Aviso:** la serie es determinística, suave y autocorrelada. Estos estadísticos **describen** dónde cambia la pendiente; los $p$ no tienen la interpretación inferencial habitual. La permutación por bloques es solo una vara de comparación (su "ruido" son residuos de una curva que no es lineal).

**Resultados:**

* **El máximo está en 2015** (TBP_C_p4: F = 9,37; TBP_C_p1: 9,10; nacimientos: 168). El F del corte 2011 es 8,40 (se reproduce exactamente el 8,3972 del test existente). La curva es una meseta entre 2013 y 2018 (F de 9,1 a 9,4): **el corte 2011 no es muy distinto, pero tampoco es el "natural"**; el cambio de pendiente de $TBP$ sigue al de $N$ (pico 2014, caída desde 2015).
* **Robustez:** con permutación por bloques, el $p$ del sup-F de $TBP$ es 0,43-0,62 (no distinguible de una tendencia lineal con ruido autocorrelado), mientras que para $N$ es ≈ 0,003. El Wald-HAC da $p$ = 0,005-0,033 (p4) y 0,010-0,060 (p1) según el bloque. **Los tests no coinciden entre sí**: la conclusión no es robusta como "quiebre estadístico".
* **Conclusión que sale:** la pendiente de $TBP$ cambia alrededor de 2014-2015 y lo hace por $N$. **Conclusión que NO sale:** que exista un "quiebre estructural significativo" en 2011 (ni en ninguna fecha) en sentido inferencial.

In [ ]:
if HAY_AV19:
    ser = AV19.set_index("cohorte").loc[:2024]
    CORTES = range(2000, 2019)
    series_q = {"TBP_C_p4": ser.TBP_C_p4, "TBP_C_p1": ser.TBP_C_p1, "N_t (nacimientos)": ser.N_t.astype(float)}
    res_q, por_corte = [], []
    for nom, sr in series_q.items():
        o, pc = tabla_quiebre(sr, CORTES, nom, bloques=(5, 15), B=399)
        res_q.append(o); por_corte.append(pc)
    QB19 = pd.DataFrame(res_q); PC19 = pd.concat(por_corte, ignore_index=True)
    with pd.option_context("display.float_format", lambda v: f"{v:,.3f}"):
        display(QB19.set_index("serie").T)
    f11 = float(PC19[(PC19.serie == "TBP_C_p4") & (PC19.corte == 2011)].F_chow.iloc[0])
    print(f"Chow en 2011 (TBP_C_p4, k=2) recalculado aquí: F = {f11:.4f}  (el test existente informa 8,3972) ", "→ coincide" if abs(f11 - 8.3972) < 0.01 else "→ NO coincide, revisar")
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
    p4 = PC19[PC19.serie == "TBP_C_p4"]
    ax[0].plot(p4.corte, p4.F_chow, "o-", color="#0072b2", label="F de Chow (tendencia) por corte")
    ax[0].axvline(2011, color="k", ls="--", lw=1, label="corte del test existente (2011)")
    ax[0].axvline(QB19.set_index("serie").loc["TBP_C_p4", "corte_supF"], color="#d55e00", ls=":", lw=2, label="máximo (sup-F)")
    ax[0].set_xticks(list(CORTES)[::2]); ax[0].set_xlabel("corte candidato (primera cohorte del 2.º tramo)"); ax[0].set_ylabel("F"); ax[0].legend(fontsize=8)
    ax[0].set_title("TBP_C_p4: el estadístico casi no distingue entre cortes 2013-2018", fontsize=10, fontweight="bold")
    for nom, col in (("TBP_C_p4", "#0072b2"), ("N_t (nacimientos)", "#d55e00")):
        q19 = PC19[PC19.serie == nom]; ax[1].plot(q19.corte, q19.F_chow / q19.F_chow.max(), "o-", color=col, label=nom + " (F / F máx.)")
    ax[1].axvline(2011, color="k", ls="--", lw=1); ax[1].axvline(2014, color="grey", ls=":", lw=1.5, label="pico de nacimientos DEIS (2014)")
    ax[1].set_xticks(list(CORTES)[::2]); ax[1].set_xlabel("corte candidato"); ax[1].set_ylabel("F normalizado"); ax[1].legend(fontsize=8)
    ax[1].set_title("El quiebre de TBP sigue al de N", fontsize=10, fontweight="bold")
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "seccion_20_supF.png"), dpi=130); plt.show()
    print("RECORDATORIO: serie determinística y autocorrelada → F, Wald y p-valores son DESCRIPTIVOS; no prueban un 'quiebre estructural' en sentido inferencial.")

### 20.9 Salidas y lista de lo que habría que descargar

Se escriben en `datos/procesados/`: `backtest_insumos.csv` (una fila por comparación; las filas `PENDIENTE DE DESCARGA` son la lista de abajo), `estabilidad_ventanas.csv` (una fila por ventana y versión) y, de apoyo, `estabilidad_quiebre.csv` (F y Wald por corte) y `backtest_bandas_montecarlo.csv`.

**Qué habría que bajar (NO se descargó; tamaños medidos con cabeceras HTTP):**

| Archivo | Tamaño (cabeceras HTTP) | Para qué |
|---|---|---|
| INDEC 2013, Serie Análisis Demográfico n.º 35 (PDF) `https://www.indec.gob.ar/ftp/cuadros/publicaciones/proyeccionesyestimaciones_nac_2010_2040.pdf` | 267.025 B (≈ 0,26 MB) | 65+ y natalidad/TGF proyectados por INDEC en 2013 (Cuadros 2 y 5) → error de J y de N contra el Censo 2022 y DEIS |
| WPP 2019 (paquete R `wpp2019_1.1-1.tar.gz`, CRAN) | 4.684.504 B (≈ 4,7 MB) | vintage 2019: horizonte ~4-5 años para 2023-24 |
| WPP 2017 (`wpp2017_1.2-3.tar.gz`, CRAN) | 4.743.882 B (≈ 4,7 MB) | vintage intermedio |
| WPP 2015 (`Archive/wpp2015/wpp2015_1.1-2.tar.gz`, CRAN) | 4.585.228 B (≈ 4,6 MB) | horizonte ~8-9 años |
| WPP 2012 (`wpp2012_2.2-1.tar.gz`, CRAN) | 4.902.348 B (≈ 4,9 MB) | horizonte ~10-12 años |
| WPP 2010 | no verificado (sin paquete en CRAN; las URL históricas de population.un.org devolvieron 404) | habría que pedirlo al archivo de la División de Población de la ONU |

Total verificable: ≈ 19,2 MB. Las URL del sitio de la ONU para los Excel históricos devolvieron 404, por eso se proponen los paquetes R de CRAN (que redistribuyen las tablas de cada revisión); el contenido exacto de cada paquete hay que confirmarlo al abrirlo.

In [ ]:
os.makedirs(DEST19, exist_ok=True)
BT19.to_csv(os.path.join(DEST19, "backtest_insumos.csv"), index=False, encoding="utf-8")
print("Escrito:", os.path.join(DEST19, "backtest_insumos.csv"), BT19.shape)
if HAY_AV19:
    EST19 = pd.concat([V4.assign(version="p4"), V1.assign(version="p1")], ignore_index=True)
    EST19 = EST19[["version"] + [c for c in EST19.columns if c != "version"]]
    EST19.to_csv(os.path.join(DEST19, "estabilidad_ventanas.csv"), index=False, encoding="utf-8")
    PC19.to_csv(os.path.join(DEST19, "estabilidad_quiebre.csv"), index=False, encoding="utf-8")
    MC19.to_csv(os.path.join(DEST19, "backtest_bandas_montecarlo.csv"), index=False, encoding="utf-8")
    print("Escrito:", os.path.join(DEST19, "estabilidad_ventanas.csv"), EST19.shape, "| estabilidad_quiebre.csv", PC19.shape, "| backtest_bandas_montecarlo.csv", MC19.shape)
print(f"A descargar (solo lista; verificable): ~{tamano_pendientes():.1f} MB en {sum(1 for p in PENDIENTES_DESCARGA if p['bytes'])} archivos + WPP 2010 (no verificado).")

### 20.10 Qué sirve validar y qué no

| Qué se prueba | Qué responde | Datos que usa | Qué NO responde |
|---|---|---|---|
| **Backtest de N**: nacimientos proyectados por la ONU (WPP 2024) vs DEIS 2023-24 | ¿Cuánto se equivocó una proyección real publicada *antes* del dato? (+9,3 % y +22,6 %) | `series_tbp_extraidas.csv` | No dice cuánto se equivocarán las proyecciones 2025+; son solo 2 puntos en un régimen de caída de la fecundidad |
| **Backtest de población total**: INDEC 2013 vs Censo 2022 | ¿Acertaron las proyecciones largas (12 años) el tamaño de la población? (+0,6 %) | xls INDEC 2013 + PDF INDEC 2025 (pp. 12 y 19) | No dice nada de N ni de 65+ por separado (errores de signo opuesto se compensan) |
| **Revisión entre vintages**: INDEC 2013 vs INDEC 2025 (2025-2040) | ¿Cuánto cambia el escenario al incorporar un censo nuevo? (+2,3 % en 2025 a +11,1 % en 2040) | xls INDEC 2013 + PDF INDEC 2025 (pp. 39-41) | No es error contra hechos: la "realidad" es otra proyección |
| **Dispersión de J entre fuentes**: ONU WPP 2024 vs INDEC 2025 (65+) | ¿Cuánta incertidumbre mínima hay en J? (RMSE 1,5 %; −0,7 a −2,5 %) | `series_tbp_extraidas.csv` + PDF INDEC 2025 | No es un error de proyección; las dos fuentes pueden equivocarse juntas |
| **Backtest de vintages anteriores** (WPP 2012/2015/2017/2019, INDEC 2013 por edad) | Cómo crece el error con el horizonte (necesario para una banda seria de N y J) | **PENDIENTE**: ver lista de descargas | Hoy no se puede responder: no hay datos en el proyecto |
| **Ventanas móviles + descomposición logarítmica** | ¿Qué factor (N, S65, Ā, τ/ρ, J) empuja la serie en cada tramo? | `tbp_modelo_avanzado_1950_2035.csv` | No es causal ni predictivo; es contabilidad exacta de lo ya calculado |
| **sup-F / Chow por corte** (y Wald-HAC, permutación por bloques) | ¿Dónde cambia la pendiente de la serie? (máximo en 2015) | ídem | **No** prueba un "quiebre estructural" inferencial: la serie es determinística y autocorrelada |
| **Split IS/OOS 2011** (test existente) | Describe que las cohortes 2011-2024 tienen TBP menor que las anteriores (−28,6 %) | ídem | No es out-of-sample (los insumos de esas cohortes son proyecciones a 2076-2089); el corte 2011 es arbitrario; los p-valores no tienen lectura inferencial |
| **Sensibilidad / tornado / Monte Carlo** (sección 16) | ¿Cuánto cambia el nivel si cambian los supuestos? | parámetros con fuente | No valida que los supuestos sean ciertos; los desvíos de N y J eran arbitrarios (se reemplazan acá) |
| **Asserts de identidad** (secciones 3, 7, 15) | ¿El código calcula bien la fórmula? | las propias series | No prueba que la fórmula describa la realidad |

### 20.11 Lectura y límites

* **Qué se puede decir:** (1) la ONU sobrestimó los nacimientos 2023-24 en 9 % y 23 %; (2) INDEC 2013 acertó el total de población a 12 años pero su escenario hoy está 2-11 % arriba del de INDEC 2025; (3) $J$ difiere ~1,5 % entre ONU e INDEC; (4) la caída de $TBP_C$ en las cohortes recientes viene de $N$, mientras que en el resto de la serie el freno principal fue $J$; (5) la pendiente cambia hacia 2014-2015.
* **Qué NO se puede decir:** que el índice "predice" algo; que el tramo 2011-2024 sea out-of-sample; que haya un quiebre estadísticamente significativo; que las bandas de 19.6 sean definitivas (σ de $N$ con 2 observaciones; escalamiento por horizonte es un supuesto).
* **Límites:** los vintages previos (WPP 2012-2019, INDEC 2013 por edad) no están descargados, así que falta medir cómo crece el error con el horizonte; el DEIS 2024 puede ser provisorio; el backtest de 65+ contra el censo usa la ONU, no una proyección de INDEC de la época.
* **Observación sobre 16.5:** con los parámetros de 16.4 se reproduce P95 = 0,269 y la probabilidad de superar la cohorte 2000 (0,312) es ≈ 0,3 % (30 de 10.000), no "< 0,1 %"; la conclusión cualitativa no cambia.

## 21. 🆕 Backtest con proyecciones antiguas (ONU 2012-2019 e INDEC 2013)

**Pregunta:** ¿cuánto se equivocaron las proyecciones de **fecundidad / nacimientos** y de **población de 65+** que existían *antes* de 2020? Es el único "fuera de muestra" legítimo del proyecto: no se valida el índice (no tiene resultado observable), sino sus **insumos** $N$ y $J$.

**Datos (descargados con autorización de la usuaria, fuentes oficiales):** paquetes R de CRAN con las tablas de la ONU (`wpp2012`, `wpp2015`, `wpp2017`, `wpp2019`) y la *Serie Análisis Demográfico n.º 35* de INDEC (2013). El repositorio guarda solo las filas de Argentina (`vintages_wpp_argentina.csv`).

> ⚠️ Los nacimientos "derivados" son una reconstrucción propia (TGF × patrón de fecundidad por edad × mujeres de cada edad), no un dato de la ONU. Se verifica que, en 2015-2020 (sin shock), el error de la reconstrucción es de −2 % a +7 %.

### 21.1 Módulo (copia de `src/tbp_vintages.py`)

In [ ]:
# ====== MÓDULO tbp_vintages (misma versión que src/tbp_vintages.py) ======
# -*- coding: utf-8 -*-
"""
tbp_vintages.py — Backtest de proyecciones ANTIGUAS (ONU WPP 2012/2015/2017/2019 e INDEC 2013) contra lo ocurrido.

Pregunta: ¿cuánto se equivocaron las proyecciones de fecundidad/nacimientos y de población de 65+ que estaban
disponibles antes de 2020? Es el único "fuera de muestra" legítimo del proyecto: se valida el INSUMO (N, J), no el índice.

Datos de entrada (todos en el repositorio):
  datos/procesados/vintages_wpp_argentina.csv   filas de Argentina extraídas de los paquetes R `wpp2012/2015/2017/2019`
                                                (CRAN; datos de la División de Población de la ONU)
  datos/procesados/series_tbp_extraidas.csv     nacimientos DEIS y población 65+ ONU 2024 (referencia)
  datos/procesados/renaper_natalidad_2012_2025.csv   TGF observada (RENAPER)
INDEC 2013 (Serie Análisis Demográfico n.º 35, Cuadros 2 y 6) está transcripta abajo con su página.

AVISOS:
  * Los nacimientos "derivados" = TGF × Σ_a (porcentaje de fecundidad por edad_a / 5) × mujeres_a, con las mujeres de
    cada grupo quinquenal promediadas entre el inicio y el fin del período. Es una reconstrucción propia, no un dato de la ONU.
    Control: en 2015-2020, sin shock, el error de esa reconstrucción es de −2 % a +7 %.
  * La "población 65+ observada" usa WPP 2024 como referencia (no es un censo por edad). Para 2025 es proyección.
  * El período 2020-2025 se compara con 2020-2024 observado; con 2025 (provisorio) la TGF media baja de 1,39 a 1,34.
"""
import os, io, urllib.request
import numpy as np
import pandas as pd

RAW_BASE_VINT = "https://raw.githubusercontent.com/carolinamsfelipe/Tp-Indicadores/main/datos/procesados"
RUTAS_VINT = ["../datos/procesados", "datos/procesados", "github_staging/datos/procesados", "datos_tbp", "."]
EDADES_MUJERES = ["15-19", "20-24", "25-29", "30-34", "35-39", "40-44", "45-49"]
LANZAMIENTO = {"wpp2012": 2012, "wpp2015": 2015, "wpp2017": 2017, "wpp2019": 2019}

# INDEC 2013 (Serie 35). Cuadro 6 (p. 17): TGF. Cuadro 2 (pp. 29-30): población de 65+ = suma de 65-69 ... 100 y más, ambos sexos.
INDEC_2013_TGF = {2010: 2.41, 2015: 2.28, 2020: 2.18, 2025: 2.10, 2030: 2.05, 2035: 2.01, 2040: 1.98}
INDEC_2013_J65 = {2018: 4982425, 2019: 5103968, 2020: 5227722, 2021: 5353272, 2022: 5480183, 2023: 5608402, 2024: 5737818, 2025: 5868496}
# INDEC 2025 (proyección base Censo 2022; pp. 39-42) para contraste de 65+
INDEC_2025_J65 = {2022: 5529167, 2023: 5610238, 2024: 5711979, 2025: 5815927}


def _leer_vint(nombre, rutas=None):
    for r in (rutas or RUTAS_VINT):
        p = os.path.join(r, nombre)
        if os.path.exists(p):
            return pd.read_csv(p)
    with urllib.request.urlopen(f"{RAW_BASE_VINT}/{nombre}") as f:
        return pd.read_csv(io.BytesIO(f.read()))


def cargar_entradas_vint(rutas=None):
    v = _leer_vint("vintages_wpp_argentina.csv", rutas)
    ser = _leer_vint("series_tbp_extraidas.csv", rutas).set_index("year")
    ren = _leer_vint("renaper_natalidad_2012_2025.csv", rutas).set_index("anio")
    return v, ser, ren


def _edad_inicio_vint(a):
    return int(str(a).replace("+", "-").split("-")[0])


def _pop_vint(v, vint, anio, edades, sexo_tablas):
    tot = 0.0
    for tab in sexo_tablas:
        d = v[(v.vintage == vint) & (v.tabla == tab) & (v.periodo.astype(str) == str(anio)) & (v.edad.isin(edades))]
        tot += d.valor.sum()
    return tot


def backtest_wpp(v, ser, ren):
    """Compara TGF, nacimientos (derivados) y población 65+ proyectados por cada edición de la ONU con lo observado."""
    tfr_obs = ren.tgf_hijos_por_mujer
    obs_tfr = {"2015-2020": tfr_obs.loc[2015:2019].mean(), "2020-2025": tfr_obs.loc[2020:2024].mean()}
    nac = ser.deis_nacimientos
    obs_b = {"2015-2020": nac.loc[2015:2019].mean(), "2020-2025": nac.loc[2020:2024].mean()}
    filas = []
    for vint, yr in LANZAMIENTO.items():
        for per in ("2015-2020", "2020-2025"):
            t = v[(v.vintage == vint) & (v.tabla == "tfrprojMed") & (v.periodo == per)]
            if t.empty:
                continue
            T = float(t.valor.iloc[0]); y0, y1 = int(per[:4]), int(per[5:])
            pct = np.array([float(v[(v.vintage == vint) & (v.tabla == "percentASFR") & (v.periodo == per) & (v.edad == a)].valor.iloc[0])
                            for a in EDADES_MUJERES]) / 100
            W = []
            for a in EDADES_MUJERES:
                w0 = _pop_vint(v, vint, y0, [a], ["popF", "popFprojMed"]); w1 = _pop_vint(v, vint, y1, [a], ["popF", "popFprojMed"])
                W.append((w0 + w1) / 2)
            births = float((T * pct / 5 * np.array(W)).sum()) * 1000
            hz = (y0 + y1) / 2 - yr
            filas.append((vint, per, "TGF (hijos por mujer)", T, obs_tfr[per], hz))
            filas.append((vint, per, "Nacimientos por año (derivado)", births, obs_b[per], hz))
        edades65 = sorted({a for a in v[(v.vintage == vint) & v.tabla.isin(["popF", "popFprojMed"])].edad.unique() if _edad_inicio_vint(a) >= 65})
        for Y in (2020, 2025):
            p65 = _pop_vint(v, vint, Y, edades65, ["popF", "popFprojMed"]) + _pop_vint(v, vint, Y, edades65, ["popM", "popMprojMed"])
            if p65 > 0:
                filas.append((vint, str(Y), "Población de 65+ (miles)", p65, ser.un_pob_65mas_miles[Y], Y - yr))
    df = pd.DataFrame(filas, columns=["edicion", "periodo", "variable", "proyectado", "observado", "horizonte_anios"])
    df["error_pct"] = (df.proyectado / df.observado - 1) * 100
    return df


def backtest_indec2013(ser, ren):
    tfr = ren.tgf_hijos_por_mujer
    filas = []
    for y in (2015, 2020, 2025):
        filas.append(("INDEC 2013", str(y), "TGF (hijos por mujer)", INDEC_2013_TGF[y], float(tfr[y]), y - 2013, "2025 provisorio" if y == 2025 else ""))
    for y in (2022, 2023, 2024, 2025):
        filas.append(("INDEC 2013", str(y), "Población de 65+ (contra INDEC 2025)", INDEC_2013_J65[y], INDEC_2025_J65[y], y - 2013, ""))
    df = pd.DataFrame(filas, columns=["edicion", "periodo", "variable", "proyectado", "observado", "horizonte_anios", "nota"])
    df["error_pct"] = (df.proyectado / df.observado - 1) * 100
    return df


def traduccion_a_tbp(tbp_c_base, errores_N=(0.17, 0.25, 0.40)):
    """TBP_C es PROPORCIONAL a N: si N de una cohorte futura se equivocara ±x %, el TBP_C cambia ±x % (resto constante)."""
    return pd.DataFrame({"error_en_N": [f"±{int(e*100)} %" for e in errores_N],
                         "TBP_C_bajo": [tbp_c_base * (1 - e) for e in errores_N],
                         "TBP_C_alto": [tbp_c_base * (1 + e) for e in errores_N]})


def autotest_vintages(verbose=True):
    v, ser, ren = cargar_entradas_vint()
    b = backtest_wpp(v, ser, ren)
    # 1) la reconstrucción de nacimientos es razonable cuando NO hubo shock (2015-2020): error entre −5 % y +10 %
    n1 = b[(b.variable.str.startswith("Nacimientos")) & (b.periodo == "2015-2020")]
    assert n1.error_pct.between(-5, 10).all(), n1
    # 2) en 2020-2025 todas las ediciones sobreestimaron la fecundidad por más de 40 %
    t2 = b[(b.variable.str.startswith("TGF")) & (b.periodo == "2020-2025")]
    assert len(t2) == 4 and (t2.error_pct > 40).all(), t2
    # 3) la población de 65+ tiene error pequeño (< 4 %) en 2020
    j20 = b[(b.variable.str.startswith("Población")) & (b.periodo == "2020")]
    assert j20.error_pct.abs().max() < 4, j20
    # 4) el error de fecundidad NO crece de forma sistemática con el horizonte (de 3,5 a 10,5 años va de +48 % a +58 %)
    assert t2.error_pct.max() - t2.error_pct.min() < 15
    i = backtest_indec2013(ser, ren)
    assert abs(i[(i.variable.str.startswith("TGF")) & (i.periodo == "2020")].error_pct.iloc[0] - 33.7) < 1.0
    assert traduccion_a_tbp(0.2, (0.1,)).TBP_C_alto.iloc[0] == 0.2 * 1.1
    if verbose:
        print("Autotest tbp_vintages OK: reconstrucción de nacimientos, errores de TGF, 65+ e INDEC 2013.")
    return b

### 21.2 Proyectado contra observado

In [ ]:
V21, SER21, REN21 = cargar_entradas_vint()
BT21 = backtest_wpp(V21, SER21, REN21)
IN21 = backtest_indec2013(SER21, REN21)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
display(BT21.round(2))
display(IN21.round(2))
t = BT21[(BT21.variable.str.startswith("TGF")) & (BT21.periodo == "2020-2025")]
print("TGF 2020-2025: proyectada", t.proyectado.round(2).tolist(), "| observada (2020-24) =", round(float(t.observado.iloc[0]), 2),
      "| error:", t.error_pct.round(0).astype(int).tolist(), "% (ediciones 2012, 2015, 2017, 2019)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
t = BT21[(BT21.variable.str.startswith("TGF")) & (BT21.periodo == "2020-2025")]
x = np.arange(len(t) + 1)
vals = list(t.proyectado) + [INDEC_2013_TGF[2020]]
ax[0].bar(x, vals, color=["#4c72b0"] * len(t) + ["#55a868"], alpha=.85)
ax[0].axhline(float(t.observado.iloc[0]), color="#c44e52", lw=2, label="observada 2020-24 (RENAPER) = %.2f" % float(t.observado.iloc[0]))
ax[0].set_xticks(x); ax[0].set_xticklabels([e.replace("wpp", "ONU ") for e in t.edicion] + ["INDEC 2013\n(año 2020)"], fontsize=8)
ax[0].set_ylim(0, 2.6); ax[0].set_title("Fecundidad 2020-2025: proyectada vs observada"); ax[0].set_ylabel("hijos por mujer"); ax[0].legend(fontsize=8)
n = BT21[(BT21.variable.str.startswith("Nacimientos")) & (BT21.periodo == "2020-2025")]
ax[1].bar(range(len(n)), n.proyectado / 1e3, color="#4c72b0", alpha=.85)
ax[1].axhline(float(n.observado.iloc[0]) / 1e3, color="#c44e52", lw=2, label="observados 2020-24 (DEIS)")
ax[1].set_xticks(range(len(n))); ax[1].set_xticklabels([e.replace("wpp", "ONU ") for e in n.edicion]); ax[1].set_title("Nacimientos por año 2020-2025 (derivados)")
ax[1].set_ylabel("miles"); ax[1].legend(fontsize=8)
j = BT21[(BT21.variable.str.startswith("Población")) & (BT21.periodo == "2020")]
ax[2].bar(range(len(j)), j.error_pct, color="#8172b2", alpha=.85); ax[2].axhline(0, color="k", lw=.8)
ax[2].set_xticks(range(len(j))); ax[2].set_xticklabels([e.replace("wpp", "ONU ") for e in j.edicion])
ax[2].set_title("Error de la población de 65+ en 2020 (%)"); ax[2].set_ylim(-12, 12)
plt.tight_layout(); os.makedirs(OUT, exist_ok=True); plt.savefig(os.path.join(OUT, "seccion_21_backtest_vintages.png"), dpi=120); plt.show()

### 21.3 Qué muestra

* **Fecundidad:** todas las ediciones de la ONU (2012, 2015, 2017, 2019) proyectaron para 2020-2025 una TGF de **2,06 a 2,20** hijos por mujer; lo observado fue **~1,39**. Error de **+48 % a +58 %**. INDEC 2013 proyectaba 2,18 para 2020 (observada 1,63) y 2,10 para 2025 (observada 1,04, provisoria).
* **Nacimientos (derivados):** **+38 % a +54 %** por encima de los observados. El error **no se achica** al acercarse el horizonte (la edición 2019, a 3-4 años, falló lo mismo que la de 2012, a 10): fue un **cambio de régimen** que nadie anticipó, no un error que crezca de a poco.
* **Población de 65+ ($J$):** error de **−2,8 % a +0,2 %** en 2020 y de **−1,1 % a +2,6 %** en 2025 (contra WPP 2024). INDEC 2013 contra INDEC 2025: de −0,9 % a +0,9 %. Las personas de 65+ de 2090 **ya nacieron**: su proyección depende de mortalidad y migración, mucho más estables.
* **Conclusión:** el insumo frágil es **$N$ (natalidad) hacia adelante**; $J$ es robusto. Esto refuerza que, para cohortes anteriores a 2025 (con $N$ observado), el indicador es sólido, y que para cohortes futuras la incertidumbre dominante es la natalidad.

### 21.4 Qué significa para el TBP de las cohortes futuras

In [ ]:
def _cargar_modelo_avanzado():
    for r in ["../datos/procesados", "datos/procesados", "github_staging/datos/procesados", "datos_tbp", "."]:
        p = os.path.join(r, "tbp_modelo_avanzado_1950_2035.csv")
        if os.path.exists(p):
            return pd.read_csv(p).set_index("cohorte")
    raise FileNotFoundError("tbp_modelo_avanzado_1950_2035.csv")
MA21 = _cargar_modelo_avanzado()
for c in (2030, 2035):
    base = float(MA21.loc[c, "TBP_C_p4_persistencia"])
    print(f"Cohorte {c} (persistencia): TBP_C = {base:.3f} años. Si N se equivocara (el resto constante):")
    display(traduccion_a_tbp(base, (0.17, 0.25, 0.40)).round(3))
print("TBP_C es proporcional a N: un error de x % en N es un error de x % en el indicador.")

**Lectura.** La banda del Monte Carlo de la sección 19 usa un error de $N$ calibrado con RENAPER/DEIS (σ ≈ 2 %) para cohortes observadas y los **tres escenarios** para las futuras. El backtest muestra que, en el último ciclo, las proyecciones oficiales sobrestimaron los nacimientos entre 38 % y 54 %. **No** significa que los escenarios actuales vayan a fallar así (ahora están anclados en la fecundidad observada de 2025), pero sí que **±17 % es un piso, no un techo**, para la incertidumbre de la natalidad de cohortes posteriores a 2025. Por eso la tabla de arriba muestra también ±25 % y ±40 %.

In [ ]:
_bt = autotest_vintages()

### 21.5 Límites

* Los nacimientos derivados son una reconstrucción propia (error de −2 % a +7 % en 2015-2020, sin shock).
* La "población de 65+ observada" es WPP 2024 (no un censo por edad); para 2025 es proyección.
* El período 2020-2025 se compara con 2020-2024 observado; incluyendo 2025 (provisorio) la TGF media observada baja de 1,39 a 1,34 y los errores suben.
* Cuatro ediciones de la ONU y una de INDEC son una muestra pequeña: no alcanza para estimar cómo crece el error con el horizonte.
* No se encontró la edición WPP 2010; las URL históricas del sitio de la ONU devuelven 404.